# Automated VWAP V5 Backtest Runner

This notebook is the V4 automated backtest runner for the VWAP Probability Band Engine project.

V4 is intended to test dynamic regime selection between existing continuation entry modules.

At this stage, V4 starts from the completed V3 notebook so the inherited baseline can be separated before adding regime classification and model routing.

It tests one defined automated strategy version on historical OHLC CSV data.

The notebook is separate from the research notebook and separate from the visual/discretionary MT5 overlay.

This notebook does not connect to MT5 and does not place live trades.

## Strategy Version

**Version:** Automated VWAP V5

V4 is intended to test a dynamic VWAP regime selector between existing continuation entry modules.

The planned V4 structure is:

- calm / clean trend uses V1-style green reclaim/rejection entries
- volatile but directional trend uses V3 second-close green reclaim/rejection entries
- chop / unclear value avoids continuation trades
- extreme or abnormal news regimes avoid fresh continuation trades at first

The inherited baseline still uses the completed V3 second-close continuation logic until V4 regime routing is added.

V4 keeps the same simulator, fixed SL / TP / BE rules, session controls, and daily risk controls.

V5 extends the V4 dynamic regime selector by adding a conditional V2 trend-health safety layer for selected volatile-trend conditions.

This version does not add failed-auction setups, orange entries, red-band reversal entries, dynamic stops, runner exits, or new TP/SL/BE logic.

## 1. Project Setup

This section imports the required libraries, detects the project root, and defines the main project folders used by the notebook.

In [ ]:
from pathlib import Path
from pprint import pprint
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


def find_project_root(start_path: Path | None = None) -> Path:
    """
    Find the project root by walking upward until the expected repository
    structure is found.

    This works whether the project was cloned with Git or downloaded as a ZIP,
    and whether the notebook is run from the project root or from notebooks/.
    """
    if start_path is None:
        start_path = Path.cwd()

    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        has_project_structure = (
            (path / "src").is_dir()
            and (path / "data").is_dir()
            and (path / "notebooks").is_dir()
        )

        has_repo_marker = (
            (path / ".git").exists()
            or (path / "README.md").exists()
            or (path / "requirements.txt").exists()
        )

        if has_project_structure and has_repo_marker:
            return path

    raise FileNotFoundError(
        "Could not find the project root. Make sure this notebook is being run "
        "from inside the VWAP-probability-band-engine project folder."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
HISTORICAL_DATA_DIR = DATA_DIR / "historical"
SRC_DIR = PROJECT_ROOT / "src"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
OUTPUT_DIR = ARTIFACTS_DIR / "automated_vwap_v5_modular_engine"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current working directory:", Path.cwd())
print("Detected project root:", PROJECT_ROOT)
print("Historical data folder:", HISTORICAL_DATA_DIR)
print("Source folder:", SRC_DIR)
print("Output folder:", OUTPUT_DIR)

## 2. Editable Strategy Configuration

This is the main control panel for the V4 backtest.

Most basic V4 experiments should be possible by changing this cell only.

The parameters are kept in one place so V4 can be tested and adjusted without changing the rest of the notebook.

At this stage, the inherited baseline still uses the completed V3 second-close continuation logic. Dynamic regime routing is added later.

V4 keeps the existing configurable red-band shift buckets:

- `< 3` = weak
- `3 to <5` = minimum
- `5 to <7` = good
- `7 to <10` = strong
- `10 to <20` = very strong
- `20 to <40` = extreme
- `40+` = abnormal crash/news regime context

In [ ]:
CONFIG = {
    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    # Main dataset used by the normal single-run backtest
    "csv_file": "US100_cash_M1_NY_session_1y.csv", 
    "dataset_name": "US100_cash_M1_NY_session_1y",

    # Datasets used by the final comparison table
    "comparison_datasets": [
        {
            "csv_file": "US100_cash_M1_NY_session_30d.csv",
            "dataset_name": "US100_cash_M1_NY_session_30d",
        },
        {
            "csv_file": "US100_cash_M1_NY_session_1y.csv",
            "dataset_name": "US100_cash_M1_NY_session_1y",
        },
        {
            "csv_file": "US100_cash_M1_NY_session_calm_2021_partial.csv",
            "dataset_name": "US100_cash_M1_NY_session_calm_2021_partial",
        },
        {
            "csv_file": "US100_cash_M1_NY_session_volatile_2022.csv",
            "dataset_name": "US100_cash_M1_NY_session_volatile_2022",
        },
        {
            "csv_file": "US100_cash_M1_NY_session_ukraine_war_2022_2023.csv",
            "dataset_name": "US100_cash_M1_NY_session_ukraine_war_2022_2023",
        },
        {
            "csv_file": "US100_cash_M1_NY_session_recent_2025_2026.csv",
            "dataset_name": "US100_cash_M1_NY_session_recent_2025_2026",
        },
        {
            "csv_file": "us100_cash_M1_new_york_2023_01_01_to_2024_01_01.csv",
            "dataset_name": "US100_cash_M1_NY_session_2023_full",
        },
        {
            "csv_file": "us100_cash_M1_new_york_2024_01_01_to_2025_01_01.csv",
            "dataset_name": "US100_cash_M1_NY_session_2024_full",
        },
    ],

    # ------------------------------------------------------------------
    # Strategy identity
    # ------------------------------------------------------------------
    "strategy_version": "automated_vwap_v5_modular_engine",
    "strategy_description": "Dynamic regime selector with conditional V2 trend-health safety layer",

    # ------------------------------------------------------------------
    # Session handling
    # ------------------------------------------------------------------
    "session_timezone": "Europe/London",
    "no_new_trades_after": "19:00",

    # ------------------------------------------------------------------
    # Direction controls
    # ------------------------------------------------------------------
    "allow_longs": True,
    "allow_shorts": True,

    # ------------------------------------------------------------------
    # Strategy filters
    # ------------------------------------------------------------------
    "strategy_family": "continuation_only",
    "strategy_filter": "v4_dynamic_regime_selector",
    "red_shift_filter_choice": "adaptive_relative_red_shift",

    # ------------------------------------------------------------------
    # Entry logic
    # ------------------------------------------------------------------
    "entry_timing": "next_bar_open",

    # ------------------------------------------------------------------
    # Fixed Nasdaq point trade levels
    # ------------------------------------------------------------------
    "sl_points": 36.0,
    "tp_points": 72.0,
    "be_trigger_points": 36.0,

    # ------------------------------------------------------------------
    # Risk controls
    # ------------------------------------------------------------------
    "risk_per_trade_pct": 1.0,
    "daily_max_consecutive_losses": 2,
    "daily_profit_cap_pct": 999.0,

    # ------------------------------------------------------------------
    # Candle quality filters
    # ------------------------------------------------------------------
    "min_body_ratio": 0.25,
    "min_close_through_green": 1.0,
    "max_extension_from_green": 8.0,

    # ------------------------------------------------------------------
    # Output settings
    # ------------------------------------------------------------------
    "save_trade_log": True,
    "save_daily_summary": True,
    "save_skipped_signals": True,
    "save_config_snapshot": True,
}

pprint(CONFIG)

## 3. Trade Level Sanity Check

This section checks that the configured stop loss, take profit, and breakeven levels behave correctly for both long and short trades.

The config stores all point distances as positive numbers.

The trade simulator will later convert those distances into the correct long or short price levels.

In [ ]:
def build_trade_levels(entry_price: float, side: str, config: dict) -> dict:
    """
    Build fixed-point Nasdaq execution levels for a long or short trade.
    """
    side = side.lower()

    sl = float(config["sl_points"])
    tp = float(config["tp_points"])
    be = float(config["be_trigger_points"])

    if side == "long":
        return {
            "side": "long",
            "entry_price": entry_price,
            "stop_price": entry_price - sl,
            "breakeven_trigger_price": entry_price + be,
            "target_price": entry_price + tp,
        }

    if side == "short":
        return {
            "side": "short",
            "entry_price": entry_price,
            "stop_price": entry_price + sl,
            "breakeven_trigger_price": entry_price - be,
            "target_price": entry_price - tp,
        }

    raise ValueError("side must be either 'long' or 'short'")


example_entry = 20000.0

level_check = {
    "long_example": build_trade_levels(example_entry, "long", CONFIG),
    "short_example": build_trade_levels(example_entry, "short", CONFIG),
}

pprint(level_check)

## 4. Data Loading

This section loads the configured OHLC CSV file.

It handles common MT5-style column names and standardises them into:

- `datetime`
- `open`
- `high`
- `low`
- `close`

Optional fields such as `tick_volume`, `spread`, and `real_volume` are preserved when available.

In [ ]:
REQUIRED_RAW_COLUMNS = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
]

COLUMN_ALIASES = {
    "datetime": [
        "datetime",
        "time",
        "timestamp",
        "date",
        "Date",
        "Time",
        "Datetime",
        "Local time",
        "Gmt time",
        "GMT time",
    ],
    "open": ["open", "Open", "OPEN"],
    "high": ["high", "High", "HIGH"],
    "low": ["low", "Low", "LOW"],
    "close": ["close", "Close", "CLOSE"],
    "tick_volume": [
        "tick_volume",
        "volume",
        "Volume",
        "tickvol",
        "Tick Volume",
        "Tick volume",
    ],
    "spread": ["spread", "Spread"],
    "real_volume": ["real_volume", "Real Volume", "real volume"],
}


def find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """
    Return the first matching column from a list of possible column names.
    """
    existing = set(df.columns)

    for col in candidates:
        if col in existing:
            return col

    lower_map = {str(col).lower(): col for col in df.columns}

    for col in candidates:
        if str(col).lower() in lower_map:
            return lower_map[str(col).lower()]

    return None


def standardise_raw_ohlc_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename common OHLC column names into the standard names used by this notebook.
    """
    out = df.copy()
    rename_map = {}

    for standard_name, aliases in COLUMN_ALIASES.items():
        matched_col = find_column(out, aliases)

        if matched_col is not None and matched_col != standard_name:
            rename_map[matched_col] = standard_name

    out = out.rename(columns=rename_map)

    return out


def validate_required_columns(df: pd.DataFrame, required_columns: list[str]) -> None:
    """
    Validate that the dataframe contains the required columns.
    """
    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(
            "Missing required columns: "
            + ", ".join(missing)
            + "\n\nAvailable columns:\n"
            + ", ".join(map(str, df.columns))
        )


def prepare_raw_ohlc_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare raw OHLC data for the automated V1 backtest.

    This function only cleans and validates raw market data.

    It does not calculate VWAP.
    It does not calculate probability bands.
    It does not change the existing VWAP engine logic.
    """
    out = standardise_raw_ohlc_columns(df)

    validate_required_columns(out, REQUIRED_RAW_COLUMNS)

    out["datetime"] = pd.to_datetime(out["datetime"], errors="coerce")
    out = out.dropna(subset=["datetime"]).copy()

    numeric_cols = ["open", "high", "low", "close"]

    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=numeric_cols).copy()

    for optional_col in ["tick_volume", "spread", "real_volume"]:
        if optional_col in out.columns:
            out[optional_col] = pd.to_numeric(out[optional_col], errors="coerce")

    out = out.sort_values("datetime").reset_index(drop=True)

    out["candle_range"] = out["high"] - out["low"]
    out["candle_body"] = (out["close"] - out["open"]).abs()
    out["body_ratio"] = np.where(
        out["candle_range"] > 0,
        out["candle_body"] / out["candle_range"],
        0.0,
    )

    return out


def list_candidate_data_files() -> list[Path]:
    """
    Find likely CSV or Parquet files inside the project.
    """
    patterns = ["*.csv", "*.parquet"]
    ignored_parts = {
        ".git",
        ".venv",
        "venv",
        "__pycache__",
        ".ipynb_checkpoints",
    }

    files = []

    for pattern in patterns:
        files.extend(PROJECT_ROOT.rglob(pattern))

    clean_files = [
        file for file in files
        if not any(part in ignored_parts for part in file.parts)
    ]

    return sorted(set(clean_files))


def resolve_data_file(config: dict) -> Path:
    """
    Resolve the configured data file.

    The config can use:
    - a file name, e.g. US100_cash_M1_NY_session_30d.csv
    - a relative path, e.g. data/historical/file.csv
    - an absolute path
    """
    configured_file = Path(config["csv_file"])

    if configured_file.is_absolute() and configured_file.exists():
        return configured_file

    direct_project_path = PROJECT_ROOT / configured_file

    if direct_project_path.exists():
        return direct_project_path

    candidate_files = list_candidate_data_files()

    matching_files = [
        file for file in candidate_files
        if file.name == configured_file.name
    ]

    if len(matching_files) == 1:
        return matching_files[0]

    if len(matching_files) > 1:
        print("Multiple matching files found:")

        for file in matching_files:
            print("-", file.relative_to(PROJECT_ROOT))

        raise ValueError(
            f"Multiple files named {configured_file.name} were found. "
            "Use a more specific relative path in CONFIG['csv_file']."
        )

    print(f"Could not find configured file: {config['csv_file']}")
    print("\nAvailable candidate files:")

    for file in candidate_files[:100]:
        print("-", file.relative_to(PROJECT_ROOT))

    raise FileNotFoundError(
        f"Could not find configured data file: {config['csv_file']}"
    )


def load_market_data(config: dict) -> tuple[pd.DataFrame, Path]:
    """
    Load the configured CSV or Parquet file and return a cleaned OHLC dataframe.
    """
    data_file = resolve_data_file(config)

    if data_file.suffix.lower() == ".csv":
        raw_df = pd.read_csv(data_file)
    elif data_file.suffix.lower() == ".parquet":
        raw_df = pd.read_parquet(data_file)
    else:
        raise ValueError(f"Unsupported file type: {data_file.suffix}")

    prepared_df = prepare_raw_ohlc_dataframe(raw_df)

    return prepared_df, data_file

In [ ]:
raw_ohlc_df, DATA_FILE = load_market_data(CONFIG)

print(f"Loaded data file: {DATA_FILE.relative_to(PROJECT_ROOT)}")
print(f"Rows loaded: {len(raw_ohlc_df):,}")
print(f"Start datetime: {raw_ohlc_df['datetime'].min()}")
print(f"End datetime: {raw_ohlc_df['datetime'].max()}")

print("\nColumns:")
print(list(raw_ohlc_df.columns))

print("\nFirst 5 rows:")
print(raw_ohlc_df.head().to_string(index=False))

## 5. Dataset Summary

This section prints a compact summary of the loaded dataset before the VWAP engine is applied.

In [ ]:
def summarise_raw_ohlc_data(df: pd.DataFrame, config: dict, data_file: Path) -> dict:
    """
    Create a compact summary of the loaded OHLC dataset.
    """
    summary = {
        "dataset_name": config["dataset_name"],
        "configured_file": config["csv_file"],
        "resolved_file": str(data_file.relative_to(PROJECT_ROOT)),
        "rows": len(df),
        "start_datetime": df["datetime"].min(),
        "end_datetime": df["datetime"].max(),
        "open_min": df["open"].min(),
        "open_max": df["open"].max(),
        "high_max": df["high"].max(),
        "low_min": df["low"].min(),
        "close_min": df["close"].min(),
        "close_max": df["close"].max(),
        "mean_candle_range": df["candle_range"].mean(),
        "median_candle_range": df["candle_range"].median(),
        "mean_body_ratio": df["body_ratio"].mean(),
        "zero_range_candles": int((df["candle_range"] <= 0).sum()),
        "duplicate_datetimes": int(df["datetime"].duplicated().sum()),
    }

    return summary


dataset_summary = summarise_raw_ohlc_data(raw_ohlc_df, CONFIG, DATA_FILE)

pprint(dataset_summary)

## 6. Ready for VWAP Engine Integration

At this point, the notebook has:

- loaded the configured dataset
- standardised OHLC column names
- parsed timestamps
- sorted candles chronologically
- created basic candle features
- printed a dataset summary

The next section will pass `raw_ohlc_df` into the existing VWAP Probability Band Engine from `src/`.

In [ ]:
print("raw_ohlc_df is ready for VWAP engine integration.")
print(f"Shape: {raw_ohlc_df.shape}")

print("\nLast 5 rows:")
print(raw_ohlc_df.tail().to_string(index=False))

## 7. Existing VWAP Engine Integration

This section passes the cleaned OHLC data through the existing VWAP Probability Band Engine from `src/`.

The aim is to reuse the existing project logic instead of rewriting or changing the model.

This section creates:

- VWAP/reference line
- sigma estimate
- upper and lower probability bands
- z-score
- zone classification
- automation-friendly band aliases

In [ ]:
import sys
import importlib


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


import src.config as engine_config_module
importlib.reload(engine_config_module)

from src.config import CONFIG as ENGINE_CONFIG
from src.reference import compute_reference
from src.sigma import compute_sigma, compute_bands
from src.zones import compute_zscore, classify_zones_series


print("Loaded existing VWAP engine modules.")
print("Engine config snapshot:")
print(f"- reference_type: {ENGINE_CONFIG.get('reference_type')}")
print(f"- vol_method: {ENGINE_CONFIG.get('vol_method')}")
print(f"- zone_thresholds: {ENGINE_CONFIG.get('zone_thresholds')}")

## 8. Engine Output Builder

This section wraps the existing VWAP engine into one clean function.

The function receives cleaned OHLC data and returns a full engine dataframe with standardised automation columns.

The strategy config remains separate from the engine config.

In [ ]:
REQUIRED_ENGINE_OUTPUT_COLUMNS = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
]


ENGINE_COLUMN_ALIASES = {
    "vwap": ["vwap", "VWAP", "reference", "ref", "reference_line"],
    "upper_green": ["upper_green", "upper_1", "upper_band_1", "band_1p", "band_1_plus", "band_1+", "z1_upper", "upper_sigma_1"],
    "upper_orange": ["upper_orange", "upper_2", "upper_band_2", "band_2p", "band_2_plus", "band_2+", "z2_upper", "upper_sigma_2"],
    "upper_red": ["upper_red", "upper_3", "upper_band_3", "band_3p", "band_3_plus", "band_3+", "z3_upper", "upper_sigma_3"],
    "lower_green": ["lower_green", "lower_1", "lower_band_1", "band_1n", "band_1m", "band_1_minus", "band_1-", "z1_lower", "lower_sigma_1"],
    "lower_orange": ["lower_orange", "lower_2", "lower_band_2", "band_2n", "band_2m", "band_2_minus", "band_2-", "z2_lower", "lower_sigma_2"],
    "lower_red": ["lower_red", "lower_3", "lower_band_3", "band_3n", "band_3m", "band_3_minus", "band_3-", "z3_lower", "lower_sigma_3"],
}


def add_engine_band_aliases(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add automation-friendly VWAP/band aliases to the existing engine output.

    This does not change model values.
    It only creates easier column names for the automated strategy layer.
    """
    out = df.copy()

    for standard_name, aliases in ENGINE_COLUMN_ALIASES.items():
        if standard_name in out.columns:
            continue

        matched_col = find_column(out, aliases)

        if matched_col is not None:
            out[standard_name] = out[matched_col]

    return out


def validate_engine_output_columns(df: pd.DataFrame) -> None:
    """
    Confirm that the engine output contains the columns needed by the automated strategy layer.
    """
    missing = [col for col in REQUIRED_ENGINE_OUTPUT_COLUMNS if col not in df.columns]

    if missing:
        raise ValueError(
            "Missing required engine output columns: "
            + ", ".join(missing)
            + "\n\nAvailable columns:\n"
            + ", ".join(map(str, df.columns))
        )

    print("All required engine output columns are available.")


def build_existing_engine_output(raw_ohlc_df: pd.DataFrame, engine_config: dict) -> pd.DataFrame:
    """
    Build VWAP probability-band output from raw OHLC data using the existing project logic.

    This function does not modify the existing model.

    It calls the existing source functions and then adds automation-friendly aliases.
    """
    df = raw_ohlc_df.copy()

    df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
    df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

    if "tick_volume" not in df.columns:
        df["tick_volume"] = 1.0

    df["tick_volume"] = (
        pd.to_numeric(df["tick_volume"], errors="coerce")
        .fillna(1.0)
        .clip(lower=1.0)
    )

    df["typical_price"] = (df["high"] + df["low"] + df["close"]) / 3.0
    df["session_date"] = df["datetime"].dt.date

    df["reference"] = compute_reference(df, engine_config)
    df["price_deviation"] = df["close"] - df["reference"]

    df["sigma"] = compute_sigma(df, engine_config)

    bands = compute_bands(df, df["sigma"])
    df = pd.concat([df, bands], axis=1)

    df["z_score"] = compute_zscore(df)
    df["zone"] = classify_zones_series(df["z_score"], engine_config["zone_thresholds"])

    df = add_engine_band_aliases(df)

    df["candle_range"] = df["high"] - df["low"]
    df["candle_body"] = (df["close"] - df["open"]).abs()
    df["body_ratio"] = np.where(
        df["candle_range"] > 0,
        df["candle_body"] / df["candle_range"],
        0.0,
    )

    validate_engine_output_columns(df)

    return df

## 9. Run Existing VWAP Engine

This section runs the existing engine on the loaded OHLC data.

The output is the full engine dataframe used by the automated V1 strategy.

In [ ]:
engine_df = build_existing_engine_output(raw_ohlc_df, ENGINE_CONFIG)

print(f"Built engine output for {len(engine_df):,} rows.")

engine_preview_cols = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "sigma",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
    "z_score",
    "zone",
]

print("\nEngine output preview:")
print(engine_df[engine_preview_cols].tail(10).to_string(index=False))

## 10. Automation-Ready DataFrame

This section prepares the engine output for the automated execution layer.

The output dataframe should contain all OHLC, VWAP, and band columns needed for feature engineering and signal generation.

In [ ]:
def prepare_automation_dataframe(engine_df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare existing VWAP engine output for the automated strategy layer.

    This function validates and cleans the columns required for automated V1 signal generation.
    """
    out = engine_df.copy()

    out = add_engine_band_aliases(out)
    validate_engine_output_columns(out)

    out["datetime"] = pd.to_datetime(out["datetime"], utc=True, errors="coerce")
    out = out.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

    numeric_cols = [
        "open",
        "high",
        "low",
        "close",
        "vwap",
        "upper_green",
        "upper_orange",
        "upper_red",
        "lower_green",
        "lower_orange",
        "lower_red",
    ]

    optional_numeric_cols = [
        "sigma",
        "z_score",
        "tick_volume",
        "spread",
        "real_volume",
    ]

    for col in numeric_cols + optional_numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=numeric_cols).reset_index(drop=True)

    out["candle_range"] = out["high"] - out["low"]
    out["candle_body"] = (out["close"] - out["open"]).abs()
    out["body_ratio"] = np.where(
        out["candle_range"] > 0,
        out["candle_body"] / out["candle_range"],
        0.0,
    )

    out["bar_index"] = np.arange(len(out))

    return out


automation_ready_df = prepare_automation_dataframe(engine_df)

print(f"Automation-ready dataframe rows: {len(automation_ready_df):,}")
print(f"Automation-ready dataframe columns: {len(automation_ready_df.columns):,}")

automation_preview_cols = [
    "bar_index",
    "datetime",
    "close",
    "vwap",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
    "body_ratio",
]

print("\nAutomation-ready preview:")
print(automation_ready_df[automation_preview_cols].tail(10).to_string(index=False))

## 11. Engine Summary

This section prints a compact summary of the VWAP engine output.

It helps confirm that the probability bands were generated correctly before signal rules are applied.

In [ ]:
def summarise_engine_output(df: pd.DataFrame, config: dict, engine_config: dict) -> dict:
    """
    Create a compact summary of the engine output.
    """
    summary = {
        "dataset_name": config["dataset_name"],
        "strategy_version": config["strategy_version"],
        "rows": len(df),
        "start_datetime": df["datetime"].min(),
        "end_datetime": df["datetime"].max(),
        "reference_type": engine_config.get("reference_type"),
        "vol_method": engine_config.get("vol_method"),
        "zone_thresholds": engine_config.get("zone_thresholds"),
        "mean_sigma": df["sigma"].mean() if "sigma" in df.columns else None,
        "median_sigma": df["sigma"].median() if "sigma" in df.columns else None,
        "mean_green_band_width": (df["upper_green"] - df["lower_green"]).mean(),
        "median_green_band_width": (df["upper_green"] - df["lower_green"]).median(),
        "mean_red_band_width": (df["upper_red"] - df["lower_red"]).mean(),
        "median_red_band_width": (df["upper_red"] - df["lower_red"]).median(),
        "zone_counts": df["zone"].value_counts(dropna=False).to_dict() if "zone" in df.columns else None,
        "missing_vwap_values": int(df["vwap"].isna().sum()),
        "missing_upper_green_values": int(df["upper_green"].isna().sum()),
        "missing_lower_green_values": int(df["lower_green"].isna().sum()),
    }

    return summary


engine_summary = summarise_engine_output(automation_ready_df, CONFIG, ENGINE_CONFIG)

pprint(engine_summary)

## 12. Ready for Automation Feature Engineering

At this point, the notebook has:

- loaded raw OHLC data
- reused the existing VWAP engine from `src/`
- created VWAP and probability bands
- added automation-friendly band aliases
- validated the automation-ready dataframe
- printed an engine summary

The next section will create V1 automation features from `automation_ready_df`.

In [ ]:
print("automation_ready_df is ready for automated feature engineering.")
print(f"Shape: {automation_ready_df.shape}")

print("\nAvailable key columns:")
key_cols = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
    "z_score",
    "zone",
]

for col in key_cols:
    print(f"- {col}: {'yes' if col in automation_ready_df.columns else 'missing'}")

## 13. Automation Feature Settings

This section adds the extra V1 feature settings used by the automated execution layer.

These settings control trend shifts, red-band shift labels, VWAP acceptance, chop detection, and band compression context.

In [ ]:
CONFIG.update(
    {
        # Feature lookbacks
        "shift_lookback": 3,
        "acceptance_lookback": 3,
        "trend_lane_lookback": 5,
        "trend_damage_lookback": 5,
        "compression_lookback": 5,
        "flat_vwap_lookback": 5,
        "vwap_cross_lookback": 8,

        # Chop / compression settings
        "flat_vwap_threshold_points": 3.0,
        "min_band_expansion_points": 0.0,
        "max_vwap_crosses_before_chop": 2,

        # Red-band shift strength buckets
        "red_shift_minimum_points": 2.5,
        "red_shift_good_points": 5.0,
        "red_shift_strong_points": 7.0,
        "red_shift_very_strong_points": 10.0,
        "red_shift_extreme_points": 15.0,
        "red_shift_very_extreme_points": 25.0,
        "red_shift_abnormal_points": 30.0,

        # V4 preliminary regime classification settings
        "v4_realised_range_lookback": 20,
        "v4_min_realised_range_periods": 10,
        "v4_high_realised_range_ratio": 1.25,
        "v4_extreme_realised_range_ratio": 2.00,
        "v4_vwap_slope_lookback": 10,
        "v4_flat_vwap_threshold_points": 3.0,
        "v4_band_width_lookback": 20,
        "v4_min_band_width_periods": 10,
        "v4_wide_band_width_ratio": 1.20,
        "v4_band_expansion_lookback": 5,
        "v4_min_band_expansion_points": 0.0,
        "v4_vwap_cross_lookback": 12,
        "v4_max_vwap_crosses_for_trend": 2,
        "v4_recent_extreme_lookback": 5,
        # V4 dynamic routing settings
        "v4_route_calm_trend_to": "v1_green_reclaim_rejection",
        "v4_route_volatile_trend_to": "v3_second_close_green_reclaim_rejection",
        "v4_route_chop_to": "no_trade",
        "v4_route_extreme_news_to": "no_trade",
        # V4 conditional V2 trend-health safety layer
        "v4_use_conditional_v2_trend_health_layer": True,
        "v4_conditional_v2_min_recent_vwap_crosses": 2,
        "v4_conditional_v2_extreme_range_ratio": 1.60,
        "v4_conditional_v2_min_directional_red_shift_points": 5.0,

        # V2 adaptive trend-health helper settings
        "v2_trend_health_lookback": 5,
        "v2_min_trend_health_periods": 3,
        "v2_min_red_shift_relative_to_average": 0.70,
        "v2_min_band_spread_change_points": 0.0,
        "v2_min_opposite_band_expansion_points": 0.0,
        "v2_trend_dead_bad_candles": 5,
        
        # V5 V2 state/score settings
        "v2_reclaim_recovery_lookback": 20,
        "v2_reclaim_acceptance_lookback": 6,
        "v2_min_reclaim_above_green_ratio": 0.65,
        "v2_min_reclaim_consecutive_green_closes": 2,
        "v2_min_pullback_green_damage_count": 3,
        "v2_vwap_slope_lookback": 5,
        "v2_flat_vwap_slope_points": 1.5,

        # V5 red-shift relative context
        "v5_red_shift_relative_lookback_bars": 20,
        "v5_red_shift_upgrade_ratio": 1.50,
        "v5_red_shift_downgrade_ratio": 0.75,
        
        # V3 second-close helper settings
        "v3_green_reentry_lookback": 4,
        "v3_trend_dead_bad_candles": 5,
        "v3_min_directional_red_shift_points": 5.0,
        "v3_band_spread_lookback": 1,
        "v3_a_tier_max_extension_from_green": 30.0,
        # V3 modular layer toggles
        "v3_use_vwap_acceptance_layer": True,
        "v3_use_v1_execution_quality_layer": True,
        "v3_use_v2_adaptive_trend_health_layer": False,
        "v3_use_directional_red_shift_layer": False,
        "v3_use_trend_not_dead_layer": True,
        "v3_require_red_bands_spreading": False,
    }
)

print("Updated V4 config with automation feature settings:")
pprint(CONFIG)

## 13A. V5 Modular Engine Controls

This section defines the control layer for the V5 modular engine.

The toggles below are designed to support isolated testing of the V1, V2, V3, regime-router, session-filter, and exit-manager components while preserving the existing V4.5-style behaviour until the modules are wired into the strategy logic.

In [ ]:
# ---------------------------------------------------------------------
# V5 modular engine configuration
# ---------------------------------------------------------------------
# These controls define the active V5 module, routing, session, and exit behaviour.

base_sl_points = float(CONFIG["sl_points"])
base_tp_points = float(CONFIG["tp_points"])
base_be_trigger_points = float(CONFIG["be_trigger_points"])

CONFIG.update(
    {
        # ------------------------------------------------------------------
        # V5 modular setup toggles
        # ------------------------------------------------------------------
        "enable_v1_s_tier": True,                  # clean S-tier continuation trades
        "enable_v2_trend_health_filter": True,     # apply V2 quality/safety checks
        "enable_v2_continuation_health": True,     # protect V1 continuation trades
        "enable_v2_reclaim_recovery_health": True, # allow V3 reclaim/recovery checks
        "enable_v3_a_tier_second_close": True,     # A-tier second-close reclaim/rejection

        # ------------------------------------------------------------------
        # V5 optional MR orange-failure entry controls
        # ------------------------------------------------------------------
        "enable_v5_mr_orange_failure_entry": False, # first MR test; OFF by default

        "mr_orange_confirm_closes": 2,
        "mr_orange_touch_lookback_bars": 3,
        "mr_use_locked_orange_touch": True,

        "mr_block_if_strong_continuation": True,
        "mr_require_weak_or_compressing_trend": True,

        "mr_target_mode": "vwap",
        "mr_sl_mode": "structure",
        "mr_max_sl_points": 18.0,
        "mr_min_target_rr": 1.2,

        "mr_entry_priority": "after_continuation",

        # ------------------------------------------------------------------
        # V5 V2 trend-health activation controls
        # ------------------------------------------------------------------
        "v2_trend_health_activation_mode": "router_gated",  # off, always, router_gated, time_gated, hybrid
        "v2_track_when_not_active": True,                   # calculate/report V2 even when not hard-blocking
        "v2_force_after_time_enabled": False,               # optional manual time override
        "v2_force_after_time": "16:30",
        "v2_force_after_timezone": "Europe/London",

        # ------------------------------------------------------------------
        # V5 router / engine-mode toggles
        # ------------------------------------------------------------------
        "engine_mode": "intelligent",              # manual = module toggles, intelligent = regime router
        "enable_regime_router": True,              # enable automatic regime routing
        "regime_lookback_minutes": 20,             # regime lookback window

        # ------------------------------------------------------------------
        # V5 session filter controls
        # ------------------------------------------------------------------
        "enable_session_filter": False,            # block trades outside chosen sessions
        "enabled_sessions": ["asia", "london", "new_york"],  # sessions allowed when filter is on
        "session_windows": {
            "asia": ("00:00", "07:00"),
            "london": ("07:00", "13:30"),
            "new_york": ("13:30", "21:00"),
        },
        "allow_out_of_session_trades": False,      # allow unknown/out-of-session trades

        # ------------------------------------------------------------------
        # V5 exit manager toggles
        # ------------------------------------------------------------------
        "enable_v5_exit_manager": True,           # use V5 exit framework
        "enable_runner_trailing": True,           # allow runner-mode logic
        "runner_mode": "always",                    # off, smart, always
        "enable_be_plus_buffer": True,             # move BE stop slightly beyond entry after +1R
        "be_plus_buffer_points": 3.0,              # BE-plus buffer in NQ points
        "enable_extreme_expansion_entries": False, # allow extreme expansion setups
        "enable_extreme_5r_tp": False,             # allow extreme setups to use 5R logic
        "enable_abnormal_news_entries": False,     # allow 30+ red-shift news regimes
        "v5_route_extreme_expansion_to": "extreme_expansion", # route extreme regimes to module

        # ------------------------------------------------------------------
        # V5 point-based exit aliases
        # These mirror existing sl_points / tp_points / be_trigger_points.
        # They are not used by the simulator yet.
        # ------------------------------------------------------------------
        "normal_sl_points": base_sl_points,
        "normal_tp_points": base_tp_points,
        "normal_be_trigger_points": base_be_trigger_points,

        # ------------------------------------------------------------------
        # V5 runner / stair trailing levels
        # R-based controls are the source of truth.
        # Point-based fields stay populated for simulator/reporting compatibility.
        # ------------------------------------------------------------------
        # Runner target and R-based trail rules.
        # To test 5R, set runner_target_r = 5.0.
        # To test 10R, set runner_target_r = 10.0.
        #
        # The extended trail rules can stay here for both.
        # If runner_target_r is 5.0, the 5R+ rules simply will not matter
        # because the trade should full-TP before reaching them.

        "runner_target_r": 7.0,

        "runner_trail_rules_r": [
            {"trigger_r": 1.0, "lock_r": 0.0, "label": "BE"},
            {"trigger_r": 2.7, "lock_r": 2.0, "label": "TRAIL_LOCK_2R"},
            {"trigger_r": 3.7, "lock_r": 3.0, "label": "TRAIL_LOCK_3R"},
            {"trigger_r": 4.7, "lock_r": 4.0, "label": "TRAIL_LOCK_4R"},
            {"trigger_r": 5.7, "lock_r": 5.0, "label": "TRAIL_LOCK_5R"},
            {"trigger_r": 6.7, "lock_r": 6.0, "label": "TRAIL_LOCK_6R"},
            #{"trigger_r": 7.7, "lock_r": 7.0, "label": "TRAIL_LOCK_7R"},
            #{"trigger_r": 8.7, "lock_r": 8.0, "label": "TRAIL_LOCK_8R"},
            #{"trigger_r": 9.7, "lock_r": 9.0, "label": "TRAIL_LOCK_9R"},
        ],


        "enable_legacy_early_runner_lock": False,
        "legacy_early_runner_lock_only_when_tp_points_above": 60.0,
        "legacy_runner_reference_sl_points": 29.0,
        "legacy_runner_lock_trigger_r": 2.7,
        "legacy_runner_lock_r": 2.0,

        "legacy_runner_lock_trigger_points": 29.0 * 2.7,
        "legacy_runner_lock_points": 29.0 * 2.0,

        "runner_tp_points": base_sl_points * 5.0,
        "extreme_tp_points": base_sl_points * 5.0,

        "trail_be_trigger_points": base_sl_points * 1.0,
        "trail_lock_2r_trigger_points": base_sl_points * 2.7,
        "trail_lock_3r_trigger_points": base_sl_points * 3.7,
        "trail_lock_4r_trigger_points": base_sl_points * 4.7,
        "trail_5r_tp_points": base_sl_points * 5.0,

        "trail_lock_2r_points": base_sl_points * 2.0,
        "trail_lock_3r_points": base_sl_points * 3.0,
        "trail_lock_4r_points": base_sl_points * 4.0,

        "runner_allowed_setup_sources": [
            "V1_S_TIER_WITH_V2_FILTER",
            "V3_A_TIER_SECOND_CLOSE",
            "V3_A_TIER_RECLAIM_RECOVERY",
            "EXTREME_EXPANSION",
        ],
        "runner_allowed_regimes": [
            "strong_clean_trend",
            "volatile_trend",
            "extreme_expansion",
            "very_extreme_expansion",
        ],
        "runner_min_red_shift_buckets": [
            "very_strong",
            "extreme",
            "very_extreme",
        ],
        "runner_block_health_states": [
            "DEAD_TREND",
            "CHOP_COMPRESSION",
        ],

        # ------------------------------------------------------------------
        # V5 optional entry red-shift floor controls
        # ------------------------------------------------------------------
        "entry_min_directional_red_shift_points": 2.0,
        "s_tier_use_entry_red_shift_floor": False,
        "s_tier_min_directional_red_shift_points": 2.0,
        "a_tier_use_entry_red_shift_floor": False,
        "a_tier_min_directional_red_shift_points": 3.0,

        # ------------------------------------------------------------------
        # V5 optional Dynamic S-tier touch diagnostics
        # ------------------------------------------------------------------
        "enable_dynamic_s_tier_touch_diagnostics": True,
        "dynamic_s_tier_touch_mode": "tag_only",  # tag_only only for now

        # ------------------------------------------------------------------
        # V5 optional S-tier clean-state filter
        # ------------------------------------------------------------------
        "enable_s_tier_clean_state_filter": True,

        "s_tier_green_failure_lookback_bars": 8,
        "s_tier_required_clean_closes_after_failure": 3,

        # ------------------------------------------------------------------
        # V5 optional A-tier retrace-entry controls
        # ------------------------------------------------------------------
        "enable_a_tier_retrace_entry": False,      # optional retrace mode for extended A-tier confirmations
        "a_tier_retrace_entry_mode": "extended_only",  # off, extended_only, always_wait_for_retrace
        "a_tier_max_direct_extension_from_green": 30.0,  # direct second-close allowed up to this extension
        "a_tier_retrace_wait_bars": 1,             # 1 = only the third close gets a chance
        "a_tier_retrace_min_points": 5.0,          # minimum pullback from second close
        "a_tier_retrace_max_points": 25.0,         # maximum controlled pullback
        "a_tier_retrace_max_extension_after_retrace": 25.0,  # max extension after retrace close
        "a_tier_retrace_must_hold_green": True,    # retrace close must still hold green
        "a_tier_retrace_allow_intrabar_green_pierce_points": 0.0,  # allowed intrabar green pierce

        # ------------------------------------------------------------------
        # V5 optional A-tier delayed-pullback entry controls
        # ------------------------------------------------------------------
        "enable_a_tier_delayed_pullback_entry": True,

        "a_tier_delayed_pullback_entry_bar": 4,
        "a_tier_delayed_pullback_min_hold_bars": 3,

        "a_tier_delayed_pullback_min_points": 5.0,
        "a_tier_delayed_pullback_max_points": 15.0,

        "a_tier_delayed_pullback_must_hold_green": True,
        "a_tier_delayed_pullback_allow_intrabar_green_pierce_points": 0.0,

        "a_tier_delayed_pullback_require_trend_health": True,

        "enable_a_tier_delayed_pullback_strength_filter": False,

        "a_tier_delayed_pullback_allowed_red_shift_buckets": ["strong", "very_strong"],

        "a_tier_delayed_pullback_block_track_only": False,
        "a_tier_delayed_pullback_require_v2_pass": False,

        # ------------------------------------------------------------------
        # V5 optional A-tier fast-reclaim entry controls
        # ------------------------------------------------------------------
        "enable_a_tier_fast_reclaim_entry": False,

        "a_tier_fast_reclaim_max_away_bars": 2,
        "a_tier_fast_reclaim_max_depth_points": 20.0,

        "a_tier_fast_reclaim_min_body_points": 10.0,
        "a_tier_fast_reclaim_min_body_ratio": 0.60,
        "a_tier_fast_reclaim_min_close_position": 0.70,

        "a_tier_fast_reclaim_require_trend_health": True,

        # ------------------------------------------------------------------
        # V5 risk / reporting controls
        # ------------------------------------------------------------------
        "max_daily_loss_r": -2,
        "max_no_tp_run_r": -6,
        "max_consecutive_sl": 2,
        "plot_session_time_chart": True,
    }
)


V5_MODULE_TOGGLE_KEYS = {
    "v1_s_tier": "enable_v1_s_tier",
    "v2_trend_health_filter": "enable_v2_trend_health_filter",
    "v2_continuation_health": "enable_v2_continuation_health",
    "v2_reclaim_recovery_health": "enable_v2_reclaim_recovery_health",
    "v3_a_tier_second_close": "enable_v3_a_tier_second_close",
    "extreme_expansion": "enable_extreme_expansion_entries",
}

V5_VALID_SESSIONS = {"asia", "london", "new_york"}
V5_ENGINE_MODES = {"manual", "intelligent"}


def v5_get_engine_mode(config: dict) -> str:
    """
    Return the active V5 engine mode.

    manual:
        Manual module toggles decide what can trade.

    intelligent:
        The regime router chooses between allowed modules.
    """
    engine_mode = str(config.get("engine_mode", "intelligent")).lower()

    if engine_mode not in V5_ENGINE_MODES:
        raise ValueError(
            f"Invalid engine_mode={engine_mode!r}. "
            f"Allowed values: {sorted(V5_ENGINE_MODES)}"
        )

    return engine_mode


def v5_use_intelligent_router(config: dict) -> bool:
    """
    Return whether V5 should use intelligent regime routing.
    """
    return (
        v5_get_engine_mode(config) == "intelligent"
        and bool(config.get("enable_regime_router", True))
    )


def v5_enabled_sessions(config: dict) -> list[str]:
    """
    Return valid enabled V5 trading sessions.

    Used by the V5 session filter and trade-tracking layer.
    """
    sessions = config.get("enabled_sessions", ["asia", "london", "new_york"])

    if isinstance(sessions, str):
        sessions = [sessions]

    return [session for session in sessions if session in V5_VALID_SESSIONS]


def v5_is_session_enabled(trade_session: str, config: dict) -> bool:
    """
    Session override helper.

    If session filtering is off, all sessions are allowed.
    If session filtering is on, disabled sessions should block future candidates.
    """
    if not config.get("enable_session_filter", False):
        return True

    if trade_session in {"unknown", "out_of_session"}:
        return bool(config.get("allow_out_of_session_trades", False))

    return trade_session in v5_enabled_sessions(config)


def v5_apply_manual_module_overrides(
    router_allowed_modules: dict[str, bool] | None,
    config: dict,
) -> dict[str, bool]:
    """
    Apply manual V5 toggles over any router decision.

    Manual False always blocks a module.
    If the regime router is disabled, manual toggles become the full source of truth.

    Used by the V5 module-gating layer.
    """
    if router_allowed_modules is None:
        router_allowed_modules = {
            module_name: True for module_name in V5_MODULE_TOGGLE_KEYS
        }

    effective_modules = dict(router_allowed_modules)

    for module_name, toggle_key in V5_MODULE_TOGGLE_KEYS.items():
        manual_enabled = bool(config.get(toggle_key, True))
        router_enabled = bool(effective_modules.get(module_name, True))
        effective_modules[module_name] = manual_enabled and router_enabled

    if not config.get("enable_regime_router", True):
        for module_name, toggle_key in V5_MODULE_TOGGLE_KEYS.items():
            effective_modules[module_name] = bool(config.get(toggle_key, True))

    return effective_modules


v5_config_summary = {
    "v1_enabled": CONFIG["enable_v1_s_tier"],
    "v2_enabled": CONFIG["enable_v2_trend_health_filter"],
    "v2_continuation_health": CONFIG["enable_v2_continuation_health"],
    "v2_reclaim_recovery_health": CONFIG["enable_v2_reclaim_recovery_health"],
    "v3_enabled": CONFIG["enable_v3_a_tier_second_close"],

    "mr_orange_failure_enabled": CONFIG["enable_v5_mr_orange_failure_entry"],
    "mr_orange_confirm_closes": CONFIG["mr_orange_confirm_closes"],
    "mr_orange_touch_lookback_bars": CONFIG["mr_orange_touch_lookback_bars"],
    "mr_use_locked_orange_touch": CONFIG["mr_use_locked_orange_touch"],
    "mr_block_if_strong_continuation": CONFIG["mr_block_if_strong_continuation"],
    "mr_require_weak_or_compressing_trend": CONFIG["mr_require_weak_or_compressing_trend"],
    "mr_target_mode": CONFIG["mr_target_mode"],
    "mr_sl_mode": CONFIG["mr_sl_mode"],
    "mr_max_sl_points": CONFIG["mr_max_sl_points"],
    "mr_min_target_rr": CONFIG["mr_min_target_rr"],
    "mr_entry_priority": CONFIG["mr_entry_priority"],

    "engine_mode": v5_get_engine_mode(CONFIG),
    "router_enabled": CONFIG["enable_regime_router"],
    "session_filter_enabled": CONFIG["enable_session_filter"],
    "enabled_sessions": v5_enabled_sessions(CONFIG),
    "exit_manager_enabled": CONFIG["enable_v5_exit_manager"],
    "runner_mode": CONFIG["runner_mode"],
    "runner_trailing_enabled": CONFIG["enable_runner_trailing"],
    "normal_sl_points": CONFIG["normal_sl_points"],
    "normal_tp_points": CONFIG["normal_tp_points"],
    "normal_be_trigger_points": CONFIG["normal_be_trigger_points"],
    "runner_tp_points": CONFIG["runner_tp_points"],
    "enable_be_plus_buffer": CONFIG["enable_be_plus_buffer"],
    "be_plus_buffer_points": CONFIG["be_plus_buffer_points"],
    "trail_be_trigger_points": CONFIG["trail_be_trigger_points"],
    "trail_lock_2r_trigger_points": CONFIG["trail_lock_2r_trigger_points"],
    "trail_lock_3r_trigger_points": CONFIG["trail_lock_3r_trigger_points"],
    "trail_lock_4r_trigger_points": CONFIG["trail_lock_4r_trigger_points"],
    "entry_min_directional_red_shift_points": CONFIG["entry_min_directional_red_shift_points"],

    "s_tier_use_entry_red_shift_floor": CONFIG["s_tier_use_entry_red_shift_floor"],
    "s_tier_min_directional_red_shift_points": CONFIG["s_tier_min_directional_red_shift_points"],
    
    "a_tier_use_entry_red_shift_floor": CONFIG["a_tier_use_entry_red_shift_floor"],
    "a_tier_min_directional_red_shift_points": CONFIG["a_tier_min_directional_red_shift_points"],
    "enable_a_tier_retrace_entry": CONFIG["enable_a_tier_retrace_entry"],
    "a_tier_retrace_entry_mode": CONFIG["a_tier_retrace_entry_mode"],
    "a_tier_max_direct_extension_from_green": CONFIG["a_tier_max_direct_extension_from_green"],
    "a_tier_retrace_wait_bars": CONFIG["a_tier_retrace_wait_bars"],
    "a_tier_retrace_min_points": CONFIG["a_tier_retrace_min_points"],
    "a_tier_retrace_max_points": CONFIG["a_tier_retrace_max_points"],
    "a_tier_retrace_max_extension_after_retrace": CONFIG["a_tier_retrace_max_extension_after_retrace"],
    "a_tier_retrace_must_hold_green": CONFIG["a_tier_retrace_must_hold_green"],
    "a_tier_retrace_allow_intrabar_green_pierce_points": CONFIG["a_tier_retrace_allow_intrabar_green_pierce_points"],

    "v2_trend_health_activation_mode": CONFIG["v2_trend_health_activation_mode"],
    "v2_track_when_not_active": CONFIG["v2_track_when_not_active"],
    "v2_force_after_time_enabled": CONFIG["v2_force_after_time_enabled"],
    "v2_force_after_time": CONFIG["v2_force_after_time"],
    "v2_force_after_timezone": CONFIG["v2_force_after_timezone"],
    
    "active_preset": CONFIG.get("active_preset", "custom"),
}

print("Added V5 modular engine controls:")
pprint(v5_config_summary)

### Dynamic S-tier touch diagnostics

A normal S-tier continuation assumes price genuinely pulled back into the green band and then closed back in the trend direction.

However, VWAP probability bands update every candle. Sometimes the final band value can shift into a candle after the candle has closed. In that case, the historical candle may look like it touched green, even though live price did not clearly pull back into the band at the time.

This notebook labels those cases as:

- `S_TIER_REAL_GREEN_TOUCH`: price touched the previous/locked green-band level.
- `S_TIER_BAND_SHIFT_TOUCH`: price only appears to have touched green using the final recalculated band value.

By default this is diagnostic only. It does not block trades unless a future test explicitly enables a blocker.

### S-tier clean-state filter

A true S-tier continuation should come from clean green support: price is already holding the trend side of green, pulls back into green, defends it, and closes back in the trend direction.

If price recently closed beyond green, the setup is no longer clean S-tier. It is more like A-tier reclaim territory. The clean-state filter keeps S-tier disabled after a green failure until price has rebuilt structure with enough clean closes back on the correct side of green.

By default this filter is OFF so baseline results are unchanged.

## 13B. V5 Quick Presets / How To Run Modes

This section provides optional presets for common V5 test modes.

The default preset is `custom`, which means the notebook uses the manual Section 13A toggles exactly as written. No preset values are applied unless `SELECTED_V5_PRESET` is changed.

Available modes:

- `custom`: uses the manual Section 13A toggles exactly as written.
- `manual_v1`: closest to testing original V1 / S-tier behaviour.
- `manual_v1_with_v2`: V1 / S-tier with V2 trend-health protection.
- `manual_v3`: tests A-tier second-close reclaim/rejection logic.
- `manual_v3_retrace_test`: tests A-tier / V3 with optional retrace entries enabled for extended second-close confirmations.
- `manual_v1_v3`: allows both V1 and V3 without intelligent regime routing.
- `intelligent_default`: allows V1/V2/V3, then lets the regime router choose the active module.
- `volatile_day`: intelligent mode with extreme expansion allowed, while abnormal-news regimes remain blocked.
- `runner_always_test`: forces every valid trade to use runner trailing. Testing only.
- `london_only_test`: verifies that the session filter blocks non-London trades.
- `manual_v1_redshift_floor`: S-tier / V1 only, with a minimum directional red-shift floor.
- `manual_v1_with_v2_redshift_floor`: S-tier / V1 with V2 protection and a minimum directional red-shift floor.
- `manual_v3_redshift_floor`: A-tier / V3 second-close logic with a minimum directional red-shift floor.

In [ ]:
# ---------------------------------------------------------------------
# V5 quick presets
# ---------------------------------------------------------------------

V5_PRESETS = {
    "manual_v1": {
        "active_preset": "manual_v1",
        "engine_mode": "manual",
        "enable_regime_router": False,
        "enable_v1_s_tier": True,
        "enable_v2_trend_health_filter": False,
        "enable_v2_continuation_health": False,
        "enable_v2_reclaim_recovery_health": False,
        "enable_v3_a_tier_second_close": False,
        "enable_extreme_expansion_entries": False,
        "enable_abnormal_news_entries": False,
        "enable_session_filter": False,
        "runner_mode": "off",
    },

    "manual_v1_with_v2": {
        "active_preset": "manual_v1_with_v2",
        "engine_mode": "manual",
        "enable_regime_router": False,
        "enable_v1_s_tier": True,
        "enable_v2_trend_health_filter": True,
        "enable_v2_continuation_health": True,
        "enable_v2_reclaim_recovery_health": False,
        "enable_v3_a_tier_second_close": False,
        "enable_extreme_expansion_entries": False,
        "enable_abnormal_news_entries": False,
        "enable_session_filter": False,
        "runner_mode": "off",
    },

    "manual_v1_redshift_floor": {
        "active_preset": "manual_v1_redshift_floor",
        "engine_mode": "manual",
        "enable_regime_router": False,
        "enable_v1_s_tier": True,
        "enable_v2_trend_health_filter": False,
        "enable_v2_continuation_health": False,
        "enable_v2_reclaim_recovery_health": False,
        "enable_v3_a_tier_second_close": False,
        "enable_extreme_expansion_entries": False,
        "enable_abnormal_news_entries": False,
        "enable_session_filter": False,
        "runner_mode": "off",
        "s_tier_use_entry_red_shift_floor": True,
        "s_tier_min_directional_red_shift_points": 2.0,
        "a_tier_use_entry_red_shift_floor": False,
    },

    "manual_v1_with_v2_redshift_floor": {
        "active_preset": "manual_v1_with_v2_redshift_floor",
        "engine_mode": "manual",
        "enable_regime_router": False,
        "enable_v1_s_tier": True,
        "enable_v2_trend_health_filter": True,
        "enable_v2_continuation_health": True,
        "enable_v2_reclaim_recovery_health": False,
        "enable_v3_a_tier_second_close": False,
        "enable_extreme_expansion_entries": False,
        "enable_abnormal_news_entries": False,
        "enable_session_filter": False,
        "runner_mode": "off",
        "s_tier_use_entry_red_shift_floor": True,
        "s_tier_min_directional_red_shift_points": 2.0,
        "a_tier_use_entry_red_shift_floor": False,
    },

    "manual_v3_redshift_floor": {
        "active_preset": "manual_v3_redshift_floor",
        "engine_mode": "manual",
        "enable_regime_router": False,
        "enable_v1_s_tier": False,
        "enable_v2_trend_health_filter": True,
        "enable_v2_continuation_health": False,
        "enable_v2_reclaim_recovery_health": True,
        "enable_v3_a_tier_second_close": True,
        "enable_extreme_expansion_entries": False,
        "enable_abnormal_news_entries": False,
        "enable_session_filter": False,
        "runner_mode": "off",
        "s_tier_use_entry_red_shift_floor": False,
        "a_tier_use_entry_red_shift_floor": True,
        "a_tier_min_directional_red_shift_points": 2.0,
    },

    "manual_v3": {
        "active_preset": "manual_v3",
        "engine_mode": "manual",
        "enable_regime_router": False,
        "enable_v1_s_tier": False,
        "enable_v2_trend_health_filter": True,
        "enable_v2_continuation_health": False,
        "enable_v2_reclaim_recovery_health": True,
        "enable_v3_a_tier_second_close": True,
        "enable_extreme_expansion_entries": False,
        "enable_abnormal_news_entries": False,
        "enable_session_filter": False,
        "runner_mode": "off",
    },

    "manual_v3_retrace_test": {
        "active_preset": "manual_v3_retrace_test",
        "engine_mode": "manual",
        "enable_regime_router": False,
        "enable_v1_s_tier": False,
        "enable_v2_trend_health_filter": True,
        "enable_v2_continuation_health": False,
        "enable_v2_reclaim_recovery_health": True,
        "enable_v3_a_tier_second_close": True,
        "enable_a_tier_retrace_entry": True,
        "a_tier_retrace_entry_mode": "extended_only",
        "enable_extreme_expansion_entries": False,
        "enable_abnormal_news_entries": False,
        "enable_session_filter": False,
        "runner_mode": "off",
    },

    "manual_v1_v3": {
        "active_preset": "manual_v1_v3",
        "engine_mode": "manual",
        "enable_regime_router": False,
        "enable_v1_s_tier": True,
        "enable_v2_trend_health_filter": True,
        "enable_v2_continuation_health": True,
        "enable_v2_reclaim_recovery_health": True,
        "enable_v3_a_tier_second_close": True,
        "enable_extreme_expansion_entries": False,
        "enable_abnormal_news_entries": False,
        "enable_session_filter": False,
        "runner_mode": "off",
    },

    "intelligent_default": {
        "active_preset": "intelligent_default",
        "engine_mode": "intelligent",
        "enable_regime_router": True,
        "enable_v1_s_tier": True,
        "enable_v2_trend_health_filter": True,
        "enable_v2_continuation_health": True,
        "enable_v2_reclaim_recovery_health": True,
        "enable_v3_a_tier_second_close": True,
        "enable_extreme_expansion_entries": False,
        "enable_abnormal_news_entries": False,
        "enable_session_filter": False,
        "runner_mode": "always",
    },

    "volatile_day": {
        "active_preset": "volatile_day",
        "engine_mode": "intelligent",
        "enable_regime_router": True,
        "enable_v1_s_tier": True,
        "enable_v2_trend_health_filter": True,
        "enable_v2_continuation_health": True,
        "enable_v2_reclaim_recovery_health": True,
        "enable_v3_a_tier_second_close": True,
        "enable_extreme_expansion_entries": True,
        "enable_abnormal_news_entries": False,
        "enable_session_filter": False,
        "runner_mode": "smart",
    },

    "runner_always_test": {
        "active_preset": "runner_always_test",
        "engine_mode": "manual",
        "enable_regime_router": False,
        "enable_v1_s_tier": True,
        "enable_v2_trend_health_filter": True,
        "enable_v2_continuation_health": True,
        "enable_v2_reclaim_recovery_health": True,
        "enable_v3_a_tier_second_close": True,
        "enable_extreme_expansion_entries": False,
        "enable_abnormal_news_entries": False,
        "enable_session_filter": False,
        "enable_v5_exit_manager": True,
        "enable_runner_trailing": True,
        "runner_mode": "always",
    },

    "london_only_test": {
        "active_preset": "london_only_test",
        "engine_mode": "manual",
        "enable_regime_router": False,
        "enable_v1_s_tier": True,
        "enable_v2_trend_health_filter": True,
        "enable_v2_continuation_health": True,
        "enable_v2_reclaim_recovery_health": True,
        "enable_v3_a_tier_second_close": True,
        "enable_extreme_expansion_entries": False,
        "enable_abnormal_news_entries": False,
        "enable_session_filter": True,
        "enabled_sessions": ["london"],
        "runner_mode": "off",
    },
}

V5_PRESET_REDSHIFT_DEFAULTS = {
    "s_tier_use_entry_red_shift_floor": False,
    "s_tier_min_directional_red_shift_points": None,
    "a_tier_use_entry_red_shift_floor": False,
    "a_tier_min_directional_red_shift_points": None,
}


def apply_v5_preset(config: dict, preset_name: str | None) -> dict:
    """
    Apply a named V5 preset by updating CONFIG toggles.

    If preset_name is 'custom' or 'none', manual Section 13A settings are preserved.
    """
    config = config.copy()

    if preset_name is None:
        config["active_preset"] = "custom"
        return config

    preset_name = str(preset_name).lower()

    if preset_name in {"custom", "none"}:
        config["active_preset"] = "custom"
        return config

    if preset_name not in V5_PRESETS:
        available_presets = sorted(list(V5_PRESETS.keys()) + ["custom", "none"])
        raise ValueError(
            f"Unknown V5 preset: {preset_name}. "
            f"Available presets: {available_presets}"
        )

    config.update(V5_PRESET_REDSHIFT_DEFAULTS)
    config.update(V5_PRESETS[preset_name])

    return config


def build_v5_preset_summary(config: dict, selected_preset: str) -> pd.DataFrame:
    """
    Build a compact summary of the active V5 preset and key mode toggles.
    """
    enabled_modules = []

    if config.get("enable_v1_s_tier", False):
        enabled_modules.append("V1")

    if config.get("enable_v2_trend_health_filter", False):
        enabled_modules.append("V2")

    if config.get("enable_v3_a_tier_second_close", False):
        enabled_modules.append("V3")

    if config.get("enable_extreme_expansion_entries", False):
        enabled_modules.append("EXTREME")

    if len(enabled_modules) == 0:
        enabled_modules = ["none"]

    return pd.DataFrame(
        [
            {
                "selected_preset": selected_preset,
                "active_preset": config.get("active_preset", "custom"),
                "engine_mode": config.get("engine_mode", "intelligent"),
                "runner_mode": config.get("runner_mode", "smart"),
                "enabled_modules": ", ".join(enabled_modules),
                "enable_v1_s_tier": config.get("enable_v1_s_tier", False),
                "enable_v2_trend_health_filter": config.get("enable_v2_trend_health_filter", False),
                "enable_v2_continuation_health": config.get("enable_v2_continuation_health", False),
                "enable_v2_reclaim_recovery_health": config.get("enable_v2_reclaim_recovery_health", False),
                "v2_trend_health_activation_mode": config.get("v2_trend_health_activation_mode", "router_gated"),
                "v2_track_when_not_active": config.get("v2_track_when_not_active", True),
                "v2_force_after_time_enabled": config.get("v2_force_after_time_enabled", False),
                "v2_force_after_time": config.get("v2_force_after_time", "16:30"),
                "enable_v3_a_tier_second_close": config.get("enable_v3_a_tier_second_close", False),
                "enable_session_filter": config.get("enable_session_filter", False),
                "enabled_sessions": config.get("enabled_sessions", []),
                "entry_min_directional_red_shift_points": config.get("entry_min_directional_red_shift_points", 2.0),
                "s_tier_use_entry_red_shift_floor": config.get("s_tier_use_entry_red_shift_floor", False),
                "s_tier_min_directional_red_shift_points": config.get("s_tier_min_directional_red_shift_points", None),
                "a_tier_use_entry_red_shift_floor": config.get("a_tier_use_entry_red_shift_floor", False),
                "a_tier_min_directional_red_shift_points": config.get("a_tier_min_directional_red_shift_points", None),
                "enable_a_tier_retrace_entry": config.get("enable_a_tier_retrace_entry", False),
                "a_tier_retrace_entry_mode": config.get("a_tier_retrace_entry_mode", "extended_only"),
                "a_tier_max_direct_extension_from_green": config.get("a_tier_max_direct_extension_from_green", 30.0),
            }
        ]
    )


# Optional preset application.
# Leave as "custom" to use the manual Section 13A toggles exactly as written.
SELECTED_V5_PRESET = "custom"

# User-facing V5 experiment logging controls.
# Leave EXPERIMENT_NAME as "auto" to generate a safe name from the active toggles.
SAVE_EXPERIMENT_LOG = False

EXPERIMENT_NAME = "safe_base2_36sl_10r_runner_legacy_lock_on"
EXPERIMENT_TAG = "runner_test"
EXPERIMENT_NOTES = (
    "Safe Base 2 runner test. Delayed pullback ON, S clean-state 8/3 ON, "
    "SL 36, original TP 72 for diagnostics, runner target 10R, "
    "extended R-based trail rules, and legacy early runner lock ON. "
    "Legacy lock uses old 29-point SL reference: trigger old 2.7R = 78.3 pts, "
    "lock old 2R = 58 pts when tp_points > 60."
)

EXPERIMENT_LOG_PATH = ARTIFACTS_DIR / "tables" / "v5_experiment_log.csv"

# Examples:
# SELECTED_V5_PRESET = "manual_v1"
# SELECTED_V5_PRESET = "manual_v1_with_v2"
# SELECTED_V5_PRESET = "manual_v3"
# SELECTED_V5_PRESET = "manual_v3_retrace_test"
# SELECTED_V5_PRESET = "manual_v1_v3"
# SELECTED_V5_PRESET = "intelligent_default"
# SELECTED_V5_PRESET = "volatile_day"
# SELECTED_V5_PRESET = "runner_always_test"
# SELECTED_V5_PRESET = "london_only_test"

def apply_v5_runner_r_settings(config: dict) -> dict:
    """
    Convert R-based runner settings into point-based fields used by the
    existing simulator/reporting.

    Default runner_target_r=5.0 reproduces the current 5R runner.
    """
    out = config.copy()

    sl_points = float(out["sl_points"])
    runner_target_r = float(out.get("runner_target_r", 5.0))

    out["runner_tp_points"] = sl_points * runner_target_r
    out["trail_5r_tp_points"] = sl_points * runner_target_r
    out["extreme_tp_points"] = sl_points * runner_target_r

    trail_rules = out.get("runner_trail_rules_r", [])

    for rule in trail_rules:
        trigger_r = float(rule["trigger_r"])
        lock_r = float(rule["lock_r"])
        label = str(rule.get("label", "")).upper()

        trigger_points = sl_points * trigger_r
        lock_points = sl_points * lock_r

        if label == "BE":
            out["trail_be_trigger_points"] = trigger_points

        elif label == "TRAIL_LOCK_2R":
            out["trail_lock_2r_trigger_points"] = trigger_points
            out["trail_lock_2r_points"] = lock_points

        elif label == "TRAIL_LOCK_3R":
            out["trail_lock_3r_trigger_points"] = trigger_points
            out["trail_lock_3r_points"] = lock_points

        elif label == "TRAIL_LOCK_4R":
            out["trail_lock_4r_trigger_points"] = trigger_points
            out["trail_lock_4r_points"] = lock_points

    legacy_reference_sl = float(out.get("legacy_runner_reference_sl_points", 29.0))
    legacy_trigger_r = float(out.get("legacy_runner_lock_trigger_r", 2.7))
    legacy_lock_r = float(out.get("legacy_runner_lock_r", 2.0))

    out["legacy_runner_lock_trigger_points"] = legacy_reference_sl * legacy_trigger_r
    out["legacy_runner_lock_points"] = legacy_reference_sl * legacy_lock_r

    return out

CONFIG = apply_v5_preset(CONFIG, SELECTED_V5_PRESET)
CONFIG = apply_v5_runner_r_settings(CONFIG)


def safe_name_part(value) -> str:
    """
    Make one component safe for CSV/log names and filenames.
    """
    text = str(value)
    text = text.replace(" ", "")
    text = text.replace(":", "-")
    text = text.replace("/", "-")
    text = text.replace("\\", "-")
    text = text.replace("\n", "-")
    text = text.replace("\t", "-")

    while "__" in text:
        text = text.replace("__", "_")

    return text


def bool_label(value) -> str:
    """
    Compact on/off label for auto-generated experiment names.
    """
    return "on" if bool(value) else "off"


def get_runner_target_r(config: dict) -> float | None:
    """
    Return active runner target in R.
    """
    if "runner_target_r" in config:
        return float(config["runner_target_r"])

    sl_points = config.get("sl_points") or config.get("base_sl_points") or None
    tp_points = config.get("trail_5r_tp_points") or config.get("runner_tp_points") or None

    if sl_points in [None, 0] or tp_points is None:
        return None

    return float(tp_points) / float(sl_points)


def make_v5_experiment_name(config: dict, selected_preset: str) -> str:
    """
    Build a compact auto-name from the active V5 preset and key toggles.
    """
    runner_target_r = get_runner_target_r(config)

    target_label = "tpNA"
    if runner_target_r is not None:
        target_label = f"tp{runner_target_r:.1f}R"

    s_floor_enabled = config.get("s_tier_use_entry_red_shift_floor", False)
    a_floor_enabled = config.get("a_tier_use_entry_red_shift_floor", False)

    s_floor_value = config.get("s_tier_min_directional_red_shift_points")
    a_floor_value = config.get("a_tier_min_directional_red_shift_points")

    s_floor_label = "sfloor_off"
    if s_floor_enabled:
        s_floor_label = f"sfloor_on_{s_floor_value}"

    a_floor_label = "afloor_off"
    if a_floor_enabled:
        a_floor_label = f"afloor_on_{a_floor_value}"

    parts = [
        selected_preset,
        f"engine_{config.get('engine_mode')}",
        f"runner_{config.get('runner_mode')}",
        target_label,
        f"v2_{config.get('v2_trend_health_activation_mode')}",
        s_floor_label,
        a_floor_label,
        f"retrace_{config.get('enable_a_tier_retrace_entry')}",
        f"session_{config.get('enable_session_filter')}",
        f"cutoff_{config.get('no_new_trades_after')}",
        f"daily{config.get('max_daily_loss_r')}",
        f"slstreak{config.get('max_consecutive_sl')}",
    ]

    return "__".join(safe_name_part(part) for part in parts)


if EXPERIMENT_NAME == "auto":
    EXPERIMENT_NAME = make_v5_experiment_name(CONFIG, SELECTED_V5_PRESET)

v5_preset_summary = build_v5_preset_summary(CONFIG, SELECTED_V5_PRESET)

print("Active V5 preset and mode summary:")
display(v5_preset_summary)

print(f"Experiment name: {EXPERIMENT_NAME}")
print(f"Experiment tag: {EXPERIMENT_TAG}")
print(f"Experiment notes: {EXPERIMENT_NOTES}")

IMPORTANT_CONFIG_KEYS = [
    "active_preset",
    "engine_mode",
    "enable_regime_router",

    "enable_v1_s_tier",
    "enable_v2_trend_health_filter",
    "enable_v2_continuation_health",
    "enable_v2_reclaim_recovery_health",
    "enable_v3_a_tier_second_close",

    "s_tier_use_entry_red_shift_floor",
    "s_tier_min_directional_red_shift_points",
    "a_tier_use_entry_red_shift_floor",
    "a_tier_min_directional_red_shift_points",

    "enable_dynamic_s_tier_touch_diagnostics",
    "dynamic_s_tier_touch_mode",
    "enable_s_tier_clean_state_filter",
    "s_tier_green_failure_lookback_bars",
    "s_tier_required_clean_closes_after_failure",

    "enable_a_tier_retrace_entry",
    "a_tier_retrace_entry_mode",

    "enable_a_tier_delayed_pullback_strength_filter",
    "a_tier_delayed_pullback_allowed_red_shift_buckets",
    "a_tier_delayed_pullback_block_track_only",
    "a_tier_delayed_pullback_require_v2_pass",

    "v2_trend_health_activation_mode",
    "v2_force_after_time_enabled",

    "tp_points",
    "runner_mode",
    "runner_target_r",
    "runner_trail_rules_r",
    "runner_tp_points",
    "trail_5r_tp_points",

    "enable_legacy_early_runner_lock",
    "legacy_early_runner_lock_only_when_tp_points_above",
    "legacy_runner_reference_sl_points",
    "legacy_runner_lock_trigger_r",
    "legacy_runner_lock_r",
    "legacy_runner_lock_trigger_points",
    "legacy_runner_lock_points",

    "enable_session_filter",
    "enabled_sessions",
    "no_new_trades_after",

    "max_daily_loss_r",
    "max_no_tp_run_r",
    "max_consecutive_sl",
]

def describe_v5_runner_settings(config: dict) -> None:
    """
    Print compact active runner settings.
    """
    runner_target_r = float(config.get("runner_target_r", 5.0))
    runner_tp_points = float(config.get("runner_tp_points", np.nan))

    trail_parts = []

    for rule in config.get("runner_trail_rules_r", []):
        trigger_r = float(rule.get("trigger_r", np.nan))
        lock_r = float(rule.get("lock_r", np.nan))
        label = str(rule.get("label", "")).upper()

        if label == "BE":
            trail_parts.append(f"{trigger_r:.1f}R -> BE")
        else:
            trail_parts.append(f"{trigger_r:.1f}R -> lock {lock_r:g}R")

    print("V5 runner settings:")
    print(f"Runner target: {runner_target_r:.1f}R")
    print(f"Runner TP points: {runner_tp_points:.1f}")
    print(f"Trail rules: {', '.join(trail_parts)}")

    if bool(config.get("enable_legacy_early_runner_lock", False)):
        print(
            "Legacy early lock: "
            f"ON when tp_points > {float(config.get('legacy_early_runner_lock_only_when_tp_points_above', 60.0)):.1f} | "
            f"trigger old {float(config.get('legacy_runner_lock_trigger_r', 2.7)):.1f}R = "
            f"{float(config.get('legacy_runner_lock_trigger_points', 78.3)):.1f} pts | "
            f"lock old {float(config.get('legacy_runner_lock_r', 2.0)):.1f}R = "
            f"{float(config.get('legacy_runner_lock_points', 58.0)):.1f} pts"
        )
    else:
        print("Legacy early lock: OFF")


describe_v5_runner_settings(CONFIG)

print("FINAL ACTIVE V5 CONFIG:")
for key in IMPORTANT_CONFIG_KEYS:
    print(f"{key}: {CONFIG.get(key)}")

## 14. Automation Feature Engineering

This section creates automation-only features from the existing VWAP engine output.

These features are used by the automated execution rules only.

They do not change the existing VWAP engine, probability bands, visual overlay, or discretionary model.

This section includes the copied V2 helper features and the inherited V3 second-close helper features.

The inherited V3 helper features remain active for the baseline until V4 regime routing is added.

This section also adds preliminary V4 regime-classification columns for diagnostics.

The preliminary V4 regime label is not used to filter or route signals yet.

In [ ]:
def consecutive_true_count(condition: pd.Series) -> pd.Series:
    """
    Count consecutive True values.

    Example:
    False, True, True, False, True -> 0, 1, 2, 0, 1
    """
    condition = condition.fillna(False).astype(bool)

    counts = []
    current_count = 0

    for value in condition:
        if value:
            current_count += 1
        else:
            current_count = 0

        counts.append(current_count)

    return pd.Series(counts, index=condition.index)


def classify_red_shift_strength(value: float, config: dict) -> str:
    """
    Classify directional red-band shift strength in Nasdaq points.

    The input should already be converted into positive trend-direction strength.
    """
    if pd.isna(value):
        return "unknown"

    if value < config["red_shift_minimum_points"]:
        return "weak"

    if value < config["red_shift_good_points"]:
        return "minimum"

    if value < config["red_shift_strong_points"]:
        return "good"

    if value < config["red_shift_very_strong_points"]:
        return "strong"

    if value < config["red_shift_extreme_points"]:
        return "very_strong"

    if value < config["red_shift_very_extreme_points"]:
        return "extreme"

    if value < config["red_shift_abnormal_points"]:
        return "very_extreme"

    return "abnormal_news"


RED_SHIFT_BUCKET_ORDER = [
    "weak",
    "minimum",
    "good",
    "strong",
    "very_strong",
    "extreme",
    "very_extreme",
    "abnormal_news",
]


def adjust_red_shift_bucket_by_relative_strength(
    bucket: str,
    shift_ratio: float,
    config: dict,
) -> str:
    """
    Adjust an absolute red-shift bucket using recent relative strength.
    """
    if bucket not in RED_SHIFT_BUCKET_ORDER:
        return bucket

    if pd.isna(shift_ratio):
        return bucket

    bucket_index = RED_SHIFT_BUCKET_ORDER.index(bucket)

    if shift_ratio >= config.get("v5_red_shift_upgrade_ratio", 1.50):
        bucket_index = min(bucket_index + 1, len(RED_SHIFT_BUCKET_ORDER) - 1)

    elif shift_ratio <= config.get("v5_red_shift_downgrade_ratio", 0.75):
        bucket_index = max(bucket_index - 1, 0)

    return RED_SHIFT_BUCKET_ORDER[bucket_index]


def classify_red_shift_strength_with_relative_context(
    value: float,
    shift_ratio: float,
    config: dict,
) -> str:
    """
    Classify red-band shift using the absolute NQ bucket and a recent relative baseline.
    """
    absolute_bucket = classify_red_shift_strength(value, config)

    return adjust_red_shift_bucket_by_relative_strength(
        bucket=absolute_bucket,
        shift_ratio=shift_ratio,
        config=config,
    )


def normalise_timestamp_to_session_time(timestamp, config: dict) -> pd.Timestamp:
    """
    Convert timestamp into the configured session timezone.
    """
    ts = pd.Timestamp(timestamp)

    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")

    return ts.tz_convert(config["session_timezone"])


def is_before_no_new_trades_cutoff(timestamp, config: dict) -> bool:
    """
    Check whether a new signal is before the configured no-new-trades cutoff.
    """
    if "no_new_trades_after" not in config or config["no_new_trades_after"] is None:
        return True

    session_ts = normalise_timestamp_to_session_time(timestamp, config)
    cutoff_time = pd.to_datetime(config["no_new_trades_after"]).time()

    return session_ts.time() < cutoff_time

In [ ]:
def add_automation_features(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Add automation-only derived features to an existing VWAP engine dataframe.

    This function uses the existing VWAP/band output and creates execution features
    for the automated V1 strategy layer.
    """
    out = df.copy()
    out = out.sort_values("datetime").reset_index(drop=True)

    shift_lookback = config["shift_lookback"]
    acceptance_lookback = config["acceptance_lookback"]
    trend_lane_lookback = config["trend_lane_lookback"]
    trend_damage_lookback = config["trend_damage_lookback"]
    compression_lookback = config["compression_lookback"]
    flat_vwap_lookback = config["flat_vwap_lookback"]
    vwap_cross_lookback = config["vwap_cross_lookback"]

    # ------------------------------------------------------------------
    # Band shifts
    # ------------------------------------------------------------------
    shift_columns = [
        "vwap",
        "upper_green",
        "upper_orange",
        "upper_red",
        "lower_green",
        "lower_orange",
        "lower_red",
    ]

    for col in shift_columns:
        out[f"{col}_shift"] = out[col] - out[col].shift(shift_lookback)

    # Directional red-band strength.
    # Long continuation strength = upper red shifting upward.
    # Short continuation strength = lower red shifting downward.
    out["bullish_red_shift_strength"] = out["upper_red_shift"]
    out["bearish_red_shift_strength"] = -out["lower_red_shift"]

    out["bullish_red_shift_label"] = out["bullish_red_shift_strength"].apply(
        lambda value: classify_red_shift_strength(value, config)
    )

    out["bearish_red_shift_label"] = out["bearish_red_shift_strength"].apply(
        lambda value: classify_red_shift_strength(value, config)
    )

    # ------------------------------------------------------------------
    # Band width / compression context
    # ------------------------------------------------------------------
    out["green_band_width"] = out["upper_green"] - out["lower_green"]
    out["orange_band_width"] = out["upper_orange"] - out["lower_orange"]
    out["red_band_width"] = out["upper_red"] - out["lower_red"]

    out["green_band_width_change"] = (
        out["green_band_width"] - out["green_band_width"].shift(compression_lookback)
    )

    out["orange_band_width_change"] = (
        out["orange_band_width"] - out["orange_band_width"].shift(compression_lookback)
    )

    out["red_band_width_change"] = (
        out["red_band_width"] - out["red_band_width"].shift(compression_lookback)
    )

    out["bands_expanding"] = out["red_band_width_change"] > config["min_band_expansion_points"]
    out["bands_compressing"] = out["red_band_width_change"] < 0

    out["bullish_opposite_band_expansion"] = -out["lower_red_shift"]
    out["bearish_opposite_band_expansion"] = out["upper_red_shift"]

    # ------------------------------------------------------------------
    # V2 adaptive trend-health helper features
    # ------------------------------------------------------------------
    v2_trend_health_lookback = config["v2_trend_health_lookback"]
    v2_min_trend_health_periods = config["v2_min_trend_health_periods"]

    # Last completed candle red-band shift.
    # Each row represents a completed candle; entries are simulated at the next bar open.
    # Long directional red shift = upper red shifting upward.
    # Short directional red shift = lower red shifting downward.
    out["v2_last_upper_red_shift"] = out["upper_red"] - out["upper_red"].shift(1)
    out["v2_last_lower_red_shift"] = out["lower_red"].shift(1) - out["lower_red"]

    out["v2_long_directional_red_shift"] = out["v2_last_upper_red_shift"]
    out["v2_short_directional_red_shift"] = out["v2_last_lower_red_shift"]

    # Recent average directional red shift.
    # Use only positive directional shifts, then shift the average by one row so
    # the current candle is compared against the previous trend-health window.
    out["v2_long_recent_avg_red_shift"] = (
        out["v2_long_directional_red_shift"]
        .clip(lower=0)
        .rolling(v2_trend_health_lookback, min_periods=v2_min_trend_health_periods)
        .mean()
        .shift(1)
    )

    out["v2_short_recent_avg_red_shift"] = (
        out["v2_short_directional_red_shift"]
        .clip(lower=0)
        .rolling(v2_trend_health_lookback, min_periods=v2_min_trend_health_periods)
        .mean()
        .shift(1)
    )

    out["v2_long_red_shift_relative_to_avg"] = np.where(
        out["v2_long_recent_avg_red_shift"] > 0,
        out["v2_long_directional_red_shift"] / out["v2_long_recent_avg_red_shift"],
        np.nan,
    )

    out["v2_short_red_shift_relative_to_avg"] = np.where(
        out["v2_short_recent_avg_red_shift"] > 0,
        out["v2_short_directional_red_shift"] / out["v2_short_recent_avg_red_shift"],
        np.nan,
    )

    out["v2_long_red_shift_adaptive_pass"] = (
        (out["v2_long_directional_red_shift"] > 0)
        & (
            out["v2_long_red_shift_relative_to_avg"]
            >= config["v2_min_red_shift_relative_to_average"]
        )
    )

    out["v2_short_red_shift_adaptive_pass"] = (
        (out["v2_short_directional_red_shift"] > 0)
        & (
            out["v2_short_red_shift_relative_to_avg"]
            >= config["v2_min_red_shift_relative_to_average"]
        )
    )

    # Band-spread helper.
    # Positive red-band width change means the red bands are spreading apart.
    out["v2_red_band_width_change_1"] = out["red_band_width"] - out["red_band_width"].shift(1)

    out["v2_red_band_width_change_window"] = (
        out["red_band_width"] - out["red_band_width"].shift(v2_trend_health_lookback)
    )

    out["v2_bands_not_compressing"] = (
        out["v2_red_band_width_change_window"] >= config["v2_min_band_spread_change_points"]
    )

    # Opposite-side expansion.
    # Long opposite expansion = lower red moving down / away.
    # Short opposite expansion = upper red moving up / away.
    out["v2_long_opposite_band_expansion"] = out["lower_red"].shift(1) - out["lower_red"]
    out["v2_short_opposite_band_expansion"] = out["upper_red"] - out["upper_red"].shift(1)

    out["v2_long_opposite_band_expansion_pass"] = (
        out["v2_long_opposite_band_expansion"] >= config["v2_min_opposite_band_expansion_points"]
    )

    out["v2_short_opposite_band_expansion_pass"] = (
        out["v2_short_opposite_band_expansion"] >= config["v2_min_opposite_band_expansion_points"]
    )

    # Trend-dead helper.
    # Long trend is damaged when price keeps closing below upper green.
    # Short trend is damaged when price keeps closing above lower green.
    out["v2_long_bad_green_close"] = out["close"] < out["upper_green"]
    out["v2_short_bad_green_close"] = out["close"] > out["lower_green"]

    out["v2_long_bad_green_close_count"] = consecutive_true_count(
        out["v2_long_bad_green_close"]
    )

    out["v2_short_bad_green_close_count"] = consecutive_true_count(
        out["v2_short_bad_green_close"]
    )

    out["v2_long_trend_dead"] = (
        out["v2_long_bad_green_close_count"] >= config["v2_trend_dead_bad_candles"]
    )

    out["v2_short_trend_dead"] = (
        out["v2_short_bad_green_close_count"] >= config["v2_trend_dead_bad_candles"]
    )

    # ------------------------------------------------------------------
    # VWAP acceptance
    # ------------------------------------------------------------------
    out["close_above_vwap"] = out["close"] > out["vwap"]
    out["close_below_vwap"] = out["close"] < out["vwap"]

    out["closes_above_vwap_count"] = (
        out["close_above_vwap"]
        .astype(int)
        .rolling(acceptance_lookback, min_periods=1)
        .sum()
    )

    out["closes_below_vwap_count"] = (
        out["close_below_vwap"]
        .astype(int)
        .rolling(acceptance_lookback, min_periods=1)
        .sum()
    )

    out["accepted_above_vwap"] = out["closes_above_vwap_count"] >= 2
    out["accepted_below_vwap"] = out["closes_below_vwap_count"] >= 2

    # V2 VWAP acceptance helper aliases for later V2 entry filtering.
    out["v2_long_vwap_acceptance_pass"] = out["accepted_above_vwap"] & (out["close"] > out["vwap"])
    out["v2_short_vwap_acceptance_pass"] = out["accepted_below_vwap"] & (out["close"] < out["vwap"])

    # Combined V2 helper flags used by the modular trend-health layer.
    # These provide the initial continuation-health baseline before V5 state labels are calculated.
    out["v2_long_trend_health_pass"] = (
        out["v2_long_vwap_acceptance_pass"]
        & out["v2_long_red_shift_adaptive_pass"]
        & out["v2_bands_not_compressing"]
        & out["v2_long_opposite_band_expansion_pass"]
        & ~out["v2_long_trend_dead"]
    )

    out["v2_short_trend_health_pass"] = (
        out["v2_short_vwap_acceptance_pass"]
        & out["v2_short_red_shift_adaptive_pass"]
        & out["v2_bands_not_compressing"]
        & out["v2_short_opposite_band_expansion_pass"]
        & ~out["v2_short_trend_dead"]
    )

    # ------------------------------------------------------------------
    # V3 second-close helper features
    # ------------------------------------------------------------------
    v3_green_reentry_lookback = config["v3_green_reentry_lookback"]
    v3_trend_dead_bad_candles = config["v3_trend_dead_bad_candles"]
    v3_band_spread_lookback = config["v3_band_spread_lookback"]

    # Directional red shift on the second-close candle.
    # Long directional red shift = upper red shifting upward.
    # Short directional red shift = lower red shifting downward.
    out["v3_long_directional_red_shift"] = out["upper_red"] - out["upper_red"].shift(1)
    out["v3_short_directional_red_shift"] = out["lower_red"].shift(1) - out["lower_red"]

    out["v3_long_red_shift_bucket"] = out["v3_long_directional_red_shift"].apply(
        lambda value: classify_red_shift_strength(value, config)
    )

    out["v3_short_red_shift_bucket"] = out["v3_short_directional_red_shift"].apply(
        lambda value: classify_red_shift_strength(value, config)
    )

    out["v3_long_directional_red_shift_pass"] = (
        out["v3_long_directional_red_shift"]
        >= config["v3_min_directional_red_shift_points"]
    )

    out["v3_short_directional_red_shift_pass"] = (
        out["v3_short_directional_red_shift"]
        >= config["v3_min_directional_red_shift_points"]
    )

    out["v3_long_abnormal_red_shift_context"] = (
        out["v3_long_red_shift_bucket"] == "abnormal_news_or_crash_regime"
    )

    out["v3_long_abnormal_red_shift_context"] = (
        out["v3_long_red_shift_bucket"] == "abnormal_news"
    )

    out["v3_short_abnormal_red_shift_context"] = (
        out["v3_short_red_shift_bucket"] == "abnormal_news"
    )
    # Green-zone loss tracking.
    # Long trend damage = closes below upper green.
    # Short trend damage = closes above lower green.
    out["v3_long_bad_green_close"] = out["close"] < out["upper_green"]
    out["v3_short_bad_green_close"] = out["close"] > out["lower_green"]

    out["v3_long_bad_green_close_count"] = consecutive_true_count(
        out["v3_long_bad_green_close"]
    )

    out["v3_short_bad_green_close_count"] = consecutive_true_count(
        out["v3_short_bad_green_close"]
    )

    out["v3_long_recent_bad_green_close_max_count"] = (
        out["v3_long_bad_green_close_count"]
        .shift(1)
        .rolling(v3_trend_dead_bad_candles, min_periods=1)
        .max()
    )

    out["v3_short_recent_bad_green_close_max_count"] = (
        out["v3_short_bad_green_close_count"]
        .shift(1)
        .rolling(v3_trend_dead_bad_candles, min_periods=1)
        .max()
    )

    out["v3_long_trend_dead"] = (
        out["v3_long_recent_bad_green_close_max_count"]
        >= config["v3_trend_dead_bad_candles"]
    )

    out["v3_short_trend_dead"] = (
        out["v3_short_recent_bad_green_close_max_count"]
        >= config["v3_trend_dead_bad_candles"]
    )

    # Recent temporary green-zone loss before the second-close candle.
    out["v3_long_recently_below_upper_green"] = (
        out["v3_long_bad_green_close"]
        .shift(1)
        .fillna(False)
        .astype(int)
        .rolling(v3_green_reentry_lookback, min_periods=1)
        .sum()
        > 0
    )

    out["v3_short_recently_above_lower_green"] = (
        out["v3_short_bad_green_close"]
        .shift(1)
        .fillna(False)
        .astype(int)
        .rolling(v3_green_reentry_lookback, min_periods=1)
        .sum()
        > 0
    )

    # First reclaim / rejection close.
    out["v3_long_first_reclaim_close"] = (
        (out["close"] > out["upper_green"])
        & out["v3_long_bad_green_close"].shift(1).fillna(False).astype(bool)
    )

    out["v3_short_first_rejection_close"] = (
        (out["close"] < out["lower_green"])
        & out["v3_short_bad_green_close"].shift(1).fillna(False).astype(bool)
    )

    # Second completed close confirmation.
    out["v3_long_second_close_confirmation"] = (
        (out["close"] > out["upper_green"])
        & out["v3_long_first_reclaim_close"].shift(1).fillna(False).astype(bool)
    )

    out["v3_short_second_close_confirmation"] = (
        (out["close"] < out["lower_green"])
        & out["v3_short_first_rejection_close"].shift(1).fillna(False).astype(bool)
    )

    # Red-band spreading quality flag.
    out["v3_red_band_width_change"] = (
        out["red_band_width"] - out["red_band_width"].shift(v3_band_spread_lookback)
    )

    out["v3_red_bands_spreading"] = out["v3_red_band_width_change"] > 0

    # Combined V3 setup helper flags.
    # Timing pass is intentionally broad so raw candidates can be inspected
    # before optional quality layers are applied.
    out["v3_long_vwap_acceptance_pass"] = (
        out["accepted_above_vwap"]
        & (out["close"] > out["vwap"])
    )

    out["v3_short_vwap_acceptance_pass"] = (
        out["accepted_below_vwap"]
        & (out["close"] < out["vwap"])
    )

    out["v3_long_second_close_timing_pass"] = (
        out["v3_long_recently_below_upper_green"]
        & out["v3_long_second_close_confirmation"]
    )

    out["v3_short_second_close_timing_pass"] = (
        out["v3_short_recently_above_lower_green"]
        & out["v3_short_second_close_confirmation"]
    )

    out["v3_long_second_close_helper_pass"] = (
        out["v3_long_second_close_timing_pass"]
        & out["v3_long_vwap_acceptance_pass"]
        & out["v3_long_directional_red_shift_pass"]
        & ~out["v3_long_trend_dead"]
    )

    out["v3_short_second_close_helper_pass"] = (
        out["v3_short_second_close_timing_pass"]
        & out["v3_short_vwap_acceptance_pass"]
        & out["v3_short_directional_red_shift_pass"]
        & ~out["v3_short_trend_dead"]
    )

    # ------------------------------------------------------------------
    # Trend-lane context
    # ------------------------------------------------------------------
    out["close_in_bullish_green_lane"] = (
        (out["close"] >= out["upper_green"])
        & (out["close"] <= out["upper_orange"])
    )

    out["close_in_bearish_green_lane"] = (
        (out["close"] <= out["lower_green"])
        & (out["close"] >= out["lower_orange"])
    )

    out["bullish_lane_close_count"] = (
        out["close_in_bullish_green_lane"]
        .astype(int)
        .rolling(trend_lane_lookback, min_periods=1)
        .sum()
    )

    out["bearish_lane_close_count"] = (
        out["close_in_bearish_green_lane"]
        .astype(int)
        .rolling(trend_lane_lookback, min_periods=1)
        .sum()
    )

    # ------------------------------------------------------------------
    # Trend damage context
    # ------------------------------------------------------------------
    out["close_below_upper_green"] = out["close"] < out["upper_green"]
    out["close_above_lower_green"] = out["close"] > out["lower_green"]

    out["consecutive_closes_below_upper_green"] = consecutive_true_count(
        out["close_below_upper_green"]
    )

    out["consecutive_closes_above_lower_green"] = consecutive_true_count(
        out["close_above_lower_green"]
    )

    out["bullish_trend_dead_by_green_loss"] = (
        out["consecutive_closes_below_upper_green"] >= trend_damage_lookback
    )

    out["bearish_trend_dead_by_green_loss"] = (
        out["consecutive_closes_above_lower_green"] >= trend_damage_lookback
    )

    out["bullish_second_close_back_above_green"] = (
        (out["close"] > out["upper_green"])
        & (out["close"].shift(1) > out["upper_green"].shift(1))
        & (out["close"].shift(2) <= out["upper_green"].shift(2))
    )

    out["bearish_second_close_back_below_green"] = (
        (out["close"] < out["lower_green"])
        & (out["close"].shift(1) < out["lower_green"].shift(1))
        & (out["close"].shift(2) >= out["lower_green"].shift(2))
    )

    # ------------------------------------------------------------------
    # VWAP crossing / chop markers
    # ------------------------------------------------------------------
    out["vwap_side"] = np.where(
        out["close"] > out["vwap"],
        1,
        np.where(out["close"] < out["vwap"], -1, 0),
    )

    out["vwap_cross"] = (
        (out["vwap_side"] != out["vwap_side"].shift(1))
        & (out["vwap_side"] != 0)
        & (out["vwap_side"].shift(1) != 0)
    )

    out["vwap_cross_count"] = (
        out["vwap_cross"]
        .astype(int)
        .rolling(vwap_cross_lookback, min_periods=1)
        .sum()
    )

    out["vwap_shift_flat_check"] = out["vwap"] - out["vwap"].shift(flat_vwap_lookback)
    out["vwap_is_flat"] = out["vwap_shift_flat_check"].abs() <= config["flat_vwap_threshold_points"]

    # Defragment after creating many feature columns.
    out = out.copy()
    out["possible_chop"] = (
        (out["vwap_cross_count"] >= config["max_vwap_crosses_before_chop"])
        & out["vwap_is_flat"]
    ) | (
        out["bands_compressing"]
        & out["vwap_is_flat"]
    )

    # ------------------------------------------------------------------
    # V5 V2 state/score trend-health layer
    # ------------------------------------------------------------------
    v2_reclaim_recovery_lookback = config["v2_reclaim_recovery_lookback"]
    v2_reclaim_acceptance_lookback = config["v2_reclaim_acceptance_lookback"]
    v2_vwap_slope_lookback = config["v2_vwap_slope_lookback"]
    v5_red_shift_relative_lookback_bars = config["v5_red_shift_relative_lookback_bars"]

    out["v5_long_avg_red_shift_20m"] = (
        out["v2_long_directional_red_shift"]
        .clip(lower=0)
        .rolling(v5_red_shift_relative_lookback_bars, min_periods=3)
        .mean()
        .shift(1)
    )

    out["v5_short_avg_red_shift_20m"] = (
        out["v2_short_directional_red_shift"]
        .clip(lower=0)
        .rolling(v5_red_shift_relative_lookback_bars, min_periods=3)
        .mean()
        .shift(1)
    )

    out["v5_long_red_shift_ratio_20m"] = np.where(
        out["v5_long_avg_red_shift_20m"] > 0,
        out["v2_long_directional_red_shift"] / out["v5_long_avg_red_shift_20m"],
        np.nan,
    )

    out["v5_short_red_shift_ratio_20m"] = np.where(
        out["v5_short_avg_red_shift_20m"] > 0,
        out["v2_short_directional_red_shift"] / out["v5_short_avg_red_shift_20m"],
        np.nan,
    )

    out["v5_long_red_shift_bucket"] = [
        classify_red_shift_strength_with_relative_context(value, ratio, config)
        for value, ratio in zip(
            out["v2_long_directional_red_shift"],
            out["v5_long_red_shift_ratio_20m"],
        )
    ]

    out["v5_short_red_shift_bucket"] = [
        classify_red_shift_strength_with_relative_context(value, ratio, config)
        for value, ratio in zip(
            out["v2_short_directional_red_shift"],
            out["v5_short_red_shift_ratio_20m"],
        )
    ]

    out["v2_vwap_slope_points"] = (
        out["vwap"] - out["vwap"].shift(v2_vwap_slope_lookback)
    )

    out["v2_vwap_flat_to_rising"] = (
        out["v2_vwap_slope_points"] >= -config["v2_flat_vwap_slope_points"]
    )

    out["v2_vwap_flat_to_falling"] = (
        out["v2_vwap_slope_points"] <= config["v2_flat_vwap_slope_points"]
    )

    out["v2_long_consecutive_above_green_count"] = consecutive_true_count(
        out["close"] > out["upper_green"]
    )

    out["v2_short_consecutive_below_green_count"] = consecutive_true_count(
        out["close"] < out["lower_green"]
    )

    out["v2_long_above_green_ratio_6"] = (
        (out["close"] > out["upper_green"])
        .astype(int)
        .rolling(v2_reclaim_acceptance_lookback, min_periods=1)
        .mean()
    )

    out["v2_short_below_green_ratio_6"] = (
        (out["close"] < out["lower_green"])
        .astype(int)
        .rolling(v2_reclaim_acceptance_lookback, min_periods=1)
        .mean()
    )

    out["v2_long_below_green_count_before_reclaim"] = (
        out["v2_long_bad_green_close_count"]
        .shift(1)
        .rolling(v2_reclaim_recovery_lookback, min_periods=1)
        .max()
    )

    out["v2_short_above_green_count_before_reclaim"] = (
        out["v2_short_bad_green_close_count"]
        .shift(1)
        .rolling(v2_reclaim_recovery_lookback, min_periods=1)
        .max()
    )

    out["v2_long_prior_bullish_acceptance"] = (
        (
            out["accepted_above_vwap"]
            & (out["close"] > out["upper_green"])
        )
        .shift(1)
        .fillna(False)
        .astype(int)
        .rolling(v2_reclaim_recovery_lookback, min_periods=1)
        .max()
        .astype(bool)
    )

    out["v2_short_prior_bearish_acceptance"] = (
        (
            out["accepted_below_vwap"]
            & (out["close"] < out["lower_green"])
        )
        .shift(1)
        .fillna(False)
        .astype(int)
        .rolling(v2_reclaim_recovery_lookback, min_periods=1)
        .max()
        .astype(bool)
    )

    out["v2_long_dead_trend_state"] = (
        out["accepted_below_vwap"]
        & (out["close"] < out["vwap"])
        & (out["v2_vwap_slope_points"] < -config["v2_flat_vwap_slope_points"])
    )

    out["v2_short_dead_trend_state"] = (
        out["accepted_above_vwap"]
        & (out["close"] > out["vwap"])
        & (out["v2_vwap_slope_points"] > config["v2_flat_vwap_slope_points"])
    )

    out["v2_long_chop_compression_state"] = (
        out["possible_chop"]
        | (
            out["bands_compressing"]
            & out["vwap_is_flat"]
        )
    )

    out["v2_short_chop_compression_state"] = out["v2_long_chop_compression_state"]

    out["v2_long_reclaim_recovery_health_pass"] = (
        out["v2_long_prior_bullish_acceptance"]
        & (
            out["v2_long_below_green_count_before_reclaim"]
            >= config["v2_min_pullback_green_damage_count"]
        )
        & (
            out["v2_long_above_green_ratio_6"]
            >= config["v2_min_reclaim_above_green_ratio"]
        )
        & (
            out["v2_long_consecutive_above_green_count"]
            >= config["v2_min_reclaim_consecutive_green_closes"]
        )
        & ~out["accepted_below_vwap"]
        & out["v2_vwap_flat_to_rising"]
        & ~out["v2_long_chop_compression_state"]
    )

    out["v2_short_reclaim_recovery_health_pass"] = (
        out["v2_short_prior_bearish_acceptance"]
        & (
            out["v2_short_above_green_count_before_reclaim"]
            >= config["v2_min_pullback_green_damage_count"]
        )
        & (
            out["v2_short_below_green_ratio_6"]
            >= config["v2_min_reclaim_above_green_ratio"]
        )
        & (
            out["v2_short_consecutive_below_green_count"]
            >= config["v2_min_reclaim_consecutive_green_closes"]
        )
        & ~out["accepted_above_vwap"]
        & out["v2_vwap_flat_to_falling"]
        & ~out["v2_short_chop_compression_state"]
    )

    out["v2_long_healthy_continuation_pass"] = (
        out["v2_long_vwap_acceptance_pass"]
        & out["v2_long_red_shift_adaptive_pass"]
        & out["v2_bands_not_compressing"]
        & out["v2_long_opposite_band_expansion_pass"]
        & ~out["v2_long_trend_dead"]
        & ~out["v2_long_chop_compression_state"]
    )

    out["v2_short_healthy_continuation_pass"] = (
        out["v2_short_vwap_acceptance_pass"]
        & out["v2_short_red_shift_adaptive_pass"]
        & out["v2_bands_not_compressing"]
        & out["v2_short_opposite_band_expansion_pass"]
        & ~out["v2_short_trend_dead"]
        & ~out["v2_short_chop_compression_state"]
    )

    out["v2_long_continuation_health_state"] = np.select(
        [
            out["v2_long_chop_compression_state"],
            out["v2_long_dead_trend_state"],
            out["v2_long_healthy_continuation_pass"],
            out["v2_long_reclaim_recovery_health_pass"],
            out["v2_long_bad_green_close_count"].between(1, 4),
            out["v2_long_bad_green_close_count"] >= config["v2_trend_dead_bad_candles"],
        ],
        [
            "CHOP_COMPRESSION",
            "DEAD_TREND",
            "HEALTHY_CONTINUATION",
            "RECLAIM_RECOVERY",
            "DAMAGED_PULLBACK",
            "DAMAGED_PULLBACK",
        ],
        default="UNCLEAR",
    )

    out["v2_short_continuation_health_state"] = np.select(
        [
            out["v2_short_chop_compression_state"],
            out["v2_short_dead_trend_state"],
            out["v2_short_healthy_continuation_pass"],
            out["v2_short_reclaim_recovery_health_pass"],
            out["v2_short_bad_green_close_count"].between(1, 4),
            out["v2_short_bad_green_close_count"] >= config["v2_trend_dead_bad_candles"],
        ],
        [
            "CHOP_COMPRESSION",
            "DEAD_TREND",
            "HEALTHY_CONTINUATION",
            "RECLAIM_RECOVERY",
            "DAMAGED_PULLBACK",
            "DAMAGED_PULLBACK",
        ],
        default="UNCLEAR",
    )

    out["v2_long_reclaim_recovery_state"] = np.where(
        out["v2_long_reclaim_recovery_health_pass"],
        "RECLAIM_RECOVERY",
        out["v2_long_continuation_health_state"],
    )

    out["v2_short_reclaim_recovery_state"] = np.where(
        out["v2_short_reclaim_recovery_health_pass"],
        "RECLAIM_RECOVERY",
        out["v2_short_continuation_health_state"],
    )

    out["v2_long_continuation_health_pass"] = (
        out["v2_long_continuation_health_state"] == "HEALTHY_CONTINUATION"
    )

    out["v2_short_continuation_health_pass"] = (
        out["v2_short_continuation_health_state"] == "HEALTHY_CONTINUATION"
    )

    # Compatibility aliases used by existing V2/V4.5 filter paths.
    # V1-style continuation uses the stricter continuation-health pass.
    out["v2_long_trend_health_pass"] = out["v2_long_continuation_health_pass"]
    out["v2_short_trend_health_pass"] = out["v2_short_continuation_health_pass"]

    # ------------------------------------------------------------------
    # V4 preliminary regime classification features
    # ------------------------------------------------------------------
    v4_realised_range_lookback = config["v4_realised_range_lookback"]
    v4_min_realised_range_periods = config["v4_min_realised_range_periods"]
    v4_vwap_slope_lookback = config["v4_vwap_slope_lookback"]
    v4_band_width_lookback = config["v4_band_width_lookback"]
    v4_min_band_width_periods = config["v4_min_band_width_periods"]
    v4_band_expansion_lookback = config["v4_band_expansion_lookback"]
    v4_vwap_cross_lookback = config["v4_vwap_cross_lookback"]
    v4_recent_extreme_lookback = config["v4_recent_extreme_lookback"]

    out["v4_realised_range_points"] = out["high"] - out["low"]

    out["v4_realised_range_average"] = (
        out["v4_realised_range_points"]
        .rolling(v4_realised_range_lookback, min_periods=v4_min_realised_range_periods)
        .mean()
        .shift(1)
    )

    out["v4_realised_range_relative_to_average"] = np.where(
        out["v4_realised_range_average"] > 0,
        out["v4_realised_range_points"] / out["v4_realised_range_average"],
        np.nan,
    )

    out["v4_high_realised_volatility"] = (
        out["v4_realised_range_relative_to_average"]
        >= config["v4_high_realised_range_ratio"]
    )

    out["v4_extreme_realised_volatility"] = (
        out["v4_realised_range_relative_to_average"]
        >= config["v4_extreme_realised_range_ratio"]
    )

    out["v4_vwap_slope_points"] = (
        out["vwap"] - out["vwap"].shift(v4_vwap_slope_lookback)
    )

    out["v4_vwap_slope_abs_points"] = out["v4_vwap_slope_points"].abs()

    out["v4_vwap_is_flat"] = (
        out["v4_vwap_slope_abs_points"]
        <= config["v4_flat_vwap_threshold_points"]
    )

    out["v4_red_band_width"] = out["red_band_width"]

    out["v4_red_band_width_average"] = (
        out["v4_red_band_width"]
        .rolling(v4_band_width_lookback, min_periods=v4_min_band_width_periods)
        .mean()
        .shift(1)
    )

    out["v4_red_band_width_relative_to_average"] = np.where(
        out["v4_red_band_width_average"] > 0,
        out["v4_red_band_width"] / out["v4_red_band_width_average"],
        np.nan,
    )

    out["v4_red_band_width_change"] = (
        out["v4_red_band_width"]
        - out["v4_red_band_width"].shift(v4_band_expansion_lookback)
    )

    out["v4_red_bands_expanding"] = (
        out["v4_red_band_width_change"] > config["v4_min_band_expansion_points"]
    )

    out["v4_red_bands_compressing"] = out["v4_red_band_width_change"] < 0

    out["v4_wide_bands"] = (
        out["v4_red_band_width_relative_to_average"]
        >= config["v4_wide_band_width_ratio"]
    )

    out["v4_distance_from_vwap_points"] = (out["close"] - out["vwap"]).abs()

    out["v4_distance_from_vwap_relative_to_green_width"] = np.where(
        out["green_band_width"] > 0,
        out["v4_distance_from_vwap_points"] / out["green_band_width"],
        np.nan,
    )

    out["v4_recent_vwap_cross_count"] = (
        out["vwap_cross"]
        .astype(int)
        .rolling(v4_vwap_cross_lookback, min_periods=1)
        .sum()
    )

    out["v4_chop_from_vwap_crosses"] = (
        out["v4_recent_vwap_cross_count"] > config["v4_max_vwap_crosses_for_trend"]
    )

    out["v4_chop_from_flat_vwap_and_compression"] = (
        out["v4_vwap_is_flat"] & out["v4_red_bands_compressing"]
    )

    out["v4_chop_or_unclear_value"] = (
        out["possible_chop"]
        | out["v4_chop_from_vwap_crosses"]
        | out["v4_chop_from_flat_vwap_and_compression"]
    )

    out["v4_bullish_directional_context"] = (
        out["accepted_above_vwap"]
        & (out["close"] > out["vwap"])
        & (out["close"] > out["upper_green"])
    )

    out["v4_bearish_directional_context"] = (
        out["accepted_below_vwap"]
        & (out["close"] < out["vwap"])
        & (out["close"] < out["lower_green"])
    )

    out["v4_directional_trend_context"] = (
        out["v4_bullish_directional_context"] | out["v4_bearish_directional_context"]
    )

    out["v4_directional_red_shift_strength"] = np.select(
        [
            out["v4_bullish_directional_context"],
            out["v4_bearish_directional_context"],
        ],
        [
            out["bullish_red_shift_strength"],
            out["bearish_red_shift_strength"],
        ],
        default=np.nan,
    )

    out["v4_directional_red_shift_bucket"] = out[
        "v4_directional_red_shift_strength"
    ].apply(lambda value: classify_red_shift_strength(value, config))

    out["v4_abnormal_red_shift_context"] = (
        out["v4_directional_red_shift_strength"]
        >= config["red_shift_abnormal_points"]
    )

    out["v4_recent_abnormal_red_shift_context"] = (
        out["v4_abnormal_red_shift_context"]
        .astype(int)
        .rolling(v4_recent_extreme_lookback, min_periods=1)
        .max()
        .astype(bool)
    )

    out["v4_extreme_news_context"] = (
        out["v4_recent_abnormal_red_shift_context"]
        | out["v4_extreme_realised_volatility"]
    )

    out["v4_volatile_directional_context"] = (
        out["v4_directional_trend_context"]
        & (
            out["v4_high_realised_volatility"]
            | out["v4_wide_bands"]
            | out["v4_red_bands_expanding"]
        )
    )

    out["v4_calm_directional_context"] = (
        out["v4_directional_trend_context"]
        & ~out["v4_high_realised_volatility"]
        & ~out["v4_wide_bands"]
        & ~out["v4_chop_or_unclear_value"]
    )

    out["v4_preliminary_regime_label"] = np.select(
        [
            out["v4_extreme_news_context"],
            out["v4_chop_or_unclear_value"] | ~out["v4_directional_trend_context"],
            out["v4_volatile_directional_context"],
            out["v4_calm_directional_context"],
        ],
        [
            "extreme_news",
            "chop",
            "volatile_trend",
            "calm_trend",
        ],
        default="chop",
    )

    # ------------------------------------------------------------------
    # V4 conditional V2 trend-health context
    # ------------------------------------------------------------------
    out["v4_conditional_v2_wide_flat_context"] = (
        out["v4_wide_bands"] & out["v4_vwap_is_flat"]
    )

    out["v4_conditional_v2_borderline_chop_context"] = (
        out["v4_recent_vwap_cross_count"]
        >= config["v4_conditional_v2_min_recent_vwap_crosses"]
    )

    out["v4_conditional_v2_range_without_expansion_context"] = (
        (
            out["v4_realised_range_relative_to_average"]
            >= config["v4_conditional_v2_extreme_range_ratio"]
        )
        & ~out["v4_red_bands_expanding"]
    )

    out["v4_conditional_v2_weak_red_shift_context"] = (
        out["v4_high_realised_volatility"]
        & (
            out["v4_directional_red_shift_strength"]
            < config["v4_conditional_v2_min_directional_red_shift_points"]
        )
    )

    out["v4_v2_trend_health_required"] = (
        config.get("v4_use_conditional_v2_trend_health_layer", True)
        & (out["v4_preliminary_regime_label"] == "volatile_trend")
        & (
            out["v4_conditional_v2_wide_flat_context"]
            | out["v4_conditional_v2_borderline_chop_context"]
            | out["v4_conditional_v2_range_without_expansion_context"]
            | out["v4_conditional_v2_weak_red_shift_context"]
        )
    )

    out["v4_conditional_v2_reason"] = np.select(
        [
            out["v4_conditional_v2_wide_flat_context"],
            out["v4_conditional_v2_borderline_chop_context"],
            out["v4_conditional_v2_range_without_expansion_context"],
            out["v4_conditional_v2_weak_red_shift_context"],
        ],
        [
            "wide_flat",
            "borderline_chop",
            "range_without_expansion",
            "weak_red_shift",
        ],
        default="not_required",
    )

    out.loc[
        ~out["v4_v2_trend_health_required"],
        "v4_conditional_v2_reason",
    ] = "not_required"

    # ------------------------------------------------------------------
    # V5 extreme expansion / abnormal-news context
    # ------------------------------------------------------------------
    out["v5_long_extreme_expansion_context"] = (
        out["v4_bullish_directional_context"]
        & out["v4_red_bands_expanding"]
        & out["v5_long_red_shift_bucket"].isin(["extreme", "very_extreme"])
        & out["v3_long_vwap_acceptance_pass"].fillna(False).astype(bool)
        & out["v2_vwap_flat_to_rising"].fillna(False).astype(bool)
        & ~out["possible_chop"]
    )

    out["v5_short_extreme_expansion_context"] = (
        out["v4_bearish_directional_context"]
        & out["v4_red_bands_expanding"]
        & out["v5_short_red_shift_bucket"].isin(["extreme", "very_extreme"])
        & out["v3_short_vwap_acceptance_pass"].fillna(False).astype(bool)
        & out["v2_vwap_flat_to_falling"].fillna(False).astype(bool)
        & ~out["possible_chop"]
    )

    out["v5_long_abnormal_news_context"] = (
        out["v5_long_red_shift_bucket"] == "abnormal_news"
    )

    out["v5_short_abnormal_news_context"] = (
        out["v5_short_red_shift_bucket"] == "abnormal_news"
    )

    out["v5_abnormal_news_context"] = (
        out["v5_long_abnormal_news_context"]
        | out["v5_short_abnormal_news_context"]
        | out["v4_abnormal_red_shift_context"].fillna(False).astype(bool)
    )

    out["v5_extreme_expansion_context"] = (
        out["v5_long_extreme_expansion_context"]
        | out["v5_short_extreme_expansion_context"]
    )

    out["v5_very_extreme_expansion_context"] = (
        out["v5_long_red_shift_bucket"].eq("very_extreme")
        | out["v5_short_red_shift_bucket"].eq("very_extreme")
    ) & out["v5_extreme_expansion_context"]

    out["v5_regime_20m"] = np.select(
        [
            out["v5_abnormal_news_context"],
            out["v5_very_extreme_expansion_context"],
            out["v5_extreme_expansion_context"],
        ],
        [
            "abnormal_news",
            "very_extreme_expansion",
            "extreme_expansion",
        ],
        default=out["v4_preliminary_regime_label"],
    )
    
    # ------------------------------------------------------------------
    # Session time context
    # ------------------------------------------------------------------
    out["session_time"] = out["datetime"].apply(
        lambda ts: normalise_timestamp_to_session_time(ts, config)
    )

    out["session_date"] = out["session_time"].dt.date
    out["session_hour"] = out["session_time"].dt.hour
    out["session_minute"] = out["session_time"].dt.minute

    out["before_no_new_trades_cutoff"] = out["datetime"].apply(
        lambda ts: is_before_no_new_trades_cutoff(ts, config)
    )

    return out

In [ ]:
features_df = add_automation_features(automation_ready_df, CONFIG)

print(f"Feature dataframe rows: {len(features_df):,}")
print(f"Columns added: {len(features_df.columns) - len(automation_ready_df.columns):,}")

feature_preview_cols = [
    "datetime",
    "session_time",
    "close",
    "vwap",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
    "vwap_shift",
    "upper_red_shift",
    "lower_red_shift",
    "bullish_red_shift_strength",
    "bearish_red_shift_strength",
    "bullish_red_shift_label",
    "bearish_red_shift_label",
    "accepted_above_vwap",
    "accepted_below_vwap",
    "vwap_cross_count",
    "possible_chop",
    "v4_realised_range_points",
    "v4_realised_range_relative_to_average",
    "v4_high_realised_volatility",
    "v4_extreme_realised_volatility",
    "v4_vwap_slope_points",
    "v4_vwap_is_flat",
    "v4_red_band_width_relative_to_average",
    "v4_red_band_width_change",
    "v4_red_bands_expanding",
    "v4_wide_bands",
    "v4_recent_vwap_cross_count",
    "v4_chop_or_unclear_value",
    "v4_directional_trend_context",
    "v4_directional_red_shift_strength",
    "v4_directional_red_shift_bucket",
    "v4_extreme_news_context",
    "v4_preliminary_regime_label",
    "v4_v2_trend_health_required",
    "v4_conditional_v2_reason",
    "before_no_new_trades_cutoff",
    "v2_long_directional_red_shift",
    "v2_long_recent_avg_red_shift",
    "v2_long_red_shift_relative_to_avg",
    "v2_long_red_shift_adaptive_pass",
    "v2_short_directional_red_shift",
    "v2_short_recent_avg_red_shift",
    "v2_short_red_shift_relative_to_avg",
    "v2_short_red_shift_adaptive_pass",
    "v2_red_band_width_change_window",
    "v2_bands_not_compressing",
    "v2_long_opposite_band_expansion",
    "v2_short_opposite_band_expansion",
    "v2_long_trend_dead",
    "v2_short_trend_dead",
    "v2_long_trend_health_pass",
    "v2_short_trend_health_pass",
]

print(features_df[feature_preview_cols].tail(20).to_string(index=False))

v3_helper_preview_cols = [
    "datetime",
    "close",
    "vwap",
    "upper_green",
    "lower_green",
    "upper_red",
    "lower_red",
    "accepted_above_vwap",
    "accepted_below_vwap",
    "v3_long_vwap_acceptance_pass",
    "v3_short_vwap_acceptance_pass",
    "v3_long_directional_red_shift",
    "v3_long_red_shift_bucket",
    "v3_long_directional_red_shift_pass",
    "v3_short_directional_red_shift",
    "v3_short_red_shift_bucket",
    "v3_short_directional_red_shift_pass",
    "v3_long_bad_green_close_count",
    "v3_short_bad_green_close_count",
    "v3_long_recent_bad_green_close_max_count",
    "v3_short_recent_bad_green_close_max_count",
    "v3_long_trend_dead",
    "v3_short_trend_dead",
    "v3_long_recently_below_upper_green",
    "v3_short_recently_above_lower_green",
    "v3_long_first_reclaim_close",
    "v3_long_second_close_confirmation",
    "v3_short_first_rejection_close",
    "v3_short_second_close_confirmation",
    "v3_red_bands_spreading",
    "v3_long_second_close_timing_pass",
    "v3_short_second_close_timing_pass",
    "v3_long_second_close_helper_pass",
    "v3_short_second_close_helper_pass",
]

print("\nV3 second-close helper preview:")
print(features_df[v3_helper_preview_cols].tail(20).to_string(index=False))

## 15. Signal Generation

This section generates the active inherited baseline signal set.

At this stage, the active logic still uses the V3 second-close green reclaim/rejection continuation setup.

For longs, the raw candidate is based on a second completed close back above upper green after a short temporary loss of upper green.

For shorts, the raw candidate is based on a second completed close back below lower green after a short temporary loss of lower green.

The simulator still enters at the next candle open and keeps the same fixed SL / TP / BE, session controls, and daily risk controls.

In [ ]:
def trade_directional_red_shift_label(df: pd.DataFrame) -> pd.Series:
    """
    Return the red-shift label in the direction of the signal.
    """
    return pd.Series(
        np.where(
            df["raw_signal_side"] == "long",
            df["bullish_red_shift_label"],
            np.where(
                df["raw_signal_side"] == "short",
                df["bearish_red_shift_label"],
                "none",
            ),
        ),
        index=df.index,
    )


def trade_directional_red_shift_strength(df: pd.DataFrame) -> pd.Series:
    """
    Return the red-shift strength in the direction of the signal.
    """
    return pd.Series(
        np.where(
            df["raw_signal_side"] == "long",
            df["bullish_red_shift_strength"],
            np.where(
                df["raw_signal_side"] == "short",
                df["bearish_red_shift_strength"],
                np.nan,
            ),
        ),
        index=df.index,
    )


def trade_directional_extension_from_green(df: pd.DataFrame) -> pd.Series:
    """
    Return extension from the relevant green band in the trade direction.
    """
    return pd.Series(
        np.where(
            df["raw_signal_side"] == "long",
            df["long_extension_from_green_points"],
            np.where(
                df["raw_signal_side"] == "short",
                df["short_extension_from_green_points"],
                np.nan,
            ),
        ),
        index=df.index,
    )


def build_v3_modular_filter_mask(
    df: pd.DataFrame,
    config: dict,
    has_candidate: pd.Series,
) -> pd.Series:
    """
    Build the modular V3 second-close continuation filter mask.
    """
    required_v3_columns = [
        "v3_long_second_close_timing_pass",
        "v3_short_second_close_timing_pass",
        "v3_long_vwap_acceptance_pass",
        "v3_short_vwap_acceptance_pass",
        "v3_long_directional_red_shift_pass",
        "v3_short_directional_red_shift_pass",
        "v3_long_trend_dead",
        "v3_short_trend_dead",
        "v3_long_v1_execution_quality_pass",
        "v3_short_v1_execution_quality_pass",
        "v2_long_trend_health_pass",
        "v2_short_trend_health_pass",
        "v3_red_bands_spreading",
    ]

    missing_v3_columns = [
        column for column in required_v3_columns if column not in df.columns
    ]

    if missing_v3_columns:
        raise KeyError(
            "Missing V3 modular filter columns: "
            + ", ".join(missing_v3_columns)
            + ". Run add_automation_features and signal feature preparation before filtering."
        )

    is_long = df["raw_signal_side"] == "long"
    is_short = df["raw_signal_side"] == "short"

    raw_signal_name_text = (
        df.get("raw_signal_name", pd.Series("NO_SIGNAL", index=df.index))
        .fillna("NO_SIGNAL")
        .astype(str)
        .str.upper()
    )

    entry_timing_type = (
        df.get("a_tier_entry_timing_type", pd.Series("UNKNOWN", index=df.index))
        .fillna("UNKNOWN")
        .astype(str)
        .str.upper()
    )

    delayed_pullback_enabled = bool(
        config.get("enable_a_tier_delayed_pullback_entry", False)
    )

    delayed_pullback_name_or_timing_pass = (
        entry_timing_type.eq("DELAYED_PULLBACK_4TH_CANDLE")
        | raw_signal_name_text.str.contains("DELAYED_PULLBACK", na=False)
    )

    long_delayed_pullback_strategy_pass = (
        delayed_pullback_enabled
        & is_long
        & df.get(
            "a_tier_delayed_pullback_long_passed",
            pd.Series(False, index=df.index),
        ).fillna(False).astype(bool)
        & delayed_pullback_name_or_timing_pass
    )

    short_delayed_pullback_strategy_pass = (
        delayed_pullback_enabled
        & is_short
        & df.get(
            "a_tier_delayed_pullback_short_passed",
            pd.Series(False, index=df.index),
        ).fillna(False).astype(bool)
        & delayed_pullback_name_or_timing_pass
    )

    delayed_pullback_strategy_pass = (
        has_candidate
        & (
            long_delayed_pullback_strategy_pass
            | short_delayed_pullback_strategy_pass
        )
    )

    long_timing_pass = (
        df["v3_long_second_close_timing_pass"].fillna(False).astype(bool)
        | df.get(
            "a_tier_fast_reclaim_long_passed",
            pd.Series(False, index=df.index),
        ).fillna(False).astype(bool)
        | long_delayed_pullback_strategy_pass
    )

    short_timing_pass = (
        df["v3_short_second_close_timing_pass"].fillna(False).astype(bool)
        | df.get(
            "a_tier_fast_reclaim_short_passed",
            pd.Series(False, index=df.index),
        ).fillna(False).astype(bool)
        | short_delayed_pullback_strategy_pass
    )

    v3_filter_pass = has_candidate & (
        (is_long & long_timing_pass)
        | (is_short & short_timing_pass)
    )

    v2_hard_filter_active = (
        df.get(
            "v2_trend_health_active",
            pd.Series(False, index=df.index),
        )
        .fillna(False)
        .astype(bool)
    )

    if config.get("v3_use_vwap_acceptance_layer", True):
        long_vwap_acceptance_pass = (
            df["v3_long_vwap_acceptance_pass"].fillna(False).astype(bool)
        )
        short_vwap_acceptance_pass = (
            df["v3_short_vwap_acceptance_pass"].fillna(False).astype(bool)
        )

        vwap_acceptance_pass = (
            (is_long & long_vwap_acceptance_pass)
            | (is_short & short_vwap_acceptance_pass)
        )

        v3_filter_pass = v3_filter_pass & vwap_acceptance_pass

    if config.get("v3_use_v1_execution_quality_layer", True):
        long_v1_quality_pass = (
            df["v3_long_v1_execution_quality_pass"].fillna(False).astype(bool)
        )
        short_v1_quality_pass = (
            df["v3_short_v1_execution_quality_pass"].fillna(False).astype(bool)
        )

        v1_quality_pass = (
            (is_long & long_v1_quality_pass)
            | (is_short & short_v1_quality_pass)
        )

        v3_filter_pass = v3_filter_pass & v1_quality_pass

    if (
        config.get("v3_use_v2_adaptive_trend_health_layer", False)
        and config.get("enable_v2_trend_health_filter", True)
        and config.get("enable_v2_reclaim_recovery_health", True)
    ):
        long_v2_health_pass = (
            df.get("v2_long_reclaim_recovery_health_pass", df["v2_long_trend_health_pass"])
            .fillna(False)
            .astype(bool)
        )

        short_v2_health_pass = (
            df.get("v2_short_reclaim_recovery_health_pass", df["v2_short_trend_health_pass"])
            .fillna(False)
            .astype(bool)
        )

        v2_health_pass = (
            (is_long & long_v2_health_pass)
            | (is_short & short_v2_health_pass)
        )

        v3_filter_pass = v3_filter_pass & (~v2_hard_filter_active | v2_health_pass)

    if config.get("v3_use_directional_red_shift_layer", True):
        long_red_shift_pass = (
            df["v3_long_directional_red_shift_pass"].fillna(False).astype(bool)
        )
        short_red_shift_pass = (
            df["v3_short_directional_red_shift_pass"].fillna(False).astype(bool)
        )

        directional_red_shift_pass = (
            (is_long & long_red_shift_pass)
            | (is_short & short_red_shift_pass)
        )

        v3_filter_pass = v3_filter_pass & directional_red_shift_pass

    if config.get("v3_use_trend_not_dead_layer", True):
        allow_reclaim_recovery_override = (
            config.get("enable_v2_trend_health_filter", True)
            and config.get("enable_v2_reclaim_recovery_health", True)
        )

        long_reclaim_recovery_pass = (
            df.get("v2_long_reclaim_recovery_health_pass", False)
            .fillna(False)
            .astype(bool)
            if allow_reclaim_recovery_override
            and "v2_long_reclaim_recovery_health_pass" in df.columns
            else pd.Series(False, index=df.index)
        )

        short_reclaim_recovery_pass = (
            df.get("v2_short_reclaim_recovery_health_pass", False)
            .fillna(False)
            .astype(bool)
            if allow_reclaim_recovery_override
            and "v2_short_reclaim_recovery_health_pass" in df.columns
            else pd.Series(False, index=df.index)
        )

        long_trend_not_dead = (
            ~df["v3_long_trend_dead"].fillna(False).astype(bool)
            | long_reclaim_recovery_pass
        )

        short_trend_not_dead = (
            ~df["v3_short_trend_dead"].fillna(False).astype(bool)
            | short_reclaim_recovery_pass
        )

        trend_not_dead_pass = (
            (is_long & long_trend_not_dead)
            | (is_short & short_trend_not_dead)
        )

        v3_filter_pass = v3_filter_pass & trend_not_dead_pass

    if config.get("v3_require_red_bands_spreading", False):
        red_bands_spreading_pass = (
            df["v3_red_bands_spreading"].fillna(False).astype(bool)
        )

        v3_filter_pass = v3_filter_pass & red_bands_spreading_pass

    return v3_filter_pass | delayed_pullback_strategy_pass


def build_v1_strategy_filter_mask(df: pd.DataFrame, config: dict) -> pd.Series:
    """
    Build the final strategy filter mask for the selected automated strategy version.
    """
    strategy_filter = config.get("strategy_filter", "baseline")

    has_candidate = df["raw_signal_name"] != "NO_SIGNAL"

    directional_red_shift_label = df["directional_red_shift_label"]
    directional_extension = df["directional_extension_from_green_points"]

    if strategy_filter in ["baseline", "no_red_shift_filter"]:
        return has_candidate

    if strategy_filter == "v2_adaptive_trend_health":
        required_v2_columns = [
            "v2_long_trend_health_pass",
            "v2_short_trend_health_pass",
        ]

        missing_v2_columns = [
            column for column in required_v2_columns if column not in df.columns
        ]

        if missing_v2_columns:
            raise KeyError(
                "Missing V2 trend-health columns: "
                + ", ".join(missing_v2_columns)
                + ". Run add_automation_features before signal generation."
            )

        long_v2_health_pass = df["v2_long_trend_health_pass"].fillna(False).astype(bool)
        short_v2_health_pass = df["v2_short_trend_health_pass"].fillna(False).astype(bool)

        v2_directional_trend_health_pass = (
            ((df["raw_signal_side"] == "long") & long_v2_health_pass)
            | ((df["raw_signal_side"] == "short") & short_v2_health_pass)
        )

        return has_candidate & v2_directional_trend_health_pass

    if strategy_filter == "v3_second_close_green_reclaim_rejection":
        return build_v3_modular_filter_mask(df, config, has_candidate)

    if strategy_filter == "v4_dynamic_regime_selector":
        required_v4_columns = [
            "v4_preliminary_regime_label",
            "v4_selected_entry_module",
        ]

        missing_v4_columns = [
            column for column in required_v4_columns if column not in df.columns
        ]

        if missing_v4_columns:
            raise KeyError(
                "Missing V4 routing columns: "
                + ", ".join(missing_v4_columns)
                + ". Run V4 signal routing before filtering."
            )

        regime = df.get(
            "v5_regime_20m",
            df["v4_preliminary_regime_label"],
        ).astype(str)

        selected_module = df["v4_selected_entry_module"].astype(str)

        v2_hard_filter_active = (
            df.get(
                "v2_trend_health_active",
                pd.Series(False, index=df.index),
            )
            .fillna(False)
            .astype(bool)
        )

        if not v5_use_intelligent_router(config):
            v1_manual_pass = (
                has_candidate
                & bool(config.get("enable_v1_s_tier", True))
                & (selected_module == "v1_green_reclaim_rejection")
            )

            v3_manual_pass = (
                bool(config.get("enable_v3_a_tier_second_close", True))
                & build_v3_modular_filter_mask(df, config, has_candidate)
                & (selected_module == "v3_second_close_green_reclaim_rejection")
            )

            v5_extreme_manual_pass = (
                bool(config.get("enable_extreme_expansion_entries", False))
                & build_v3_modular_filter_mask(df, config, has_candidate)
                & regime.isin(
                    [
                        "extreme_expansion",
                        "very_extreme_expansion",
                        "abnormal_news",
                    ]
                )
                & (selected_module == "extreme_expansion")
            )

            if not bool(config.get("enable_abnormal_news_entries", False)):
                v5_extreme_manual_pass = (
                    v5_extreme_manual_pass
                    & (regime != "abnormal_news")
                )

            return v1_manual_pass | v3_manual_pass | v5_extreme_manual_pass

        v1_calm_route_pass = (
            has_candidate
            & bool(config.get("enable_v1_s_tier", True))
            & (regime == "calm_trend")
            & (selected_module == "v1_green_reclaim_rejection")
        )

        v3_volatile_route_pass = (
            bool(config.get("enable_v3_a_tier_second_close", True))
            & build_v3_modular_filter_mask(df, config, has_candidate)
            & (regime == "volatile_trend")
            & (selected_module == "v3_second_close_green_reclaim_rejection")
        )

        v5_extreme_route_pass = (
            bool(config.get("enable_extreme_expansion_entries", False))
            & build_v3_modular_filter_mask(df, config, has_candidate)
            & regime.isin(["extreme_expansion", "very_extreme_expansion", "abnormal_news"])
            & (selected_module == "extreme_expansion")
        )

        if not bool(config.get("enable_abnormal_news_entries", False)):
            v5_extreme_route_pass = v5_extreme_route_pass & (regime != "abnormal_news")

        if (
            config.get("v4_use_conditional_v2_trend_health_layer", True)
            and config.get("enable_v2_trend_health_filter", True)
        ):
            required_conditional_v2_columns = [
                "v4_v2_trend_health_required",
                "v2_long_trend_health_pass",
                "v2_short_trend_health_pass",
            ]

            missing_conditional_v2_columns = [
                column
                for column in required_conditional_v2_columns
                if column not in df.columns
            ]

            if missing_conditional_v2_columns:
                raise KeyError(
                    "Missing V4 conditional V2 columns: "
                    + ", ".join(missing_conditional_v2_columns)
                )

            v2_health_required = (
                df["v4_v2_trend_health_required"].fillna(False).astype(bool)
            )

            long_v2_health_pass = (
                df.get(
                    "v2_long_reclaim_recovery_health_pass",
                    df["v2_long_trend_health_pass"],
                )
                .fillna(False)
                .astype(bool)
            )

            short_v2_health_pass = (
                df.get(
                    "v2_short_reclaim_recovery_health_pass",
                    df["v2_short_trend_health_pass"],
                )
                .fillna(False)
                .astype(bool)
            )

            directional_v2_health_pass = (
                ((df["raw_signal_side"] == "long") & long_v2_health_pass)
                | ((df["raw_signal_side"] == "short") & short_v2_health_pass)
            )

            conditional_v2_pass = ~v2_health_required | directional_v2_health_pass

            if config.get("enable_v2_reclaim_recovery_health", True):
                v3_volatile_route_pass = (v3_volatile_route_pass & (~v2_hard_filter_active | conditional_v2_pass))

        return v1_calm_route_pass | v3_volatile_route_pass | v5_extreme_route_pass

    if strategy_filter == "weak_red_shift_only":
        return has_candidate & (directional_red_shift_label == "weak")

    if strategy_filter == "exclude_extreme_red_shift":
        return has_candidate & ~directional_red_shift_label.isin(
            ["extreme", "very_extreme", "abnormal_news"]
        )

    if strategy_filter == "exclude_strong_plus_red_shift":
        return has_candidate & ~directional_red_shift_label.isin(
            ["strong", "very_strong", "extreme", "very_extreme", "abnormal_news"]
        )
    
    if strategy_filter == "extension_le_6":
        return has_candidate & (directional_extension <= 6.0)

    if strategy_filter == "extension_le_4":
        return has_candidate & (directional_extension <= 4.0)

    if strategy_filter == "body_ratio_ge_0_50":
        return has_candidate & (df["body_ratio"] >= 0.50)

    raise ValueError(
        f"Unknown strategy_filter: {strategy_filter}. "
        "Use baseline, no_red_shift_filter, v2_adaptive_trend_health, "
        "v3_second_close_green_reclaim_rejection, v4_dynamic_regime_selector, "
        "weak_red_shift_only, exclude_extreme_red_shift, "
        "exclude_strong_plus_red_shift, extension_le_6, extension_le_4, "
        "or body_ratio_ge_0_50."
    )

### V5 MR Orange Failure setup

This setup is a first mean-reversion test.

It does not trade orange touches blindly. It waits for price to touch/pierce orange, fail to continue, and confirm rejection with clean closes.

MR is kept separate from S-tier and A-tier continuation logic because it has a different purpose, target, and risk profile.

### V5 Signal and Trade Tracking Helpers

This section standardises setup, regime, session, and exit metadata for the V5 modular engine.

The tracking fields are designed to make each trade auditable without changing signal generation, routing, execution, or risk-control behaviour.

In [ ]:
# ---------------------------------------------------------------------
# V5 signal and trade tracking helpers
# ---------------------------------------------------------------------

def v5_safe_bool(value, default: bool = False) -> bool:
    """
    Convert nullable pandas / numpy values into a safe boolean.
    """
    if value is None:
        return default

    try:
        if pd.isna(value):
            return default
    except (TypeError, ValueError):
        pass

    return bool(value)


def v5_trade_session_from_timestamp(timestamp, config: dict) -> str:
    """
    Label the trading session for a timestamp using CONFIG['session_windows'].

    Returns one of:
    asia, london, new_york, out_of_session, unknown
    """
    if timestamp is None:
        return "unknown"

    try:
        if pd.isna(timestamp):
            return "unknown"
    except (TypeError, ValueError):
        pass

    session_windows = config.get("session_windows", {})

    if not isinstance(session_windows, dict) or len(session_windows) == 0:
        return "unknown"

    try:
        session_ts = normalise_timestamp_to_session_time(timestamp, config)
        current_time = session_ts.time()
    except Exception:
        return "unknown"

    valid_sessions = globals().get(
        "V5_VALID_SESSIONS",
        {"asia", "london", "new_york"},
    )

    for session_name, window in session_windows.items():
        if session_name not in valid_sessions:
            continue

        try:
            start_raw, end_raw = window
            start_time = pd.to_datetime(start_raw).time()
            end_time = pd.to_datetime(end_raw).time()
        except Exception:
            continue

        if start_time <= end_time:
            in_window = start_time <= current_time < end_time
        else:
            in_window = current_time >= start_time or current_time < end_time

        if in_window:
            return session_name

    return "out_of_session"


def v5_session_enabled_for_trade(trade_session: str, config: dict) -> bool:
    """
    Return whether a session is enabled under the current V5 session settings.
    """
    if "v5_is_session_enabled" in globals():
        return bool(v5_is_session_enabled(trade_session, config))

    if not config.get("enable_session_filter", False):
        return True

    if trade_session in {"unknown", "out_of_session"}:
        return bool(config.get("allow_out_of_session_trades", False))

    enabled_sessions = config.get("enabled_sessions", ["asia", "london", "new_york"])

    return trade_session in enabled_sessions


def v5_entry_time_context_fields(entry_time, config: dict) -> dict:
    """
    Build entry-time and session tracking fields.
    """
    trade_session = v5_trade_session_from_timestamp(entry_time, config)
    session_enabled = v5_session_enabled_for_trade(trade_session, config)

    fields = {
        "entry_hour": np.nan,
        "entry_minute": np.nan,
        "entry_time_of_day": "unknown",
        "trade_session": trade_session,
        "session_enabled": session_enabled,
        "blocked_by_session": not session_enabled,
    }

    try:
        session_ts = normalise_timestamp_to_session_time(entry_time, config)
        fields["entry_hour"] = int(session_ts.hour)
        fields["entry_minute"] = int(session_ts.minute)
        fields["entry_time_of_day"] = session_ts.strftime("%H:%M")
    except Exception:
        pass

    return fields

def v5_selected_module_key(selected_module: str) -> str:
    """
    Map internal selected-module labels into compact V5 module keys.
    """
    selected_module = str(selected_module)

    if selected_module == "v1_green_reclaim_rejection":
        return "v1_s_tier"

    if selected_module == "v3_second_close_green_reclaim_rejection":
        return "v3_a_tier_second_close"

    if selected_module == "extreme_expansion":
        return "extreme_expansion"

    return "none"


def v5_module_toggle_block_reason(row: pd.Series, config: dict) -> str | None:
    """
    Return a manual module-toggle block reason, or None if the module is allowed.
    """
    if str(row.get("raw_signal_name", "NO_SIGNAL")) == "NO_SIGNAL":
        return None

    selected_module = str(row.get("v4_selected_entry_module", "none"))
    module_key = v5_selected_module_key(selected_module)
    regime = str(row.get("v5_regime_20m", row.get("v4_preliminary_regime_label", "")))

    if module_key == "v1_s_tier" and not bool(config.get("enable_v1_s_tier", True)):
        return "BLOCKED_BY_V1_TOGGLE"

    if (
        module_key == "v3_a_tier_second_close"
        and not bool(config.get("enable_v3_a_tier_second_close", True))
    ):
        return "BLOCKED_BY_V3_TOGGLE"

    if module_key == "extreme_expansion":
        if regime == "abnormal_news" and not bool(config.get("enable_abnormal_news_entries", False)):
            return "BLOCKED_BY_ABNORMAL_NEWS"

        if not bool(config.get("enable_extreme_expansion_entries", False)):
            return "BLOCKED_BY_MANUAL_TOGGLE"

    if module_key == "none":
        return "BLOCKED_BY_ROUTER"

    return None


def v5_directional_continuation_health_pass(row: pd.Series) -> bool:
    """
    Directional continuation-health pass used for V1-style continuation checks.
    """
    side = str(row.get("raw_signal_side", "none"))

    if side == "long":
        return v5_safe_bool(row.get("v2_long_trend_health_pass", False))

    if side == "short":
        return v5_safe_bool(row.get("v2_short_trend_health_pass", False))

    return False


def v5_directional_reclaim_recovery_health_pass(row: pd.Series) -> bool:
    """
    Directional reclaim/recovery-health pass used for V3-style recovery checks.
    """
    side = str(row.get("raw_signal_side", "none"))

    if side == "long":
        return v5_safe_bool(
            row.get(
                "v2_long_reclaim_recovery_health_pass",
                row.get("v2_long_trend_health_pass", False),
            )
        )

    if side == "short":
        return v5_safe_bool(
            row.get(
                "v2_short_reclaim_recovery_health_pass",
                row.get("v2_short_trend_health_pass", False),
            )
        )

    return False


V5_V2_TREND_HEALTH_ACTIVATION_MODES = {
    "off",
    "always",
    "router_gated",
    "time_gated",
    "hybrid",
}


def v5_get_v2_trend_health_activation_mode(config: dict) -> str:
    """
    Return the active V2 trend-health activation mode.
    """
    mode = str(
        config.get("v2_trend_health_activation_mode", "router_gated")
    ).lower()

    if mode not in V5_V2_TREND_HEALTH_ACTIVATION_MODES:
        raise ValueError(
            f"Invalid v2_trend_health_activation_mode={mode!r}. "
            f"Allowed values: {sorted(V5_V2_TREND_HEALTH_ACTIVATION_MODES)}"
        )

    return mode


def v5_is_after_configured_time(
    timestamp,
    time_str: str,
    timezone_str: str,
) -> bool:
    """
    Return whether a timestamp is at or after the configured local time.

    Naive timestamps are treated as already being in the notebook/session timezone.
    Timezone-aware timestamps are converted to the requested timezone.
    """
    if timestamp is None or pd.isna(timestamp):
        return False

    ts = pd.Timestamp(timestamp)

    if ts.tzinfo is not None:
        ts = ts.tz_convert(timezone_str)

    hour_text, minute_text = str(time_str).split(":")
    force_minutes = int(hour_text) * 60 + int(minute_text)
    timestamp_minutes = int(ts.hour) * 60 + int(ts.minute)

    return timestamp_minutes >= force_minutes


def v5_router_requires_v2_trend_health(row: pd.Series, config: dict) -> bool:
    """
    Return whether the router/regime context wants V2 trend-health protection.

    V2 is meant to be a safety layer. Clean trend regimes do not need to
    hard-block by default, but questionable/choppy/damaged regimes can.
    """
    regime = str(
        row.get(
            "v5_regime_20m",
            row.get("v4_preliminary_regime_label", "unknown"),
        )
    ).lower()

    model_route = str(row.get("v4_model_route", "unknown")).lower()

    router_required_regimes = {
        "questionable_trend",
        "damaged_pullback",
        "chop_compression",
        "chop",
        "unclear",
        "abnormal_news",
        "late_session_risk",
    }

    router_safe_regimes = {
        "calm_trend",
        "strong_clean_trend",
        "volatile_trend",
        "extreme_expansion",
        "very_extreme_expansion",
    }

    if regime in router_required_regimes:
        return True

    if regime in router_safe_regimes:
        return False

    if "chop" in model_route or "no_trade" in model_route:
        return True

    if "questionable" in regime or "damaged" in regime or "unclear" in regime:
        return True

    return False


def v5_v2_trend_health_activation_state(row: pd.Series, config: dict) -> dict:
    """
    Return whether V2 should be active as a hard filter for this row,
    and why.

    This helper is scaffolded in Commit 9.1.
    Commit 9.2 wires this state into V2 blocking.
    """
    mode = v5_get_v2_trend_health_activation_mode(config)

    router_required = v5_router_requires_v2_trend_health(row, config)

    timestamp = row.get(
        "datetime",
        row.get("signal_time", row.get("entry_time", None)),
    )

    force_after_time_active = (
        bool(config.get("v2_force_after_time_enabled", False))
        and v5_is_after_configured_time(
            timestamp=timestamp,
            time_str=config.get("v2_force_after_time", "16:30"),
            timezone_str=config.get("v2_force_after_timezone", "Europe/London"),
        )
    )

    if not bool(config.get("enable_v2_trend_health_filter", True)):
        active = False
        reason = "OFF"

    elif mode == "off":
        active = False
        reason = "OFF"

    elif mode == "always":
        active = True
        reason = "ALWAYS"

    elif mode == "router_gated":
        active = router_required
        reason = "ROUTER_REQUIRED" if active else "TRACK_ONLY"

    elif mode == "time_gated":
        active = force_after_time_active
        reason = "TIME_OVERRIDE" if active else "TRACK_ONLY"

    elif mode == "hybrid":
        active = router_required or force_after_time_active

        if router_required and force_after_time_active:
            reason = "ROUTER_OR_TIME"
        elif router_required:
            reason = "ROUTER_REQUIRED"
        elif force_after_time_active:
            reason = "TIME_OVERRIDE"
        else:
            reason = "TRACK_ONLY"

    else:
        active = False
        reason = "OFF"

    if not active and not bool(config.get("v2_track_when_not_active", True)):
        reason = "OFF"

    return {
        "v2_trend_health_active": bool(active),
        "v2_activation_reason": reason,
        "v2_router_required": bool(router_required),
        "v2_force_after_time_active": bool(force_after_time_active),
    }

def add_v5_v2_activation_fields(out: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Add V2 activation-state fields to the signal dataframe.

    V2 can be tracked all day, but only becomes a hard blocker when
    v2_trend_health_active is True.
    """
    out = out.copy()

    activation_df = out.apply(
        lambda row: v5_v2_trend_health_activation_state(row, config),
        axis=1,
        result_type="expand",
    )

    for column in activation_df.columns:
        out[column] = activation_df[column]

    return out

def v5_v2_health_failure_reason(row: pd.Series, config: dict) -> str | None:
    """
    Return the underlying V2 health failure reason, ignoring activation mode.

    This tells us whether V2 would fail if it were active as a hard filter.
    """
    if str(row.get("raw_signal_name", "NO_SIGNAL")) == "NO_SIGNAL":
        return None

    if not bool(config.get("enable_v2_trend_health_filter", True)):
        return None

    v2_required = v5_safe_bool(row.get("v4_v2_trend_health_required", False))

    if not v2_required:
        return None

    selected_module = str(row.get("v4_selected_entry_module", "none"))
    module_key = v5_selected_module_key(selected_module)

    if module_key == "v1_s_tier":
        if not bool(config.get("enable_v2_continuation_health", True)):
            return None

        if not v5_directional_continuation_health_pass(row):
            return "BLOCKED_BY_V2_CONTINUATION_HEALTH"

    if module_key == "v3_a_tier_second_close":
        if not bool(config.get("enable_v2_reclaim_recovery_health", True)):
            return None

        if not v5_directional_reclaim_recovery_health_pass(row):
            return "BLOCKED_BY_V2_RECLAIM_RECOVERY_HEALTH"

    return None


def v5_v2_toggle_block_reason(row: pd.Series, config: dict) -> str | None:
    """
    Return a V2 hard-block reason, or None.

    V2 only blocks when the activation state says it is active.
    If V2 fails while inactive, it is tracked-only and does not block.
    """
    health_failure_reason = v5_v2_health_failure_reason(row, config)

    if health_failure_reason is None:
        return None

    v2_active = v5_safe_bool(
        row.get(
            "v2_trend_health_active",
            v5_v2_trend_health_activation_state(row, config)["v2_trend_health_active"],
        )
    )

    if not v2_active:
        return None

    return health_failure_reason

def v5_entry_red_shift_min_points(config: dict, setup_family: str) -> float:
    """
    Return the active minimum directional red-shift floor for a setup family.
    """
    base_min_shift = float(config.get("entry_min_directional_red_shift_points", 0.0))

    if setup_family == "s_tier":
        override = config.get("s_tier_min_directional_red_shift_points", None)

    elif setup_family == "a_tier":
        override = config.get("a_tier_min_directional_red_shift_points", None)

    else:
        override = None

    if override is None or pd.isna(override):
        return base_min_shift

    return float(override)


def v5_entry_directional_red_shift_points(row: pd.Series) -> float:
    """
    Return directional red-shift points for the candidate side.

    Longs use upper red shifting upward.
    Shorts use lower red shifting downward.
    """
    selected_module = str(row.get("v4_selected_entry_module", "none"))
    module_key = v5_selected_module_key(selected_module)

    if module_key == "v3_a_tier_second_close":
        value = row.get("v3_directional_red_shift", np.nan)

        if not pd.isna(value):
            return float(value)

    value = row.get("directional_red_shift_strength", np.nan)

    if not pd.isna(value):
        return float(value)

    side = str(row.get("raw_signal_side", "none"))

    if side == "long":
        value = row.get("bullish_red_shift_strength", row.get("upper_red_shift", np.nan))

        if not pd.isna(value):
            return float(value)

    if side == "short":
        value = row.get("bearish_red_shift_strength", np.nan)

        if not pd.isna(value):
            return float(value)

        lower_red_shift = row.get("lower_red_shift", np.nan)

        if not pd.isna(lower_red_shift):
            return float(-lower_red_shift)

    return np.nan


def v5_entry_red_shift_floor_block_reason(row: pd.Series, config: dict) -> str | None:
    """
    Return an optional entry red-shift-floor block reason.

    This is an optional quality gate only. It does not change the underlying
    V1/S-tier or V3/A-tier entry definitions.
    """
    if str(row.get("raw_signal_name", "NO_SIGNAL")) == "NO_SIGNAL":
        return None

    selected_module = str(row.get("v4_selected_entry_module", "none"))
    module_key = v5_selected_module_key(selected_module)

    if module_key == "v1_s_tier":
        if not bool(config.get("s_tier_use_entry_red_shift_floor", False)):
            return None

        min_shift = v5_entry_red_shift_min_points(config, "s_tier")
        directional_shift = v5_entry_directional_red_shift_points(row)

        if pd.isna(directional_shift) or directional_shift < min_shift:
            return "BLOCKED_BY_S_TIER_REDSHIFT_FLOOR"

    if module_key == "v3_a_tier_second_close":
        if not bool(config.get("a_tier_use_entry_red_shift_floor", False)):
            return None

        min_shift = v5_entry_red_shift_min_points(config, "a_tier")
        directional_shift = v5_entry_directional_red_shift_points(row)

        if pd.isna(directional_shift) or directional_shift < min_shift:
            return "BLOCKED_BY_A_TIER_REDSHIFT_FLOOR"

    return None

V5_A_TIER_RETRACE_ENTRY_MODES = {
    "off",
    "extended_only",
    "always_wait_for_retrace",
}


def v5_get_a_tier_retrace_entry_mode(config: dict) -> str:
    """
    Return the active A-tier retrace-entry mode.
    """
    mode = str(config.get("a_tier_retrace_entry_mode", "extended_only")).lower()

    if mode not in V5_A_TIER_RETRACE_ENTRY_MODES:
        raise ValueError(
            f"Invalid a_tier_retrace_entry_mode={mode!r}. "
            f"Allowed values: {sorted(V5_A_TIER_RETRACE_ENTRY_MODES)}"
        )

    if not bool(config.get("enable_a_tier_retrace_entry", False)):
        return "off"

    return mode


def add_v5_a_tier_fast_reclaim_fields(out: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Add optional A-tier fast-reclaim diagnostics and candidate flags.

    Toggle-off behaviour:
    - all fast-reclaim pass flags stay False
    - existing second-close entries remain unchanged
    """
    epsilon = 1e-9

    enabled = bool(config.get("enable_a_tier_fast_reclaim_entry", False))
    max_away_bars = max(1, int(config.get("a_tier_fast_reclaim_max_away_bars", 2)))
    max_depth_points = float(config.get("a_tier_fast_reclaim_max_depth_points", 20.0))

    min_body_points = float(config.get("a_tier_fast_reclaim_min_body_points", 10.0))
    min_body_ratio = float(config.get("a_tier_fast_reclaim_min_body_ratio", 0.60))
    min_close_position = float(config.get("a_tier_fast_reclaim_min_close_position", 0.70))

    require_trend_health = bool(
        config.get("a_tier_fast_reclaim_require_trend_health", True)
    )

    long_away_from_green = (out["close"] < out["upper_green"]).fillna(False)
    short_away_from_green = (out["close"] > out["lower_green"]).fillna(False)

    long_away_bars = (
        consecutive_true_count(long_away_from_green)
        .shift(1)
        .fillna(0)
        .astype(int)
    )

    short_away_bars = (
        consecutive_true_count(short_away_from_green)
        .shift(1)
        .fillna(0)
        .astype(int)
    )

    long_depth = (out["upper_green"] - out["low"]).clip(lower=0.0)
    short_depth = (out["high"] - out["lower_green"]).clip(lower=0.0)

    long_depth_away_only = long_depth.where(long_away_from_green, 0.0)
    short_depth_away_only = short_depth.where(short_away_from_green, 0.0)

    long_depth_candidates = [
        long_depth_away_only.shift(i)
        for i in range(1, max_away_bars + 1)
    ]

    short_depth_candidates = [
        short_depth_away_only.shift(i)
        for i in range(1, max_away_bars + 1)
    ]

    long_max_depth = (
        pd.concat(long_depth_candidates, axis=1)
        .max(axis=1)
        .fillna(0.0)
    )

    short_max_depth = (
        pd.concat(short_depth_candidates, axis=1)
        .max(axis=1)
        .fillna(0.0)
    )

    body_points = (out["close"] - out["open"]).abs()
    range_points = (out["high"] - out["low"]).abs()
    body_ratio = body_points / range_points.clip(lower=epsilon)

    long_close_position = (
        (out["close"] - out["low"])
        / range_points.clip(lower=epsilon)
    )

    short_close_position = (
        (out["high"] - out["close"])
        / range_points.clip(lower=epsilon)
    )

    long_trend_health_pass = (
        out.get(
            "v2_long_reclaim_recovery_health_pass",
            out.get("v2_long_trend_health_pass", pd.Series(False, index=out.index)),
        )
        .fillna(False)
        .astype(bool)
    )

    short_trend_health_pass = (
        out.get(
            "v2_short_reclaim_recovery_health_pass",
            out.get("v2_short_trend_health_pass", pd.Series(False, index=out.index)),
        )
        .fillna(False)
        .astype(bool)
    )

    if not require_trend_health:
        long_trend_health_pass = pd.Series(True, index=out.index)
        short_trend_health_pass = pd.Series(True, index=out.index)

    long_pass = (
        enabled
        & long_away_bars.between(1, max_away_bars)
        & (long_max_depth <= max_depth_points)
        & (out["close"] > out["upper_green"])
        & (out["close"] > out["open"])
        & (body_points >= min_body_points)
        & (body_ratio >= min_body_ratio)
        & (long_close_position >= min_close_position)
        & out.get("v3_long_vwap_acceptance_pass", pd.Series(True, index=out.index)).fillna(False).astype(bool)
        & out.get("v3_long_directional_red_shift_pass", pd.Series(True, index=out.index)).fillna(False).astype(bool)
        & ~out.get("v3_long_trend_dead", pd.Series(False, index=out.index)).fillna(False).astype(bool)
        & long_trend_health_pass
    )

    short_pass = (
        enabled
        & short_away_bars.between(1, max_away_bars)
        & (short_max_depth <= max_depth_points)
        & (out["close"] < out["lower_green"])
        & (out["close"] < out["open"])
        & (body_points >= min_body_points)
        & (body_ratio >= min_body_ratio)
        & (short_close_position >= min_close_position)
        & out.get("v3_short_vwap_acceptance_pass", pd.Series(True, index=out.index)).fillna(False).astype(bool)
        & out.get("v3_short_directional_red_shift_pass", pd.Series(True, index=out.index)).fillna(False).astype(bool)
        & ~out.get("v3_short_trend_dead", pd.Series(False, index=out.index)).fillna(False).astype(bool)
        & short_trend_health_pass
    )

    out["a_tier_fast_reclaim_long_passed"] = long_pass
    out["a_tier_fast_reclaim_short_passed"] = short_pass
    out["a_tier_fast_reclaim_passed"] = long_pass | short_pass

    out["a_tier_fast_reclaim_away_bars"] = np.where(
        long_pass,
        long_away_bars,
        np.where(short_pass, short_away_bars, np.nan),
    )

    out["a_tier_fast_reclaim_max_depth_points"] = np.where(
        long_pass,
        long_max_depth,
        np.where(short_pass, short_max_depth, np.nan),
    )

    out["a_tier_fast_reclaim_body_points"] = np.where(
        long_pass | short_pass,
        body_points,
        np.nan,
    )

    out["a_tier_fast_reclaim_body_ratio"] = np.where(
        long_pass | short_pass,
        body_ratio,
        np.nan,
    )

    out["a_tier_fast_reclaim_close_position"] = np.where(
        long_pass,
        long_close_position,
        np.where(short_pass, short_close_position, np.nan),
    )

    return out

def add_v5_a_tier_delayed_pullback_fields(out: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Add optional A-tier delayed-pullback diagnostics and candidate flags.

    This is separate from:
    - V3_A_TIER_SECOND_CLOSE
    - V5_A_TIER_FAST_RECLAIM
    - existing A-tier retrace-entry execution logic

    Toggle-off behaviour:
    - structural candidates can still be diagnosed
    - passed flags stay False
    - baseline entries remain unchanged
    """
    enabled = bool(config.get("enable_a_tier_delayed_pullback_entry", False))

    entry_bar = max(2, int(config.get("a_tier_delayed_pullback_entry_bar", 4)))
    min_hold_bars = max(1, int(config.get("a_tier_delayed_pullback_min_hold_bars", 3)))
    hold_lookback = min(min_hold_bars, entry_bar - 1)

    min_pullback_points = float(config.get("a_tier_delayed_pullback_min_points", 5.0))
    max_pullback_points = float(config.get("a_tier_delayed_pullback_max_points", 15.0))

    must_hold_green = bool(config.get("a_tier_delayed_pullback_must_hold_green", True))
    pierce_points = float(
        config.get("a_tier_delayed_pullback_allow_intrabar_green_pierce_points", 0.0)
    )

    require_trend_health = bool(
        config.get("a_tier_delayed_pullback_require_trend_health", True)
    )

    long_in_green = (out["close"] > out["upper_green"]).fillna(False)
    short_in_green = (out["close"] < out["lower_green"]).fillna(False)

    long_dipped_below_green = (out["close"] < out["upper_green"]).fillna(False)
    short_pushed_above_green = (out["close"] > out["lower_green"]).fillna(False)

    long_green_hold_bars = consecutive_true_count(long_in_green).fillna(0).astype(int)
    short_green_hold_bars = consecutive_true_count(short_in_green).fillna(0).astype(int)

    long_had_required_hold = (
        long_green_hold_bars.shift(1).fillna(0).astype(int) >= min_hold_bars
    )

    short_had_required_hold = (
        short_green_hold_bars.shift(1).fillna(0).astype(int) >= min_hold_bars
    )

    long_previous_dip_before_sequence = (
        long_dipped_below_green.shift(entry_bar).fillna(False).astype(bool)
    )

    short_previous_push_before_sequence = (
        short_pushed_above_green.shift(entry_bar).fillna(False).astype(bool)
    )

    long_candidate = (
        (long_green_hold_bars == entry_bar)
        & long_had_required_hold
        & long_previous_dip_before_sequence
    )

    short_candidate = (
        (short_green_hold_bars == entry_bar)
        & short_had_required_hold
        & short_previous_push_before_sequence
    )

    recent_long_high_after_reclaim = (
        out["high"]
        .shift(1)
        .rolling(hold_lookback, min_periods=1)
        .max()
    )

    recent_short_low_after_reclaim = (
        out["low"]
        .shift(1)
        .rolling(hold_lookback, min_periods=1)
        .min()
    )

    long_pullback_points = recent_long_high_after_reclaim - out["low"]
    short_pullback_points = out["high"] - recent_short_low_after_reclaim

    long_pullback_size_pass = (
        long_pullback_points.between(min_pullback_points, max_pullback_points)
    )

    short_pullback_size_pass = (
        short_pullback_points.between(min_pullback_points, max_pullback_points)
    )

    long_held_green = (
        out["low"] >= (out["upper_green"] - pierce_points)
    ).fillna(False)

    short_held_green = (
        out["high"] <= (out["lower_green"] + pierce_points)
    ).fillna(False)

    if not must_hold_green:
        long_held_green = pd.Series(True, index=out.index)
        short_held_green = pd.Series(True, index=out.index)

    long_trend_health_pass = (
        out.get(
            "v2_long_reclaim_recovery_health_pass",
            out.get("v2_long_trend_health_pass", pd.Series(False, index=out.index)),
        )
        .fillna(False)
        .astype(bool)
    )

    short_trend_health_pass = (
        out.get(
            "v2_short_reclaim_recovery_health_pass",
            out.get("v2_short_trend_health_pass", pd.Series(False, index=out.index)),
        )
        .fillna(False)
        .astype(bool)
    )

    if not require_trend_health:
        long_trend_health_pass = pd.Series(True, index=out.index)
        short_trend_health_pass = pd.Series(True, index=out.index)

    long_a_tier_filter_pass = (
        out.get("v3_long_vwap_acceptance_pass", pd.Series(True, index=out.index))
        .fillna(False)
        .astype(bool)
        & out.get("v3_long_directional_red_shift_pass", pd.Series(True, index=out.index))
        .fillna(False)
        .astype(bool)
        & ~out.get("v3_long_trend_dead", pd.Series(False, index=out.index))
        .fillna(False)
        .astype(bool)
    )

    short_a_tier_filter_pass = (
        out.get("v3_short_vwap_acceptance_pass", pd.Series(True, index=out.index))
        .fillna(False)
        .astype(bool)
        & out.get("v3_short_directional_red_shift_pass", pd.Series(True, index=out.index))
        .fillna(False)
        .astype(bool)
        & ~out.get("v3_short_trend_dead", pd.Series(False, index=out.index))
        .fillna(False)
        .astype(bool)
    )

    # ------------------------------------------------------------------
    # Optional delayed-pullback strength/trend filter
    # ------------------------------------------------------------------
    strength_filter_enabled = bool(
        config.get("enable_a_tier_delayed_pullback_strength_filter", False)
    )

    allowed_red_shift_buckets = {
        str(bucket).lower()
        for bucket in config.get(
            "a_tier_delayed_pullback_allowed_red_shift_buckets",
            ["strong", "very_strong"],
        )
    }

    block_track_only = bool(
        config.get("a_tier_delayed_pullback_block_track_only", True)
    )

    require_v2_pass = bool(
        config.get("a_tier_delayed_pullback_require_v2_pass", True)
    )

    long_red_shift_bucket = (
        out.get(
            "v3_long_red_shift_bucket",
            out.get("bullish_red_shift_label", pd.Series("unknown", index=out.index)),
        )
        .fillna("unknown")
        .astype(str)
        .str.lower()
    )

    short_red_shift_bucket = (
        out.get(
            "v3_short_red_shift_bucket",
            out.get("bearish_red_shift_label", pd.Series("unknown", index=out.index)),
        )
        .fillna("unknown")
        .astype(str)
        .str.lower()
    )

    v2_activation_reason = (
        out.get(
            "v2_activation_reason",
            pd.Series("TRACK_ONLY", index=out.index),
        )
        .fillna("TRACK_ONLY")
        .astype(str)
        .str.upper()
    )

    long_v2_pass_used = (
        out.get(
            "v2_long_reclaim_recovery_health_pass",
            out.get("v2_long_trend_health_pass", pd.Series(False, index=out.index)),
        )
        .fillna(False)
        .astype(bool)
    )

    short_v2_pass_used = (
        out.get(
            "v2_short_reclaim_recovery_health_pass",
            out.get("v2_short_trend_health_pass", pd.Series(False, index=out.index)),
        )
        .fillna(False)
        .astype(bool)
    )

    long_strength_bucket_pass = long_red_shift_bucket.isin(allowed_red_shift_buckets)
    short_strength_bucket_pass = short_red_shift_bucket.isin(allowed_red_shift_buckets)

    track_only_pass = (
        ~v2_activation_reason.eq("TRACK_ONLY")
        if block_track_only
        else pd.Series(True, index=out.index)
    )

    long_strength_v2_pass = (
        long_v2_pass_used
        if require_v2_pass
        else pd.Series(True, index=out.index)
    )

    short_strength_v2_pass = (
        short_v2_pass_used
        if require_v2_pass
        else pd.Series(True, index=out.index)
    )

    if strength_filter_enabled:
        long_strength_filter_pass = (
            long_strength_bucket_pass
            & track_only_pass
            & long_strength_v2_pass
        )

        short_strength_filter_pass = (
            short_strength_bucket_pass
            & track_only_pass
            & short_strength_v2_pass
        )
    else:
        long_strength_filter_pass = pd.Series(True, index=out.index)
        short_strength_filter_pass = pd.Series(True, index=out.index)

    delayed_strength_filter_pass = pd.Series(
        np.where(
            long_candidate,
            long_strength_filter_pass,
            np.where(short_candidate, short_strength_filter_pass, False),
        ),
        index=out.index,
    ).astype(bool)

    delayed_strength_red_shift_bucket = pd.Series(
        np.where(
            long_candidate,
            long_red_shift_bucket,
            np.where(short_candidate, short_red_shift_bucket, "unknown"),
        ),
        index=out.index,
    )

    delayed_strength_v2_pass_used = pd.Series(
        np.where(
            long_candidate,
            long_v2_pass_used,
            np.where(short_candidate, short_v2_pass_used, False),
        ),
        index=out.index,
    ).astype(bool)

    delayed_strength_block_reason = pd.Series(
        "NOT_CANDIDATE",
        index=out.index,
        dtype="object",
    )

    delayed_candidate_for_strength = long_candidate | short_candidate

    delayed_strength_block_reason.loc[
        delayed_candidate_for_strength & ~strength_filter_enabled
    ] = "STRENGTH_FILTER_DISABLED"

    delayed_strength_block_reason.loc[
        delayed_candidate_for_strength
        & strength_filter_enabled
        & delayed_strength_filter_pass
    ] = "PASSED"

    delayed_strength_block_reason.loc[
        delayed_candidate_for_strength
        & strength_filter_enabled
        & ~pd.Series(
            np.where(
                long_candidate,
                long_strength_bucket_pass,
                np.where(short_candidate, short_strength_bucket_pass, False),
            ),
            index=out.index,
        ).astype(bool)
    ] = "DELAYED_PULLBACK_BUCKET_BLOCKED"

    delayed_strength_block_reason.loc[
        delayed_candidate_for_strength
        & strength_filter_enabled
        & block_track_only
        & v2_activation_reason.eq("TRACK_ONLY")
    ] = "DELAYED_PULLBACK_TRACK_ONLY_BLOCKED"

    delayed_strength_block_reason.loc[
        delayed_candidate_for_strength
        & strength_filter_enabled
        & require_v2_pass
        & ~delayed_strength_v2_pass_used
    ] = "DELAYED_PULLBACK_V2_PASS_BLOCKED"

    long_pass = (
        enabled
        & long_candidate
        & long_pullback_size_pass
        & long_held_green
        & long_trend_health_pass
        & long_a_tier_filter_pass
        & long_strength_filter_pass
    )

    short_pass = (
        enabled
        & short_candidate
        & short_pullback_size_pass
        & short_held_green
        & short_trend_health_pass
        & short_a_tier_filter_pass
        & short_strength_filter_pass
    )

    delayed_candidate = long_candidate | short_candidate
    delayed_passed = long_pass | short_pass

    delayed_points = np.where(
        long_candidate,
        long_pullback_points,
        np.where(short_candidate, short_pullback_points, np.nan),
    )

    delayed_hold_bars = np.where(
        long_candidate,
        long_green_hold_bars,
        np.where(short_candidate, short_green_hold_bars, np.nan),
    )

    delayed_held_green = np.where(
        long_candidate,
        long_held_green,
        np.where(short_candidate, short_held_green, False),
    )

    delayed_pullback_size_pass = pd.Series(
        np.where(
            long_candidate,
            long_pullback_size_pass,
            np.where(short_candidate, short_pullback_size_pass, False),
        ),
        index=out.index,
    ).astype(bool)

    delayed_held_green_pass = pd.Series(
        delayed_held_green,
        index=out.index,
    ).astype(bool)

    delayed_trend_health_pass = pd.Series(
        np.where(
            long_candidate,
            long_trend_health_pass,
            np.where(short_candidate, short_trend_health_pass, False),
        ),
        index=out.index,
    ).astype(bool)

    delayed_a_tier_filter_pass = pd.Series(
        np.where(
            long_candidate,
            long_a_tier_filter_pass,
            np.where(short_candidate, short_a_tier_filter_pass, False),
        ),
        index=out.index,
    ).astype(bool)

    block_reason = pd.Series("not_candidate", index=out.index, dtype="object")

    block_reason.loc[delayed_candidate & ~enabled] = "disabled"

    block_reason.loc[
        delayed_candidate
        & enabled
        & ~delayed_pullback_size_pass
    ] = "failed_pullback_size"

    block_reason.loc[
        delayed_candidate
        & enabled
        & delayed_pullback_size_pass
        & ~delayed_held_green_pass
    ] = "failed_hold_green"

    block_reason.loc[
        delayed_candidate
        & enabled
        & delayed_pullback_size_pass
        & delayed_held_green_pass
        & ~delayed_trend_health_pass
    ] = "failed_trend_health"

    block_reason.loc[
        delayed_candidate
        & enabled
        & delayed_pullback_size_pass
        & delayed_held_green_pass
        & delayed_trend_health_pass
        & ~delayed_strength_filter_pass
    ] = delayed_strength_block_reason

    block_reason.loc[
        delayed_candidate
        & enabled
        & delayed_pullback_size_pass
        & delayed_held_green_pass
        & delayed_trend_health_pass
        & delayed_strength_filter_pass
        & ~delayed_a_tier_filter_pass
    ] = "failed_a_tier_filters"

    block_reason.loc[delayed_passed] = "passed"

    out["a_tier_delayed_pullback_long_passed"] = long_pass
    out["a_tier_delayed_pullback_short_passed"] = short_pass
    out["a_tier_delayed_pullback_candidate"] = delayed_candidate
    out["a_tier_delayed_pullback_passed"] = delayed_passed

    out["a_tier_delayed_pullback_entry_bar"] = np.where(
        delayed_candidate,
        entry_bar,
        np.nan,
    )

    out["a_tier_delayed_pullback_hold_bars"] = delayed_hold_bars
    out["a_tier_delayed_pullback_points"] = delayed_points
    out["a_tier_delayed_pullback_held_green"] = delayed_held_green_pass
    out["a_tier_delayed_pullback_block_reason"] = block_reason

    out["a_tier_delayed_pullback_strength_filter_pass"] = np.where(
        delayed_candidate,
        delayed_strength_filter_pass,
        False,
    )

    out["a_tier_delayed_pullback_strength_filter_block_reason"] = (
        delayed_strength_block_reason
    )

    out["a_tier_delayed_pullback_red_shift_bucket"] = (
        delayed_strength_red_shift_bucket
    )

    out["a_tier_delayed_pullback_v2_activation_reason"] = np.where(
        delayed_candidate,
        v2_activation_reason,
        "UNKNOWN",
    )

    out["a_tier_delayed_pullback_v2_pass_used"] = np.where(
        delayed_candidate,
        delayed_strength_v2_pass_used,
        False,
    )

    return out

def add_v5_a_tier_diagnostic_columns(out: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Add diagnostic-only A-tier candle-quality columns.

    These columns do not affect signal validity or trade generation.
    They only describe the current V3/A-tier second-close setup quality.
    """
    out = out.copy()
    epsilon = 1e-9

    lookback = int(
        config.get(
            "v3_green_reentry_lookback",
            config.get("v2_reclaim_recovery_lookback", 20),
        )
    )

    selected_module = out.get(
        "v4_selected_entry_module",
        pd.Series("none", index=out.index),
    ).astype(str)

    raw_side = out.get(
        "raw_signal_side",
        pd.Series("none", index=out.index),
    ).astype(str)

    raw_signal_name = out.get(
        "raw_signal_name",
        pd.Series("NO_SIGNAL", index=out.index),
    ).astype(str)

    a_tier_candidate = (
        (selected_module == "v3_second_close_green_reclaim_rejection")
        & raw_side.isin(["long", "short"])
        & (raw_signal_name != "NO_SIGNAL")
    )

    long_candidate = a_tier_candidate & (raw_side == "long")
    short_candidate = a_tier_candidate & (raw_side == "short")

    # ------------------------------------------------------------------
    # Temporary green-loss diagnostics before the second-close candle
    # ------------------------------------------------------------------
    long_below_green = out.get(
        "v3_long_bad_green_close",
        out["close"] < out["upper_green"],
    ).fillna(False).astype(bool)

    short_above_green = out.get(
        "v3_short_bad_green_close",
        out["close"] > out["lower_green"],
    ).fillna(False).astype(bool)

    long_depth_below_green = (out["upper_green"] - out["low"]).clip(lower=0.0)
    short_depth_above_green = (out["high"] - out["lower_green"]).clip(lower=0.0)

    long_loss_count = (
        long_below_green.shift(1)
        .fillna(False)
        .astype(int)
        .rolling(lookback, min_periods=1)
        .sum()
    )

    short_loss_count = (
        short_above_green.shift(1)
        .fillna(False)
        .astype(int)
        .rolling(lookback, min_periods=1)
        .sum()
    )

    long_max_depth = (
        long_depth_below_green.shift(1)
        .rolling(lookback, min_periods=1)
        .max()
    )

    short_max_depth = (
        short_depth_above_green.shift(1)
        .rolling(lookback, min_periods=1)
        .max()
    )

    # ------------------------------------------------------------------
    # First reclaim/rejection candle diagnostics
    # Current V3 second-close entry uses the previous candle as reclaim.
    # ------------------------------------------------------------------
    reclaim_open = out["open"].shift(1)
    reclaim_high = out["high"].shift(1)
    reclaim_low = out["low"].shift(1)
    reclaim_close = out["close"].shift(1)

    reclaim_body_points = (reclaim_close - reclaim_open).abs()
    reclaim_range_points = (reclaim_high - reclaim_low).abs()
    reclaim_body_ratio = reclaim_body_points / reclaim_range_points.clip(lower=epsilon)

    reclaim_long_close_position = (
        (reclaim_close - reclaim_low)
        / reclaim_range_points.clip(lower=epsilon)
    )

    reclaim_short_close_position = (
        (reclaim_high - reclaim_close)
        / reclaim_range_points.clip(lower=epsilon)
    )

    # ------------------------------------------------------------------
    # Second-close candle diagnostics
    # ------------------------------------------------------------------
    second_close_body_points = (out["close"] - out["open"]).abs()
    second_close_range_points = (out["high"] - out["low"]).abs()
    second_close_body_ratio = (
        second_close_body_points
        / second_close_range_points.clip(lower=epsilon)
    )

    second_close_long_position = (
        (out["close"] - out["low"])
        / second_close_range_points.clip(lower=epsilon)
    )

    second_close_short_position = (
        (out["high"] - out["close"])
        / second_close_range_points.clip(lower=epsilon)
    )

    long_extension_from_green = out["close"] - out["upper_green"]
    short_extension_from_green = out["lower_green"] - out["close"]

    directional_red_shift_points = out.get(
        "v3_directional_red_shift",
        out.get(
            "v5_entry_directional_red_shift_points",
            pd.Series(np.nan, index=out.index),
        ),
    )

    directional_red_shift_bucket = out.get(
        "v3_directional_red_shift_bucket",
        out.get(
            "red_shift_bucket",
            pd.Series("UNKNOWN", index=out.index),
        ),
    )

    # ------------------------------------------------------------------
    # Safe defaults for all rows
    # ------------------------------------------------------------------
    out["a_tier_is_candidate"] = a_tier_candidate
    out["a_tier_side"] = np.where(a_tier_candidate, raw_side, "UNKNOWN")
    fast_reclaim_candidate = (
        out.get(
            "a_tier_fast_reclaim_passed",
            pd.Series(False, index=out.index),
        )
        .fillna(False)
        .astype(bool)
    )

    delayed_pullback_candidate = (
        out.get(
            "a_tier_delayed_pullback_passed",
            pd.Series(False, index=out.index),
        )
        .fillna(False)
        .astype(bool)
    )

    out["a_tier_entry_timing_type"] = np.select(
        [
            a_tier_candidate & fast_reclaim_candidate,
            a_tier_candidate & delayed_pullback_candidate,
            a_tier_candidate,
        ],
        [
            "FIRST_CLOSE_FAST_RECLAIM",
            "DELAYED_PULLBACK_4TH_CANDLE",
            "SECOND_CLOSE",
        ],
        default="UNKNOWN",
    )

    out["a_tier_candles_below_green_count"] = np.where(
        long_candidate,
        long_loss_count,
        np.where(short_candidate, short_loss_count, np.nan),
    )

    out["a_tier_max_depth_below_green_points"] = np.where(
        long_candidate,
        long_max_depth,
        np.where(short_candidate, short_max_depth, np.nan),
    )

    out["a_tier_reclaim_body_points"] = np.where(
        a_tier_candidate,
        reclaim_body_points,
        np.nan,
    )

    out["a_tier_reclaim_range_points"] = np.where(
        a_tier_candidate,
        reclaim_range_points,
        np.nan,
    )

    out["a_tier_reclaim_body_ratio"] = np.where(
        a_tier_candidate,
        reclaim_body_ratio,
        np.nan,
    )

    out["a_tier_reclaim_close_position"] = np.where(
        long_candidate,
        reclaim_long_close_position,
        np.where(short_candidate, reclaim_short_close_position, np.nan),
    )

    out["a_tier_second_close_body_points"] = np.where(
        a_tier_candidate,
        second_close_body_points,
        np.nan,
    )

    out["a_tier_second_close_range_points"] = np.where(
        a_tier_candidate,
        second_close_range_points,
        np.nan,
    )

    out["a_tier_second_close_body_ratio"] = np.where(
        a_tier_candidate,
        second_close_body_ratio,
        np.nan,
    )

    out["a_tier_second_close_position"] = np.where(
        long_candidate,
        second_close_long_position,
        np.where(short_candidate, second_close_short_position, np.nan),
    )

    out["a_tier_second_close_extension_from_green"] = np.where(
        long_candidate,
        long_extension_from_green,
        np.where(short_candidate, short_extension_from_green, np.nan),
    )

    out["a_tier_entry_directional_red_shift_points"] = np.where(
        a_tier_candidate,
        directional_red_shift_points,
        np.nan,
    )

    out["a_tier_entry_red_shift_bucket"] = np.where(
        a_tier_candidate,
        directional_red_shift_bucket.astype(str),
        "UNKNOWN",
    )

    # ------------------------------------------------------------------
    # A-tier retrace-entry diagnostics
    # ------------------------------------------------------------------
    out["a_tier_retrace_required"] = False
    out["a_tier_retrace_pass"] = False
    out["a_tier_retrace_points"] = np.nan
    out["a_tier_retrace_close_extension_from_green"] = np.nan
    out["a_tier_retrace_hold_green"] = False
    out["a_tier_retrace_fail_reason"] = "NOT_RETRACE_ENTRY"

    return out

def add_v5_a_tier_diagnostic_buckets(out: pd.DataFrame) -> pd.DataFrame:
    """
    Add readable diagnostic buckets for A-tier candle-quality analysis.

    These buckets are diagnostic only and do not affect entries or exits.
    """
    out = out.copy()

    a_tier_candidate = (
        out.get("a_tier_is_candidate", pd.Series(False, index=out.index))
        .fillna(False)
        .astype(bool)
    )

    depth = pd.to_numeric(
        out.get("a_tier_max_depth_below_green_points", np.nan),
        errors="coerce",
    )

    extension = pd.to_numeric(
        out.get("a_tier_second_close_extension_from_green", np.nan),
        errors="coerce",
    )

    reclaim_body = pd.to_numeric(
        out.get("a_tier_reclaim_body_points", np.nan),
        errors="coerce",
    )

    second_close_body = pd.to_numeric(
        out.get("a_tier_second_close_body_points", np.nan),
        errors="coerce",
    )

    reclaim_body_ratio = pd.to_numeric(
        out.get("a_tier_reclaim_body_ratio", np.nan),
        errors="coerce",
    )

    reclaim_close_position = pd.to_numeric(
        out.get("a_tier_reclaim_close_position", np.nan),
        errors="coerce",
    )

    out["a_tier_depth_bucket"] = np.select(
        [
            a_tier_candidate & depth.notna() & (depth < 5),
            a_tier_candidate & depth.notna() & (depth >= 5) & (depth < 15),
            a_tier_candidate & depth.notna() & (depth >= 15) & (depth < 30),
            a_tier_candidate & depth.notna() & (depth >= 30),
        ],
        [
            "shallow",
            "normal",
            "deep",
            "very_deep",
        ],
        default="unknown",
    )

    out["a_tier_extension_bucket"] = np.select(
        [
            a_tier_candidate & extension.notna() & (extension < 0),
            a_tier_candidate & extension.notna() & (extension >= 0) & (extension < 5),
            a_tier_candidate & extension.notna() & (extension >= 5) & (extension < 15),
            a_tier_candidate & extension.notna() & (extension >= 15) & (extension < 30),
            a_tier_candidate & extension.notna() & (extension >= 30),
        ],
        [
            "below_green",
            "barely_reclaimed",
            "clean",
            "extended",
            "chase_risk",
        ],
        default="unknown",
    )

    out["a_tier_reclaim_body_bucket"] = np.select(
        [
            a_tier_candidate & reclaim_body.notna() & (reclaim_body < 5),
            a_tier_candidate & reclaim_body.notna() & (reclaim_body >= 5) & (reclaim_body < 15),
            a_tier_candidate & reclaim_body.notna() & (reclaim_body >= 15) & (reclaim_body < 30),
            a_tier_candidate & reclaim_body.notna() & (reclaim_body >= 30),
        ],
        [
            "small",
            "normal",
            "strong",
            "huge",
        ],
        default="unknown",
    )

    out["a_tier_second_close_body_bucket"] = np.select(
        [
            a_tier_candidate & second_close_body.notna() & (second_close_body < 5),
            a_tier_candidate & second_close_body.notna() & (second_close_body >= 5) & (second_close_body < 15),
            a_tier_candidate & second_close_body.notna() & (second_close_body >= 15) & (second_close_body < 30),
            a_tier_candidate & second_close_body.notna() & (second_close_body >= 30),
        ],
        [
            "small",
            "normal",
            "strong",
            "huge",
        ],
        default="unknown",
    )

    out["a_tier_reclaim_quality_bucket"] = np.select(
        [
            a_tier_candidate
            & reclaim_body_ratio.notna()
            & reclaim_close_position.notna()
            & (reclaim_body_ratio >= 0.55)
            & (reclaim_close_position >= 0.70),

            a_tier_candidate
            & reclaim_body_ratio.notna()
            & reclaim_close_position.notna()
            & (reclaim_body_ratio >= 0.35)
            & (reclaim_close_position >= 0.55),

            a_tier_candidate
            & reclaim_body_ratio.notna()
            & reclaim_close_position.notna()
            & (
                (reclaim_body_ratio < 0.35)
                | (reclaim_close_position < 0.55)
            ),
        ],
        [
            "strong_reclaim",
            "clean_reclaim",
            "weak_reclaim",
        ],
        default="unknown",
    )

    return out

def v5_a_tier_second_close_extension_from_green(row: pd.Series, side: str) -> float:
    """
    Return second-close extension from green for an A-tier candidate.

    Long:
        close - upper_green

    Short:
        lower_green - close
    """
    existing_value = row.get("a_tier_second_close_extension_from_green", np.nan)

    if not pd.isna(existing_value):
        return float(existing_value)

    close = row.get("close", np.nan)

    if side == "long":
        upper_green = row.get("upper_green", np.nan)

        if pd.isna(close) or pd.isna(upper_green):
            return np.nan

        return float(close - upper_green)

    if side == "short":
        lower_green = row.get("lower_green", np.nan)

        if pd.isna(close) or pd.isna(lower_green):
            return np.nan

        return float(lower_green - close)

    return np.nan


def v5_a_tier_direct_entry_allowed(row: pd.Series, config: dict) -> bool:
    """
    Return whether an A-tier candidate can use normal second-close entry.

    This is scaffolding for Commit 8.2 and is not wired into signal validity yet.
    """
    mode = v5_get_a_tier_retrace_entry_mode(config)

    if mode == "off":
        return True

    if mode == "always_wait_for_retrace":
        return False

    side = str(row.get("raw_signal_side", row.get("a_tier_side", "none")))
    extension = v5_a_tier_second_close_extension_from_green(row, side)

    if pd.isna(extension):
        return True

    max_direct_extension = float(
        config.get("a_tier_max_direct_extension_from_green", 30.0)
    )

    return float(extension) <= max_direct_extension


def v5_a_tier_retrace_entry_required(row: pd.Series, config: dict) -> bool:
    """
    Return whether an A-tier candidate should wait for retrace confirmation.

    This is scaffolding for Commit 8.2 and is not wired into signal validity yet.
    """
    mode = v5_get_a_tier_retrace_entry_mode(config)

    if mode == "off":
        return False

    if mode == "always_wait_for_retrace":
        return True

    side = str(row.get("raw_signal_side", row.get("a_tier_side", "none")))
    extension = v5_a_tier_second_close_extension_from_green(row, side)

    if pd.isna(extension):
        return False

    max_direct_extension = float(
        config.get("a_tier_max_direct_extension_from_green", 30.0)
    )

    return float(extension) > max_direct_extension


def v5_a_tier_retrace_entry_passes(
    second_close_row: pd.Series,
    retrace_row: pd.Series,
    side: str,
    config: dict,
) -> dict:
    """
    Evaluate whether a third-close/retrace candle passes A-tier retrace rules.

    This helper is scaffolded in Commit 8.1 and will be wired into signal
    generation in Commit 8.2.
    """
    side = str(side)

    second_close = float(second_close_row.get("close", np.nan))
    retrace_close = float(retrace_row.get("close", np.nan))
    retrace_low = float(retrace_row.get("low", np.nan))
    retrace_high = float(retrace_row.get("high", np.nan))

    min_retrace_points = float(config.get("a_tier_retrace_min_points", 5.0))
    max_retrace_points = float(config.get("a_tier_retrace_max_points", 25.0))
    max_extension_after_retrace = float(
        config.get("a_tier_retrace_max_extension_after_retrace", 25.0)
    )
    must_hold_green = bool(config.get("a_tier_retrace_must_hold_green", True))
    allowed_pierce_points = float(
        config.get("a_tier_retrace_allow_intrabar_green_pierce_points", 0.0)
    )

    if side == "long":
        retrace_green = float(retrace_row.get("upper_green", np.nan))

        retrace_points = second_close - retrace_low
        close_extension_from_green = retrace_close - retrace_green
        hold_green = retrace_close >= retrace_green
        intrabar_hold_green = retrace_low >= retrace_green - allowed_pierce_points

    elif side == "short":
        retrace_green = float(retrace_row.get("lower_green", np.nan))

        retrace_points = retrace_high - second_close
        close_extension_from_green = retrace_green - retrace_close
        hold_green = retrace_close <= retrace_green
        intrabar_hold_green = retrace_high <= retrace_green + allowed_pierce_points

    else:
        return {
            "a_tier_retrace_pass": False,
            "a_tier_retrace_points": np.nan,
            "a_tier_retrace_close_extension_from_green": np.nan,
            "a_tier_retrace_hold_green": False,
            "a_tier_retrace_fail_reason": "INVALID_SIDE",
        }

    if (
        pd.isna(retrace_points)
        or pd.isna(close_extension_from_green)
        or pd.isna(retrace_green)
    ):
        return {
            "a_tier_retrace_pass": False,
            "a_tier_retrace_points": np.nan,
            "a_tier_retrace_close_extension_from_green": np.nan,
            "a_tier_retrace_hold_green": False,
            "a_tier_retrace_fail_reason": "MISSING_RETRACE_DATA",
        }

    if must_hold_green and not hold_green:
        fail_reason = "BLOCKED_BY_A_TIER_RETRACE_LOST_GREEN"

    elif not intrabar_hold_green:
        fail_reason = "BLOCKED_BY_A_TIER_RETRACE_LOST_GREEN"

    elif retrace_points < min_retrace_points:
        fail_reason = "BLOCKED_BY_A_TIER_RETRACE_NO_PULLBACK"

    elif retrace_points > max_retrace_points:
        fail_reason = "BLOCKED_BY_A_TIER_RETRACE_TOO_DEEP"

    elif close_extension_from_green > max_extension_after_retrace:
        fail_reason = "BLOCKED_BY_A_TIER_RETRACE_TOO_EXTENDED"

    else:
        fail_reason = "VALID_RETRACE"

    retrace_pass = fail_reason == "VALID_RETRACE"

    return {
        "a_tier_retrace_pass": retrace_pass,
        "a_tier_retrace_points": float(retrace_points),
        "a_tier_retrace_close_extension_from_green": float(close_extension_from_green),
        "a_tier_retrace_hold_green": bool(hold_green and intrabar_hold_green),
        "a_tier_retrace_fail_reason": fail_reason,
    }

def v5_apply_a_tier_retrace_execution(
    df: pd.DataFrame,
    signal_index: int,
    entry_index: int,
    config: dict,
) -> dict:
    """
    Apply optional A-tier retrace-entry execution logic.

    Default behaviour is unchanged when enable_a_tier_retrace_entry=False.

    If retrace is required, the current second-close signal does not enter
    immediately. The next candle must produce a controlled retrace/hold, and
    the trade then enters on the following candle open.
    """
    signal_row = df.loc[signal_index]
    side = str(signal_row.get("raw_signal_side", "none"))
    selected_module = str(signal_row.get("v4_selected_entry_module", "none"))

    default_result = {
        "use_trade": True,
        "entry_index": entry_index,
        "a_tier_entry_timing_type": signal_row.get(
            "a_tier_entry_timing_type",
            "SECOND_CLOSE",
        ),
        "a_tier_retrace_required": False,
        "a_tier_retrace_pass": False,
        "a_tier_retrace_points": np.nan,
        "a_tier_retrace_close_extension_from_green": np.nan,
        "a_tier_retrace_hold_green": False,
        "a_tier_retrace_fail_reason": "NOT_RETRACE_ENTRY",
        "skip_reason": None,
    }

    if selected_module != "v3_second_close_green_reclaim_rejection":
        return default_result

    if side not in {"long", "short"}:
        return default_result

    mode = v5_get_a_tier_retrace_entry_mode(config)

    if mode == "off":
        return default_result

    retrace_required = v5_a_tier_retrace_entry_required(signal_row, config)
    direct_allowed = v5_a_tier_direct_entry_allowed(signal_row, config)

    if direct_allowed and not retrace_required:
        return default_result

    wait_bars = int(config.get("a_tier_retrace_wait_bars", 1))

    if wait_bars != 1:
        raise ValueError(
            "Only a_tier_retrace_wait_bars=1 is currently supported."
        )

    retrace_index = signal_index + wait_bars

    if retrace_index not in df.index:
        return {
            **default_result,
            "use_trade": False,
            "a_tier_retrace_required": True,
            "a_tier_retrace_fail_reason": "BLOCKED_BY_A_TIER_RETRACE_NO_NEXT_BAR",
            "skip_reason": "BLOCKED_BY_A_TIER_RETRACE_NO_NEXT_BAR",
        }

    new_entry_index = retrace_index + 1

    if new_entry_index not in df.index:
        return {
            **default_result,
            "use_trade": False,
            "a_tier_retrace_required": True,
            "a_tier_retrace_fail_reason": "BLOCKED_BY_A_TIER_RETRACE_NO_ENTRY_BAR",
            "skip_reason": "BLOCKED_BY_A_TIER_RETRACE_NO_ENTRY_BAR",
        }

    retrace_row = df.loc[retrace_index]

    retrace_result = v5_a_tier_retrace_entry_passes(
        second_close_row=signal_row,
        retrace_row=retrace_row,
        side=side,
        config=config,
    )

    if not retrace_result["a_tier_retrace_pass"]:
        return {
            **default_result,
            "use_trade": False,
            "a_tier_entry_timing_type": "RETRACE_ENTRY",
            "a_tier_retrace_required": True,
            "a_tier_retrace_pass": False,
            "a_tier_retrace_points": retrace_result["a_tier_retrace_points"],
            "a_tier_retrace_close_extension_from_green": retrace_result[
                "a_tier_retrace_close_extension_from_green"
            ],
            "a_tier_retrace_hold_green": retrace_result["a_tier_retrace_hold_green"],
            "a_tier_retrace_fail_reason": retrace_result["a_tier_retrace_fail_reason"],
            "skip_reason": retrace_result["a_tier_retrace_fail_reason"],
        }

    return {
        **default_result,
        "use_trade": True,
        "entry_index": new_entry_index,
        "a_tier_entry_timing_type": "RETRACE_ENTRY",
        "a_tier_retrace_required": True,
        "a_tier_retrace_pass": True,
        "a_tier_retrace_points": retrace_result["a_tier_retrace_points"],
        "a_tier_retrace_close_extension_from_green": retrace_result[
            "a_tier_retrace_close_extension_from_green"
        ],
        "a_tier_retrace_hold_green": retrace_result["a_tier_retrace_hold_green"],
        "a_tier_retrace_fail_reason": "VALID_RETRACE",
        "skip_reason": None,
    }

def v5_signal_gate_fields(row: pd.Series, config: dict) -> dict:
    """
    Build V5 module/V2 gate fields for one signal row.
    """
    module_block_reason = v5_module_toggle_block_reason(row, config)
    v2_health_failure_reason = v5_v2_health_failure_reason(row, config)
    v2_block_reason = v5_v2_toggle_block_reason(row, config)
    v2_activation_state = v5_v2_trend_health_activation_state(row, config)
    red_shift_floor_block_reason = v5_entry_red_shift_floor_block_reason(row, config)

    module_pass = module_block_reason is None
    v2_pass = v2_block_reason is None
    v2_required = v5_safe_bool(row.get("v4_v2_trend_health_required", False))
    v2_active = bool(v2_activation_state["v2_trend_health_active"])

    if not bool(config.get("enable_v2_trend_health_filter", True)) or not v2_required:
        v2_filter_result = "NOT_APPLIED"
        v2_filter_tracked_only = False

    elif v2_active:
        v2_filter_result = "FAIL" if v2_health_failure_reason is not None else "PASS"
        v2_filter_tracked_only = False

    else:
        v2_filter_result = "TRACK_ONLY"
        v2_filter_tracked_only = bool(config.get("v2_track_when_not_active", True))

    red_shift_floor_pass = red_shift_floor_block_reason is None

    selected_module = str(row.get("v4_selected_entry_module", "none"))
    module_key = v5_selected_module_key(selected_module)

    if module_key == "v1_s_tier":
        entry_red_shift_min_points = v5_entry_red_shift_min_points(config, "s_tier")
        entry_red_shift_floor_applied = bool(
            config.get("s_tier_use_entry_red_shift_floor", False)
        )

    elif module_key == "v3_a_tier_second_close":
        entry_red_shift_min_points = v5_entry_red_shift_min_points(config, "a_tier")
        entry_red_shift_floor_applied = bool(
            config.get("a_tier_use_entry_red_shift_floor", False)
        )

    else:
        entry_red_shift_min_points = np.nan
        entry_red_shift_floor_applied = False

    return {
        "v5_module_toggle_pass": module_pass,
        "v5_module_block_reason": module_block_reason,
        "v5_v2_toggle_pass": v2_pass,
        "v5_v2_block_reason": v2_block_reason,
        "v5_entry_red_shift_floor_pass": red_shift_floor_pass,
        "v5_entry_red_shift_floor_block_reason": red_shift_floor_block_reason,
        "v5_entry_red_shift_floor_applied": entry_red_shift_floor_applied,
        "v5_entry_directional_red_shift_points": v5_entry_directional_red_shift_points(row),
        "v5_entry_red_shift_min_points": entry_red_shift_min_points,
        "v5_signal_gate_pass": module_pass and v2_pass and red_shift_floor_pass,
        "v2_trend_health_active": v2_activation_state["v2_trend_health_active"],
        "v2_activation_reason": v2_activation_state["v2_activation_reason"],
        "v2_router_required": v2_activation_state["v2_router_required"],
        "v2_force_after_time_active": v2_activation_state["v2_force_after_time_active"],
        "v2_filter_result": v2_filter_result,
        "v2_filter_tracked_only": v2_filter_tracked_only,
        "v2_health_failure_reason": v2_health_failure_reason,
    }


def v5_directional_v2_health_pass(row: pd.Series) -> bool:
    """
    Return the directional V2 trend-health pass/fail value for a signal row.
    """
    side = str(row.get("raw_signal_side", row.get("v1_signal_side", "none")))

    if side == "long":
        return v5_safe_bool(row.get("v2_long_trend_health_pass", False))

    if side == "short":
        return v5_safe_bool(row.get("v2_short_trend_health_pass", False))

    return False


def v5_v2_health_state_from_signal_row(row: pd.Series) -> str:
    """
    Build a compact V2 health-state label for tracking.
    """
    side = str(row.get("raw_signal_side", row.get("v1_signal_side", "none")))
    selected_module = str(row.get("v4_selected_entry_module", "not_available"))

    if side == "long":
        if selected_module == "v3_second_close_green_reclaim_rejection":
            return str(row.get("v2_long_reclaim_recovery_state", "UNKNOWN"))

        return str(row.get("v2_long_continuation_health_state", "UNKNOWN"))

    if side == "short":
        if selected_module == "v3_second_close_green_reclaim_rejection":
            return str(row.get("v2_short_reclaim_recovery_state", "UNKNOWN"))

        return str(row.get("v2_short_continuation_health_state", "UNKNOWN"))

    return "NO_SIGNAL"


def v5_red_shift_bucket_from_signal_row(row: pd.Series) -> str:
    """
    Return the best available directional red-shift bucket for a signal row.
    """
    side = str(row.get("raw_signal_side", row.get("v1_signal_side", "none")))

    if side == "long":
        bucket = str(row.get("v5_long_red_shift_bucket", "unknown"))
    elif side == "short":
        bucket = str(row.get("v5_short_red_shift_bucket", "unknown"))
    else:
        bucket = "none"

    if bucket not in {"none", "nan", "unknown", "not_available"}:
        return bucket

    v3_bucket = str(row.get("v3_directional_red_shift_bucket", "none"))

    if v3_bucket not in {"none", "nan", "unknown", "not_available"}:
        return v3_bucket

    directional_bucket = str(row.get("directional_red_shift_label", "unknown"))

    if directional_bucket not in {"nan", "none"}:
        return directional_bucket

    return "unknown"


def v5_setup_source_from_signal_row(row: pd.Series) -> str:
    """
    Map existing V4.5 signal/router fields into V5 setup-source labels.
    """
    raw_signal_name = str(row.get("raw_signal_name", "NO_SIGNAL"))
    final_signal_name = str(row.get("v1_signal_name", raw_signal_name))
    block_reason = str(row.get("v1_signal_block_reason", ""))
    selected_module = str(row.get("v4_selected_entry_module", "not_available"))
    router_decision = str(row.get("v4_model_route", "not_available"))

    if final_signal_name == "NO_SIGNAL":
        if raw_signal_name == "NO_SIGNAL":
            return "NO_SIGNAL"

        if block_reason == "DIRECTION_DISABLED":
            return "BLOCKED_BY_MANUAL_TOGGLE"

        if block_reason == "SESSION_CUTOFF":
            return "BLOCKED_BY_SESSION"
        
        if block_reason.startswith("BLOCKED_BY_"):
            return block_reason

        if block_reason == "STRATEGY_FILTER_BLOCKED":
            if (
                v5_safe_bool(row.get("v4_v2_trend_health_required", False))
                and not v5_directional_v2_health_pass(row)
            ):
                return "BLOCKED_BY_TREND_HEALTH"

            if selected_module == "none" or router_decision.endswith("no_trade"):
                return "BLOCKED_BY_ROUTER"

            return "BLOCKED_BY_STRATEGY_FILTER"

        return "BLOCKED_CANDIDATE"

    if selected_module == "v1_green_reclaim_rejection":
        if v5_safe_bool(row.get("v4_v2_trend_health_required", False)):
            return "V1_S_TIER_WITH_V2_FILTER"

        return "V1_S_TIER"

    if selected_module == "extreme_expansion":
        return "EXTREME_EXPANSION"
    
    if selected_module == "v3_second_close_green_reclaim_rejection":
        final_signal_name_text = str(
            row.get("v1_signal_name", row.get("raw_signal_name", "NO_SIGNAL"))
        ).upper()

        entry_timing_type = str(
            row.get("a_tier_entry_timing_type", "UNKNOWN")
        ).upper()

        if (
            entry_timing_type == "FIRST_CLOSE_FAST_RECLAIM"
            or "FAST_RECLAIM" in final_signal_name_text
        ):
            return "V5_A_TIER_FAST_RECLAIM"

        if (
            entry_timing_type == "DELAYED_PULLBACK_4TH_CANDLE"
            or "DELAYED_PULLBACK" in final_signal_name_text
        ):
            return "V5_A_TIER_DELAYED_PULLBACK"

        return "V3_A_TIER_SECOND_CLOSE"

    if "FAST_RECLAIM" in final_signal_name:
        return "V5_A_TIER_FAST_RECLAIM"

    if "DELAYED_PULLBACK" in final_signal_name:
        return "V5_A_TIER_DELAYED_PULLBACK"

    if "V3" in final_signal_name:
        return "V3_A_TIER_SECOND_CLOSE"

    if "GREEN_RECLAIM" in final_signal_name or "GREEN_REJECTION" in final_signal_name:
        return "V1_S_TIER"

    return "UNKNOWN_SETUP_SOURCE"


def v5_setup_family_from_source(setup_source: str) -> str:
    """
    Group setup-source labels into broader setup families.
    """
    setup_source = str(setup_source)

    if setup_source.startswith("V1_S_TIER_WITH_V2"):
        return "V1_WITH_V2_FILTER"

    if setup_source.startswith("V1_S_TIER"):
        return "V1"
    
    if setup_source.startswith("V5_A_TIER"):
        return "V5_A_TIER"

    if setup_source.startswith("V3_A_TIER"):
        return "V3"

    if setup_source.startswith("EXTREME"):
        return "EXTREME_EXPANSION"

    if setup_source.startswith("BLOCKED"):
        return "BLOCKED"

    if setup_source == "NO_SIGNAL":
        return "NONE"

    return "UNKNOWN"


def v5_signal_tracking_fields(row: pd.Series) -> dict:
    """
    Build setup, regime, router, V2, and red-shift tracking fields for one row.
    """
    setup_source = v5_setup_source_from_signal_row(row)

    return {
        "setup_source": setup_source,
        "setup_family": v5_setup_family_from_source(setup_source),
        "engine_mode": row.get("engine_mode", CONFIG.get("engine_mode", "intelligent")),
        "regime_20m": row.get("v5_regime_20m", row.get("v4_preliminary_regime_label", "not_available"),
                              ),
        "router_decision": row.get("v4_model_route", "not_available"),
        "v2_health_state": v5_v2_health_state_from_signal_row(row),
        "v2_filter_applied": v5_safe_bool(
            row.get("v4_v2_trend_health_required", False)
        ),
        "v2_trend_health_active": row.get("v2_trend_health_active", False),
        "v2_activation_reason": row.get("v2_activation_reason", "TRACK_ONLY"),
        "v2_router_required": row.get("v2_router_required", False),
        "v2_force_after_time_active": row.get("v2_force_after_time_active", False),
        "v2_filter_result": row.get("v2_filter_result", "NOT_APPLIED"),
        "v2_filter_tracked_only": row.get("v2_filter_tracked_only", False),
        "v2_health_failure_reason": row.get("v2_health_failure_reason", None),

        "entry_red_shift_floor_applied": row.get("v5_entry_red_shift_floor_applied", False),
        "entry_directional_red_shift_points": row.get("v5_entry_directional_red_shift_points", np.nan),
        "entry_red_shift_min_points": row.get("v5_entry_red_shift_min_points", np.nan),
        "red_shift_bucket": v5_red_shift_bucket_from_signal_row(row),

        "s_tier_current_green_touch": row.get("s_tier_current_green_touch", False),
        "s_tier_locked_green_touch": row.get("s_tier_locked_green_touch", False),
        "s_tier_band_shift_touch": row.get("s_tier_band_shift_touch", False),
        "s_tier_touch_type": row.get("s_tier_touch_type", "NOT_S_TIER"),
        "s_tier_clean_state_recent_failure_seen": row.get("s_tier_clean_state_recent_failure_seen", False),
        "s_tier_clean_state_clean_closes_after_failure": row.get("s_tier_clean_state_clean_closes_after_failure", np.nan),
        "s_tier_clean_state_pass": row.get("s_tier_clean_state_pass", False),
        "s_tier_clean_state_blocked": row.get("s_tier_clean_state_blocked", False),
        "s_tier_clean_state_block_reason": row.get("s_tier_clean_state_block_reason", "NOT_BLOCKED"),

        "a_tier_is_candidate": row.get("a_tier_is_candidate", False),
        "a_tier_side": row.get("a_tier_side", "UNKNOWN"),
        "a_tier_entry_timing_type": row.get("a_tier_entry_timing_type", "UNKNOWN"),
        
        "a_tier_fast_reclaim_away_bars": row.get("a_tier_fast_reclaim_away_bars", np.nan),
        "a_tier_fast_reclaim_max_depth_points": row.get("a_tier_fast_reclaim_max_depth_points", np.nan),
        "a_tier_fast_reclaim_body_points": row.get("a_tier_fast_reclaim_body_points", np.nan),
        "a_tier_fast_reclaim_body_ratio": row.get("a_tier_fast_reclaim_body_ratio", np.nan),
        "a_tier_fast_reclaim_close_position": row.get("a_tier_fast_reclaim_close_position", np.nan),
        "a_tier_fast_reclaim_passed": row.get("a_tier_fast_reclaim_passed", False),

        "a_tier_delayed_pullback_candidate": row.get("a_tier_delayed_pullback_candidate", False),
        "a_tier_delayed_pullback_passed": row.get("a_tier_delayed_pullback_passed", False),
        "a_tier_delayed_pullback_entry_bar": row.get("a_tier_delayed_pullback_entry_bar", np.nan),
        "a_tier_delayed_pullback_hold_bars": row.get("a_tier_delayed_pullback_hold_bars", np.nan),
        "a_tier_delayed_pullback_points": row.get("a_tier_delayed_pullback_points", np.nan),
        "a_tier_delayed_pullback_held_green": row.get("a_tier_delayed_pullback_held_green", False),
        "a_tier_delayed_pullback_block_reason": row.get("a_tier_delayed_pullback_block_reason", "UNKNOWN"),
        "a_tier_delayed_pullback_strength_filter_pass": row.get("a_tier_delayed_pullback_strength_filter_pass", False),
        "a_tier_delayed_pullback_strength_filter_block_reason": row.get("a_tier_delayed_pullback_strength_filter_block_reason", "UNKNOWN"),
        "a_tier_delayed_pullback_red_shift_bucket": row.get("a_tier_delayed_pullback_red_shift_bucket", "unknown"),
        "a_tier_delayed_pullback_v2_activation_reason": row.get("a_tier_delayed_pullback_v2_activation_reason", "UNKNOWN"),
        "a_tier_delayed_pullback_v2_pass_used": row.get("a_tier_delayed_pullback_v2_pass_used", False),

        "a_tier_candles_below_green_count": row.get("a_tier_candles_below_green_count", np.nan),
        "a_tier_max_depth_below_green_points": row.get("a_tier_max_depth_below_green_points", np.nan),
        "a_tier_reclaim_body_points": row.get("a_tier_reclaim_body_points", np.nan),
        "a_tier_reclaim_range_points": row.get("a_tier_reclaim_range_points", np.nan),
        "a_tier_reclaim_body_ratio": row.get("a_tier_reclaim_body_ratio", np.nan),
        "a_tier_reclaim_close_position": row.get("a_tier_reclaim_close_position", np.nan),
        "a_tier_second_close_body_points": row.get("a_tier_second_close_body_points", np.nan),
        "a_tier_second_close_range_points": row.get("a_tier_second_close_range_points", np.nan),
        "a_tier_second_close_body_ratio": row.get("a_tier_second_close_body_ratio", np.nan),
        "a_tier_second_close_position": row.get("a_tier_second_close_position", np.nan),
        "a_tier_second_close_extension_from_green": row.get("a_tier_second_close_extension_from_green", np.nan),
        "a_tier_entry_directional_red_shift_points": row.get("a_tier_entry_directional_red_shift_points", np.nan),
        "a_tier_entry_red_shift_bucket": row.get("a_tier_entry_red_shift_bucket", "UNKNOWN"),
        "a_tier_depth_bucket": row.get("a_tier_depth_bucket", "unknown"),
        "a_tier_extension_bucket": row.get("a_tier_extension_bucket", "unknown"),
        "a_tier_reclaim_body_bucket": row.get("a_tier_reclaim_body_bucket", "unknown"),
        "a_tier_second_close_body_bucket": row.get("a_tier_second_close_body_bucket", "unknown"),
        "a_tier_reclaim_quality_bucket": row.get("a_tier_reclaim_quality_bucket", "unknown"),
        "a_tier_retrace_required": row.get("a_tier_retrace_required", False),
        "a_tier_retrace_pass": row.get("a_tier_retrace_pass", False),
        "a_tier_retrace_points": row.get("a_tier_retrace_points", np.nan),
        "a_tier_retrace_close_extension_from_green": row.get(
            "a_tier_retrace_close_extension_from_green",
            np.nan,
        ),
        "a_tier_retrace_hold_green": row.get("a_tier_retrace_hold_green", False),
        "a_tier_retrace_fail_reason": row.get(
            "a_tier_retrace_fail_reason",
            "NOT_RETRACE_ENTRY",
        ),
    }


def v5_exit_category_from_result(result: str, result_reason: str) -> str:
    """
    Standardise exit categories for later V5 exit-manager reporting.
    """
    result = str(result)
    result_reason = str(result_reason)

    if result_reason == "FORCE_CLOSED_AT_DATASET_END":
        return "TIME_EXIT"

    if result == "TP":
        return "TP"

    if result == "SL":
        return "SL_FULL"

    if result == "BE":
        return "BE_TRAIL"

    if result in {"CLOSE_PROFIT", "CLOSE_LOSS", "CLOSE_FLAT"}:
        return "TIME_EXIT"

    return "UNKNOWN_EXIT_CATEGORY"


def v5_exit_reason_from_result(result: str, result_reason: str) -> str:
    """
    Standardise detailed exit reasons.
    """
    exit_category = v5_exit_category_from_result(result, result_reason)

    if exit_category == "TP":
        return "TP_2R"

    if exit_category == "SL_FULL":
        return "SL_FULL"

    if exit_category == "BE_TRAIL":
        return "BE_TRAIL"

    if exit_category == "TIME_EXIT":
        return "TIME_EXIT"

    return str(result_reason)


def v5_calculate_trade_excursions(
    df: pd.DataFrame,
    entry_index: int,
    exit_index: int,
    entry_price: float,
    side: str,
) -> tuple[float, float]:
    """
    Calculate max favourable and adverse movement in points during a trade.
    """
    if entry_index is None or exit_index is None:
        return np.nan, np.nan

    trade_slice = df.loc[entry_index:exit_index].copy()

    if trade_slice.empty:
        return np.nan, np.nan

    if side == "long":
        max_favourable_points = float((trade_slice["high"] - entry_price).max())
        max_adverse_points = float((entry_price - trade_slice["low"]).max())

    elif side == "short":
        max_favourable_points = float((entry_price - trade_slice["low"]).max())
        max_adverse_points = float((trade_slice["high"] - entry_price).max())

    else:
        return np.nan, np.nan

    return max(0.0, max_favourable_points), max(0.0, max_adverse_points)


def v5_build_candidate_tracker_fields(
    row: pd.Series,
    config: dict,
    entry_time=None,
    blocked_reason: str | None = None,
) -> dict:
    """
    Build standard tracker fields for skipped or blocked candidates.
    """
    if entry_time is None:
        entry_time = row.get("datetime", None)

    fields = v5_signal_tracking_fields(row)
    fields.update(v5_entry_time_context_fields(entry_time, config))

    if blocked_reason == "BLOCKED_BY_SESSION":
        fields["setup_source"] = "BLOCKED_BY_SESSION"
        fields["setup_family"] = "BLOCKED"
    
    fields.update(
        {
            "exit_mode": "NO_TRADE",
            "exit_reason": "NO_TRADE",
            "exit_category": "NO_TRADE",
            "tp_points": float(config["tp_points"]),
            "sl_points": float(config["sl_points"]),
            "runner_enabled": False,
            "trail_activated": False,
            "trail_lock_level": "NONE",
            "max_favourable_points": np.nan,
            "max_adverse_points": np.nan,
            "blocked_reason": blocked_reason,
        }
    )

    return fields


def v5_build_trade_tracker_fields(
    df: pd.DataFrame,
    signal_row: pd.Series,
    entry_index: int,
    exit_index: int,
    entry_price: float,
    side: str,
    result: str,
    result_reason: str,
    config: dict,
    exit_plan: dict | None = None,
) -> dict:
    """
    Build standard V5 tracker fields for executed trades.
    """
    entry_time = df.loc[entry_index, "datetime"] if entry_index in df.index else None

    max_favourable_points, max_adverse_points = v5_calculate_trade_excursions(
        df=df,
        entry_index=entry_index,
        exit_index=exit_index,
        entry_price=entry_price,
        side=side,
    )

    if exit_plan is None:
        exit_plan = v5_build_exit_plan(
            entry_price=entry_price,
            side=side,
            signal_row=signal_row,
            config=config,
        )

    exit_category = v5_exit_category_from_result(result, result_reason)
    trade_trail_lock_level = v5_trail_lock_level_from_result_reason(result_reason)

    fields = v5_signal_tracking_fields(signal_row)
    fields.update(v5_entry_time_context_fields(entry_time, config))

    fields.update(
        {
            "exit_mode": exit_plan["exit_mode"],
            "exit_reason": v5_exit_reason_from_result(result, result_reason),
            "exit_category": exit_category,
            "tp_points": float(exit_plan["tp_points"]),
            "sl_points": float(exit_plan["sl_points"]),
            "runner_enabled": bool(exit_plan.get("runner_enabled", False)),
            "trail_activated": bool(
                exit_category in {"BE_TRAIL", "TRAILED_SL_PROFIT"}
                or trade_trail_lock_level != "NONE"
            ),
            "trail_lock_level": trade_trail_lock_level,
            "max_favourable_points": max_favourable_points,
            "max_adverse_points": max_adverse_points,
            "blocked_reason": None,
        }
    )

    return fields

In [ ]:
def add_v5_s_tier_touch_diagnostics(out: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Add Dynamic S-tier green-touch diagnostics and optional S-tier clean-state filter.

    Dynamic touch diagnostics are diagnostic-only by default:
    - dynamic_s_tier_touch_mode = tag_only

    Clean-state filter is optional and default-off:
    - enable_s_tier_clean_state_filter = False
    """
    diagnostics_enabled = bool(
        config.get("enable_dynamic_s_tier_touch_diagnostics", True)
    )

    touch_mode = str(
        config.get("dynamic_s_tier_touch_mode", "tag_only")
    ).lower()

    enable_clean_state_filter = bool(
        config.get("enable_s_tier_clean_state_filter", False)
    )

    clean_state_lookback_bars = max(
        1,
        int(config.get("s_tier_green_failure_lookback_bars", 8)),
    )

    required_clean_closes_after_failure = max(
        1,
        int(config.get("s_tier_required_clean_closes_after_failure", 3)),
    )

    raw_signal_name = (
        out.get("raw_signal_name", pd.Series("NO_SIGNAL", index=out.index))
        .fillna("NO_SIGNAL")
        .astype(str)
    )

    raw_signal_side = (
        out.get("raw_signal_side", pd.Series("none", index=out.index))
        .fillna("none")
        .astype(str)
    )

    selected_module = (
        out.get("v4_selected_entry_module", pd.Series("none", index=out.index))
        .fillna("none")
        .astype(str)
    )

    is_s_tier_raw_candidate = (
        raw_signal_name.ne("NO_SIGNAL")
        & selected_module.eq("v1_green_reclaim_rejection")
        & raw_signal_side.isin(["long", "short"])
    )

    # Dynamic S-tier diagnostic:
    # A band-assisted touch means the current candle appears to touch green
    # only because the green band shifted into the candle by the final
    # closed-candle calculation. We tag this separately before deciding
    # whether it should ever be blocked.
    long_current_touch = (out["low"] <= out["upper_green"]).fillna(False)
    long_locked_touch = (out["low"] <= out["upper_green"].shift(1)).fillna(False)

    short_current_touch = (out["high"] >= out["lower_green"]).fillna(False)
    short_locked_touch = (out["high"] >= out["lower_green"].shift(1)).fillna(False)

    directional_current_touch = pd.Series(False, index=out.index)
    directional_locked_touch = pd.Series(False, index=out.index)

    directional_current_touch.loc[raw_signal_side.eq("long")] = long_current_touch.loc[
        raw_signal_side.eq("long")
    ]

    directional_current_touch.loc[raw_signal_side.eq("short")] = short_current_touch.loc[
        raw_signal_side.eq("short")
    ]

    directional_locked_touch.loc[raw_signal_side.eq("long")] = long_locked_touch.loc[
        raw_signal_side.eq("long")
    ]

    directional_locked_touch.loc[raw_signal_side.eq("short")] = short_locked_touch.loc[
        raw_signal_side.eq("short")
    ]

    band_shift_touch = (
        diagnostics_enabled
        & is_s_tier_raw_candidate
        & directional_current_touch
        & ~directional_locked_touch
    )

    s_tier_touch_type = pd.Series("NOT_S_TIER", index=out.index, dtype="object")

    s_tier_touch_type.loc[
        diagnostics_enabled
        & is_s_tier_raw_candidate
        & ~band_shift_touch
    ] = "S_TIER_REAL_GREEN_TOUCH"

    s_tier_touch_type.loc[band_shift_touch] = "S_TIER_BAND_SHIFT_TOUCH"

    if not diagnostics_enabled:
        s_tier_touch_type.loc[
            is_s_tier_raw_candidate
        ] = "S_TIER_TOUCH_DIAGNOSTICS_DISABLED"

    # ------------------------------------------------------------------
    # Optional S-tier clean-state filter
    # ------------------------------------------------------------------
    # Uses previous candles only. After a close beyond green, S-tier is
    # considered unclean until enough consecutive closes rebuild structure
    # on the correct side of green.
    long_green_failure = (out["close"] < out["upper_green"]).fillna(False)
    short_green_failure = (out["close"] > out["lower_green"]).fillna(False)

    long_clean_close = (out["close"] > out["upper_green"]).fillna(False)
    short_clean_close = (out["close"] < out["lower_green"]).fillna(False)

    long_recent_failure_seen = (
        long_green_failure
        .shift(1)
        .fillna(False)
        .rolling(clean_state_lookback_bars, min_periods=1)
        .max()
        .fillna(False)
        .astype(bool)
    )

    short_recent_failure_seen = (
        short_green_failure
        .shift(1)
        .fillna(False)
        .rolling(clean_state_lookback_bars, min_periods=1)
        .max()
        .fillna(False)
        .astype(bool)
    )

    long_clean_closes_after_failure = (
        consecutive_true_count(long_clean_close)
        .shift(1)
        .fillna(0)
        .astype(int)
    )

    short_clean_closes_after_failure = (
        consecutive_true_count(short_clean_close)
        .shift(1)
        .fillna(0)
        .astype(int)
    )

    directional_clean_state_recent_failure_seen = pd.Series(False, index=out.index)
    directional_clean_closes_after_failure = pd.Series(0, index=out.index, dtype="int64")

    directional_clean_state_recent_failure_seen.loc[raw_signal_side.eq("long")] = (
        long_recent_failure_seen.loc[raw_signal_side.eq("long")]
    )

    directional_clean_state_recent_failure_seen.loc[raw_signal_side.eq("short")] = (
        short_recent_failure_seen.loc[raw_signal_side.eq("short")]
    )

    directional_clean_closes_after_failure.loc[raw_signal_side.eq("long")] = (
        long_clean_closes_after_failure.loc[raw_signal_side.eq("long")]
    )

    directional_clean_closes_after_failure.loc[raw_signal_side.eq("short")] = (
        short_clean_closes_after_failure.loc[raw_signal_side.eq("short")]
    )

    clean_state_pass = (
        ~directional_clean_state_recent_failure_seen
        | (
            directional_clean_closes_after_failure
            >= required_clean_closes_after_failure
        )
    )

    clean_state_blocked = (
        enable_clean_state_filter
        & is_s_tier_raw_candidate
        & ~clean_state_pass
    )

    out["s_tier_current_green_touch"] = np.where(
        is_s_tier_raw_candidate,
        directional_current_touch,
        False,
    )

    out["s_tier_locked_green_touch"] = np.where(
        is_s_tier_raw_candidate,
        directional_locked_touch,
        False,
    )

    out["s_tier_band_shift_touch"] = np.where(
        is_s_tier_raw_candidate,
        band_shift_touch,
        False,
    )

    out["s_tier_touch_type"] = s_tier_touch_type

    out["s_tier_clean_state_recent_failure_seen"] = np.where(
        is_s_tier_raw_candidate,
        directional_clean_state_recent_failure_seen,
        False,
    )

    out["s_tier_clean_state_clean_closes_after_failure"] = np.where(
        is_s_tier_raw_candidate,
        directional_clean_closes_after_failure,
        np.nan,
    )

    out["s_tier_clean_state_pass"] = np.where(
        is_s_tier_raw_candidate,
        clean_state_pass,
        False,
    )

    out["s_tier_clean_state_blocked"] = clean_state_blocked

    out["s_tier_clean_state_block_reason"] = np.where(
        clean_state_blocked,
        "BLOCKED_S_TIER_NOT_CLEAN_AFTER_GREEN_FAILURE",
        "NOT_BLOCKED",
    )

    out["dynamic_s_tier_touch_mode"] = touch_mode

    return out

def add_v1_green_signals(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Add automated green reclaim / rejection signals for the selected strategy version.

    The active V4 layer can route between V1-style green reclaim/rejection entries
    and V3 second-close green reclaim/rejection entries.

    The final signal layer then applies:
    - long/short direction controls
    - selected strategy filter
    - no-new-trades-after time filter
    """
    out = df.copy()

    strategy_filter = config.get("strategy_filter", "baseline")
    use_v3_second_close = strategy_filter == "v3_second_close_green_reclaim_rejection"
    use_v4_dynamic_router = strategy_filter == "v4_dynamic_regime_selector"

    engine_mode = v5_get_engine_mode(config)
    use_intelligent_router = use_v4_dynamic_router and v5_use_intelligent_router(config)
    use_manual_module_routing = use_v4_dynamic_router and not use_intelligent_router

    min_body_ratio = config["min_body_ratio"]
    min_close_through_green = config["min_close_through_green"]
    max_extension_from_green = config["max_extension_from_green"]

    # ------------------------------------------------------------------
    # Shared green-band distance features
    # ------------------------------------------------------------------
    out["long_touched_upper_green"] = out["low"] <= out["upper_green"]
    out["long_closed_above_upper_green"] = out["close"] > out["upper_green"]

    out["long_close_through_green_points"] = out["close"] - out["upper_green"]
    out["long_extension_from_green_points"] = out["close"] - out["upper_green"]

    out["long_close_through_green_valid"] = (
        out["long_close_through_green_points"] >= min_close_through_green
    )

    out["long_extension_valid"] = (
        out["long_extension_from_green_points"] <= max_extension_from_green
    )

    out["v3_long_a_tier_extension_valid"] = (
        out["long_extension_from_green_points"]
        <= config["v3_a_tier_max_extension_from_green"]
    )

    out["long_body_valid"] = out["body_ratio"] >= min_body_ratio
    out["long_not_orange_chase"] = out["close"] < out["upper_orange"]

    out["short_touched_lower_green"] = out["high"] >= out["lower_green"]
    out["short_closed_below_lower_green"] = out["close"] < out["lower_green"]

    out["short_close_through_green_points"] = out["lower_green"] - out["close"]
    out["short_extension_from_green_points"] = out["lower_green"] - out["close"]

    out["short_close_through_green_valid"] = (
        out["short_close_through_green_points"] >= min_close_through_green
    )

    out["short_extension_valid"] = (
        out["short_extension_from_green_points"] <= max_extension_from_green
    )

    out["v3_short_a_tier_extension_valid"] = (
        out["short_extension_from_green_points"]
        <= config["v3_a_tier_max_extension_from_green"]
    )

    out["short_body_valid"] = out["body_ratio"] >= min_body_ratio
    out["short_not_orange_chase"] = out["close"] > out["lower_orange"]

    out["v3_long_v1_execution_quality_pass"] = (
        out["long_close_through_green_valid"]
        & out["v3_long_a_tier_extension_valid"]
        & out["long_body_valid"]
        & out["long_not_orange_chase"]
        & ~out["possible_chop"]
    )

    out["v3_short_v1_execution_quality_pass"] = (
        out["short_close_through_green_valid"]
        & out["v3_short_a_tier_extension_valid"]
        & out["short_body_valid"]
        & out["short_not_orange_chase"]
        & ~out["possible_chop"]
    )

    out = add_v5_a_tier_fast_reclaim_fields(out, config)
    out = add_v5_a_tier_delayed_pullback_fields(out, config)
    
    # ------------------------------------------------------------------
    # Raw signal structure
    # ------------------------------------------------------------------
    if use_v4_dynamic_router:
        required_v4_columns = [
            "v4_preliminary_regime_label",
            "v3_long_second_close_timing_pass",
            "v3_short_second_close_timing_pass",
            "v3_long_vwap_acceptance_pass",
            "v3_short_vwap_acceptance_pass",
            "v3_long_second_close_confirmation",
            "v3_short_second_close_confirmation",
            "v3_long_directional_red_shift_pass",
            "v3_short_directional_red_shift_pass",
            "v3_long_trend_dead",
            "v3_short_trend_dead",
        ]

        missing_v4_columns = [
            column for column in required_v4_columns if column not in out.columns
        ]

        if missing_v4_columns:
            raise KeyError(
                "Missing V4 routing columns: "
                + ", ".join(missing_v4_columns)
                + ". Run add_automation_features before signal generation."
            )

        router_regime = out.get(
            "v5_regime_20m",
            out["v4_preliminary_regime_label"],
        ).astype(str)

        calm_regime = router_regime == "calm_trend"
        volatile_regime = router_regime == "volatile_trend"
        chop_regime = router_regime == "chop"
        extreme_expansion_regime = router_regime.isin(
            ["extreme_expansion", "very_extreme_expansion"]
        )
        abnormal_news_regime = router_regime == "abnormal_news"

        calm_uses_v1 = (
            config.get("v4_route_calm_trend_to", "v1_green_reclaim_rejection")
            == "v1_green_reclaim_rejection"
        )

        volatile_uses_v3 = (
            config.get(
                "v4_route_volatile_trend_to",
                "v3_second_close_green_reclaim_rejection",
            )
            == "v3_second_close_green_reclaim_rejection"
        )

        chop_uses_no_trade = config.get("v4_route_chop_to", "no_trade") == "no_trade"

        extreme_entries_enabled = bool(
            config.get("enable_extreme_expansion_entries", False)
        )
        abnormal_news_entries_enabled = bool(
            config.get("enable_abnormal_news_entries", False)
        )

        extreme_expansion_uses_module = (
            config.get("v5_route_extreme_expansion_to", "extreme_expansion")
            == "extreme_expansion"
        )

        if use_manual_module_routing:
            out["v4_calm_trend_v1_route_active"] = pd.Series(True, index=out.index)
            out["v4_volatile_trend_v3_route_active"] = pd.Series(True, index=out.index)
            out["v4_chop_no_trade_route_active"] = pd.Series(False, index=out.index)

            out["v5_extreme_expansion_route_active"] = (
                extreme_entries_enabled
                & extreme_expansion_uses_module
                & out["v5_extreme_expansion_context"].fillna(False).astype(bool)
                & ~abnormal_news_regime
            )

            out["v5_abnormal_news_route_active"] = (
                extreme_entries_enabled
                & abnormal_news_entries_enabled
                & extreme_expansion_uses_module
                & abnormal_news_regime
            )

            out["v4_extreme_news_no_trade_route_active"] = pd.Series(False, index=out.index)
            out["v4_model_route"] = "manual_module_selection"

        else:
            out["v4_calm_trend_v1_route_active"] = calm_regime & calm_uses_v1
            out["v4_volatile_trend_v3_route_active"] = volatile_regime & volatile_uses_v3
            out["v4_chop_no_trade_route_active"] = chop_regime & chop_uses_no_trade

            out["v5_extreme_expansion_route_active"] = (
                extreme_expansion_regime
                & extreme_entries_enabled
                & extreme_expansion_uses_module
            )

            out["v5_abnormal_news_route_active"] = (
                abnormal_news_regime
                & extreme_entries_enabled
                & abnormal_news_entries_enabled
                & extreme_expansion_uses_module
            )

            out["v4_extreme_news_no_trade_route_active"] = (
                (extreme_expansion_regime & ~out["v5_extreme_expansion_route_active"])
                | (abnormal_news_regime & ~out["v5_abnormal_news_route_active"])
            )

            out["v4_model_route"] = np.select(
                [
                    out["v4_calm_trend_v1_route_active"],
                    out["v4_volatile_trend_v3_route_active"],
                    out["v5_extreme_expansion_route_active"],
                    out["v5_abnormal_news_route_active"],
                    out["v4_chop_no_trade_route_active"],
                    out["v4_extreme_news_no_trade_route_active"],
                ],
                [
                    "calm_trend_v1",
                    "volatile_trend_v3",
                    "extreme_expansion",
                    "abnormal_news_enabled_extreme_expansion",
                    "chop_no_trade",
                    "extreme_or_abnormal_no_trade",
                ],
                default="no_trade",
            )

        v1_long_candidate = (
            out["accepted_above_vwap"]
            & (out["close"] > out["vwap"])
            & out["long_touched_upper_green"]
            & out["long_closed_above_upper_green"]
            & out["long_close_through_green_valid"]
            & out["long_extension_valid"]
            & out["long_body_valid"]
            & out["long_not_orange_chase"]
            & ~out["possible_chop"]
        )

        v1_short_candidate = (
            out["accepted_below_vwap"]
            & (out["close"] < out["vwap"])
            & out["short_touched_lower_green"]
            & out["short_closed_below_lower_green"]
            & out["short_close_through_green_valid"]
            & out["short_extension_valid"]
            & out["short_body_valid"]
            & out["short_not_orange_chase"]
            & ~out["possible_chop"]
        )

        v3_long_second_close_candidate = (
            out["v3_long_second_close_timing_pass"].fillna(False).astype(bool)
        )

        v3_short_second_close_candidate = (
            out["v3_short_second_close_timing_pass"].fillna(False).astype(bool)
        )

        v3_long_fast_reclaim_candidate = (
            out["a_tier_fast_reclaim_long_passed"].fillna(False).astype(bool)
        )

        v3_short_fast_reclaim_candidate = (
            out["a_tier_fast_reclaim_short_passed"].fillna(False).astype(bool)
        )

        v3_long_delayed_pullback_candidate = (
            out["a_tier_delayed_pullback_long_passed"].fillna(False).astype(bool)
        )

        v3_short_delayed_pullback_candidate = (
            out["a_tier_delayed_pullback_short_passed"].fillna(False).astype(bool)
        )

        v3_long_candidate = (
            v3_long_second_close_candidate
            | v3_long_fast_reclaim_candidate
            | v3_long_delayed_pullback_candidate
        )

        v3_short_candidate = (
            v3_short_second_close_candidate
            | v3_short_fast_reclaim_candidate
            | v3_short_delayed_pullback_candidate
        )

        out["v4_long_v1_calm_candidate"] = (
            out["v4_calm_trend_v1_route_active"] & v1_long_candidate
        )

        out["v4_short_v1_calm_candidate"] = (
            out["v4_calm_trend_v1_route_active"] & v1_short_candidate
        )

        out["v4_long_v3_volatile_candidate"] = (
            out["v4_volatile_trend_v3_route_active"] & v3_long_candidate
        )

        out["v4_short_v3_volatile_candidate"] = (
            out["v4_volatile_trend_v3_route_active"] & v3_short_candidate
        )

        out["v4_long_v3_fast_reclaim_candidate"] = (
            out["v4_volatile_trend_v3_route_active"]
            & v3_long_fast_reclaim_candidate
        )

        out["v4_short_v3_fast_reclaim_candidate"] = (
            out["v4_volatile_trend_v3_route_active"]
            & v3_short_fast_reclaim_candidate
        )

        out["v4_long_v3_delayed_pullback_candidate"] = (
            out["v4_volatile_trend_v3_route_active"]
            & v3_long_delayed_pullback_candidate
        )

        out["v4_short_v3_delayed_pullback_candidate"] = (
            out["v4_volatile_trend_v3_route_active"]
            & v3_short_delayed_pullback_candidate
        )

        out["v5_long_extreme_expansion_candidate"] = (
            (
                out["v5_extreme_expansion_route_active"]
                | out["v5_abnormal_news_route_active"]
            )
            & v3_long_candidate
            & out["v5_long_extreme_expansion_context"].fillna(False).astype(bool)
        )

        out["v5_short_extreme_expansion_candidate"] = (
            (
                out["v5_extreme_expansion_route_active"]
                | out["v5_abnormal_news_route_active"]
            )
            & v3_short_candidate
            & out["v5_short_extreme_expansion_context"].fillna(False).astype(bool)
        )

        out["raw_long_green_reclaim"] = (
            out["v4_long_v1_calm_candidate"]
            | out["v4_long_v3_volatile_candidate"]
            | out["v5_long_extreme_expansion_candidate"]
        )

        out["raw_short_green_rejection"] = (
            out["v4_short_v1_calm_candidate"]
            | out["v4_short_v3_volatile_candidate"]
            | out["v5_short_extreme_expansion_candidate"]
        )

        out["v4_selected_entry_module"] = np.select(
            [
                out["v4_long_v1_calm_candidate"] | out["v4_short_v1_calm_candidate"],
                out["v4_long_v3_volatile_candidate"] | out["v4_short_v3_volatile_candidate"],
                out["v5_long_extreme_expansion_candidate"] | out["v5_short_extreme_expansion_candidate"],
            ],
            [
                "v1_green_reclaim_rejection",
                "v3_second_close_green_reclaim_rejection",
                "extreme_expansion",
            ],
            default="none",
        )

        long_signal_name = np.select(
            [
                out["v4_long_v1_calm_candidate"],
                out["v4_long_v3_fast_reclaim_candidate"],
                out["v4_long_v3_delayed_pullback_candidate"],
                out["v4_long_v3_volatile_candidate"],
                out["v5_long_extreme_expansion_candidate"],
            ],
            [
                "LONG_V5_CALM_TREND_V1_GREEN_RECLAIM",
                "LONG_V5_A_TIER_FAST_RECLAIM",
                "LONG_V5_A_TIER_DELAYED_PULLBACK",
                "LONG_V5_VOLATILE_TREND_V3_SECOND_CLOSE_GREEN_RECLAIM",
                "LONG_V5_EXTREME_EXPANSION",
            ],
            default="LONG_V5_DYNAMIC_GREEN_RECLAIM",
        )

        short_signal_name = np.select(
            [
                out["v4_short_v1_calm_candidate"],
                out["v4_short_v3_fast_reclaim_candidate"],
                out["v4_short_v3_delayed_pullback_candidate"],
                out["v4_short_v3_volatile_candidate"],
                out["v5_short_extreme_expansion_candidate"],
            ],
            [
                "SHORT_V5_CALM_TREND_V1_GREEN_REJECTION",
                "SHORT_V5_A_TIER_FAST_RECLAIM",
                "SHORT_V5_A_TIER_DELAYED_PULLBACK",
                "SHORT_V5_VOLATILE_TREND_V3_SECOND_CLOSE_GREEN_REJECTION",
                "SHORT_V5_EXTREME_EXPANSION",
            ],
            default="SHORT_V5_DYNAMIC_GREEN_REJECTION",
        )

    elif use_v3_second_close:
        required_v3_columns = [
            "v3_long_second_close_timing_pass",
            "v3_short_second_close_timing_pass",
            "v3_long_vwap_acceptance_pass",
            "v3_short_vwap_acceptance_pass",
            "v3_long_second_close_confirmation",
            "v3_short_second_close_confirmation",
            "v3_long_directional_red_shift_pass",
            "v3_short_directional_red_shift_pass",
            "v3_long_trend_dead",
            "v3_short_trend_dead",
        ]

        missing_v3_columns = [
            column for column in required_v3_columns if column not in out.columns
        ]

        if missing_v3_columns:
            raise KeyError(
                "Missing V3 second-close columns: "
                + ", ".join(missing_v3_columns)
                + ". Run add_automation_features before signal generation."
            )

        out["raw_long_green_reclaim"] = (
            out["v3_long_second_close_timing_pass"].fillna(False).astype(bool)
        )

        out["raw_short_green_rejection"] = (
            out["v3_short_second_close_timing_pass"].fillna(False).astype(bool)
        )

        out["v4_model_route"] = "not_used"
        out["v4_selected_entry_module"] = "v3_second_close_green_reclaim_rejection"

        long_signal_name = "LONG_V3_SECOND_CLOSE_GREEN_RECLAIM"
        short_signal_name = "SHORT_V3_SECOND_CLOSE_GREEN_REJECTION"

    else:
        out["raw_long_green_reclaim"] = (
            out["accepted_above_vwap"]
            & (out["close"] > out["vwap"])
            & out["long_touched_upper_green"]
            & out["long_closed_above_upper_green"]
            & out["long_close_through_green_valid"]
            & out["long_extension_valid"]
            & out["long_body_valid"]
            & out["long_not_orange_chase"]
            & ~out["possible_chop"]
        )

        out["raw_short_green_rejection"] = (
            out["accepted_below_vwap"]
            & (out["close"] < out["vwap"])
            & out["short_touched_lower_green"]
            & out["short_closed_below_lower_green"]
            & out["short_close_through_green_valid"]
            & out["short_extension_valid"]
            & out["short_body_valid"]
            & out["short_not_orange_chase"]
            & ~out["possible_chop"]
        )

        out["v4_model_route"] = "not_used"
        out["v4_selected_entry_module"] = "v1_green_reclaim_rejection"

        long_signal_name = "LONG_GREEN_RECLAIM"
        short_signal_name = "SHORT_GREEN_REJECTION"

    # ------------------------------------------------------------------
    # Raw signal labels
    # ------------------------------------------------------------------
    out["raw_signal_side"] = np.select(
        [
            out["raw_long_green_reclaim"],
            out["raw_short_green_rejection"],
        ],
        [
            "long",
            "short",
        ],
        default="none",
    )

    out["raw_signal_name"] = np.select(
        [
            out["raw_long_green_reclaim"],
            out["raw_short_green_rejection"],
        ],
        [
            long_signal_name,
            short_signal_name,
        ],
        default="NO_SIGNAL",
    )

    out["directional_red_shift_label"] = trade_directional_red_shift_label(out)
    out["directional_red_shift_strength"] = trade_directional_red_shift_strength(out)
    out["directional_extension_from_green_points"] = trade_directional_extension_from_green(out)

    # V3 trade-direction helper columns.
    out["v3_directional_red_shift"] = np.where(
        out["raw_signal_side"] == "long",
        out["v3_long_directional_red_shift"],
        np.where(
            out["raw_signal_side"] == "short",
            out["v3_short_directional_red_shift"],
            np.nan,
        ),
    )

    out["v3_directional_red_shift_bucket"] = np.where(
        out["raw_signal_side"] == "long",
        out["v3_long_red_shift_bucket"],
        np.where(
            out["raw_signal_side"] == "short",
            out["v3_short_red_shift_bucket"],
            "none",
        ),
    )

    out["v3_second_close_pass"] = np.where(
        out["raw_signal_side"] == "long",
        out["v3_long_second_close_confirmation"],
        np.where(
            out["raw_signal_side"] == "short",
            out["v3_short_second_close_confirmation"],
            False,
        ),
    )

    out["v3_directional_trend_dead"] = np.where(
        out["raw_signal_side"] == "long",
        out["v3_long_trend_dead"],
        np.where(
            out["raw_signal_side"] == "short",
            out["v3_short_trend_dead"],
            False,
        ),
    )

    # ------------------------------------------------------------------
    # V5 A-tier diagnostics
    # ------------------------------------------------------------------
    out = add_v5_a_tier_diagnostic_columns(out, config)
    out = add_v5_a_tier_diagnostic_buckets(out)
    # ------------------------------------------------------------------

    # ------------------------------------------------------------------
    # V5 Dynamic S-tier diagnostics / optional blocker
    # ------------------------------------------------------------------
    out = add_v5_s_tier_touch_diagnostics(out, config)

    out["engine_mode"] = engine_mode

    # ------------------------------------------------------------------
    # V5 module and V2 toggle gates
    # ------------------------------------------------------------------
    v5_gate_df = out.apply(
        lambda row: v5_signal_gate_fields(row, config),
        axis=1,
        result_type="expand",
    )

    for column in v5_gate_df.columns:
        out[column] = v5_gate_df[column]

    # ------------------------------------------------------------------
    # Direction controls
    # ------------------------------------------------------------------
    out["direction_allowed"] = (
        ((out["raw_signal_side"] == "long") & bool(config["allow_longs"]))
        | ((out["raw_signal_side"] == "short") & bool(config["allow_shorts"]))
        | (out["raw_signal_side"] == "none")
    )

    # ------------------------------------------------------------------
    # V5 V2 trend-health activation state
    # ------------------------------------------------------------------
    out = add_v5_v2_activation_fields(out, config)
    
    # ------------------------------------------------------------------
    # Strategy filter
    # ------------------------------------------------------------------
    out["strategy_filter_pass"] = build_v1_strategy_filter_mask(out, config)

    out.loc[
        out.get(
            "s_tier_clean_state_blocked",
            pd.Series(False, index=out.index),
        )
        .fillna(False)
        .astype(bool),
        "strategy_filter_pass",
    ] = False

    # ------------------------------------------------------------------
    # Final signal
    # ------------------------------------------------------------------
    out["v1_signal_valid"] = (
        (out["raw_signal_name"] != "NO_SIGNAL")
        & out["direction_allowed"]
        & out["v5_module_toggle_pass"]
        & out["v5_v2_toggle_pass"]
        & out["v5_entry_red_shift_floor_pass"]
        & out["strategy_filter_pass"]
        & out["before_no_new_trades_cutoff"]
    )

    out["v1_signal_side"] = np.where(
        out["v1_signal_valid"],
        out["raw_signal_side"],
        "none",
    )

    out["v1_signal_name"] = np.where(
        out["v1_signal_valid"],
        out["raw_signal_name"],
        "NO_SIGNAL",
    )

    out["v1_signal_block_reason"] = "NO_RAW_SIGNAL"

    out.loc[
        (out["raw_signal_name"] != "NO_SIGNAL") & ~out["direction_allowed"],
        "v1_signal_block_reason",
    ] = "DIRECTION_DISABLED"

    out.loc[
        (out["raw_signal_name"] != "NO_SIGNAL")
        & out["direction_allowed"]
        & ~out["v5_module_toggle_pass"],
        "v1_signal_block_reason",
    ] = out.loc[
        (out["raw_signal_name"] != "NO_SIGNAL")
        & out["direction_allowed"]
        & ~out["v5_module_toggle_pass"],
        "v5_module_block_reason",
    ]

    out.loc[
        (out["raw_signal_name"] != "NO_SIGNAL")
        & out["direction_allowed"]
        & out["v5_module_toggle_pass"]
        & ~out["v5_v2_toggle_pass"],
        "v1_signal_block_reason",
    ] = out.loc[
        (out["raw_signal_name"] != "NO_SIGNAL")
        & out["direction_allowed"]
        & out["v5_module_toggle_pass"]
        & ~out["v5_v2_toggle_pass"],
        "v5_v2_block_reason",
    ]

    out.loc[
        (out["raw_signal_name"] != "NO_SIGNAL")
        & out["direction_allowed"]
        & out["v5_module_toggle_pass"]
        & out["v5_v2_toggle_pass"]
        & ~out["v5_entry_red_shift_floor_pass"],
        "v1_signal_block_reason",
    ] = out.loc[
        (out["raw_signal_name"] != "NO_SIGNAL")
        & out["direction_allowed"]
        & out["v5_module_toggle_pass"]
        & out["v5_v2_toggle_pass"]
        & ~out["v5_entry_red_shift_floor_pass"],
        "v5_entry_red_shift_floor_block_reason",
    ]
    
    out.loc[
        (out["raw_signal_name"] != "NO_SIGNAL")
        & out["direction_allowed"]
        & out["v5_module_toggle_pass"]
        & out["v5_v2_toggle_pass"]
        & out["v5_entry_red_shift_floor_pass"]
        & ~out["strategy_filter_pass"],
        "v1_signal_block_reason",
    ] = "STRATEGY_FILTER_BLOCKED"

    out.loc[
        (out["raw_signal_name"] != "NO_SIGNAL")
        & out["direction_allowed"]
        & out["v5_module_toggle_pass"]
        & out["v5_v2_toggle_pass"]
        & out["v5_entry_red_shift_floor_pass"]
        & out.get(
            "s_tier_clean_state_blocked",
            pd.Series(False, index=out.index),
        )
        .fillna(False)
        .astype(bool),
        "v1_signal_block_reason",
    ] = "BLOCKED_S_TIER_NOT_CLEAN_AFTER_GREEN_FAILURE"

    out.loc[
        (out["raw_signal_name"] != "NO_SIGNAL")
        & out["direction_allowed"]
        & out["v5_module_toggle_pass"]
        & out["v5_v2_toggle_pass"]
        & out["v5_entry_red_shift_floor_pass"]
        & out["strategy_filter_pass"]
        & ~out["before_no_new_trades_cutoff"],
        "v1_signal_block_reason",
    ] = "SESSION_CUTOFF"

    out.loc[out["v1_signal_valid"], "v1_signal_block_reason"] = "VALID_SIGNAL"

    # ------------------------------------------------------------------
    # V5 tracking labels
    # ------------------------------------------------------------------
    v5_tracking_df = out.apply(
        v5_signal_tracking_fields,
        axis=1,
        result_type="expand",
    )

    for column in v5_tracking_df.columns:
        out[column] = v5_tracking_df[column]

    return out

### V5 Exit Manager Framework

This section defines the exit-management framework used by the V5 modular engine.

The default exit mode preserves the existing fixed TP, SL, and breakeven behaviour. Additional exit modes can be layered onto this framework without changing the baseline execution model.

In [ ]:
# ---------------------------------------------------------------------
# V5 exit manager framework
# ---------------------------------------------------------------------

V5_EXIT_CATEGORIES = {
    "TP": "TP",
    "SL": "SL_FULL",
    "BE": "BE_TRAIL",
    "TRAIL": "TRAILED_SL_PROFIT",
    "CLOSE_PROFIT": "TIME_EXIT",
    "CLOSE_LOSS": "TIME_EXIT",
    "CLOSE_FLAT": "TIME_EXIT",
}

def v5_normal_exit_points(config: dict) -> dict:
    """
    Return normal fixed-exit point distances.

    When the V5 exit manager is disabled, this mirrors the existing config.
    When it is enabled, the normal aliases become the source for fixed exits.
    """
    if config.get("enable_v5_exit_manager", False):
        return {
            "sl_points": float(config["normal_sl_points"]),
            "tp_points": float(config["normal_tp_points"]),
            "be_trigger_points": float(config["normal_be_trigger_points"]),
        }

    return {
        "sl_points": float(config["sl_points"]),
        "tp_points": float(config["tp_points"]),
        "be_trigger_points": float(config["be_trigger_points"]),
    }

V5_TRAIL_PROFIT_REASONS = {
    "TRAIL_LOCK_2R",
    "TRAIL_LOCK_3R",
    "TRAIL_LOCK_4R",
}


def v5_is_extreme_expansion_signal(signal_row: pd.Series) -> bool:
    """
    Return whether a signal is an extreme-expansion setup.
    """
    selected_module = str(signal_row.get("v4_selected_entry_module", ""))
    setup_source = str(signal_row.get("setup_source", ""))
    model_route = str(signal_row.get("v4_model_route", ""))

    return (
        selected_module == "extreme_expansion"
        or setup_source == "EXTREME_EXPANSION"
        or "extreme_expansion" in model_route
    )

def v5_should_use_runner(signal_row: pd.Series, config: dict) -> bool:
    """
    Decide whether a trade should use the runner stair-trailing exit mode.

    runner_mode:
        off    -> fixed 2R exits only
        smart  -> runner only for approved strong contexts
        always -> every valid executed trade uses runner logic
    """
    if not bool(config.get("enable_v5_exit_manager", False)):
        return False

    if not bool(config.get("enable_runner_trailing", False)):
        return False

    runner_mode = str(config.get("runner_mode", "smart")).lower()

    if runner_mode not in {"off", "smart", "always"}:
        raise ValueError("runner_mode must be one of: off, smart, always")

    if runner_mode == "off":
        return False

    if runner_mode == "always":
        return True

    setup_source = str(
        signal_row.get(
            "setup_source",
            v5_setup_source_from_signal_row(signal_row)
            if "v5_setup_source_from_signal_row" in globals()
            else "UNKNOWN_SETUP_SOURCE",
        )
    )

    regime_20m = str(
        signal_row.get(
            "regime_20m",
            signal_row.get("v5_regime_20m", signal_row.get("v4_preliminary_regime_label", "")),
        )
    )

    red_shift_bucket = str(
        signal_row.get(
            "red_shift_bucket",
            v5_red_shift_bucket_from_signal_row(signal_row)
            if "v5_red_shift_bucket_from_signal_row" in globals()
            else "unknown",
        )
    )

    v2_health_state = str(
        signal_row.get(
            "v2_health_state",
            v5_v2_health_state_from_signal_row(signal_row)
            if "v5_v2_health_state_from_signal_row" in globals()
            else "UNKNOWN",
        )
    )

    allowed_setups = set(config.get("runner_allowed_setup_sources", []))
    allowed_regimes = set(config.get("runner_allowed_regimes", []))
    allowed_buckets = set(config.get("runner_min_red_shift_buckets", []))
    blocked_health_states = set(config.get("runner_block_health_states", []))

    if v2_health_state in blocked_health_states:
        return False

    if setup_source == "EXTREME_EXPANSION":
        return bool(config.get("enable_extreme_5r_tp", False))

    setup_allowed = setup_source in allowed_setups
    regime_allowed = regime_20m in allowed_regimes
    bucket_allowed = red_shift_bucket in allowed_buckets

    return setup_allowed and (regime_allowed or bucket_allowed)


def v5_runner_exit_points(config: dict) -> dict:
    """
    Return runner exit point distances.

    Point-based compatibility fields are still used by the simulator,
    but they are now derived from R-based runner settings.
    """
    return {
        "sl_points": float(config["normal_sl_points"]),
        "tp_points": float(config["trail_5r_tp_points"]),
        "be_trigger_points": float(config["trail_be_trigger_points"]),
        "lock_2r_trigger_points": float(config["trail_lock_2r_trigger_points"]),
        "lock_3r_trigger_points": float(config["trail_lock_3r_trigger_points"]),
        "lock_4r_trigger_points": float(config["trail_lock_4r_trigger_points"]),
        "lock_2r_points": float(config["trail_lock_2r_points"]),
        "lock_3r_points": float(config["trail_lock_3r_points"]),
        "lock_4r_points": float(config["trail_lock_4r_points"]),
    }


def v5_runner_lock_level_from_label(label: str) -> str:
    """
    Convert runner rule labels into internal lock-level names.
    """
    label = str(label).upper()

    if label == "BE":
        return "BE"

    if label.startswith("TRAIL_LOCK_"):
        return label.replace("TRAIL_", "", 1)

    if label.startswith("LOCK_"):
        return label

    return label


def v5_runner_target_result_reason(config: dict) -> str:
    """
    Return a compact target-hit reason for the active runner target.
    """
    target_r = float(config.get("runner_target_r", 5.0))

    if target_r.is_integer():
        return f"TP_{int(target_r)}R"

    return f"TP_{target_r:g}R"

def v5_be_stop_price(
    entry_price: float,
    side: str,
    config: dict,
) -> float:
    """
    Return the BE or BE-plus stop price.

    BE-plus is only used after the +1R runner trigger has been reached.
    """
    entry_price = float(entry_price)

    if not bool(config.get("enable_be_plus_buffer", False)):
        return entry_price

    buffer_points = float(config.get("be_plus_buffer_points", 0.0))

    if buffer_points <= 0:
        return entry_price

    if side == "long":
        return entry_price + buffer_points

    if side == "short":
        return entry_price - buffer_points

    raise ValueError("side must be either 'long' or 'short'")

def v5_runner_trail_state(
    max_favourable_points: float,
    entry_price: float,
    side: str,
    config: dict,
) -> dict:
    """
    Convert max favourable excursion into the active runner trail state.

    Uses generic R-based trail rules while preserving the old 1R / 2.7R /
    3.7R / 4.7R default behaviour.
    """
    max_favourable_points = float(max_favourable_points)
    entry_price = float(entry_price)
    normal_sl_points = float(config.get("normal_sl_points", config.get("sl_points", 29.0)))

    lock_level = "NONE"
    locked_points = None

    trail_rules = config.get("runner_trail_rules_r", [])

    sorted_rules = sorted(
        trail_rules,
        key=lambda rule: float(rule.get("trigger_r", 0.0)),
    )

    for rule in sorted_rules:
        trigger_r = float(rule.get("trigger_r", 0.0))
        lock_r = float(rule.get("lock_r", 0.0))
        label = str(rule.get("label", "")).upper()

        trigger_points = normal_sl_points * trigger_r

        if max_favourable_points < trigger_points:
            continue

        if label == "BE":
            if (
                bool(config.get("enable_be_plus_buffer", False))
                and float(config.get("be_plus_buffer_points", 0.0)) > 0
            ):
                candidate_lock_level = "BE_PLUS"
                candidate_locked_points = float(config.get("be_plus_buffer_points", 0.0))
            else:
                candidate_lock_level = "BE"
                candidate_locked_points = 0.0
        else:
            candidate_lock_level = v5_runner_lock_level_from_label(label)
            candidate_locked_points = normal_sl_points * lock_r

        if locked_points is None or candidate_locked_points > locked_points:
            lock_level = candidate_lock_level
            locked_points = candidate_locked_points

    legacy_lock_active = (
        bool(config.get("enable_legacy_early_runner_lock", False))
        and float(config.get("tp_points", 0.0))
        > float(config.get("legacy_early_runner_lock_only_when_tp_points_above", 60.0))
        and str(config.get("runner_mode", "off")).lower() != "off"
    )

    if legacy_lock_active:
        legacy_trigger_points = float(config.get("legacy_runner_lock_trigger_points", 78.3))
        legacy_lock_points = float(config.get("legacy_runner_lock_points", 58.0))

        if max_favourable_points >= legacy_trigger_points:
            if locked_points is None or legacy_lock_points > locked_points:
                lock_level = "LOCK_LEGACY_OLD_2R"
                locked_points = legacy_lock_points

    if lock_level == "NONE":
        if side == "long":
            stop_price = entry_price - normal_sl_points
        elif side == "short":
            stop_price = entry_price + normal_sl_points
        else:
            raise ValueError("side must be either 'long' or 'short'")

    elif lock_level == "BE_PLUS":
        stop_price = v5_be_stop_price(entry_price, side, config)

    elif side == "long":
        stop_price = entry_price + float(locked_points)

    elif side == "short":
        stop_price = entry_price - float(locked_points)

    else:
        raise ValueError("side must be either 'long' or 'short'")

    return {
        "lock_level": lock_level,
        "locked_points": locked_points,
        "stop_price": float(stop_price),
        "max_favourable_points": max_favourable_points,
    }

def v5_tighten_runner_stop(
    current_stop: float,
    candidate_stop: float,
    side: str,
) -> tuple[float, bool]:
    """
    Tighten a runner stop without ever widening risk.

    Long stops can only move upward.
    Short stops can only move downward.
    """
    current_stop = float(current_stop)
    candidate_stop = float(candidate_stop)

    if side == "long":
        new_stop = max(current_stop, candidate_stop)
        widened = new_stop < current_stop

    elif side == "short":
        new_stop = min(current_stop, candidate_stop)
        widened = new_stop > current_stop

    else:
        raise ValueError("side must be either 'long' or 'short'")

    return float(new_stop), bool(widened)


def v5_runner_stop_result(lock_level: str) -> dict:
    """
    Convert the active runner lock level into an exit result/reason.
    """
    lock_level = str(lock_level).upper()

    if lock_level == "BE_PLUS":
        return {
            "result": "BE",
            "result_reason": "BE_PLUS_1R_TRAIL",
        }

    if lock_level == "BE":
        return {
            "result": "BE",
            "result_reason": "BE_TRAIL",
        }

    if lock_level == "LOCK_LEGACY_OLD_2R":
        return {
            "result": "SL",
            "result_reason": "TRAIL_LOCK_LEGACY_OLD_2R",
        }

    if lock_level.startswith("LOCK_"):
        return {
            "result": "SL",
            "result_reason": f"TRAIL_{lock_level}",
        }

    return {
        "result": "SL",
        "result_reason": "SL",
    }


def v5_trail_lock_level_from_result_reason(result_reason: str) -> str:
    """
    Convert detailed exit reason into a compact trail-lock label.
    """
    result_reason = str(result_reason)

    if result_reason.startswith("TRAIL_LOCK_"):
        return result_reason

    if result_reason == "BE_TRAIL" or "BREAKEVEN" in result_reason:
        return "BE"

    return "NONE"


def v5_select_exit_mode(signal_row: pd.Series, config: dict) -> str:
    """
    Select the exit mode for a trade.
    """
    if v5_should_use_runner(signal_row, config):
        return "RUNNER_STAIR_TRAIL"

    if config.get("enable_v5_exit_manager", False):
        return "NORMAL_FIXED_TP_SL_BE"

    return "FIXED_TP_SL_BE"


def v5_build_exit_plan(
    entry_price: float,
    side: str,
    signal_row: pd.Series,
    config: dict,
) -> dict:
    """
    Build the V5 exit plan for a trade.
    """
    exit_mode = v5_select_exit_mode(signal_row, config)

    if exit_mode == "RUNNER_STAIR_TRAIL":
        exit_points = v5_runner_exit_points(config)
        runner_enabled = True
    else:
        exit_points = v5_normal_exit_points(config)
        runner_enabled = False

    level_config = {
        **config,
        "sl_points": exit_points["sl_points"],
        "tp_points": exit_points["tp_points"],
        "be_trigger_points": exit_points["be_trigger_points"],
    }

    levels = build_trade_levels(entry_price, side, level_config)

    return {
        "exit_mode": exit_mode,
        "levels": levels,
        "sl_points": exit_points["sl_points"],
        "tp_points": exit_points["tp_points"],
        "be_trigger_points": exit_points["be_trigger_points"],
        "runner_enabled": runner_enabled,
        "trail_activated": False,
        "trail_lock_level": "NONE",
    }


def v5_exit_category_from_result(result: str, result_reason: str) -> str:
    """
    Map raw result/reason into clearer V5 exit categories.
    """
    result = str(result)
    result_reason = str(result_reason)

    if result == "TP":
        return "TP"

    if result_reason == "BE_PLUS_1R_TRAIL":
        return "BE_PLUS"

    if result_reason in {"BE_TRAIL", "BE", "BREAKEVEN"} or result == "BE":
        return "BE_FLAT"

    if result_reason.startswith("TRAIL_LOCK_"):
        return "TRAILED_SL_PROFIT"

    if result in {"TIME_EXIT", "TIME"} or result_reason == "TIME_EXIT":
        return "TIME_EXIT"

    if result == "SL":
        return "SL_FULL"

    return "UNKNOWN_EXIT"


def v5_exit_reason_from_result(result: str, result_reason: str) -> str:
    """
    Standardise detailed exit reasons for V5 reporting.
    """
    result = str(result)
    result_reason = str(result_reason)

    if result_reason.startswith("TP_"):
        return result_reason

    if result_reason.startswith("TRAIL_LOCK_"):
        return result_reason

    if result_reason in {
        "SL_FULL",
        "BE_TRAIL",
        "TIME_EXIT",
    }:
        return result_reason

    exit_category = v5_exit_category_from_result(result, result_reason)

    if exit_category == "TP":
        return "TP_2R"

    if exit_category == "SL_FULL":
        return "SL_FULL"

    if exit_category == "BE_TRAIL":
        return "BE_TRAIL"

    if exit_category == "TIME_EXIT":
        return "TIME_EXIT"

    if exit_category == "TRAILED_SL_PROFIT":
        return result_reason

    return str(result_reason)

In [ ]:
signals_df = add_v1_green_signals(features_df, CONFIG)

raw_signal_count = int((signals_df["raw_signal_name"] != "NO_SIGNAL").sum())
final_signal_count = int((signals_df["v1_signal_name"] != "NO_SIGNAL").sum())

print(f"Raw V4 routed candidate signals: {raw_signal_count:,}")
print(f"Final V4 routed signals after filters: {final_signal_count:,}")

print("\nRaw signal counts:")
print(signals_df["raw_signal_name"].value_counts(dropna=False).to_string())

print("\nFinal signal counts:")
print(signals_df["v1_signal_name"].value_counts(dropna=False).to_string())

print("\nBlock reasons:")
print(signals_df["v1_signal_block_reason"].value_counts(dropna=False).to_string())

### Active V4 Dynamic Regime Routing Experiment

The active V4 strategy now routes continuation entries by the preliminary regime label.

Calm trend conditions use V1-style green reclaim/rejection entries.

Volatile directional trend conditions use V3 second-close green reclaim/rejection entries.

Chop and extreme-news conditions do not open fresh continuation trades.

The simulator, fixed SL / TP / BE rules, session controls, and daily risk controls remain unchanged.

In [ ]:
raw_v4 = signals_df[signals_df["raw_signal_name"] != "NO_SIGNAL"].copy()

v4_raw_routing_summary = (
    raw_v4.groupby(
        [
            "v4_preliminary_regime_label",
            "v4_model_route",
            "v4_selected_entry_module",
            "raw_signal_side",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="raw_candidates")
    .sort_values(
        [
            "v4_preliminary_regime_label",
            "v4_model_route",
            "v4_selected_entry_module",
            "raw_signal_side",
        ]
    )
)

v4_raw_routing_summary

In [ ]:
final_v4 = signals_df[signals_df["v1_signal_name"] != "NO_SIGNAL"].copy()

v4_final_routing_summary = (
    final_v4.groupby(
        [
            "v4_preliminary_regime_label",
            "v4_model_route",
            "v4_selected_entry_module",
            "v1_signal_side",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="final_signals")
    .sort_values(
        [
            "v4_preliminary_regime_label",
            "v4_model_route",
            "v4_selected_entry_module",
            "v1_signal_side",
        ]
    )
)

v4_final_routing_summary

In [ ]:
v4_signal_block_summary = (
    signals_df[signals_df["raw_signal_name"] != "NO_SIGNAL"]
    .groupby(
        [
            "v4_preliminary_regime_label",
            "v4_model_route",
            "v4_selected_entry_module",
            "v1_signal_block_reason",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="signals")
    .sort_values(
        [
            "v4_preliminary_regime_label",
            "v4_model_route",
            "v4_selected_entry_module",
            "v1_signal_block_reason",
        ]
    )
)

v4_signal_block_summary

## 16. Signal Preview

This section previews the final signals that will be passed into the trade simulator.

In [ ]:
signal_rows = signals_df[signals_df["v1_signal_name"] != "NO_SIGNAL"].copy()

signal_preview_cols = [
    "datetime",
    "session_time",
    "v1_signal_name",
    "v1_signal_side",
    "v4_preliminary_regime_label",
    "v4_model_route",
    "v4_selected_entry_module",
    "v4_v2_trend_health_required",
    "v4_conditional_v2_reason",
    "v4_v2_trend_health_required",
    "v4_conditional_v2_reason",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "upper_green",
    "lower_green",
    "upper_red",
    "lower_red",
    "accepted_above_vwap",
    "accepted_below_vwap",
    "v3_long_vwap_acceptance_pass",
    "v3_short_vwap_acceptance_pass",
    "v3_directional_red_shift",
    "v3_directional_red_shift_bucket",
    "v3_second_close_pass",
    "v3_directional_trend_dead",
    "v3_red_bands_spreading",
    "v3_long_recently_below_upper_green",
    "v3_short_recently_above_lower_green",
    "v3_long_second_close_confirmation",
    "v3_short_second_close_confirmation",
    "long_extension_from_green_points",
    "short_extension_from_green_points",
    "v3_long_a_tier_extension_valid",
    "v3_short_a_tier_extension_valid",
    "before_no_new_trades_cutoff",
]

print(f"Final signal rows: {len(signal_rows):,}")

if len(signal_rows) > 0:
    print(signal_rows[signal_preview_cols].tail(50).to_string(index=False))
else:
    print("No final signals found with the current config.")

## 17. Skipped Signal Candidates

This section shows raw candidates that were blocked by direction controls, the selected strategy filter, or the session cutoff.

These are not trade results.

They are signal-generation diagnostics.

In [ ]:
skipped_signal_candidates_df = signals_df[
    (signals_df["raw_signal_name"] != "NO_SIGNAL")
    & (signals_df["v1_signal_name"] == "NO_SIGNAL")
].copy()

skipped_preview_cols = [
    "datetime",
    "session_time",
    "raw_signal_name",
    "raw_signal_side",
    "v4_preliminary_regime_label",
    "v4_model_route",
    "v4_selected_entry_module",
    "v1_signal_block_reason",
    "body_ratio",
    "directional_red_shift_label",
    "directional_red_shift_strength",
    "v2_long_trend_health_pass",
    "v2_short_trend_health_pass",
    "v2_long_red_shift_adaptive_pass",
    "v2_short_red_shift_adaptive_pass",
    "v2_bands_not_compressing",
    "v2_long_opposite_band_expansion_pass",
    "v2_short_opposite_band_expansion_pass",
    "v2_long_trend_dead",
    "v2_short_trend_dead",
    "v2_trend_health_active",
    "v2_activation_reason",
    "v2_router_required",
    "v2_force_after_time_active",
    "v2_filter_result",
    "v2_filter_tracked_only",
    "v2_health_failure_reason",
    "directional_extension_from_green_points",
    "before_no_new_trades_cutoff",
    "v5_entry_red_shift_floor_applied",
    "v5_entry_directional_red_shift_points",
    "v5_entry_red_shift_min_points",
    "v5_entry_red_shift_floor_block_reason",
]

print(f"Skipped raw candidates: {len(skipped_signal_candidates_df):,}")

if len(skipped_signal_candidates_df) > 0:
    print(skipped_signal_candidates_df[skipped_preview_cols].tail(50).to_string(index=False))
else:
    print("No raw candidates were skipped.")

## Filter Diagnostics

This section explains how the active strategy filter changes the raw green reclaim/rejection candidate set.

The diagnostics separate raw candidates into accepted and blocked groups, then show which parts of the active filter removed candidates.

This section is diagnostic only. It does not change entries, exits, risk rules, or simulation results.

In [ ]:
def build_v2_filter_diagnostics(signals_df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarise how the V2 adaptive trend-health filter affects raw signal candidates.
    """
    required_columns = [
        "raw_signal_name",
        "raw_signal_side",
        "v1_signal_name",
        "v2_long_vwap_acceptance_pass",
        "v2_short_vwap_acceptance_pass",
        "v2_long_red_shift_adaptive_pass",
        "v2_short_red_shift_adaptive_pass",
        "v2_bands_not_compressing",
        "v2_long_opposite_band_expansion_pass",
        "v2_short_opposite_band_expansion_pass",
        "v2_long_trend_dead",
        "v2_short_trend_dead",
    ]

    missing_columns = [
        column for column in required_columns if column not in signals_df.columns
    ]

    if missing_columns:
        raise KeyError(
            "Missing columns required for V2 filter diagnostics: "
            + ", ".join(missing_columns)
        )

    raw_candidates = signals_df[signals_df["raw_signal_name"] != "NO_SIGNAL"].copy()

    if raw_candidates.empty:
        return pd.DataFrame(
            [
                {
                    "Side": "all",
                    "Raw Candidates": 0,
                    "Accepted by V2": 0,
                    "Blocked by V2": 0,
                    "Blocked: VWAP Acceptance": 0,
                    "Blocked: Adaptive Red Shift": 0,
                    "Blocked: Band Compression": 0,
                    "Blocked: Opposite Expansion": 0,
                    "Blocked: Trend Dead": 0,
                    "Accepted Rate": "0.00%",
                }
            ]
        )

    raw_candidates["v2_side_vwap_acceptance_pass"] = np.where(
        raw_candidates["raw_signal_side"] == "long",
        raw_candidates["v2_long_vwap_acceptance_pass"],
        raw_candidates["v2_short_vwap_acceptance_pass"],
    )

    raw_candidates["v2_side_red_shift_adaptive_pass"] = np.where(
        raw_candidates["raw_signal_side"] == "long",
        raw_candidates["v2_long_red_shift_adaptive_pass"],
        raw_candidates["v2_short_red_shift_adaptive_pass"],
    )

    raw_candidates["v2_side_opposite_band_expansion_pass"] = np.where(
        raw_candidates["raw_signal_side"] == "long",
        raw_candidates["v2_long_opposite_band_expansion_pass"],
        raw_candidates["v2_short_opposite_band_expansion_pass"],
    )

    raw_candidates["v2_side_trend_dead"] = np.where(
        raw_candidates["raw_signal_side"] == "long",
        raw_candidates["v2_long_trend_dead"],
        raw_candidates["v2_short_trend_dead"],
    )

    raw_candidates["v2_accepted"] = raw_candidates["v1_signal_name"] != "NO_SIGNAL"
    raw_candidates["v2_blocked"] = ~raw_candidates["v2_accepted"]

    rows = []

    for side_label, side_df in [
        ("all", raw_candidates),
        ("long", raw_candidates[raw_candidates["raw_signal_side"] == "long"]),
        ("short", raw_candidates[raw_candidates["raw_signal_side"] == "short"]),
    ]:
        raw_count = len(side_df)
        accepted_count = int(side_df["v2_accepted"].sum())
        blocked_count = int(side_df["v2_blocked"].sum())

        if raw_count == 0:
            accepted_rate = 0.0
        else:
            accepted_rate = accepted_count / raw_count

        blocked_df = side_df[side_df["v2_blocked"]].copy()

        rows.append(
            {
                "Side": side_label,
                "Raw Candidates": raw_count,
                "Accepted by V2": accepted_count,
                "Blocked by V2": blocked_count,
                "Blocked: VWAP Acceptance": int(
                    (~blocked_df["v2_side_vwap_acceptance_pass"].fillna(False)).sum()
                ),
                "Blocked: Adaptive Red Shift": int(
                    (~blocked_df["v2_side_red_shift_adaptive_pass"].fillna(False)).sum()
                ),
                "Blocked: Band Compression": int(
                    (~blocked_df["v2_bands_not_compressing"].fillna(False)).sum()
                ),
                "Blocked: Opposite Expansion": int(
                    (~blocked_df["v2_side_opposite_band_expansion_pass"].fillna(False)).sum()
                ),
                "Blocked: Trend Dead": int(
                    blocked_df["v2_side_trend_dead"].fillna(False).sum()
                ),
                "Accepted Rate": f"{accepted_rate:.2%}",
            }
        )

    return pd.DataFrame(rows)


v2_filter_diagnostics_table = build_v2_filter_diagnostics(signals_df)
v2_filter_diagnostics_table

In [ ]:
v2_blocked_candidates_preview_cols = [
    "timestamp",
    "raw_signal_name",
    "raw_signal_side",
    "v1_signal_name",
    "v1_signal_block_reason",
    "close",
    "vwap",
    "upper_green",
    "lower_green",
    "v2_long_vwap_acceptance_pass",
    "v2_short_vwap_acceptance_pass",
    "v2_long_red_shift_adaptive_pass",
    "v2_short_red_shift_adaptive_pass",
    "v2_long_red_shift_relative_to_avg",
    "v2_short_red_shift_relative_to_avg",
    "v2_bands_not_compressing",
    "v2_long_opposite_band_expansion_pass",
    "v2_short_opposite_band_expansion_pass",
    "v2_long_trend_dead",
    "v2_short_trend_dead",
]

v2_blocked_candidates_preview_cols = [
    column for column in v2_blocked_candidates_preview_cols if column in signals_df.columns
]

v2_blocked_candidates_df = signals_df[
    (signals_df["raw_signal_name"] != "NO_SIGNAL")
    & (signals_df["v1_signal_name"] == "NO_SIGNAL")
].copy()

if v2_blocked_candidates_df.empty:
    print("No raw candidates were blocked by the active filter.")
else:
    print(f"Raw candidates blocked by active filter: {len(v2_blocked_candidates_df):,}")
    display(v2_blocked_candidates_df[v2_blocked_candidates_preview_cols].tail(20))

## 18. Ready for Trade Simulation

At this point, the notebook has:

- created automation-only features
- generated green reclaim / rejection candidates
- applied the selected strategy filter
- created final signals
- created a skipped signal candidate log

The next section will simulate fixed-point trades from `signals_df`.

In [ ]:
print("signals_df is ready for trade simulation.")
print(f"Shape: {signals_df.shape}")
print(f"Final signals: {(signals_df['v1_signal_name'] != 'NO_SIGNAL').sum():,}")

required_trade_sim_columns = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "v1_signal_name",
    "v1_signal_side",
]

for col in required_trade_sim_columns:
    print(f"- {col}: {'yes' if col in signals_df.columns else 'missing'}")

## 19. Trade Simulation Settings

This section adds execution-specific settings for the automated V1 backtest.

The simulator uses the final V1 signals and turns them into fixed-point trades.

In [ ]:
CONFIG.update(
    {
        # Trade simulation behaviour
        "one_trade_at_a_time": True,
        "same_bar_exit_priority": "sl_first",
        "force_close_at_dataset_end": True,

        # Optional execution assumptions
        "entry_slippage_points": 0.0,
        "exit_slippage_points": 0.0,
    }
)

print("Updated V4 config with trade simulation settings:")
pprint(CONFIG)

## 20. Fixed-Point Trade Simulator

This section defines the bar-by-bar trade simulator.

The V1 simulator assumes:

- a valid signal appears on the signal candle
- entry happens at the next candle open
- SL, TP, and BE are fixed Nasdaq-point distances
- BE moves the stop to entry after the configured favourable move
- only one trade is open at a time

In [ ]:
def get_session_date(timestamp, config: dict):
    """
    Convert a timestamp into the configured session timezone and return the session date.
    """
    return normalise_timestamp_to_session_time(timestamp, config).date()


def apply_entry_slippage(entry_price: float, side: str, config: dict) -> float:
    """
    Apply optional adverse entry slippage.
    """
    slippage = float(config.get("entry_slippage_points", 0.0))

    if side == "long":
        return entry_price + slippage

    if side == "short":
        return entry_price - slippage

    raise ValueError("side must be either 'long' or 'short'")


def apply_exit_slippage(exit_price: float, side: str, result: str, config: dict) -> float:
    """
    Apply optional adverse exit slippage.

    For now this is simple and symmetric.
    """
    slippage = float(config.get("exit_slippage_points", 0.0))

    if slippage == 0:
        return exit_price

    if side == "long":
        if result in ["SL", "BE"]:
            return exit_price - slippage
        return exit_price - slippage

    if side == "short":
        if result in ["SL", "BE"]:
            return exit_price + slippage
        return exit_price + slippage

    raise ValueError("side must be either 'long' or 'short'")


def calculate_trade_points(entry_price: float, exit_price: float, side: str) -> float:
    """
    Calculate trade result in Nasdaq points.
    """
    if side == "long":
        return exit_price - entry_price

    if side == "short":
        return entry_price - exit_price

    raise ValueError("side must be either 'long' or 'short'")


def classify_result_from_points(points: float, config: dict) -> str:
    """
    Classify a result if the trade is force-closed rather than hitting TP/SL/BE.
    """
    if points > 0:
        return "CLOSE_PROFIT"

    if points < 0:
        return "CLOSE_LOSS"

    return "CLOSE_FLAT"

In [ ]:
def simulate_single_v1_trade(df: pd.DataFrame, signal_index: int, config: dict) -> tuple[dict | None, dict | None]:
    """
    Simulate one V1/V5 modular trade from a valid signal.

    Returns:
    - trade dictionary if a trade was taken
    - skipped dictionary if the signal could not be executed
    """
    if config["entry_timing"] != "next_bar_open":
        raise ValueError("This simulator currently supports entry_timing='next_bar_open' only.")

    signal_row = df.loc[signal_index]

    side = signal_row["v1_signal_side"]
    signal_name = signal_row["v1_signal_name"]
    signal_time = signal_row["datetime"]

    entry_index = signal_index + 1

    if entry_index >= len(df):
        skipped = {
            "signal_index": signal_index,
            "signal_time": signal_time,
            "signal_name": signal_name,
            "side": side,
            "skip_reason": "NO_NEXT_BAR_FOR_ENTRY",
        }
        return None, skipped

    retrace_execution = v5_apply_a_tier_retrace_execution(
        df=df,
        signal_index=signal_index,
        entry_index=entry_index,
        config=config,
    )

    if not retrace_execution["use_trade"]:
        skipped = {
            "signal_index": signal_index,
            "signal_time": signal_time,
            "signal_name": signal_name,
            "side": side,
            "entry_index": retrace_execution.get("entry_index", entry_index),
            "entry_time": pd.NaT,
            "skip_reason": retrace_execution["skip_reason"],
            "a_tier_entry_timing_type": retrace_execution["a_tier_entry_timing_type"],
            "a_tier_retrace_required": retrace_execution["a_tier_retrace_required"],
            "a_tier_retrace_pass": retrace_execution["a_tier_retrace_pass"],
            "a_tier_retrace_points": retrace_execution["a_tier_retrace_points"],
            "a_tier_retrace_close_extension_from_green": retrace_execution[
                "a_tier_retrace_close_extension_from_green"
            ],
            "a_tier_retrace_hold_green": retrace_execution["a_tier_retrace_hold_green"],
            "a_tier_retrace_fail_reason": retrace_execution["a_tier_retrace_fail_reason"],
        }
        return None, skipped

    entry_index = retrace_execution["entry_index"]

    entry_row = df.loc[entry_index]
    entry_time = entry_row["datetime"]

    if not is_before_no_new_trades_cutoff(entry_time, config):
        skipped = {
            "signal_index": signal_index,
            "signal_time": signal_time,
            "signal_name": signal_name,
            "side": side,
            "entry_index": entry_index,
            "entry_time": entry_time,
            "skip_reason": "ENTRY_AFTER_SESSION_CUTOFF",
        }
        return None, skipped

    entry_trade_session = v5_trade_session_from_timestamp(entry_time, config)

    if not v5_session_enabled_for_trade(entry_trade_session, config):
        skipped = {
            "signal_index": signal_index,
            "signal_time": signal_time,
            "signal_name": signal_name,
            "side": side,
            "entry_index": entry_index,
            "entry_time": entry_time,
            "trade_session": entry_trade_session,
            "skip_reason": "BLOCKED_BY_SESSION",
        }
        return None, skipped
    
    raw_entry_price = float(entry_row["open"])
    entry_price = apply_entry_slippage(raw_entry_price, side, config)

    exit_plan = v5_build_exit_plan(
        entry_price=entry_price,
        side=side,
        signal_row=signal_row,
        config=config,
    )

    levels = exit_plan["levels"]

    stop_price = float(levels["stop_price"])
    target_price = float(levels["target_price"])
    breakeven_trigger_price = float(levels["breakeven_trigger_price"])

    runner_enabled = bool(exit_plan.get("runner_enabled", False))
    active_runner_stop_price = stop_price
    runner_stop_widened = False
    runner_stop_updates = 0

    be_active = False
    be_activated_index = None

    trail_activated = False
    trail_lock_level = "NONE"
    trail_lock_activated_index = None
    max_favourable_points_running = 0.0

    exit_index = None
    exit_time = None
    exit_price = None
    result = None
    result_reason = None

    same_bar_priority = config.get("same_bar_exit_priority", "sl_first")

    for bar_index in range(entry_index, len(df)):
        bar = df.loc[bar_index]

        high = float(bar["high"])
        low = float(bar["low"])
        bar_time = bar["datetime"]

        if runner_enabled:
            stop_for_this_bar = active_runner_stop_price
            lock_level_for_this_bar = trail_lock_level

            if side == "long":
                stop_hit = low <= stop_for_this_bar
                target_hit = high >= target_price

                if stop_hit and target_hit:
                    if same_bar_priority == "tp_first":
                        exit_index = bar_index
                        exit_time = bar_time
                        exit_price = target_price
                        result = "TP"
                        result_reason = v5_runner_target_result_reason(config)
                    else:
                        stop_result = v5_runner_stop_result(lock_level_for_this_bar)
                        exit_index = bar_index
                        exit_time = bar_time
                        exit_price = stop_for_this_bar
                        result = stop_result["result"]
                        result_reason = stop_result["result_reason"]
                    break

                if stop_hit:
                    stop_result = v5_runner_stop_result(lock_level_for_this_bar)
                    exit_index = bar_index
                    exit_time = bar_time
                    exit_price = stop_for_this_bar
                    result = stop_result["result"]
                    result_reason = stop_result["result_reason"]
                    break

                if target_hit:
                    exit_index = bar_index
                    exit_time = bar_time
                    exit_price = target_price
                    result = "TP"
                    result_reason = v5_runner_target_result_reason(config)
                    break

                bar_favourable_points = max(0.0, high - entry_price)
                max_favourable_points_running = max(
                    max_favourable_points_running,
                    bar_favourable_points,
                )

            elif side == "short":
                stop_hit = high >= stop_for_this_bar
                target_hit = low <= target_price

                if stop_hit and target_hit:
                    if same_bar_priority == "tp_first":
                        exit_index = bar_index
                        exit_time = bar_time
                        exit_price = target_price
                        result = "TP"
                        result_reason = v5_runner_target_result_reason(config)
                    else:
                        stop_result = v5_runner_stop_result(lock_level_for_this_bar)
                        exit_index = bar_index
                        exit_time = bar_time
                        exit_price = stop_for_this_bar
                        result = stop_result["result"]
                        result_reason = stop_result["result_reason"]
                    break

                if stop_hit:
                    stop_result = v5_runner_stop_result(lock_level_for_this_bar)
                    exit_index = bar_index
                    exit_time = bar_time
                    exit_price = stop_for_this_bar
                    result = stop_result["result"]
                    result_reason = stop_result["result_reason"]
                    break

                if target_hit:
                    exit_index = bar_index
                    exit_time = bar_time
                    exit_price = target_price
                    result = "TP"
                    result_reason = v5_runner_target_result_reason(config)
                    break

                bar_favourable_points = max(0.0, entry_price - low)
                max_favourable_points_running = max(
                    max_favourable_points_running,
                    bar_favourable_points,
                )

            else:
                raise ValueError("side must be either 'long' or 'short'")

            # Trail updates happen only after the candle survives.
            # The updated stop is active from the next candle onward.
            trail_state = v5_runner_trail_state(
                max_favourable_points=max_favourable_points_running,
                entry_price=entry_price,
                side=side,
                config=config,
            )

            candidate_stop_price = float(trail_state["stop_price"])
            candidate_lock_level = str(trail_state["lock_level"])

            new_stop_price, widened = v5_tighten_runner_stop(
                current_stop=active_runner_stop_price,
                candidate_stop=candidate_stop_price,
                side=side,
            )

            runner_stop_widened = runner_stop_widened or widened

            if new_stop_price != active_runner_stop_price:
                active_runner_stop_price = new_stop_price
                runner_stop_updates += 1

            if candidate_lock_level != "NONE":
                trail_activated = True

                if not be_active:
                    be_active = True
                    be_activated_index = bar_index

                if candidate_lock_level != trail_lock_level:
                    trail_lock_level = candidate_lock_level
                    trail_lock_activated_index = bar_index

        else:
            if side == "long":
                original_stop_hit = low <= stop_price
                target_hit = high >= target_price
                breakeven_trigger_hit = high >= breakeven_trigger_price

                if not be_active:
                    if original_stop_hit and target_hit:
                        if same_bar_priority == "tp_first":
                            exit_index = bar_index
                            exit_time = bar_time
                            exit_price = target_price
                            result = "TP"
                            result_reason = "TP_AND_SL_SAME_BAR_TP_FIRST"
                        else:
                            exit_index = bar_index
                            exit_time = bar_time
                            exit_price = stop_price
                            result = "SL"
                            result_reason = "TP_AND_SL_SAME_BAR_SL_FIRST"
                        break

                    if original_stop_hit:
                        exit_index = bar_index
                        exit_time = bar_time
                        exit_price = stop_price
                        result = "SL"
                        result_reason = "STOP_LOSS_HIT"
                        break

                    if target_hit:
                        exit_index = bar_index
                        exit_time = bar_time
                        exit_price = target_price
                        result = "TP"
                        result_reason = "TAKE_PROFIT_HIT"
                        break

                    if breakeven_trigger_hit:
                        be_active = True
                        be_activated_index = bar_index
                        continue

                else:
                    breakeven_stop_hit = low <= entry_price
                    target_hit = high >= target_price

                    if breakeven_stop_hit and target_hit:
                        if same_bar_priority == "tp_first":
                            exit_index = bar_index
                            exit_time = bar_time
                            exit_price = target_price
                            result = "TP"
                            result_reason = "TP_AND_BE_SAME_BAR_TP_FIRST"
                        else:
                            exit_index = bar_index
                            exit_time = bar_time
                            exit_price = entry_price
                            result = "BE"
                            result_reason = "TP_AND_BE_SAME_BAR_BE_FIRST"
                        break

                    if breakeven_stop_hit:
                        exit_index = bar_index
                        exit_time = bar_time
                        exit_price = entry_price
                        result = "BE"
                        result_reason = "BREAKEVEN_STOP_HIT"
                        break

                    if target_hit:
                        exit_index = bar_index
                        exit_time = bar_time
                        exit_price = target_price
                        result = "TP"
                        result_reason = "TAKE_PROFIT_HIT_AFTER_BE_ACTIVE"
                        break

            elif side == "short":
                original_stop_hit = high >= stop_price
                target_hit = low <= target_price
                breakeven_trigger_hit = low <= breakeven_trigger_price

                if not be_active:
                    if original_stop_hit and target_hit:
                        if same_bar_priority == "tp_first":
                            exit_index = bar_index
                            exit_time = bar_time
                            exit_price = target_price
                            result = "TP"
                            result_reason = "TP_AND_SL_SAME_BAR_TP_FIRST"
                        else:
                            exit_index = bar_index
                            exit_time = bar_time
                            exit_price = stop_price
                            result = "SL"
                            result_reason = "TP_AND_SL_SAME_BAR_SL_FIRST"
                        break

                    if original_stop_hit:
                        exit_index = bar_index
                        exit_time = bar_time
                        exit_price = stop_price
                        result = "SL"
                        result_reason = "STOP_LOSS_HIT"
                        break

                    if target_hit:
                        exit_index = bar_index
                        exit_time = bar_time
                        exit_price = target_price
                        result = "TP"
                        result_reason = "TAKE_PROFIT_HIT"
                        break

                    if breakeven_trigger_hit:
                        be_active = True
                        be_activated_index = bar_index
                        continue

                else:
                    breakeven_stop_hit = high >= entry_price
                    target_hit = low <= target_price

                    if breakeven_stop_hit and target_hit:
                        if same_bar_priority == "tp_first":
                            exit_index = bar_index
                            exit_time = bar_time
                            exit_price = target_price
                            result = "TP"
                            result_reason = "TP_AND_BE_SAME_BAR_TP_FIRST"
                        else:
                            exit_index = bar_index
                            exit_time = bar_time
                            exit_price = entry_price
                            result = "BE"
                            result_reason = "TP_AND_BE_SAME_BAR_BE_FIRST"
                        break

                    if breakeven_stop_hit:
                        exit_index = bar_index
                        exit_time = bar_time
                        exit_price = entry_price
                        result = "BE"
                        result_reason = "BREAKEVEN_STOP_HIT"
                        break

                    if target_hit:
                        exit_index = bar_index
                        exit_time = bar_time
                        exit_price = target_price
                        result = "TP"
                        result_reason = "TAKE_PROFIT_HIT_AFTER_BE_ACTIVE"
                        break

            else:
                raise ValueError("side must be either 'long' or 'short'")

    if exit_index is None:
        if not bool(config.get("force_close_at_dataset_end", True)):
            skipped = {
                "signal_index": signal_index,
                "signal_time": signal_time,
                "signal_name": signal_name,
                "side": side,
                "entry_index": entry_index,
                "entry_time": entry_time,
                "skip_reason": "TRADE_STILL_OPEN_AT_DATASET_END",
            }
            return None, skipped

        final_bar = df.iloc[-1]
        exit_index = int(final_bar.name)
        exit_time = final_bar["datetime"]
        exit_price = float(final_bar["close"])

        raw_points = calculate_trade_points(entry_price, exit_price, side)
        result = classify_result_from_points(raw_points, config)
        result_reason = "FORCE_CLOSED_AT_DATASET_END"

    exit_price = apply_exit_slippage(exit_price, side, result, config)

    points = calculate_trade_points(entry_price, exit_price, side)
    r_multiple = points / float(exit_plan["sl_points"])
    account_pct = r_multiple * float(config["risk_per_trade_pct"])

    bars_held = exit_index - entry_index + 1

    trade = {
        "signal_index": signal_index,
        "signal_time": signal_time,
        "signal_name": signal_name,
        "v4_preliminary_regime_label": signal_row.get("v4_preliminary_regime_label", "not_available"),
        "v4_model_route": signal_row.get("v4_model_route", "not_available"),
        "v4_selected_entry_module": signal_row.get("v4_selected_entry_module", "not_available"),
        "v4_v2_trend_health_required": signal_row.get("v4_v2_trend_health_required", False),
        "v4_conditional_v2_reason": signal_row.get("v4_conditional_v2_reason", "not_available"),
        "side": side,

        "entry_index": entry_index,
        "entry_time": entry_time,
        "entry_price": entry_price,

        "exit_index": exit_index,
        "exit_time": exit_time,
        "exit_price": exit_price,

        "result": result,
        "result_reason": result_reason,

        "points": points,
        "r_multiple": r_multiple,
        "account_pct": account_pct,
        "bars_held": bars_held,

        "stop_price_initial": stop_price,
        "target_price": target_price,
        "breakeven_trigger_price": breakeven_trigger_price,
        "breakeven_activated": be_active,
        "breakeven_activated_index": be_activated_index,
        "runner_trailing_enabled": runner_enabled,
        "runner_trail_activated": trail_activated,
        "runner_trail_lock_level": trail_lock_level,
        "runner_trail_lock_activated_index": trail_lock_activated_index,
        "runner_stop_widened": runner_stop_widened,
        "runner_stop_updates": runner_stop_updates,
        "runner_mode": config.get("runner_mode", "smart"),

        "session_date": get_session_date(entry_time, config),
        "strategy_version": config["strategy_version"],
        "strategy_filter": config["strategy_filter"],
    }

    trade.update(
        v5_build_trade_tracker_fields(
            df=df,
            signal_row=signal_row,
            entry_index=entry_index,
            exit_index=exit_index,
            entry_price=entry_price,
            side=side,
            result=result,
            result_reason=result_reason,
            config=config,
            exit_plan=exit_plan,
        )
    )

    trade.update(
        {
            "a_tier_entry_timing_type": retrace_execution["a_tier_entry_timing_type"],
            "a_tier_retrace_required": retrace_execution["a_tier_retrace_required"],
            "a_tier_retrace_pass": retrace_execution["a_tier_retrace_pass"],
            "a_tier_retrace_points": retrace_execution["a_tier_retrace_points"],
            "a_tier_retrace_close_extension_from_green": retrace_execution[
                "a_tier_retrace_close_extension_from_green"
            ],
            "a_tier_retrace_hold_green": retrace_execution["a_tier_retrace_hold_green"],
            "a_tier_retrace_fail_reason": retrace_execution["a_tier_retrace_fail_reason"],
        }
    )

    if retrace_execution["a_tier_entry_timing_type"] == "RETRACE_ENTRY":
        trade["setup_source"] = "V3_A_TIER_RETRACE_ENTRY"
        trade["setup_family"] = "V3_A_TIER"

    return trade, None

## 21. Daily Risk Controls

This section applies the V1 daily controls during simulation.

The controls are applied before opening each new trade:

- do not open new trades after the daily profit cap is reached
- do not open new trades after the daily consecutive-loss limit is reached
- do not open overlapping trades when one-trade-at-a-time mode is enabled

In [ ]:
def initialise_daily_state() -> dict:
    """
    Initialise one session day's trading state.
    """
    return {
        "final_signals_seen": 0,
        "trades_taken": 0,
        "skipped_execution_signals": 0,

        "tp_count": 0,
        "sl_count": 0,
        "be_count": 0,
        "other_exit_count": 0,

        "realised_points": 0.0,
        "realised_r": 0.0,
        "realised_account_pct": 0.0,

        "consecutive_losses": 0,
        "max_consecutive_losses_seen": 0,

        "daily_stop_triggered": False,
        "daily_stop_reason": None,
    }


def get_or_create_daily_state(daily_states: dict, session_date) -> dict:
    """
    Retrieve or initialise the state for a session date.
    """
    if session_date not in daily_states:
        daily_states[session_date] = initialise_daily_state()

    return daily_states[session_date]


def get_daily_block_reason(day_state: dict, config: dict) -> str | None:
    """
    Return the reason a new trade should be blocked by daily risk controls.
    """
    if day_state["realised_account_pct"] >= float(config["daily_profit_cap_pct"]):
        return "DAILY_PROFIT_CAP_REACHED"

    if day_state["consecutive_losses"] >= int(config["daily_max_consecutive_losses"]):
        return "DAILY_MAX_CONSECUTIVE_LOSSES_REACHED"

    return None


def update_daily_state_after_trade(day_state: dict, trade: dict, config: dict) -> None:
    """
    Update daily state after a trade has closed.
    """
    result = trade["result"]

    day_state["trades_taken"] += 1
    day_state["realised_points"] += float(trade["points"])
    day_state["realised_r"] += float(trade["r_multiple"])
    day_state["realised_account_pct"] += float(trade["account_pct"])

    if result == "TP":
        day_state["tp_count"] += 1
    elif result == "SL":
        day_state["sl_count"] += 1
    elif result == "BE":
        day_state["be_count"] += 1
    else:
        day_state["other_exit_count"] += 1

    if float(trade["r_multiple"]) < 0:
        day_state["consecutive_losses"] += 1
    else:
        day_state["consecutive_losses"] = 0

    day_state["max_consecutive_losses_seen"] = max(
        day_state["max_consecutive_losses_seen"],
        day_state["consecutive_losses"],
    )

    block_reason = get_daily_block_reason(day_state, config)

    if block_reason is not None:
        day_state["daily_stop_triggered"] = True
        day_state["daily_stop_reason"] = block_reason

In [ ]:
def build_execution_skip_record(
    row: pd.Series,
    signal_index: int,
    skip_reason: str,
    config: dict,
    entry_index: int | None = None,
    entry_time=None,
) -> dict:
    """
    Build a standard skipped execution signal record.
    """
    record = {
        "signal_index": signal_index,
        "signal_time": row["datetime"],
        "signal_name": row["v1_signal_name"],
        "v4_preliminary_regime_label": row.get("v4_preliminary_regime_label", "not_available"),
        "v4_model_route": row.get("v4_model_route", "not_available"),
        "v4_selected_entry_module": row.get("v4_selected_entry_module", "not_available"),
        "side": row["v1_signal_side"],
        "skip_reason": skip_reason,
        "session_date": get_session_date(row["datetime"], config),
        "strategy_version": config["strategy_version"],
        "strategy_filter": config["strategy_filter"],
    }

    if entry_index is not None:
        record["entry_index"] = entry_index

    if entry_time is not None:
        record["entry_time"] = entry_time

    record.update(
        v5_build_candidate_tracker_fields(
            row=row,
            config=config,
            entry_time=entry_time,
            blocked_reason=skip_reason,
        )
    )

    return record


def run_v1_trade_simulation_with_daily_controls(
    df: pd.DataFrame,
    config: dict,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Run the full V1 trade simulation with daily risk controls.

    This function processes valid V1 signals in chronological order.
    """
    trades = []
    skipped_execution_signals = []
    daily_states = {}

    latest_open_trade_exit_index = -1

    valid_signal_indices = df.index[df["v1_signal_name"] != "NO_SIGNAL"].tolist()

    for signal_index in valid_signal_indices:
        row = df.loc[signal_index]

        entry_index = signal_index + 1

        if entry_index < len(df):
            entry_time = df.loc[entry_index, "datetime"]
            session_date = get_session_date(entry_time, config)
        else:
            entry_time = None
            session_date = get_session_date(row["datetime"], config)

        day_state = get_or_create_daily_state(daily_states, session_date)
        day_state["final_signals_seen"] += 1

        # One-trade-at-a-time block.
        if bool(config.get("one_trade_at_a_time", True)) and entry_index <= latest_open_trade_exit_index:
            skip_record = build_execution_skip_record(
                row=row,
                signal_index=signal_index,
                skip_reason="ACTIVE_TRADE_ALREADY_OPEN",
                config=config,
                entry_index=entry_index,
                entry_time=entry_time,
            )

            skipped_execution_signals.append(skip_record)
            day_state["skipped_execution_signals"] += 1
            continue

        # Daily risk block.
        daily_block_reason = get_daily_block_reason(day_state, config)

        if daily_block_reason is not None:
            skip_record = build_execution_skip_record(
                row=row,
                signal_index=signal_index,
                skip_reason=daily_block_reason,
                config=config,
                entry_index=entry_index,
                entry_time=entry_time,
            )

            skipped_execution_signals.append(skip_record)
            day_state["skipped_execution_signals"] += 1
            continue

        trade, skipped = simulate_single_v1_trade(df, signal_index, config)

        if skipped is not None:
            skipped = {
                **skipped,
                "session_date": session_date,
                "strategy_version": config["strategy_version"],
                "strategy_filter": config["strategy_filter"],
            }

            skipped.update(
                v5_build_candidate_tracker_fields(
                    row=row,
                    config=config,
                    entry_time=skipped.get("entry_time", entry_time),
                    blocked_reason=skipped.get("skip_reason", "EXECUTION_SKIPPED"),
                )
            )

            skipped_execution_signals.append(skipped)
            day_state["skipped_execution_signals"] += 1
            continue

        trades.append(trade)
        latest_open_trade_exit_index = max(latest_open_trade_exit_index, int(trade["exit_index"]))

        update_daily_state_after_trade(day_state, trade, config)

    trades_df = pd.DataFrame(trades)
    skipped_execution_signals_df = pd.DataFrame(skipped_execution_signals)

    daily_summary_records = []

    for session_date, state in sorted(daily_states.items(), key=lambda item: item[0]):
        record = {"session_date": session_date}
        record.update(state)
        daily_summary_records.append(record)

    daily_summary_df = pd.DataFrame(daily_summary_records)

    return trades_df, daily_summary_df, skipped_execution_signals_df

In [ ]:
# ---------------------------------------------------------------------
# V5 original fixed-TP hit diagnostics
# ---------------------------------------------------------------------
def _v5_trade_direction_from_row(row: pd.Series) -> str:
    """
    Infer trade direction safely for reporting-only diagnostics.
    """
    for col in ["direction", "side", "trade_side", "signal_side", "v1_signal_side"]:
        if col in row.index:
            value = str(row.get(col, "")).lower()
            if value in {"long", "short"}:
                return value

    for col in ["signal_name", "raw_signal_name", "v1_signal_name"]:
        if col in row.index:
            value = str(row.get(col, "")).upper()

            if value.startswith("LONG"):
                return "long"

            if value.startswith("SHORT"):
                return "short"

    return "unknown"


def add_v5_original_tp_hit_diagnostics(
    trades: pd.DataFrame,
    price_df: pd.DataFrame,
    config: dict,
) -> pd.DataFrame:
    """
    Add reporting-only diagnostics for whether each executed trade touched
    the original fixed TP level from config['tp_points'] before its official exit.

    This does not change entries, exits, runner logic, stops, or official results.
    """
    if trades is None:
        return pd.DataFrame()

    out = trades.copy()

    out["original_tp_points"] = np.nan
    out["original_tp_r"] = np.nan
    out["original_tp_price"] = np.nan
    out["hit_original_tp_before_exit"] = False
    out["original_tp_hit_time"] = pd.Series(
        [pd.NaT] * len(out),
        index=out.index,
        dtype="object",
    )
    out["bars_to_original_tp"] = np.nan

    if out.empty:
        return out

    if price_df is None or len(price_df) == 0:
        return out

    required_price_cols = {"datetime", "high", "low"}

    if not required_price_cols.issubset(set(price_df.columns)):
        return out

    if "entry_price" not in out.columns:
        return out

    if "entry_time" not in out.columns or "exit_time" not in out.columns:
        return out

    original_tp_points = float(config.get("tp_points", np.nan))

    if not np.isfinite(original_tp_points) or original_tp_points <= 0:
        return out

    default_sl_points = float(config.get("sl_points", np.nan))

    prices = price_df[["datetime", "high", "low"]].copy()
    prices["datetime"] = pd.to_datetime(prices["datetime"], errors="coerce")
    prices = prices.dropna(subset=["datetime"]).reset_index(drop=True)

    out["entry_time"] = pd.to_datetime(out["entry_time"], errors="coerce")
    out["exit_time"] = pd.to_datetime(out["exit_time"], errors="coerce")

    for trade_idx, trade in out.iterrows():
        entry_time = trade.get("entry_time", pd.NaT)
        exit_time = trade.get("exit_time", pd.NaT)
        entry_price = pd.to_numeric(trade.get("entry_price", np.nan), errors="coerce")

        if pd.isna(entry_time) or pd.isna(exit_time):
            continue

        if not np.isfinite(entry_price):
            continue

        direction = _v5_trade_direction_from_row(trade)

        if direction not in {"long", "short"}:
            continue

        stop_price = pd.to_numeric(
            trade.get("stop_price_initial", trade.get("sl_price", np.nan)),
            errors="coerce",
        )

        if np.isfinite(stop_price) and abs(entry_price - stop_price) > 0:
            risk_points = abs(entry_price - stop_price)
        elif np.isfinite(default_sl_points) and default_sl_points > 0:
            risk_points = default_sl_points
        else:
            risk_points = np.nan

        if direction == "long":
            original_tp_price = entry_price + original_tp_points
        else:
            original_tp_price = entry_price - original_tp_points

        original_tp_r = (
            original_tp_points / risk_points
            if np.isfinite(risk_points) and risk_points > 0
            else np.nan
        )

        out.at[trade_idx, "original_tp_points"] = original_tp_points
        out.at[trade_idx, "original_tp_r"] = original_tp_r
        out.at[trade_idx, "original_tp_price"] = original_tp_price

        trade_life = prices[
            (prices["datetime"] >= entry_time)
            & (prices["datetime"] <= exit_time)
        ].copy()

        if trade_life.empty:
            continue

        if direction == "long":
            hit_rows = trade_life[trade_life["high"] >= original_tp_price]
        else:
            hit_rows = trade_life[trade_life["low"] <= original_tp_price]

        if hit_rows.empty:
            continue

        first_hit_time = hit_rows.iloc[0]["datetime"]
        bars_to_original_tp = int(
            trade_life.index.get_loc(hit_rows.index[0]) + 1
            if hit_rows.index[0] in trade_life.index
            else len(trade_life.loc[trade_life["datetime"] <= first_hit_time])
        )

        out.at[trade_idx, "hit_original_tp_before_exit"] = True
        out.at[trade_idx, "original_tp_hit_time"] = first_hit_time
        out.at[trade_idx, "bars_to_original_tp"] = bars_to_original_tp

    return out

## 22. Run V1 Trade Simulation

This section runs the full automated V1 trade simulation.

The output is:

- `trades_df`
- `daily_summary_df`
- `skipped_execution_signals_df`

In [ ]:
trades_df, daily_summary_df, skipped_execution_signals_df = run_v1_trade_simulation_with_daily_controls(
    signals_df,
    CONFIG,
)

trades_df = add_v5_original_tp_hit_diagnostics(
    trades_df,
    signals_df,
    CONFIG,
)

print(f"Final V1 signals: {(signals_df['v1_signal_name'] != 'NO_SIGNAL').sum():,}")
print(f"Trades taken: {len(trades_df):,}")
print(f"Skipped execution signals: {len(skipped_execution_signals_df):,}")

if len(trades_df) > 0:
    print("\nTrade result counts:")
    print(trades_df["result"].value_counts(dropna=False).to_string())

    print("\nTrade log preview:")
    preview_cols = [
        "signal_time",
        "signal_name",
        "side",
        "entry_time",
        "entry_price",
        "exit_time",
        "exit_price",
        "result",
        "points",
        "r_multiple",
        "account_pct",
        "bars_held",
        "session_date",
    ]

    print(trades_df[preview_cols].tail(30).to_string(index=False))
else:
    print("No trades were taken.")

In [ ]:
# ------------------------------------------------------------------
# V5 delayed-pullback signal pipeline diagnostic
# ------------------------------------------------------------------
delayed_raw_mask = (
    signals_df["raw_signal_name"]
    .fillna("NO_SIGNAL")
    .astype(str)
    .str.contains("DELAYED_PULLBACK", na=False)
)

delayed_final_mask = (
    signals_df["v1_signal_name"]
    .fillna("NO_SIGNAL")
    .astype(str)
    .str.contains("DELAYED_PULLBACK", na=False)
)

delayed_strategy_blocked_mask = (
    delayed_raw_mask
    & ~signals_df["strategy_filter_pass"].fillna(False).astype(bool)
)

if trades_df is not None and len(trades_df) > 0:
    trade_setup_source = (
        trades_df.get("setup_source", pd.Series("", index=trades_df.index))
        .fillna("")
        .astype(str)
    )

    trade_signal_name = (
        trades_df.get("signal_name", pd.Series("", index=trades_df.index))
        .fillna("")
        .astype(str)
    )

    delayed_executed_mask = (
        trade_setup_source.eq("V5_A_TIER_DELAYED_PULLBACK")
        | trade_signal_name.str.contains("DELAYED_PULLBACK", na=False)
    )
else:
    delayed_executed_mask = pd.Series(dtype=bool)

delayed_pullback_pipeline_diagnostic = pd.DataFrame(
    [
        {
            "metric": "raw_delayed_pullback_candidates",
            "count": int(delayed_raw_mask.sum()),
        },
        {
            "metric": "delayed_pullback_blocked_by_strategy_filter",
            "count": int(delayed_strategy_blocked_mask.sum()),
        },
        {
            "metric": "final_delayed_pullback_signals",
            "count": int(delayed_final_mask.sum()),
        },
        {
            "metric": "executed_delayed_pullback_trades",
            "count": int(delayed_executed_mask.sum()),
        },
    ]
)

display(delayed_pullback_pipeline_diagnostic)

if delayed_raw_mask.any():
    delayed_pullback_debug_cols = [
        "datetime",
        "raw_signal_name",
        "v1_signal_name",
        "strategy_filter_pass",
        "v1_signal_block_reason",
        "a_tier_entry_timing_type",
        "a_tier_delayed_pullback_candidate",
        "a_tier_delayed_pullback_passed",
        "a_tier_delayed_pullback_entry_bar",
        "a_tier_delayed_pullback_hold_bars",
        "a_tier_delayed_pullback_points",
        "a_tier_delayed_pullback_held_green",
        "a_tier_delayed_pullback_block_reason",
    ]

    delayed_pullback_debug_cols = [
        col for col in delayed_pullback_debug_cols if col in signals_df.columns
    ]

    display(signals_df.loc[delayed_raw_mask, delayed_pullback_debug_cols])
else:
    print("No raw delayed pullback candidates in this run.")

In [ ]:
# ------------------------------------------------------------------
# V5 delayed-pullback strength-filter diagnostic
# ------------------------------------------------------------------
raw_signal_name = (
    signals_df.get("raw_signal_name", pd.Series("NO_SIGNAL", index=signals_df.index))
    .fillna("NO_SIGNAL")
    .astype(str)
)

final_signal_name = (
    signals_df.get("v1_signal_name", pd.Series("NO_SIGNAL", index=signals_df.index))
    .fillna("NO_SIGNAL")
    .astype(str)
)

raw_delayed_pullback_mask = raw_signal_name.str.contains(
    "DELAYED_PULLBACK",
    na=False,
)

final_delayed_pullback_mask = final_signal_name.str.contains(
    "DELAYED_PULLBACK",
    na=False,
)

strength_filter_pass = (
    signals_df.get(
        "a_tier_delayed_pullback_strength_filter_pass",
        pd.Series(False, index=signals_df.index),
    )
    .fillna(False)
    .astype(bool)
)

strength_filter_blocked_mask = (
    raw_delayed_pullback_mask
    & ~strength_filter_pass
    & bool(CONFIG.get("enable_a_tier_delayed_pullback_strength_filter", False))
)

delayed_trade_total_r = 0.0
delayed_trade_avg_r = 0.0
delayed_trade_max_dd_r = 0.0
executed_delayed_pullback_trades = 0

if trades_df is not None and len(trades_df) > 0:
    trade_report = trades_df.copy()

    trade_setup_source = (
        trade_report.get("setup_source", pd.Series("", index=trade_report.index))
        .fillna("")
        .astype(str)
    )

    trade_signal_name = (
        trade_report.get("signal_name", pd.Series("", index=trade_report.index))
        .fillna("")
        .astype(str)
    )

    delayed_trade_mask = (
        trade_setup_source.eq("V5_A_TIER_DELAYED_PULLBACK")
        | trade_signal_name.str.contains("DELAYED_PULLBACK", na=False)
    )

    delayed_trades = trade_report.loc[delayed_trade_mask].copy()
    executed_delayed_pullback_trades = int(len(delayed_trades))

    if executed_delayed_pullback_trades > 0:
        delayed_r = pd.to_numeric(
            delayed_trades["r_multiple"],
            errors="coerce",
        ).fillna(0.0)

        delayed_equity = delayed_r.cumsum()
        delayed_drawdown = delayed_equity - delayed_equity.cummax()

        delayed_trade_total_r = float(delayed_r.sum())
        delayed_trade_avg_r = float(delayed_r.mean())
        delayed_trade_max_dd_r = float(delayed_drawdown.min())

delayed_pullback_strength_summary = pd.DataFrame(
    [
        {
            "metric": "raw delayed pullback candidates",
            "value": int(raw_delayed_pullback_mask.sum()),
        },
        {
            "metric": "strength-filter passed",
            "value": int((raw_delayed_pullback_mask & strength_filter_pass).sum()),
        },
        {
            "metric": "strength-filter blocked",
            "value": int(strength_filter_blocked_mask.sum()),
        },
        {
            "metric": "final delayed pullback signals",
            "value": int(final_delayed_pullback_mask.sum()),
        },
        {
            "metric": "executed delayed pullback trades",
            "value": executed_delayed_pullback_trades,
        },
        {
            "metric": "delayed pullback Total R",
            "value": delayed_trade_total_r,
        },
        {
            "metric": "delayed pullback Avg R",
            "value": delayed_trade_avg_r,
        },
        {
            "metric": "delayed pullback Max DD R",
            "value": delayed_trade_max_dd_r,
        },
    ]
)

display(delayed_pullback_strength_summary)

debug_cols = [
    "datetime",
    "raw_signal_name",
    "v1_signal_name",
    "strategy_filter_pass",
    "v1_signal_block_reason",
    "a_tier_delayed_pullback_candidate",
    "a_tier_delayed_pullback_passed",
    "a_tier_delayed_pullback_strength_filter_pass",
    "a_tier_delayed_pullback_strength_filter_block_reason",
    "a_tier_delayed_pullback_red_shift_bucket",
    "a_tier_delayed_pullback_v2_activation_reason",
    "a_tier_delayed_pullback_v2_pass_used",
    "a_tier_delayed_pullback_block_reason",
]

debug_cols = [col for col in debug_cols if col in signals_df.columns]

if raw_delayed_pullback_mask.any():
    display(signals_df.loc[raw_delayed_pullback_mask, debug_cols])
else:
    print("No raw delayed pullback candidates in this run.")

## 23. Daily Summary Preview

This section shows the daily risk-control summary.

It helps confirm whether the daily loss-streak rule or profit-cap rule stopped trading.

In [ ]:
if len(daily_summary_df) > 0:
    print("Daily summary:")
    print(daily_summary_df.to_string(index=False))
else:
    print("No daily summary available because no valid signals were processed.")

## 24. Skipped Execution Signals

This section shows valid V1 signals that were not executed because of execution or daily-risk rules.

These are different from raw signal candidates blocked during signal generation.

In [ ]:
if len(skipped_execution_signals_df) > 0:
    print("Skipped execution signal reasons:")
    print(skipped_execution_signals_df["skip_reason"].value_counts(dropna=False).to_string())

    print("\nSkipped execution preview:")
    print(skipped_execution_signals_df.tail(50).to_string(index=False))
else:
    print("No valid signals were skipped during execution.")

## 25. Ready for Results Summary and Exports

At this point, the notebook has:

- generated V1 signals
- simulated trades at next bar open
- applied fixed SL, TP, and BE logic
- applied one-trade-at-a-time execution handling
- applied daily max consecutive loss controls
- applied daily profit cap controls
- produced a trade log
- produced a daily summary
- produced skipped execution signal diagnostics

The next section will create the final compact performance summary and export the outputs.

In [ ]:
print("V3 trade simulation is complete.")

print(f"trades_df shape: {trades_df.shape}")
print(f"daily_summary_df shape: {daily_summary_df.shape}")
print(f"skipped_execution_signals_df shape: {skipped_execution_signals_df.shape}")

## 26. Performance Summary

This section creates the main compact performance summary for the automated V4 backtest.

The summary includes signal counts, trade counts, TP/SL/BE rates, total points, total R, account percentage return, average R, average bars held, and skipped signal diagnostics.

In [ ]:
def safe_divide(numerator: float, denominator: float) -> float:
    """
    Safely divide two numbers and return 0.0 when the denominator is zero.
    """
    if denominator == 0:
        return 0.0

    return numerator / denominator


def count_result(trades: pd.DataFrame, result_name: str) -> int:
    """
    Count a specific trade result safely.
    """
    if trades is None or len(trades) == 0 or "result" not in trades.columns:
        return 0

    return int((trades["result"] == result_name).sum())

def infer_v5_managed_exit_category(row: pd.Series) -> str:
    """
    Infer the managed V5 exit category from current or legacy trade fields.
    """
    r_multiple = pd.to_numeric(row.get("r_multiple", 0.0), errors="coerce")

    if pd.isna(r_multiple):
        r_multiple = 0.0

    exit_category = str(row.get("exit_category", "")).upper()
    exit_reason = str(row.get("exit_reason", row.get("result_reason", ""))).upper()
    result = str(row.get("result", "")).upper()

    if exit_category in {
        "TP",
        "SL_FULL",
        "BE_FLAT",
        "BE_PLUS",
        "TRAILED_SL_PROFIT",
        "TIME_EXIT",
    }:
        return exit_category

    if exit_category in {"BE_TRAIL", "BE"}:
        return "BE_PLUS" if float(r_multiple) > 0 else "BE_FLAT"

    if exit_reason == "BE_PLUS_1R_TRAIL":
        return "BE_PLUS"

    if exit_reason in {"BE_TRAIL", "BE", "BREAKEVEN"}:
        return "BE_PLUS" if float(r_multiple) > 0 else "BE_FLAT"

    if exit_reason.startswith("TRAIL_LOCK_"):
        return "TRAILED_SL_PROFIT"

    if result == "TP":
        return "TP"

    if result == "SL":
        return "SL_FULL"

    if result == "BE":
        return "BE_PLUS" if float(r_multiple) > 0 else "BE_FLAT"

    if result in {"CLOSE_PROFIT", "CLOSE_LOSS", "CLOSE_FLAT", "TIME_EXIT", "TIME"}:
        return "TIME_EXIT"

    if exit_reason == "TIME_EXIT":
        return "TIME_EXIT"

    return "UNKNOWN_EXIT"


def build_v5_managed_exit_metrics(trades: pd.DataFrame) -> dict:
    """
    Build managed-exit metrics for comparison reporting.

    TP remains pure full-target hits.
    BE_PLUS and TRAILED_SL_PROFIT count as positive exits, not pure TP.
    """
    managed_categories = [
        "TP",
        "SL_FULL",
        "BE_FLAT",
        "BE_PLUS",
        "TRAILED_SL_PROFIT",
        "TIME_EXIT",
    ]

    if trades is None or len(trades) == 0:
        return {
            "target_tp_count": 0,
            "sl_full_count": 0,
            "be_flat_count": 0,
            "be_plus_count": 0,
            "trailed_profit_count": 0,
            "time_exit_count": 0,
            "target_tp_rate": 0.0,
            "positive_exit_rate": 0.0,
            "win_rate_ex_flat_be": 0.0,
            "non_loss_rate": 0.0,
        }

    report = trades.copy()
    report["r_multiple"] = pd.to_numeric(
        report.get("r_multiple", 0.0),
        errors="coerce",
    ).fillna(0.0)

    report["managed_exit_category"] = report.apply(
        infer_v5_managed_exit_category,
        axis=1,
    )

    total_trades = int(len(report))
    counts = report["managed_exit_category"].value_counts().to_dict()

    positive_exit_count = int((report["r_multiple"] > 0).sum())
    non_loss_count = int((report["r_multiple"] >= 0).sum())
    non_flat_count = int((report["r_multiple"] != 0).sum())

    return {
        "target_tp_count": int(counts.get("TP", 0)),
        "sl_full_count": int(counts.get("SL_FULL", 0)),
        "be_flat_count": int(counts.get("BE_FLAT", 0)),
        "be_plus_count": int(counts.get("BE_PLUS", 0)),
        "trailed_profit_count": int(counts.get("TRAILED_SL_PROFIT", 0)),
        "time_exit_count": int(counts.get("TIME_EXIT", 0)),
        "target_tp_rate": safe_divide(int(counts.get("TP", 0)), total_trades),
        "full_sl_rate": safe_divide(int(counts.get("SL_FULL", 0)), total_trades),
        "positive_exit_rate": safe_divide(positive_exit_count, total_trades),
        "win_rate_ex_flat_be": safe_divide(positive_exit_count, non_flat_count),
        "non_loss_rate": safe_divide(non_loss_count, total_trades),
    }


def build_performance_summary(
    raw_ohlc_df: pd.DataFrame,
    signals_df: pd.DataFrame,
    trades_df: pd.DataFrame,
    skipped_signal_candidates_df: pd.DataFrame,
    skipped_execution_signals_df: pd.DataFrame,
    config: dict,
    data_file: Path,
) -> dict:
    """
    Build the main compact performance summary for the automated V4 backtest.
    """
    number_of_rows = len(raw_ohlc_df)

    raw_signal_count = int((signals_df["raw_signal_name"] != "NO_SIGNAL").sum())
    final_signal_count = int((signals_df["v1_signal_name"] != "NO_SIGNAL").sum())

    number_of_trades = len(trades_df)

    tp_count = count_result(trades_df, "TP")
    sl_count = count_result(trades_df, "SL")
    be_count = count_result(trades_df, "BE")
    managed_exit_metrics = build_v5_managed_exit_metrics(trades_df)

    original_tp_hits = 0
    original_tp_hit_rate = 0.0

    if trades_df is not None and len(trades_df) > 0:
        if "hit_original_tp_before_exit" in trades_df.columns:
            original_tp_hits = int(
                trades_df["hit_original_tp_before_exit"]
                .fillna(False)
                .astype(bool)
                .sum()
            )

            original_tp_hit_rate = original_tp_hits / len(trades_df)

    close_profit_count = count_result(trades_df, "CLOSE_PROFIT")
    close_loss_count = count_result(trades_df, "CLOSE_LOSS")
    close_flat_count = count_result(trades_df, "CLOSE_FLAT")

    tp_rate = safe_divide(tp_count, number_of_trades)
    sl_rate = safe_divide(sl_count, number_of_trades)
    be_rate = safe_divide(be_count, number_of_trades)

    win_rate_excluding_be = safe_divide(tp_count, tp_count + sl_count)

    if number_of_trades > 0:
        total_points = float(trades_df["points"].sum())
        total_r = float(trades_df["r_multiple"].sum())
        average_r = float(trades_df["r_multiple"].mean())
        total_account_pct = float(trades_df["account_pct"].sum())
        average_bars_held = float(trades_df["bars_held"].mean())
    else:
        total_points = 0.0
        total_r = 0.0
        average_r = 0.0
        total_account_pct = 0.0
        average_bars_held = 0.0

    skipped_generation_count = len(skipped_signal_candidates_df)
    skipped_execution_count = len(skipped_execution_signals_df)
    total_skipped_signals = skipped_generation_count + skipped_execution_count

    summary = {
        "dataset_name": config["dataset_name"],
        "strategy_version": config["strategy_version"],
        "strategy_filter": config["strategy_filter"],
        "csv_file": config["csv_file"],
        "resolved_data_file": str(data_file.relative_to(PROJECT_ROOT)),

        "number_of_rows": number_of_rows,
        "number_of_raw_signals": raw_signal_count,
        "number_of_final_signals": final_signal_count,
        "number_of_trades": number_of_trades,

        "tp_count": tp_count,
        "sl_count": sl_count,
        "be_count": be_count,
        "close_profit_count": close_profit_count,
        "close_loss_count": close_loss_count,
        "close_flat_count": close_flat_count,
        "target_tp_count": managed_exit_metrics["target_tp_count"],
        "original_tp_hits": original_tp_hits,
        "sl_full_count": managed_exit_metrics["sl_full_count"],
        "be_flat_count": managed_exit_metrics["be_flat_count"],
        "be_plus_count": managed_exit_metrics["be_plus_count"],
        "trailed_profit_count": managed_exit_metrics["trailed_profit_count"],
        "time_exit_count": managed_exit_metrics["time_exit_count"],

        "tp_rate": tp_rate,
        "sl_rate": sl_rate,
        "be_rate": be_rate,
        "win_rate_excluding_be": win_rate_excluding_be,
        "target_tp_rate": managed_exit_metrics["target_tp_rate"],
        "original_tp_hit_rate": original_tp_hit_rate,
        "full_sl_rate": managed_exit_metrics["full_sl_rate"],
        "positive_exit_rate": managed_exit_metrics["positive_exit_rate"],
        "win_rate_ex_flat_be": managed_exit_metrics["win_rate_ex_flat_be"],
        "non_loss_rate": managed_exit_metrics["non_loss_rate"],

        "total_points": total_points,
        "total_r": total_r,
        "average_r": average_r,
        "total_account_pct": total_account_pct,
        "average_bars_held": average_bars_held,

        "skipped_generation_signals": skipped_generation_count,
        "skipped_execution_signals": skipped_execution_count,
        "total_skipped_signals": total_skipped_signals,

        "sl_points": config["sl_points"],
        "tp_points": config["tp_points"],
        "be_trigger_points": config["be_trigger_points"],
        "risk_per_trade_pct": config["risk_per_trade_pct"],
        "daily_max_consecutive_losses": config["daily_max_consecutive_losses"],
        "daily_profit_cap_pct": config["daily_profit_cap_pct"],
        "no_new_trades_after": config["no_new_trades_after"],
        "session_timezone": config["session_timezone"],
    }

    return summary


performance_summary = build_performance_summary(
    raw_ohlc_df=raw_ohlc_df,
    signals_df=signals_df,
    trades_df=trades_df,
    skipped_signal_candidates_df=skipped_signal_candidates_df,
    skipped_execution_signals_df=skipped_execution_signals_df,
    config=CONFIG,
    data_file=DATA_FILE,
)

pprint(performance_summary)

## 27. Clear Backtest Result Printout

This section prints the V4 result summary in a readable format.

In [ ]:
def print_performance_summary(summary: dict) -> None:
    """
    Print the V4 performance summary in a clean readable format.
    """
    print("=" * 90)
    print("AUTOMATED VWAP V5 MODULAR ENGINE SUMMARY")
    print("=" * 90)

    print("\nDataset")
    print("-" * 90)
    print(f"Dataset name:                 {summary['dataset_name']}")
    print(f"CSV file:                     {summary['csv_file']}")
    print(f"Resolved data file:           {summary['resolved_data_file']}")
    print(f"Rows:                         {summary['number_of_rows']:,}")

    print("\nStrategy")
    print("-" * 90)
    print(f"Strategy version:             {summary['strategy_version']}")
    print(f"Strategy filter:              {summary['strategy_filter']}")
    print(f"SL points:                    {summary['sl_points']}")
    print(f"TP points:                    {summary['tp_points']}")
    print(f"BE trigger points:            {summary['be_trigger_points']}")
    print(f"Risk per trade:               {summary['risk_per_trade_pct']}%")
    print(f"Daily max consecutive losses: {summary['daily_max_consecutive_losses']}")
    print(f"Daily profit cap:             {summary['daily_profit_cap_pct']}%")
    print(f"No new trades after:          {summary['no_new_trades_after']} {summary['session_timezone']}")

    print("\nSignals")
    print("-" * 90)
    print(f"Raw signal candidates:        {summary['number_of_raw_signals']:,}")
    print(f"Final signals:                {summary['number_of_final_signals']:,}")
    print(f"Trades taken:                 {summary['number_of_trades']:,}")
    print(f"Skipped generation signals:   {summary['skipped_generation_signals']:,}")
    print(f"Skipped execution signals:    {summary['skipped_execution_signals']:,}")
    print(f"Total skipped signals:        {summary['total_skipped_signals']:,}")

    print("\nTrade Results")
    print("-" * 90)
    print(f"TP count:                     {summary['tp_count']:,}")
    print(f"SL count:                     {summary['sl_count']:,}")
    print(f"BE count:                     {summary['be_count']:,}")
    print(f"Close profit count:           {summary['close_profit_count']:,}")
    print(f"Close loss count:             {summary['close_loss_count']:,}")
    print(f"Close flat count:             {summary['close_flat_count']:,}")

    print("\nRates")
    print("-" * 90)
    print(f"TP rate:                      {summary['tp_rate']:.2%}")
    print(f"SL rate:                      {summary['sl_rate']:.2%}")
    print(f"BE rate:                      {summary['be_rate']:.2%}")
    print(f"Win rate excluding BE:        {summary['win_rate_excluding_be']:.2%}")

    print("\nPerformance")
    print("-" * 90)
    print(f"Total points:                 {summary['total_points']:.2f}")
    print(f"Total R:                      {summary['total_r']:.2f}R")
    print(f"Average R:                    {summary['average_r']:.3f}R")
    print(f"Total account %:              {summary['total_account_pct']:.2f}%")
    print(f"Average bars held:            {summary['average_bars_held']:.2f}")

    print("=" * 90)


print_performance_summary(performance_summary)

## 28. Daily Summary

This section prints the daily performance and daily risk-control behaviour.

In [ ]:
daily_display_cols = [
    "session_date",
    "final_signals_seen",
    "trades_taken",
    "skipped_execution_signals",
    "tp_count",
    "sl_count",
    "be_count",
    "other_exit_count",
    "realised_points",
    "realised_r",
    "realised_account_pct",
    "consecutive_losses",
    "max_consecutive_losses_seen",
    "daily_stop_triggered",
    "daily_stop_reason",
]

print("Daily summary:")

if daily_summary_df is not None and len(daily_summary_df) > 0:
    existing_daily_cols = [col for col in daily_display_cols if col in daily_summary_df.columns]
    print(daily_summary_df[existing_daily_cols].to_string(index=False))
else:
    print("No daily summary available.")

## 29. Signal Diagnostics Summary

This section summarises which signal candidates were blocked during signal generation and which valid signals were skipped during execution.

In [ ]:
print("Signal generation block reasons:")
if "v1_signal_block_reason" in signals_df.columns:
    print(signals_df["v1_signal_block_reason"].value_counts(dropna=False).to_string())
else:
    print("No signal-generation block reason column found.")

print("\nSkipped execution reasons:")
if skipped_execution_signals_df is not None and len(skipped_execution_signals_df) > 0:
    print(skipped_execution_signals_df["skip_reason"].value_counts(dropna=False).to_string())
else:
    print("No skipped execution signals.")

### V5 Regime and Router Diagnostics

This section summarises the active V5 regime, router, setup-source, and exit-management behaviour.

It checks how often each preliminary regime appears, how many candidates and final signals each route produces, and how the executed trades perform by regime and selected entry module.

The diagnostics do not change signal generation or trade simulation.

In [ ]:
def build_v4_regime_diagnostics(
    signals_df: pd.DataFrame,
    trades_df: pd.DataFrame,
) -> dict:
    """
    Build compact V4 regime and routing diagnostics.
    """
    required_signal_columns = [
        "v4_preliminary_regime_label",
        "v4_model_route",
        "v4_selected_entry_module",
        "raw_signal_name",
        "raw_signal_side",
        "v1_signal_name",
        "v1_signal_side",
        "v1_signal_block_reason",
    ]

    missing_signal_columns = [
        column for column in required_signal_columns if column not in signals_df.columns
    ]

    if missing_signal_columns:
        raise KeyError(
            "Missing V4 diagnostic signal columns: "
            + ", ".join(missing_signal_columns)
        )

    diagnostic_signals = signals_df.copy()

    diagnostic_signals["has_raw_candidate"] = (
        diagnostic_signals["raw_signal_name"] != "NO_SIGNAL"
    )

    diagnostic_signals["has_final_signal"] = (
        diagnostic_signals["v1_signal_name"] != "NO_SIGNAL"
    )

    total_bars = len(diagnostic_signals)

    regime_bar_summary = (
        diagnostic_signals.groupby("v4_preliminary_regime_label", dropna=False)
        .agg(
            bars=("datetime", "size"),
            raw_candidates=("has_raw_candidate", "sum"),
            final_signals=("has_final_signal", "sum"),
        )
        .reset_index()
        .sort_values("bars", ascending=False)
    )

    regime_bar_summary["bar_share"] = np.where(
        total_bars > 0,
        regime_bar_summary["bars"] / total_bars,
        0.0,
    )

    regime_bar_summary["raw_candidates_per_1000_bars"] = np.where(
        regime_bar_summary["bars"] > 0,
        1000 * regime_bar_summary["raw_candidates"] / regime_bar_summary["bars"],
        0.0,
    )

    regime_bar_summary["final_signals_per_1000_bars"] = np.where(
        regime_bar_summary["bars"] > 0,
        1000 * regime_bar_summary["final_signals"] / regime_bar_summary["bars"],
        0.0,
    )

    raw_candidates = diagnostic_signals[
        diagnostic_signals["has_raw_candidate"]
    ].copy()

    if len(raw_candidates) > 0:
        signal_route_summary = (
            raw_candidates.groupby(
                [
                    "v4_preliminary_regime_label",
                    "v4_model_route",
                    "v4_selected_entry_module",
                    "raw_signal_side",
                ],
                dropna=False,
            )
            .agg(
                raw_candidates=("raw_signal_name", "size"),
                final_signals=("has_final_signal", "sum"),
            )
            .reset_index()
            .sort_values(
                [
                    "v4_preliminary_regime_label",
                    "v4_model_route",
                    "v4_selected_entry_module",
                    "raw_signal_side",
                ]
            )
        )

        signal_route_summary["blocked_signals"] = (
            signal_route_summary["raw_candidates"]
            - signal_route_summary["final_signals"]
        )

        signal_block_summary = (
            raw_candidates.groupby(
                [
                    "v4_preliminary_regime_label",
                    "v4_model_route",
                    "v4_selected_entry_module",
                    "v1_signal_block_reason",
                ],
                dropna=False,
            )
            .size()
            .reset_index(name="signals")
            .sort_values(
                [
                    "v4_preliminary_regime_label",
                    "v4_model_route",
                    "v4_selected_entry_module",
                    "signals",
                ],
                ascending=[True, True, True, False],
            )
        )
    else:
        signal_route_summary = pd.DataFrame()
        signal_block_summary = pd.DataFrame()

    trade_route_rows = []

    required_trade_columns = [
        "v4_preliminary_regime_label",
        "v4_model_route",
        "v4_selected_entry_module",
        "side",
        "result",
        "r_multiple",
        "points",
        "bars_held",
    ]

    if trades_df is not None and len(trades_df) > 0:
        missing_trade_columns = [
            column for column in required_trade_columns if column not in trades_df.columns
        ]

        if missing_trade_columns:
            raise KeyError(
                "Missing V4 diagnostic trade columns: "
                + ", ".join(missing_trade_columns)
            )

        for group_key, group in trades_df.groupby(
            [
                "v4_preliminary_regime_label",
                "v4_model_route",
                "v4_selected_entry_module",
                "side",
            ],
            dropna=False,
        ):
            (
                regime_label,
                model_route,
                selected_entry_module,
                side,
            ) = group_key

            trade_count = len(group)
            tp_count = int((group["result"] == "TP").sum())
            sl_count = int((group["result"] == "SL").sum())
            be_count = int((group["result"] == "BE").sum())

            trade_route_rows.append(
                {
                    "v4_preliminary_regime_label": regime_label,
                    "v4_model_route": model_route,
                    "v4_selected_entry_module": selected_entry_module,
                    "side": side,
                    "trades": trade_count,
                    "tp": tp_count,
                    "sl": sl_count,
                    "be": be_count,
                    "tp_rate": safe_divide(tp_count, trade_count),
                    "sl_rate": safe_divide(sl_count, trade_count),
                    "total_r": float(group["r_multiple"].sum()),
                    "average_r": float(group["r_multiple"].mean()),
                    "total_points": float(group["points"].sum()),
                    "average_bars_held": float(group["bars_held"].mean()),
                }
            )

    trade_route_performance = pd.DataFrame(trade_route_rows)

    if len(trade_route_performance) > 0:
        trade_route_performance = trade_route_performance.sort_values(
            [
                "v4_preliminary_regime_label",
                "v4_model_route",
                "v4_selected_entry_module",
                "side",
            ]
        ).reset_index(drop=True)

    return {
        "regime_bar_summary": regime_bar_summary,
        "signal_route_summary": signal_route_summary,
        "signal_block_summary": signal_block_summary,
        "trade_route_performance": trade_route_performance,
    }


v4_regime_diagnostics = build_v4_regime_diagnostics(
    signals_df=signals_df,
    trades_df=trades_df,
)

v4_regime_bar_summary = v4_regime_diagnostics["regime_bar_summary"]
v4_signal_route_summary = v4_regime_diagnostics["signal_route_summary"]
v4_signal_block_summary = v4_regime_diagnostics["signal_block_summary"]
v4_trade_route_performance = v4_regime_diagnostics["trade_route_performance"]

print("V4 regime bar summary")
display(v4_regime_bar_summary)

print("V4 signal route summary")
display(v4_signal_route_summary)

print("V4 signal block summary")
display(v4_signal_block_summary)

print("V4 trade route performance")
display(v4_trade_route_performance)

## 30. Output Exports

This section saves the key V1 backtest outputs.

Saved outputs include:

- performance summary
- trade log
- daily summary
- skipped signal candidates
- skipped execution signals
- config snapshot

In [ ]:
def make_safe_filename_part(value: str) -> str:
    """
    Convert a string into a safe filename component.
    """
    return (
        str(value)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace("-", "_")
    )


def json_default(value):
    """
    JSON serializer for pandas/numpy/datetime values.
    """
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()

    if hasattr(value, "item"):
        return value.item()

    if hasattr(value, "isoformat"):
        return value.isoformat()

    return str(value)


def export_v3_outputs(
    performance_summary: dict,
    config: dict,
    trades_df: pd.DataFrame,
    daily_summary_df: pd.DataFrame,
    skipped_signal_candidates_df: pd.DataFrame,
    skipped_execution_signals_df: pd.DataFrame,
    output_dir: Path,
) -> dict:
    """
    Export the main V3 backtest outputs.
    """
    dataset_part = make_safe_filename_part(config["dataset_name"])
    filter_part = make_safe_filename_part(config["strategy_filter"])

    run_name = f"{filter_part}__{dataset_part}"

    run_output_dir = output_dir / run_name
    run_output_dir.mkdir(parents=True, exist_ok=True)

    performance_summary_path = run_output_dir / "performance_summary.csv"
    performance_summary_json_path = run_output_dir / "performance_summary.json"
    config_snapshot_path = run_output_dir / "config_snapshot.json"
    trade_log_path = run_output_dir / "trade_log.csv"
    daily_summary_path = run_output_dir / "daily_summary.csv"
    skipped_signal_candidates_path = run_output_dir / "skipped_signal_candidates.csv"
    skipped_execution_signals_path = run_output_dir / "skipped_execution_signals.csv"
    v4_regime_bar_summary_path = run_output_dir / "v4_regime_bar_summary.csv"
    v4_signal_route_summary_path = run_output_dir / "v4_signal_route_summary.csv"
    v4_signal_block_summary_path = run_output_dir / "v4_signal_block_summary.csv"
    v4_trade_route_performance_path = run_output_dir / "v4_trade_route_performance.csv"
    v5_setup_source_breakdown_path = run_output_dir / "v5_setup_source_breakdown.csv"
    v5_exit_mode_breakdown_path = run_output_dir / "v5_exit_mode_breakdown.csv"
    v5_exit_category_breakdown_path = run_output_dir / "v5_exit_category_breakdown.csv"
    v5_regime_breakdown_path = run_output_dir / "v5_regime_breakdown.csv"
    v5_session_breakdown_path = run_output_dir / "v5_session_breakdown.csv"
    v5_session_setup_breakdown_path = run_output_dir / "v5_session_setup_breakdown.csv"

    pd.DataFrame([performance_summary]).to_csv(performance_summary_path, index=False)

    with open(performance_summary_json_path, "w", encoding="utf-8") as file:
        json.dump(performance_summary, file, indent=4, default=json_default)

    with open(config_snapshot_path, "w", encoding="utf-8") as file:
        json.dump(config, file, indent=4, default=json_default)

    if trades_df is not None and len(trades_df) > 0:
        trades_df.to_csv(trade_log_path, index=False)
    else:
        pd.DataFrame().to_csv(trade_log_path, index=False)

    if daily_summary_df is not None and len(daily_summary_df) > 0:
        daily_summary_df.to_csv(daily_summary_path, index=False)
    else:
        pd.DataFrame().to_csv(daily_summary_path, index=False)

    if skipped_signal_candidates_df is not None and len(skipped_signal_candidates_df) > 0:
        skipped_signal_candidates_df.to_csv(skipped_signal_candidates_path, index=False)
    else:
        pd.DataFrame().to_csv(skipped_signal_candidates_path, index=False)

    if skipped_execution_signals_df is not None and len(skipped_execution_signals_df) > 0:
        skipped_execution_signals_df.to_csv(skipped_execution_signals_path, index=False)
    else:
        pd.DataFrame().to_csv(skipped_execution_signals_path, index=False)

    if "v4_regime_bar_summary" in globals() and len(v4_regime_bar_summary) > 0:
        v4_regime_bar_summary.to_csv(v4_regime_bar_summary_path, index=False)
    else:
        pd.DataFrame().to_csv(v4_regime_bar_summary_path, index=False)

    if "v4_signal_route_summary" in globals() and len(v4_signal_route_summary) > 0:
        v4_signal_route_summary.to_csv(v4_signal_route_summary_path, index=False)
    else:
        pd.DataFrame().to_csv(v4_signal_route_summary_path, index=False)

    if "v4_signal_block_summary" in globals() and len(v4_signal_block_summary) > 0:
        v4_signal_block_summary.to_csv(v4_signal_block_summary_path, index=False)
    else:
        pd.DataFrame().to_csv(v4_signal_block_summary_path, index=False)

    if "v4_trade_route_performance" in globals() and len(v4_trade_route_performance) > 0:
        v4_trade_route_performance.to_csv(v4_trade_route_performance_path, index=False)
    else:
        pd.DataFrame().to_csv(v4_trade_route_performance_path, index=False)

    output_paths = {
        "run_output_dir": run_output_dir,
        "performance_summary_csv": performance_summary_path,
        "performance_summary_json": performance_summary_json_path,
        "config_snapshot_json": config_snapshot_path,
        "trade_log_csv": trade_log_path,
        "daily_summary_csv": daily_summary_path,
        "skipped_signal_candidates_csv": skipped_signal_candidates_path,
        "skipped_execution_signals_csv": skipped_execution_signals_path,
        "v4_regime_bar_summary_csv": v4_regime_bar_summary_path,
        "v4_signal_route_summary_csv": v4_signal_route_summary_path,
        "v4_signal_block_summary_csv": v4_signal_block_summary_path,
        "v4_trade_route_performance_csv": v4_trade_route_performance_path,
        "v5_setup_source_breakdown_csv": v5_setup_source_breakdown_path,
        "v5_exit_mode_breakdown_csv": v5_exit_mode_breakdown_path,
        "v5_exit_category_breakdown_csv": v5_exit_category_breakdown_path,
        "v5_regime_breakdown_csv": v5_regime_breakdown_path,
        "v5_session_breakdown_csv": v5_session_breakdown_path,
        "v5_session_setup_breakdown_csv": v5_session_setup_breakdown_path,
    }

    v5_export_tables = {
        "v5_setup_source_breakdown_csv": (
            "v5_setup_source_breakdown",
            v5_setup_source_breakdown_path,
        ),
        "v5_exit_mode_breakdown_csv": (
            "v5_exit_mode_breakdown",
            v5_exit_mode_breakdown_path,
        ),
        "v5_exit_category_breakdown_csv": (
            "v5_exit_category_breakdown",
            v5_exit_category_breakdown_path,
        ),
        "v5_regime_breakdown_csv": (
            "v5_regime_breakdown",
            v5_regime_breakdown_path,
        ),
        "v5_session_breakdown_csv": (
            "v5_session_breakdown",
            v5_session_breakdown_path,
        ),
        "v5_session_setup_breakdown_csv": (
            "v5_session_setup_breakdown",
            v5_session_setup_breakdown_path,
        ),
    }

    for _, (global_name, path) in v5_export_tables.items():
        table = globals().get(global_name, pd.DataFrame())
        if table is not None and len(table) > 0:
            table.to_csv(path, index=False)
        else:
            pd.DataFrame().to_csv(path, index=False)

    return output_paths


output_paths = export_v3_outputs(
    performance_summary=performance_summary,
    config=CONFIG,
    trades_df=trades_df,
    daily_summary_df=daily_summary_df,
    skipped_signal_candidates_df=skipped_signal_candidates_df,
    skipped_execution_signals_df=skipped_execution_signals_df,
    output_dir=OUTPUT_DIR,
)

print("Saved V5 outputs:")

for name, path in output_paths.items():
    print(f"{name}: {path.relative_to(PROJECT_ROOT)}")

## 31. Final Trade Log Preview

This section prints the final trade log preview.

Use this to quickly inspect the most recent trades from the completed V4 backtest.

In [ ]:
final_trade_preview_cols = [
    "signal_time",
    "signal_name",
    "setup_source",
    "setup_family",
    "s_tier_touch_type",
    "s_tier_current_green_touch",
    "s_tier_locked_green_touch",
    "s_tier_band_shift_touch",
    "s_tier_clean_state_recent_failure_seen",
    "s_tier_clean_state_clean_closes_after_failure",
    "s_tier_clean_state_pass",
    "s_tier_clean_state_blocked",
    
    "a_tier_is_candidate",
    "a_tier_side",
    "a_tier_entry_timing_type",

    "a_tier_fast_reclaim_away_bars",
    "a_tier_fast_reclaim_max_depth_points",
    "a_tier_fast_reclaim_body_points",
    "a_tier_fast_reclaim_body_ratio",
    "a_tier_fast_reclaim_close_position",
    "a_tier_fast_reclaim_passed",

    "a_tier_delayed_pullback_candidate",
    "a_tier_delayed_pullback_passed",
    "a_tier_delayed_pullback_entry_bar",
    "a_tier_delayed_pullback_hold_bars",
    "a_tier_delayed_pullback_points",
    "a_tier_delayed_pullback_held_green",
    "a_tier_delayed_pullback_block_reason",
    "a_tier_delayed_pullback_strength_filter_pass",
    "a_tier_delayed_pullback_strength_filter_block_reason",
    "a_tier_delayed_pullback_red_shift_bucket",
    "a_tier_delayed_pullback_v2_activation_reason",
    "a_tier_delayed_pullback_v2_pass_used",

    "a_tier_candles_below_green_count",
    "a_tier_max_depth_below_green_points",
    "a_tier_reclaim_body_ratio",
    "a_tier_reclaim_close_position",
    "a_tier_second_close_body_ratio",
    "a_tier_second_close_position",
    "a_tier_second_close_extension_from_green",
    "a_tier_entry_directional_red_shift_points",
    "a_tier_entry_red_shift_bucket",
    "a_tier_depth_bucket",
    "a_tier_extension_bucket",
    "a_tier_reclaim_body_bucket",
    "a_tier_second_close_body_bucket",
    "a_tier_reclaim_quality_bucket",
    "a_tier_retrace_required",
    "a_tier_retrace_pass",
    "a_tier_retrace_points",
    "a_tier_retrace_close_extension_from_green",
    "a_tier_retrace_hold_green",
    "a_tier_retrace_fail_reason",

    "original_tp_points",
    "original_tp_r",
    "original_tp_price",
    "hit_original_tp_before_exit",
    "original_tp_hit_time",
    "bars_to_original_tp",

    "regime_20m",
    "router_decision",
    "v2_health_state",
    "v2_filter_applied",
    "red_shift_bucket",
    "side",
    "entry_time",
    "entry_time_of_day",
    "trade_session",
    "session_enabled",
    "blocked_by_session",
    "entry_price",
    "exit_time",
    "exit_price",
    "result",
    "result_reason",
    "exit_mode",
    "exit_reason",
    "exit_category",
    "points",
    "r_multiple",
    "account_pct",
    "bars_held",
    "tp_points",
    "sl_points",
    "runner_enabled",
    "trail_activated",
    "trail_lock_level",
    "max_favourable_points",
    "max_adverse_points",
    "session_date",
]

print("Final trade log preview:")

if trades_df is not None and len(trades_df) > 0:
    existing_preview_cols = [col for col in final_trade_preview_cols if col in trades_df.columns]
    print(trades_df[existing_preview_cols].tail(50).to_string(index=False))
else:
    print("No trades were taken.")

## 32. V5 Run Complete

The automated VWAP V5 backtest runner has completed.

This notebook now produces:

- cleaned market data
- existing VWAP engine output
- automation features
- inherited green reclaim / rejection signal candidates
- filtered signals
- simulated trades
- daily risk-control summary
- compact performance summary
- exported result files

In [ ]:
print("Automated VWAP V5 modular engine run complete.")
print(f"Strategy version: {CONFIG['strategy_version']}")
print(f"Dataset: {CONFIG['dataset_name']}")
print(f"Trades taken: {performance_summary['number_of_trades']:,}")
print(f"Total R: {performance_summary['total_r']:.2f}R")
print(f"Total account %: {performance_summary['total_account_pct']:.2f}%")
print(f"Output folder: {output_paths['run_output_dir'].relative_to(PROJECT_ROOT)}")

In [ ]:
# Final compact V5 results table

if "v5_overall_results" in globals() and len(v5_overall_results) > 0:
    compact_results = v5_format_reporting_table(
        v5_overall_results[
            [
                "dataset",
                "trades",
                "TP",
                "SL_FULL",
                "BE_TRAIL",
                "TRAILED_SL_PROFIT",
                "TIME_EXIT",
                "WR ex BE",
                "total_R",
                "avg_R",
                "max_DD_R",
                "max_consecutive_SL_FULL",
                "worst_no_TP_run_R",
            ]
        ]
    )
else:
    compact_results = pd.DataFrame(
        [
            {
                "dataset": performance_summary["dataset_name"],
                "trades": performance_summary["number_of_trades"],
                "TP": performance_summary["tp_count"],
                "SL_FULL": performance_summary["sl_count"],
                "BE_TRAIL": performance_summary["be_count"],
                "TRAILED_SL_PROFIT": 0,
                "TIME_EXIT": (
                    performance_summary["close_profit_count"]
                    + performance_summary["close_loss_count"]
                    + performance_summary["close_flat_count"]
                ),
                "WR ex BE": f"{performance_summary['win_rate_excluding_be']:.2%}",
                "total_R": f"{performance_summary['total_r']:.2f}R",
                "avg_R": f"{performance_summary['average_r']:.3f}R",
                "max_DD_R": "n/a",
                "max_consecutive_SL_FULL": "n/a",
                "worst_no_TP_run_R": "n/a",
            }
        ]
    )

compact_results

## V5 Modular Results Breakdown

This section prints compact V5 reporting tables using the trade-tracking metadata.

The breakdowns separate setup source, exit mode, exit category, regime, and trading session so the model can be reviewed without flooding the notebook with a full trade log.

In [ ]:
# ---------------------------------------------------------------------
# V5 compact reporting helpers
# ---------------------------------------------------------------------

V5_EXIT_CATEGORY_ORDER = [
    "TP",
    "SL_FULL",
    "BE_TRAIL",
    "TRAILED_SL_PROFIT",
    "TIME_EXIT",
]


def v5_safe_trades_df(trades: pd.DataFrame) -> pd.DataFrame:
    """
    Return a trade dataframe with the V5 reporting columns present.
    """
    if trades is None or len(trades) == 0:
        return pd.DataFrame()

    out = trades.copy()

    defaults = {
        "setup_source": "UNKNOWN_SETUP_SOURCE",
        "setup_family": "UNKNOWN",
        "engine_mode": CONFIG.get("engine_mode", "intelligent"),
        "regime_20m": "not_available",
        "router_decision": "not_available",
        "v2_health_state": "not_available",
        "v2_trend_health_active": False,
        "v2_activation_reason": "TRACK_ONLY",
        "v2_router_required": False,
        "v2_force_after_time_active": False,
        "v2_filter_result": "NOT_APPLIED",
        "v2_filter_tracked_only": False,
        "v2_health_failure_reason": None,

        "red_shift_bucket": "unknown",
        
        "s_tier_current_green_touch": False,
        "s_tier_locked_green_touch": False,
        "s_tier_band_shift_touch": False,
        "s_tier_touch_type": "NOT_S_TIER",
        "s_tier_clean_state_recent_failure_seen": False,
        "s_tier_clean_state_clean_closes_after_failure": np.nan,
        "s_tier_clean_state_pass": False,
        "s_tier_clean_state_blocked": False,
        "s_tier_clean_state_block_reason": "NOT_BLOCKED",

        "trade_session": "unknown",
        "exit_mode": "FIXED_TP_SL_BE",
        "exit_reason": "UNKNOWN_EXIT_REASON",
        "exit_category": "UNKNOWN_EXIT_CATEGORY",
        "r_multiple": 0.0,
        "original_tp_points": np.nan,
        "original_tp_r": np.nan,
        "original_tp_price": np.nan,
        "hit_original_tp_before_exit": False,
        "original_tp_hit_time": pd.NaT,
        "bars_to_original_tp": np.nan,
        "points": 0.0,
        "entry_time": pd.NaT,
        "runner_enabled": False,
        "runner_stop_widened": False,
        "runner_stop_updates": 0,
        "runner_mode": CONFIG.get("runner_mode", "smart"),
        "enable_be_plus_buffer": CONFIG.get("enable_be_plus_buffer", True),
        "be_plus_buffer_points": CONFIG.get("be_plus_buffer_points", 3.0),

        "a_tier_is_candidate": False,
        "a_tier_side": "UNKNOWN",
        "a_tier_entry_timing_type": "UNKNOWN",
        "a_tier_candles_below_green_count": np.nan,
        "a_tier_max_depth_below_green_points": np.nan,
        "a_tier_reclaim_body_points": np.nan,
        "a_tier_reclaim_range_points": np.nan,
        "a_tier_reclaim_body_ratio": np.nan,
        "a_tier_reclaim_close_position": np.nan,
        "a_tier_second_close_body_points": np.nan,
        "a_tier_second_close_range_points": np.nan,
        "a_tier_second_close_body_ratio": np.nan,
        "a_tier_second_close_position": np.nan,
        "a_tier_second_close_extension_from_green": np.nan,
        "a_tier_entry_directional_red_shift_points": np.nan,
        "a_tier_entry_red_shift_bucket": "UNKNOWN",
        "a_tier_depth_bucket": "unknown",
        "a_tier_extension_bucket": "unknown",
        "a_tier_reclaim_body_bucket": "unknown",
        "a_tier_second_close_body_bucket": "unknown",
        "a_tier_reclaim_quality_bucket": "unknown",
        "a_tier_retrace_required": False,
        "a_tier_retrace_pass": False,
        "a_tier_retrace_points": np.nan,
        "a_tier_retrace_close_extension_from_green": np.nan,
        "a_tier_retrace_hold_green": False,
        "a_tier_retrace_fail_reason": "NOT_RETRACE_ENTRY",

        "a_tier_fast_reclaim_away_bars": np.nan,
        "a_tier_fast_reclaim_max_depth_points": np.nan,
        "a_tier_fast_reclaim_body_points": np.nan,
        "a_tier_fast_reclaim_body_ratio": np.nan,
        "a_tier_fast_reclaim_close_position": np.nan,
        "a_tier_fast_reclaim_passed": False,

        "a_tier_delayed_pullback_candidate": False,
        "a_tier_delayed_pullback_passed": False,
        "a_tier_delayed_pullback_entry_bar": np.nan,
        "a_tier_delayed_pullback_hold_bars": np.nan,
        "a_tier_delayed_pullback_points": np.nan,
        "a_tier_delayed_pullback_held_green": False,
        "a_tier_delayed_pullback_block_reason": "UNKNOWN",
        "a_tier_delayed_pullback_strength_filter_pass": False,
        "a_tier_delayed_pullback_strength_filter_block_reason": "UNKNOWN",
        "a_tier_delayed_pullback_red_shift_bucket": "unknown",
        "a_tier_delayed_pullback_v2_activation_reason": "UNKNOWN",
        "a_tier_delayed_pullback_v2_pass_used": False,
    }

    for column, default_value in defaults.items():
        if column not in out.columns:
            out[column] = default_value

    out["r_multiple"] = pd.to_numeric(out["r_multiple"], errors="coerce").fillna(0.0)
    out["points"] = pd.to_numeric(out["points"], errors="coerce").fillna(0.0)

    return out


def v5_max_drawdown_r(r_values: pd.Series) -> float:
    """
    Calculate max closed-trade drawdown in R.
    """
    if r_values is None or len(r_values) == 0:
        return 0.0

    equity = r_values.astype(float).cumsum()
    peak = equity.cummax()
    drawdown = equity - peak

    return float(drawdown.min())


def v5_max_consecutive_sl_full(exit_categories: pd.Series) -> int:
    """
    Calculate max consecutive full stop losses.
    """
    max_streak = 0
    current_streak = 0

    for value in exit_categories.astype(str):
        if value == "SL_FULL":
            current_streak += 1
            max_streak = max(max_streak, current_streak)
        else:
            current_streak = 0

    return int(max_streak)


def v5_worst_no_tp_run_r(group: pd.DataFrame) -> float:
    """
    Calculate worst run in R without a TP.
    """
    if group is None or len(group) == 0:
        return 0.0

    current_run = 0.0
    worst_run = 0.0

    ordered = group.sort_values("entry_time").copy()

    for _, trade in ordered.iterrows():
        r_value = float(trade.get("r_multiple", 0.0))
        exit_category = str(trade.get("exit_category", ""))

        if exit_category == "TP":
            current_run = 0.0
        else:
            current_run += r_value
            worst_run = min(worst_run, current_run)

    return float(worst_run)


def v5_win_rate_ex_be(group: pd.DataFrame) -> float:
    """
    Win rate excluding breakeven trail exits.
    """
    if group is None or len(group) == 0:
        return np.nan

    eligible = group[group["exit_category"] != "BE_TRAIL"]

    if len(eligible) == 0:
        return np.nan

    return float((eligible["exit_category"] == "TP").mean())


def v5_exit_category_counts(group: pd.DataFrame) -> dict:
    """
    Return standard V5 exit-category counts.
    """
    counts = group["exit_category"].value_counts(dropna=False).to_dict()

    return {
        category: int(counts.get(category, 0))
        for category in V5_EXIT_CATEGORY_ORDER
    }


def v5_build_reporting_row(group: pd.DataFrame) -> dict:
    """
    Build one compact V5 reporting row.
    """
    exit_counts = v5_exit_category_counts(group)

    return {
        "trades": int(len(group)),
        **exit_counts,
        "WR ex BE": v5_win_rate_ex_be(group),
        "total_R": float(group["r_multiple"].sum()),
        "avg_R": float(group["r_multiple"].mean()) if len(group) > 0 else 0.0,
        "max_DD_R": v5_max_drawdown_r(group["r_multiple"]),
        "max_consecutive_SL_FULL": v5_max_consecutive_sl_full(group["exit_category"]),
        "worst_no_TP_run_R": v5_worst_no_tp_run_r(group),
    }


def v5_build_breakdown_table(
    trades: pd.DataFrame,
    group_cols: list[str],
) -> pd.DataFrame:
    """
    Build a compact grouped V5 reporting table.
    """
    trades = v5_safe_trades_df(trades)

    if trades.empty:
        return pd.DataFrame()

    rows = []

    for group_key, group in trades.groupby(group_cols, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        row = {
            column: value
            for column, value in zip(group_cols, group_key)
        }

        row.update(v5_build_reporting_row(group))
        rows.append(row)

    breakdown = pd.DataFrame(rows)

    if "total_R" in breakdown.columns:
        breakdown = breakdown.sort_values(
            ["total_R", "trades"],
            ascending=[False, False],
        )

    return breakdown.reset_index(drop=True)


def v5_format_reporting_table(table: pd.DataFrame) -> pd.DataFrame:
    """
    Format percentage and R columns for compact notebook display.
    """
    if table is None or len(table) == 0:
        return pd.DataFrame()

    out = table.copy()

    if "WR ex BE" in out.columns:
        out["WR ex BE"] = out["WR ex BE"].map(
            lambda value: "n/a" if pd.isna(value) else f"{value:.2%}"
        )

    for column in ["total_R", "avg_R", "max_DD_R", "worst_no_TP_run_R"]:
        if column in out.columns:
            out[column] = out[column].map(lambda value: f"{float(value):.2f}R")

    return out

def v5_build_positive_exit_summary(trades: pd.DataFrame) -> pd.DataFrame:
    """
    Build clearer exit-quality metrics for fixed and runner exits.

    Target TP rate = how often full target was hit.
    Positive exit rate = how often the trade made money.
    WR ex flat BE = positive exits after removing true zero-R BE exits.
    Total R and avg R remain the main performance truth.
    """
    trades = v5_safe_trades_df(trades)

    if trades.empty:
        return pd.DataFrame(
            [
                {
                    "closed_trades": 0,
                    "target_tp_count": 0,
                    "target_tp_rate": 0.0,
                    "be_flat_count": 0,
                    "be_plus_count": 0,
                    "trailed_profit_count": 0,
                    "full_sl_count": 0,
                    "positive_exit_count": 0,
                    "positive_exit_rate": 0.0,
                    "win_rate_ex_flat_be": 0.0,
                    "non_loss_count": 0,
                    "non_loss_rate": 0.0,
                    "total_R": 0.0,
                    "avg_R": 0.0,
                }
            ]
        )

    report = trades.copy()
    report["r_multiple"] = pd.to_numeric(report["r_multiple"], errors="coerce").fillna(0.0)
    report["exit_category"] = report["exit_category"].fillna("UNKNOWN_EXIT").astype(str)

    total_trades = int(len(report))

    target_tp_count = int((report["exit_category"] == "TP").sum())
    be_flat_count = int((report["exit_category"] == "BE_FLAT").sum())
    be_plus_count = int((report["exit_category"] == "BE_PLUS").sum())
    trailed_profit_count = int((report["exit_category"] == "TRAILED_SL_PROFIT").sum())
    full_sl_count = int((report["exit_category"] == "SL_FULL").sum())

    positive_exit_count = int((report["r_multiple"] > 0).sum())
    non_loss_count = int((report["r_multiple"] >= 0).sum())

    non_flat_count = max(total_trades - be_flat_count, 0)

    return pd.DataFrame(
        [
            {
                "closed_trades": total_trades,
                "target_tp_count": target_tp_count,
                "target_tp_rate": target_tp_count / total_trades if total_trades else 0.0,
                "be_flat_count": be_flat_count,
                "be_plus_count": be_plus_count,
                "trailed_profit_count": trailed_profit_count,
                "full_sl_count": full_sl_count,
                "positive_exit_count": positive_exit_count,
                "positive_exit_rate": positive_exit_count / total_trades if total_trades else 0.0,
                "win_rate_ex_flat_be": (
                    positive_exit_count / non_flat_count
                    if non_flat_count
                    else 0.0
                ),
                "non_loss_count": non_loss_count,
                "non_loss_rate": non_loss_count / total_trades if total_trades else 0.0,
                "total_R": float(report["r_multiple"].sum()),
                "avg_R": float(report["r_multiple"].mean()) if total_trades else 0.0,
            }
        ]
    )

def v5_bucket_max_drawdown_r(group: pd.DataFrame) -> float:
    """
    Calculate simple max drawdown in R for a grouped set of trades.
    """
    if group.empty or "r_multiple" not in group.columns:
        return 0.0

    ordered = group.copy()

    if "entry_time" in ordered.columns:
        ordered = ordered.sort_values("entry_time")

    cumulative_r = pd.to_numeric(
        ordered["r_multiple"],
        errors="coerce",
    ).fillna(0.0).cumsum()

    running_peak = cumulative_r.cummax()
    drawdown = cumulative_r - running_peak

    return float(drawdown.min()) if len(drawdown) else 0.0


def v5_filter_a_tier_trades(trades: pd.DataFrame) -> pd.DataFrame:
    """
    Return executed A-tier/V3 trades for diagnostic reporting.
    """
    trades = v5_safe_trades_df(trades)

    if trades.empty:
        return trades.copy()

    setup_source = trades.get(
        "setup_source",
        pd.Series("", index=trades.index),
    ).fillna("").astype(str)

    a_tier_flag = trades.get(
        "a_tier_is_candidate",
        pd.Series(False, index=trades.index),
    ).fillna(False).astype(bool)

    return trades[
        a_tier_flag
        | setup_source.str.contains("V3_A_TIER", na=False)
    ].copy()


def v5_build_a_tier_bucket_performance_table(
    trades: pd.DataFrame,
    bucket_col: str,
) -> pd.DataFrame:
    """
    Build compact A-tier performance by one diagnostic bucket.
    """
    a_tier_trades = v5_filter_a_tier_trades(trades)

    output_cols = [
        bucket_col,
        "trades",
        "TP",
        "SL_FULL",
        "BE_FLAT",
        "BE_PLUS",
        "TRAILED_SL_PROFIT",
        "TIME_EXIT",
        "total_R",
        "avg_R",
        "positive_exit_rate",
        "full_sl_rate",
        "max_drawdown_R",
    ]

    if a_tier_trades.empty or bucket_col not in a_tier_trades.columns:
        return pd.DataFrame(columns=output_cols)

    report = a_tier_trades.copy()
    report[bucket_col] = report[bucket_col].fillna("unknown").astype(str)
    report["exit_category"] = report["exit_category"].fillna("UNKNOWN_EXIT").astype(str)
    report["r_multiple"] = pd.to_numeric(report["r_multiple"], errors="coerce").fillna(0.0)

    grouped = report.groupby(bucket_col, dropna=False)

    summary = grouped.agg(
        trades=("r_multiple", "size"),
        total_R=("r_multiple", "sum"),
        avg_R=("r_multiple", "mean"),
        positive_exit_rate=("r_multiple", lambda x: float((x > 0).mean())),
        full_sl_rate=("exit_category", lambda x: float((x == "SL_FULL").mean())),
    ).reset_index()

    exit_counts = (
        report.pivot_table(
            index=bucket_col,
            columns="exit_category",
            values="r_multiple",
            aggfunc="size",
            fill_value=0,
        )
        .reset_index()
    )

    for exit_col in ["TP", "SL_FULL", "BE_FLAT", "BE_PLUS", "TRAILED_SL_PROFIT", "TIME_EXIT"]:
        if exit_col not in exit_counts.columns:
            exit_counts[exit_col] = 0

    drawdowns = (
        grouped.apply(v5_bucket_max_drawdown_r)
        .rename("max_drawdown_R")
        .reset_index()
    )

    result = (
        summary
        .merge(
            exit_counts[
                [
                    bucket_col,
                    "TP",
                    "SL_FULL",
                    "BE_FLAT",
                    "BE_PLUS",
                    "TRAILED_SL_PROFIT",
                    "TIME_EXIT",
                ]
            ],
            on=bucket_col,
            how="left",
        )
        .merge(drawdowns, on=bucket_col, how="left")
    )

    result = result[output_cols]

    return result.sort_values(["total_R", "trades"], ascending=[True, False])


def v5_build_a_tier_quick_summary(
    trades: pd.DataFrame,
    signals: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build a compact A-tier diagnostic headline summary.
    """
    trades = v5_safe_trades_df(trades)
    a_tier_trades = v5_filter_a_tier_trades(trades)

    setup_source = trades.get(
        "setup_source",
        pd.Series("", index=trades.index),
    ).fillna("").astype(str)

    s_tier_trades = trades[
        setup_source.str.contains("V1_S_TIER", na=False)
    ].copy()

    a_tier_total_r = (
        float(pd.to_numeric(a_tier_trades["r_multiple"], errors="coerce").fillna(0.0).sum())
        if not a_tier_trades.empty
        else 0.0
    )

    s_tier_total_r = (
        float(pd.to_numeric(s_tier_trades["r_multiple"], errors="coerce").fillna(0.0).sum())
        if not s_tier_trades.empty
        else 0.0
    )

    worst_bucket = "not_available"
    best_bucket = "not_available"

    extension_table = v5_build_a_tier_bucket_performance_table(
        a_tier_trades,
        "a_tier_extension_bucket",
    )

    if not extension_table.empty:
        worst_bucket = str(extension_table.sort_values("total_R").iloc[0]["a_tier_extension_bucket"])
        best_bucket = str(extension_table.sort_values("total_R", ascending=False).iloc[0]["a_tier_extension_bucket"])

    most_common_a_tier_block_reason = "not_available"

    if signals is not None and len(signals) > 0 and "v1_signal_block_reason" in signals.columns:
        signal_source = signals.get(
            "setup_source",
            pd.Series("", index=signals.index),
        ).fillna("").astype(str)

        signal_a_tier_flag = signals.get(
            "a_tier_is_candidate",
            pd.Series(False, index=signals.index),
        ).fillna(False).astype(bool)

        blocked_a_tier = signals[
            signal_a_tier_flag
            | signal_source.str.contains("V3_A_TIER", na=False)
        ].copy()

        blocked_a_tier = blocked_a_tier[
            blocked_a_tier["v1_signal_block_reason"].astype(str) != "VALID_SIGNAL"
        ]

        if not blocked_a_tier.empty:
            most_common_a_tier_block_reason = str(
                blocked_a_tier["v1_signal_block_reason"]
                .fillna("UNKNOWN")
                .astype(str)
                .value_counts()
                .idxmax()
            )

    return pd.DataFrame(
        [
            {
                "a_tier_trades": int(len(a_tier_trades)),
                "a_tier_total_R": a_tier_total_r,
                "s_tier_trades": int(len(s_tier_trades)),
                "s_tier_total_R": s_tier_total_r,
                "worst_a_tier_extension_bucket_by_total_R": worst_bucket,
                "best_a_tier_extension_bucket_by_total_R": best_bucket,
                "most_common_a_tier_block_reason": most_common_a_tier_block_reason,
            }
        ]
    )

def v5_add_a_tier_retrace_reporting_buckets(trades: pd.DataFrame) -> pd.DataFrame:
    """
    Add compact retrace reporting buckets to A-tier trades.

    These buckets are reporting-only and do not affect strategy behaviour.
    """
    trades = v5_safe_trades_df(trades).copy()

    retrace_points = pd.to_numeric(
        trades.get("a_tier_retrace_points", np.nan),
        errors="coerce",
    )

    retrace_extension = pd.to_numeric(
        trades.get("a_tier_retrace_close_extension_from_green", np.nan),
        errors="coerce",
    )

    retrace_entry = (
        trades.get(
            "a_tier_entry_timing_type",
            pd.Series("UNKNOWN", index=trades.index),
        )
        .fillna("UNKNOWN")
        .astype(str)
        == "RETRACE_ENTRY"
    )

    trades["a_tier_retrace_points_bucket"] = np.select(
        [
            retrace_entry & retrace_points.notna() & (retrace_points < 5),
            retrace_entry & retrace_points.notna() & (retrace_points >= 5) & (retrace_points < 15),
            retrace_entry & retrace_points.notna() & (retrace_points >= 15) & (retrace_points <= 25),
            retrace_entry & retrace_points.notna() & (retrace_points > 25),
        ],
        [
            "too_small",
            "controlled",
            "deep_controlled",
            "too_deep",
        ],
        default="not_retrace_entry",
    )

    trades["a_tier_retrace_close_extension_bucket"] = np.select(
        [
            retrace_entry & retrace_extension.notna() & (retrace_extension < 0),
            retrace_entry & retrace_extension.notna() & (retrace_extension >= 0) & (retrace_extension < 5),
            retrace_entry & retrace_extension.notna() & (retrace_extension >= 5) & (retrace_extension < 15),
            retrace_entry & retrace_extension.notna() & (retrace_extension >= 15) & (retrace_extension <= 25),
            retrace_entry & retrace_extension.notna() & (retrace_extension > 25),
        ],
        [
            "lost_green",
            "near_green",
            "clean_hold",
            "extended_hold",
            "too_extended",
        ],
        default="not_retrace_entry",
    )

    return trades


def v5_build_a_tier_retrace_performance_table(
    trades: pd.DataFrame,
    bucket_col: str,
) -> pd.DataFrame:
    """
    Build compact A-tier retrace performance by one timing/retrace bucket.
    """
    trades = v5_add_a_tier_retrace_reporting_buckets(trades)

    return v5_build_a_tier_bucket_performance_table(
        trades,
        bucket_col,
    )


def v5_build_a_tier_retrace_fail_summary(
    skipped_execution: pd.DataFrame,
) -> pd.DataFrame:
    """
    Summarise blocked A-tier retrace candidates by fail reason.
    """
    if skipped_execution is None or len(skipped_execution) == 0:
        return pd.DataFrame(
            columns=[
                "a_tier_retrace_fail_reason",
                "blocked_candidates",
                "avg_retrace_points",
                "avg_retrace_close_extension_from_green",
            ]
        )

    skipped = skipped_execution.copy()

    if "a_tier_retrace_fail_reason" not in skipped.columns:
        return pd.DataFrame(
            columns=[
                "a_tier_retrace_fail_reason",
                "blocked_candidates",
                "avg_retrace_points",
                "avg_retrace_close_extension_from_green",
            ]
        )

    skipped["a_tier_retrace_fail_reason"] = (
        skipped["a_tier_retrace_fail_reason"]
        .fillna("NOT_RETRACE_ENTRY")
        .astype(str)
    )

    retrace_blocked = skipped[
        skipped["a_tier_retrace_fail_reason"].str.contains(
            "A_TIER_RETRACE",
            na=False,
        )
    ].copy()

    if retrace_blocked.empty:
        return pd.DataFrame(
            columns=[
                "a_tier_retrace_fail_reason",
                "blocked_candidates",
                "avg_retrace_points",
                "avg_retrace_close_extension_from_green",
            ]
        )

    retrace_blocked["a_tier_retrace_points"] = pd.to_numeric(
        retrace_blocked.get("a_tier_retrace_points", np.nan),
        errors="coerce",
    )

    retrace_blocked["a_tier_retrace_close_extension_from_green"] = pd.to_numeric(
        retrace_blocked.get("a_tier_retrace_close_extension_from_green", np.nan),
        errors="coerce",
    )

    return (
        retrace_blocked.groupby("a_tier_retrace_fail_reason", dropna=False)
        .agg(
            blocked_candidates=("a_tier_retrace_fail_reason", "size"),
            avg_retrace_points=("a_tier_retrace_points", "mean"),
            avg_retrace_close_extension_from_green=(
                "a_tier_retrace_close_extension_from_green",
                "mean",
            ),
        )
        .reset_index()
        .sort_values("blocked_candidates", ascending=False)
    )


def v5_build_a_tier_retrace_quick_summary(
    trades: pd.DataFrame,
    skipped_execution: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build a compact headline summary for A-tier retrace mode.
    """
    trades = v5_add_a_tier_retrace_reporting_buckets(trades)
    a_tier_trades = v5_filter_a_tier_trades(trades)

    if a_tier_trades.empty:
        second_close_trades = 0
        retrace_trades = 0
        second_close_total_r = 0.0
        retrace_total_r = 0.0
    else:
        timing_type = (
            a_tier_trades["a_tier_entry_timing_type"]
            .fillna("UNKNOWN")
            .astype(str)
        )

        second_close = a_tier_trades[timing_type == "SECOND_CLOSE"]
        retrace = a_tier_trades[timing_type == "RETRACE_ENTRY"]

        second_close_trades = int(len(second_close))
        retrace_trades = int(len(retrace))

        second_close_total_r = float(
            pd.to_numeric(second_close["r_multiple"], errors="coerce")
            .fillna(0.0)
            .sum()
        )

        retrace_total_r = float(
            pd.to_numeric(retrace["r_multiple"], errors="coerce")
            .fillna(0.0)
            .sum()
        )

    fail_summary = v5_build_a_tier_retrace_fail_summary(skipped_execution)

    if fail_summary.empty:
        blocked_retrace_candidates = 0
        most_common_retrace_fail_reason = "none"
    else:
        blocked_retrace_candidates = int(fail_summary["blocked_candidates"].sum())
        most_common_retrace_fail_reason = str(
            fail_summary.sort_values(
                "blocked_candidates",
                ascending=False,
            ).iloc[0]["a_tier_retrace_fail_reason"]
        )

    return pd.DataFrame(
        [
            {
                "enable_a_tier_retrace_entry": CONFIG.get("enable_a_tier_retrace_entry", False),
                "a_tier_retrace_entry_mode": CONFIG.get("a_tier_retrace_entry_mode", "extended_only"),
                "second_close_trades": second_close_trades,
                "second_close_total_R": second_close_total_r,
                "retrace_entry_trades": retrace_trades,
                "retrace_entry_total_R": retrace_total_r,
                "blocked_retrace_candidates": blocked_retrace_candidates,
                "most_common_retrace_fail_reason": most_common_retrace_fail_reason,
            }
        ]
    )

def v5_build_v2_activation_performance_table(
    trades: pd.DataFrame,
    group_cols: list[str],
) -> pd.DataFrame:
    """
    Build compact V2 activation performance by selected grouping columns.
    """
    trades = v5_safe_trades_df(trades)

    output_cols = (
        group_cols
        + [
            "trades",
            "TP",
            "SL_FULL",
            "BE_FLAT",
            "BE_PLUS",
            "TRAILED_SL_PROFIT",
            "TIME_EXIT",
            "total_R",
            "avg_R",
            "positive_exit_rate",
            "full_sl_rate",
        ]
    )

    if trades.empty:
        return pd.DataFrame(columns=output_cols)

    missing_cols = [col for col in group_cols if col not in trades.columns]

    if missing_cols:
        return pd.DataFrame(columns=output_cols)

    report = trades.copy()

    for col in group_cols:
        report[col] = report[col].fillna("UNKNOWN").astype(str)

    report["exit_category"] = report["exit_category"].fillna("UNKNOWN_EXIT").astype(str)
    report["r_multiple"] = pd.to_numeric(report["r_multiple"], errors="coerce").fillna(0.0)

    grouped = report.groupby(group_cols, dropna=False)

    summary = grouped.agg(
        trades=("r_multiple", "size"),
        total_R=("r_multiple", "sum"),
        avg_R=("r_multiple", "mean"),
        positive_exit_rate=("r_multiple", lambda x: float((x > 0).mean())),
        full_sl_rate=("exit_category", lambda x: float((x == "SL_FULL").mean())),
    ).reset_index()

    exit_counts = (
        report.pivot_table(
            index=group_cols,
            columns="exit_category",
            values="r_multiple",
            aggfunc="size",
            fill_value=0,
        )
        .reset_index()
    )

    for exit_col in ["TP", "SL_FULL", "BE_FLAT", "BE_PLUS", "TRAILED_SL_PROFIT", "TIME_EXIT"]:
        if exit_col not in exit_counts.columns:
            exit_counts[exit_col] = 0

    result = summary.merge(
        exit_counts[
            group_cols
            + [
                "TP",
                "SL_FULL",
                "BE_FLAT",
                "BE_PLUS",
                "TRAILED_SL_PROFIT",
                "TIME_EXIT",
            ]
        ],
        on=group_cols,
        how="left",
    )

    result = result[output_cols]

    return result.sort_values(["total_R", "trades"], ascending=[True, False])


def v5_build_v2_candidate_activation_summary(
    signals: pd.DataFrame,
    trades: pd.DataFrame,
) -> pd.DataFrame:
    """
    Summarise V2 activation candidate counts, blocked counts, and executed trades.
    """
    output_cols = [
        "v2_activation_reason",
        "candidate_count",
        "blocked_count",
        "executed_trades",
    ]

    if signals is None or len(signals) == 0:
        return pd.DataFrame(columns=output_cols)

    signal_report = signals.copy()

    if "v2_activation_reason" not in signal_report.columns:
        return pd.DataFrame(columns=output_cols)

    signal_report["v2_activation_reason"] = (
        signal_report["v2_activation_reason"]
        .fillna("UNKNOWN")
        .astype(str)
    )

    raw_signal_name = signal_report.get(
        "raw_signal_name",
        pd.Series("NO_SIGNAL", index=signal_report.index),
    ).fillna("NO_SIGNAL").astype(str)

    candidate_mask = raw_signal_name != "NO_SIGNAL"

    block_reason = signal_report.get(
        "v1_signal_block_reason",
        pd.Series("UNKNOWN", index=signal_report.index),
    ).fillna("UNKNOWN").astype(str)

    blocked_mask = candidate_mask & block_reason.str.startswith("BLOCKED_BY_V2")

    candidate_summary = (
        signal_report[candidate_mask]
        .groupby("v2_activation_reason", dropna=False)
        .agg(candidate_count=("v2_activation_reason", "size"))
        .reset_index()
    )

    blocked_summary = (
        signal_report[blocked_mask]
        .groupby("v2_activation_reason", dropna=False)
        .agg(blocked_count=("v2_activation_reason", "size"))
        .reset_index()
    )

    trade_report = v5_safe_trades_df(trades)

    if trade_report.empty or "v2_activation_reason" not in trade_report.columns:
        executed_summary = pd.DataFrame(
            columns=["v2_activation_reason", "executed_trades"]
        )
    else:
        trade_report["v2_activation_reason"] = (
            trade_report["v2_activation_reason"]
            .fillna("UNKNOWN")
            .astype(str)
        )

        executed_summary = (
            trade_report
            .groupby("v2_activation_reason", dropna=False)
            .agg(executed_trades=("v2_activation_reason", "size"))
            .reset_index()
        )

    result = (
        candidate_summary
        .merge(blocked_summary, on="v2_activation_reason", how="outer")
        .merge(executed_summary, on="v2_activation_reason", how="outer")
        .fillna(0)
    )

    for col in ["candidate_count", "blocked_count", "executed_trades"]:
        result[col] = result[col].astype(int)

    return result[output_cols].sort_values("candidate_count", ascending=False)


def v5_build_session_coverage_summary(
    signals: pd.DataFrame,
    trades: pd.DataFrame,
    config: dict,
) -> pd.DataFrame:
    """
    Show actual data coverage by session.

    This separates session labels/filter settings from actual CSV coverage.
    """
    output_cols = [
        "session",
        "bar_count",
        "first_timestamp",
        "last_timestamp",
        "raw_signals",
        "final_signals",
        "executed_trades",
        "total_R",
    ]

    if signals is None or len(signals) == 0:
        return pd.DataFrame(columns=output_cols)

    signal_report = signals.copy()

    signal_report["datetime"] = pd.to_datetime(
        signal_report["datetime"],
        errors="coerce",
    )

    signal_report["session"] = signal_report["datetime"].apply(
        lambda ts: v5_trade_session_from_timestamp(ts, config)
    )

    raw_signal_name = signal_report.get(
        "raw_signal_name",
        pd.Series("NO_SIGNAL", index=signal_report.index),
    ).fillna("NO_SIGNAL").astype(str)

    final_signal = signal_report.get(
        "v1_signal_valid",
        pd.Series(False, index=signal_report.index),
    ).fillna(False).astype(bool)

    signal_report["raw_signal_flag"] = raw_signal_name != "NO_SIGNAL"
    signal_report["final_signal_flag"] = final_signal

    bar_summary = (
        signal_report
        .groupby("session", dropna=False)
        .agg(
            bar_count=("datetime", "size"),
            first_timestamp=("datetime", "min"),
            last_timestamp=("datetime", "max"),
            raw_signals=("raw_signal_flag", "sum"),
            final_signals=("final_signal_flag", "sum"),
        )
        .reset_index()
    )

    trade_report = v5_safe_trades_df(trades)

    if trade_report.empty:
        trade_summary = pd.DataFrame(
            columns=["session", "executed_trades", "total_R"]
        )
    else:
        if "trade_session" in trade_report.columns:
            trade_report["session"] = (
                trade_report["trade_session"]
                .fillna("unknown")
                .astype(str)
            )
        else:
            trade_report["session"] = "unknown"

        trade_report["r_multiple"] = pd.to_numeric(
            trade_report["r_multiple"],
            errors="coerce",
        ).fillna(0.0)

        trade_summary = (
            trade_report
            .groupby("session", dropna=False)
            .agg(
                executed_trades=("r_multiple", "size"),
                total_R=("r_multiple", "sum"),
            )
            .reset_index()
        )

    result = (
        bar_summary
        .merge(trade_summary, on="session", how="left")
        .fillna(
            {
                "executed_trades": 0,
                "total_R": 0.0,
            }
        )
    )

    result["executed_trades"] = result["executed_trades"].astype(int)
    result["raw_signals"] = result["raw_signals"].astype(int)
    result["final_signals"] = result["final_signals"].astype(int)

    return result[output_cols].sort_values("session")


v5_report_trades_df = v5_safe_trades_df(trades_df)

if len(v5_report_trades_df) > 0:
    v5_overall_results = pd.DataFrame(
        [
            {
                "dataset": CONFIG["dataset_name"],
                **v5_build_reporting_row(v5_report_trades_df),
            }
        ]
    )

    v5_setup_source_breakdown = v5_build_breakdown_table(
        v5_report_trades_df,
        ["setup_source"],
    )

    v5_setup_timing_breakdown = v5_build_breakdown_table(
        v5_report_trades_df,
        ["setup_source", "a_tier_entry_timing_type"],
    )

    v5_engine_mode_breakdown = v5_build_breakdown_table(
        v5_report_trades_df,
        ["engine_mode", "setup_source"],
    )

    v5_exit_mode_breakdown = v5_build_breakdown_table(
        v5_report_trades_df,
        ["exit_mode"],
    )

    v5_exit_category_breakdown = v5_build_breakdown_table(
        v5_report_trades_df,
        ["exit_category"],
    )

    v5_regime_breakdown = v5_build_breakdown_table(
        v5_report_trades_df,
        ["regime_20m"],
    )

    v5_session_breakdown = v5_build_breakdown_table(
        v5_report_trades_df,
        ["trade_session"],
    )

    v5_session_setup_breakdown = v5_build_breakdown_table(
        v5_report_trades_df,
        ["trade_session", "setup_source"],
    )

    print("V5 overall result table:")
    display(v5_format_reporting_table(v5_overall_results))

    v5_positive_exit_summary = v5_build_positive_exit_summary(v5_report_trades_df)

    print("V5 positive-exit reporting summary:")
    print("Target TP rate = full target hit rate.")
    print("Positive exit rate = share of trades that made money.")
    print("WR ex flat BE = positive exits after removing true zero-R BE exits.")
    print("Total R and avg R remain the actual performance summary.")
    display(v5_positive_exit_summary.round(4))

    print("V5 breakdown by setup source:")
    display(v5_format_reporting_table(v5_setup_source_breakdown))

    print("V5 breakdown by setup source and A-tier timing type:")
    display(v5_format_reporting_table(v5_setup_timing_breakdown))

    print("V5 breakdown by engine mode and setup source:")
    display(v5_format_reporting_table(v5_engine_mode_breakdown))

    print("V5 breakdown by exit mode:")
    display(v5_format_reporting_table(v5_exit_mode_breakdown))

    print("V5 breakdown by exit category:")
    display(v5_format_reporting_table(v5_exit_category_breakdown))

    print("V5 breakdown by 20-minute regime:")
    display(v5_format_reporting_table(v5_regime_breakdown))

    # ---------------------------------------------------------------------
    # V5 A-tier diagnostic performance report
    # ---------------------------------------------------------------------

    v5_a_tier_trades_df = v5_filter_a_tier_trades(v5_report_trades_df)

    v5_a_tier_extension_breakdown = v5_build_a_tier_bucket_performance_table(
        v5_report_trades_df,
        "a_tier_extension_bucket",
    )

    v5_a_tier_depth_breakdown = v5_build_a_tier_bucket_performance_table(
        v5_report_trades_df,
        "a_tier_depth_bucket",
    )

    v5_a_tier_reclaim_quality_breakdown = v5_build_a_tier_bucket_performance_table(
        v5_report_trades_df,
        "a_tier_reclaim_quality_bucket",
    )

    v5_a_tier_red_shift_breakdown = v5_build_a_tier_bucket_performance_table(
        v5_report_trades_df,
        "a_tier_entry_red_shift_bucket",
    )

    v5_a_tier_timing_breakdown = v5_build_a_tier_bucket_performance_table(
        v5_report_trades_df,
        "a_tier_entry_timing_type",
    )

    v5_a_tier_quick_summary = v5_build_a_tier_quick_summary(
        v5_report_trades_df,
        signals_df,
    )

    print("V5 A-tier diagnostic quick summary:")
    display(v5_a_tier_quick_summary.round(4))

    print("A-tier performance by extension bucket:")
    display(v5_a_tier_extension_breakdown.round(4))

    print("A-tier performance by depth bucket:")
    display(v5_a_tier_depth_breakdown.round(4))

    print("A-tier performance by reclaim quality bucket:")
    display(v5_a_tier_reclaim_quality_breakdown.round(4))

    print("A-tier performance by red-shift bucket:")
    display(v5_a_tier_red_shift_breakdown.round(4))

    print("A-tier performance by entry timing type:")
    display(v5_a_tier_timing_breakdown.round(4))

    # ---------------------------------------------------------------------
    # V5 A-tier retrace-entry report
    # ---------------------------------------------------------------------

    v5_retrace_report_trades_df = v5_add_a_tier_retrace_reporting_buckets(
        v5_report_trades_df,
    )

    v5_a_tier_retrace_quick_summary = v5_build_a_tier_retrace_quick_summary(
        v5_retrace_report_trades_df,
        skipped_execution_signals_df,
    )

    v5_a_tier_retrace_timing_breakdown = v5_build_a_tier_retrace_performance_table(
        v5_retrace_report_trades_df,
        "a_tier_entry_timing_type",
    )

    v5_a_tier_retrace_points_breakdown = v5_build_a_tier_retrace_performance_table(
        v5_retrace_report_trades_df,
        "a_tier_retrace_points_bucket",
    )

    v5_a_tier_retrace_extension_breakdown = v5_build_a_tier_retrace_performance_table(
        v5_retrace_report_trades_df,
        "a_tier_retrace_close_extension_bucket",
    )

    v5_a_tier_retrace_fail_summary = v5_build_a_tier_retrace_fail_summary(
        skipped_execution_signals_df,
    )

    print("V5 A-tier retrace quick summary:")
    display(v5_a_tier_retrace_quick_summary.round(4))

    print("A-tier performance by entry timing type:")
    display(v5_a_tier_retrace_timing_breakdown.round(4))

    print("A-tier retrace performance by retrace-points bucket:")
    display(v5_a_tier_retrace_points_breakdown.round(4))

    print("A-tier retrace performance by close-extension bucket:")
    display(v5_a_tier_retrace_extension_breakdown.round(4))

    print("A-tier blocked retrace candidates by fail reason:")
    display(v5_a_tier_retrace_fail_summary.round(4))

    # ---------------------------------------------------------------------
    # V5 V2 activation report
    # ---------------------------------------------------------------------

    v5_v2_activation_reason_breakdown = v5_build_v2_activation_performance_table(
        v5_report_trades_df,
        ["v2_activation_reason"],
    )

    v5_v2_active_vs_tracked_breakdown = v5_build_v2_activation_performance_table(
        v5_report_trades_df,
        ["v2_trend_health_active", "v2_filter_tracked_only"],
    )

    v5_v2_block_reason_breakdown = v5_build_v2_activation_performance_table(
        v5_report_trades_df,
        ["v2_health_failure_reason"],
    )

    v5_v2_setup_source_breakdown = v5_build_v2_activation_performance_table(
        v5_report_trades_df,
        ["setup_source", "v2_activation_reason"],
    )

    v5_v2_session_breakdown = v5_build_v2_activation_performance_table(
        v5_report_trades_df,
        ["trade_session", "v2_activation_reason"],
    )

    v5_v2_candidate_activation_summary = v5_build_v2_candidate_activation_summary(
        signals_df,
        v5_report_trades_df,
    )

    print("V5 V2 activation candidate summary:")
    display(v5_v2_candidate_activation_summary)

    print("V5 V2 activation reason performance:")
    display(v5_v2_activation_reason_breakdown.round(4))

    print("V5 V2 active vs tracked-only performance:")
    display(v5_v2_active_vs_tracked_breakdown.round(4))

    print("V5 V2 health-failure reason performance:")
    display(v5_v2_block_reason_breakdown.round(4))

    print("V5 V2 activation by setup source:")
    display(v5_v2_setup_source_breakdown.round(4).head(20))

    print("V5 V2 activation by trade session:")
    display(v5_v2_session_breakdown.round(4))

    # ---------------------------------------------------------------------
    # V5 session coverage report
    # ---------------------------------------------------------------------

    v5_session_coverage_summary = v5_build_session_coverage_summary(
        signals_df,
        v5_report_trades_df,
        CONFIG,
    )

    print("V5 session coverage summary:")
    print(
        "This table shows actual CSV coverage by session label. "
        "Enabled sessions only control filtering; they do not create missing session data."
    )
    print(
        f"Session filter enabled: {CONFIG.get('enable_session_filter', False)} | "
        f"Enabled sessions: {CONFIG.get('enabled_sessions', [])} | "
        f"No-new-trades cutoff: {CONFIG.get('no_new_trades_after', 'not_set')}"
    )
    display(v5_session_coverage_summary.round(4))

    print("V5 breakdown by trade session:")
    display(v5_format_reporting_table(v5_session_breakdown))

    print("V5 breakdown by trade session and setup source:")
    display(v5_format_reporting_table(v5_session_setup_breakdown))

else:
    print("No V5 trade breakdown available because no trades were taken.")

In [ ]:
# ---------------------------------------------------------------------
# V5 Dynamic S-tier touch diagnostic summary
# ---------------------------------------------------------------------
def v5_s_tier_touch_summary(
    signals: pd.DataFrame,
    trades: pd.DataFrame,
) -> dict:
    """
    Summarise S-tier touch type performance and optional recent green-break blocks.
    """
    trades = v5_safe_trades_df(trades)

    if signals is None or len(signals) == 0:
        signals = pd.DataFrame()

    if trades.empty:
        touch_performance = pd.DataFrame(
            columns=[
                "s_tier_touch_type",
                "trades",
                "total_R",
                "avg_R",
            ]
        )
    else:
        s_trades = trades[
            trades["setup_source"].astype(str).str.contains("V1_S_TIER", na=False)
        ].copy()

        if s_trades.empty:
            touch_performance = pd.DataFrame(
                columns=[
                    "s_tier_touch_type",
                    "trades",
                    "total_R",
                    "avg_R",
                ]
            )
        else:
            touch_performance = (
                s_trades.groupby("s_tier_touch_type", dropna=False)
                .agg(
                    trades=("r_multiple", "size"),
                    total_R=("r_multiple", "sum"),
                    avg_R=("r_multiple", "mean"),
                )
                .reset_index()
                .sort_values("trades", ascending=False)
            )

    recent_green_break_blocked_count = 0

    if not signals.empty and "v1_signal_block_reason" in signals.columns:
        recent_green_break_blocked_count = int(
            (
                signals["v1_signal_block_reason"]
                == "BLOCKED_S_TIER_RECENT_GREEN_BREAK"
            ).sum()
        )

    s_tier_real_touch_count = 0
    s_tier_band_shift_touch_count = 0

    if not trades.empty and "s_tier_touch_type" in trades.columns:
        s_tier_real_touch_count = int(
            (trades["s_tier_touch_type"] == "S_TIER_REAL_GREEN_TOUCH").sum()
        )

        s_tier_band_shift_touch_count = int(
            (trades["s_tier_touch_type"] == "S_TIER_BAND_SHIFT_TOUCH").sum()
        )

    summary = pd.DataFrame(
        [
            {
                "metric": "S-tier real green touch count",
                "value": s_tier_real_touch_count,
            },
            {
                "metric": "S-tier band-shift touch count",
                "value": s_tier_band_shift_touch_count,
            },
            {
                "metric": "S-tier recent green-break blocked count",
                "value": recent_green_break_blocked_count,
            },
        ]
    )

    return {
        "summary": summary,
        "touch_performance": touch_performance,
    }


v5_s_tier_touch_report = v5_s_tier_touch_summary(signals_df, trades_df)

print("V5 Dynamic S-tier touch summary:")
display(v5_s_tier_touch_report["summary"])

print("V5 S-tier performance by touch type:")
display(v5_s_tier_touch_report["touch_performance"].round(3))

# ---------------------------------------------------------------------
# V5 S-tier clean-state diagnostic summary
# ---------------------------------------------------------------------
def v5_s_tier_clean_state_summary(
    signals: pd.DataFrame,
    trades: pd.DataFrame,
) -> dict:
    trades = v5_safe_trades_df(trades)

    if signals is None or len(signals) == 0:
        signals = pd.DataFrame()

    clean_state_blocked_count = 0

    if not signals.empty and "v1_signal_block_reason" in signals.columns:
        clean_state_blocked_count = int(
            (
                signals["v1_signal_block_reason"]
                == "BLOCKED_S_TIER_NOT_CLEAN_AFTER_GREEN_FAILURE"
            ).sum()
        )

    if trades.empty:
        passed_trade_count = 0
        passed_total_r = 0.0
        passed_avg_r = 0.0
    else:
        s_trades = trades[
            trades["setup_source"].astype(str).str.contains("V1_S_TIER", na=False)
        ].copy()

        clean_pass_trades = s_trades[
            s_trades["s_tier_clean_state_pass"].fillna(False).astype(bool)
        ].copy()

        passed_trade_count = int(len(clean_pass_trades))

        if passed_trade_count > 0:
            passed_r = pd.to_numeric(
                clean_pass_trades["r_multiple"],
                errors="coerce",
            ).fillna(0.0)

            passed_total_r = float(passed_r.sum())
            passed_avg_r = float(passed_r.mean())
        else:
            passed_total_r = 0.0
            passed_avg_r = 0.0

    clean_state_summary = pd.DataFrame(
        [
            {
                "metric": "S-tier clean-state passed trade count",
                "value": passed_trade_count,
            },
            {
                "metric": "S-tier clean-state blocked signal count",
                "value": clean_state_blocked_count,
            },
            {
                "metric": "S-tier clean-state passed Total R",
                "value": passed_total_r,
            },
            {
                "metric": "S-tier clean-state passed Avg R",
                "value": passed_avg_r,
            },
        ]
    )

    block_reason_summary = pd.DataFrame()

    if not signals.empty and "v1_signal_block_reason" in signals.columns:
        block_reason_summary = (
            signals.loc[
                signals["v1_signal_block_reason"].eq(
                    "BLOCKED_S_TIER_NOT_CLEAN_AFTER_GREEN_FAILURE"
                ),
                "v1_signal_block_reason",
            ]
            .value_counts()
            .rename_axis("block_reason")
            .reset_index(name="count")
        )

    return {
        "summary": clean_state_summary,
        "block_reason_summary": block_reason_summary,
    }


v5_s_tier_clean_state_report = v5_s_tier_clean_state_summary(
    signals_df,
    trades_df,
)

print("V5 S-tier clean-state summary:")
display(v5_s_tier_clean_state_report["summary"].round(3))

print("V5 S-tier clean-state block reasons:")
display(v5_s_tier_clean_state_report["block_reason_summary"])

In [ ]:
# ---------------------------------------------------------------------
# V5 toggle and session sanity summary
# ---------------------------------------------------------------------

def v5_toggle_sanity_summary(
    trades: pd.DataFrame,
    skipped_execution: pd.DataFrame,
    config: dict,
) -> dict:
    """
    Compact sanity summary for V5 module and session gates.
    """
    trades = v5_safe_trades_df(trades)

    skipped = (
        skipped_execution.copy()
        if skipped_execution is not None and len(skipped_execution) > 0
        else pd.DataFrame()
    )

    setup_sources = (
        sorted(trades["setup_source"].dropna().astype(str).unique().tolist())
        if len(trades) > 0 and "setup_source" in trades.columns
        else []
    )

    sessions = (
        sorted(trades["trade_session"].dropna().astype(str).unique().tolist())
        if len(trades) > 0 and "trade_session" in trades.columns
        else []
    )

    blocked_by_session_count = 0

    if len(skipped) > 0:
        if "blocked_reason" in skipped.columns:
            blocked_by_session_count += int(
                (skipped["blocked_reason"] == "BLOCKED_BY_SESSION").sum()
            )

        if "skip_reason" in skipped.columns:
            blocked_by_session_count += int(
                (skipped["skip_reason"] == "BLOCKED_BY_SESSION").sum()
            )

    return {
        "selected_preset": globals().get("SELECTED_V5_PRESET", "custom"),
        "active_preset": config.get("active_preset", "custom"),
        "v1_enabled": config.get("enable_v1_s_tier", True),
        "v2_filter_enabled": config.get("enable_v2_trend_health_filter", True),
        "v2_continuation_enabled": config.get("enable_v2_continuation_health", True),
        "v2_reclaim_enabled": config.get("enable_v2_reclaim_recovery_health", True),
        "v2_trend_health_activation_mode": config.get("v2_trend_health_activation_mode", "router_gated"),
        "v2_track_when_not_active": config.get("v2_track_when_not_active", True),
        "v2_force_after_time_enabled": config.get("v2_force_after_time_enabled", False),
        "v2_force_after_time": config.get("v2_force_after_time", "16:30"),
        "v2_force_after_timezone": config.get("v2_force_after_timezone", "Europe/London"),
        "v3_enabled": config.get("enable_v3_a_tier_second_close", True),
        "engine_mode": v5_get_engine_mode(config),
        "router_enabled": config.get("enable_regime_router", True),
        "session_filter_enabled": config.get("enable_session_filter", False),
        "enabled_sessions": config.get("enabled_sessions", []),
        "executed_trades": int(len(trades)),
        "executed_setup_sources": setup_sources,
        "executed_sessions": sessions,
        "session_blocked_candidates": blocked_by_session_count,
        "entry_min_directional_red_shift_points": config.get("entry_min_directional_red_shift_points", 2.0),
        "s_tier_use_entry_red_shift_floor": config.get("s_tier_use_entry_red_shift_floor", False),
        "s_tier_min_directional_red_shift_points": config.get("s_tier_min_directional_red_shift_points", None),
        "a_tier_use_entry_red_shift_floor": config.get("a_tier_use_entry_red_shift_floor", False),
        "a_tier_min_directional_red_shift_points": config.get("a_tier_min_directional_red_shift_points", None),
        "enable_a_tier_retrace_entry": config.get("enable_a_tier_retrace_entry", False),

        "enable_dynamic_s_tier_touch_diagnostics": config.get("enable_dynamic_s_tier_touch_diagnostics", True),
        "dynamic_s_tier_touch_mode": config.get("dynamic_s_tier_touch_mode", "tag_only"),
        "enable_s_tier_clean_state_filter": config.get("enable_s_tier_clean_state_filter", False),
        "s_tier_green_failure_lookback_bars": config.get("s_tier_green_failure_lookback_bars", 8),
        "s_tier_required_clean_closes_after_failure": config.get("s_tier_required_clean_closes_after_failure", 3),

        "enable_a_tier_delayed_pullback_entry": config.get("enable_a_tier_delayed_pullback_entry", False),
        "a_tier_delayed_pullback_entry_bar": config.get("a_tier_delayed_pullback_entry_bar", 4),
        "a_tier_delayed_pullback_min_hold_bars": config.get("a_tier_delayed_pullback_min_hold_bars", 3),
        "a_tier_delayed_pullback_min_points": config.get("a_tier_delayed_pullback_min_points", 5.0),
        "a_tier_delayed_pullback_max_points": config.get("a_tier_delayed_pullback_max_points", 15.0),

        "a_tier_retrace_entry_mode": config.get("a_tier_retrace_entry_mode", "extended_only"),
        "a_tier_max_direct_extension_from_green": config.get("a_tier_max_direct_extension_from_green", 30.0),
        "a_tier_retrace_wait_bars": config.get("a_tier_retrace_wait_bars", 1),
    }


v5_toggle_sanity = v5_toggle_sanity_summary(
    trades_df,
    skipped_execution_signals_df,
    CONFIG,
)

print("V5 toggle/session sanity summary:")
pprint(v5_toggle_sanity)

In [ ]:
# ---------------------------------------------------------------------
# V5 red-shift floor block summary
# ---------------------------------------------------------------------

def v5_redshift_floor_block_summary(signals: pd.DataFrame) -> pd.DataFrame:
    """
    Count candidates blocked by the optional S-tier/A-tier red-shift floors.
    """
    if signals is None or len(signals) == 0:
        return pd.DataFrame()

    if "v1_signal_block_reason" not in signals.columns:
        return pd.DataFrame()

    block_reasons = [
        "BLOCKED_BY_S_TIER_REDSHIFT_FLOOR",
        "BLOCKED_BY_A_TIER_REDSHIFT_FLOOR",
    ]

    blocked = signals[
        signals["v1_signal_block_reason"].isin(block_reasons)
    ].copy()

    if blocked.empty:
        return pd.DataFrame(
            columns=[
                "v1_signal_block_reason",
                "raw_signal_side",
                "v4_selected_entry_module",
                "blocked_candidates",
                "avg_directional_red_shift_points",
                "avg_required_min_shift_points",
            ]
        )

    return (
        blocked.groupby(
            [
                "v1_signal_block_reason",
                "raw_signal_side",
                "v4_selected_entry_module",
            ],
            dropna=False,
        )
        .agg(
            blocked_candidates=("datetime", "size"),
            avg_directional_red_shift_points=(
                "v5_entry_directional_red_shift_points",
                "mean",
            ),
            avg_required_min_shift_points=(
                "v5_entry_red_shift_min_points",
                "mean",
            ),
        )
        .reset_index()
        .sort_values("blocked_candidates", ascending=False)
    )


v5_redshift_floor_blocks = v5_redshift_floor_block_summary(signals_df)

print("V5 optional red-shift floor blocked candidates:")
display(v5_redshift_floor_blocks.round(3))

In [ ]:
# ---------------------------------------------------------------------
# V5 runner sanity summary
# ---------------------------------------------------------------------

def v5_runner_sanity_tables(trades: pd.DataFrame, config: dict) -> dict:
    """
    Compact sanity tables for runner-mode and trailing-stop behaviour.
    """
    trades = v5_safe_trades_df(trades)

    if trades.empty:
        return {
            "runner_config": pd.DataFrame(),
            "runner_exit_summary": pd.DataFrame(),
            "runner_approval_summary": pd.DataFrame(),
        }

    runner_config = pd.DataFrame(
        [
            {
                "enable_v5_exit_manager": config.get("enable_v5_exit_manager", False),
                "enable_runner_trailing": config.get("enable_runner_trailing", False),
                "runner_mode": config.get("runner_mode", "smart"),
                "sl_points_1R": float(config["normal_sl_points"]),
                "trail_BE_trigger": float(config["trail_be_trigger_points"]),
                "BE_trigger_is_1R": (
                    float(config["trail_be_trigger_points"])
                    == float(config["normal_sl_points"])
                ),
                "enable_be_plus_buffer": config.get("enable_be_plus_buffer", True),
                "be_plus_buffer_points": float(config.get("be_plus_buffer_points", 3.0)),
                "runner_stop_widened_count": int(
                    trades.get("runner_stop_widened", False).fillna(False).astype(bool).sum()
                    if "runner_stop_widened" in trades.columns
                    else 0
                ),
                "runner_trades": int(
                    trades.get("runner_enabled", False).fillna(False).astype(bool).sum()
                    if "runner_enabled" in trades.columns
                    else 0
                ),
                "all_trades": int(len(trades)),
            }
        ]
    )

    runner_exit_summary = (
        trades.groupby(["exit_mode", "exit_category"], dropna=False)
        .agg(
            trades=("r_multiple", "size"),
            total_R=("r_multiple", "sum"),
            avg_R=("r_multiple", "mean"),
        )
        .reset_index()
        .sort_values(["exit_mode", "exit_category"])
    )

    runner_approval_summary = (
        trades.groupby(
            [
                "runner_enabled",
                "setup_source",
                "regime_20m",
                "red_shift_bucket",
            ],
            dropna=False,
        )
        .agg(
            trades=("r_multiple", "size"),
            total_R=("r_multiple", "sum"),
            avg_R=("r_multiple", "mean"),
        )
        .reset_index()
        .sort_values(["runner_enabled", "total_R"], ascending=[False, False])
    )

    return {
        "runner_config": runner_config,
        "runner_exit_summary": runner_exit_summary,
        "runner_approval_summary": runner_approval_summary,
    }


v5_runner_sanity = v5_runner_sanity_tables(trades_df, CONFIG)

print("V5 runner configuration sanity:")
display(v5_runner_sanity["runner_config"])

print("V5 runner exit summary:")
display(v5_runner_sanity["runner_exit_summary"])

print("V5 runner approval summary:")
display(v5_runner_sanity["runner_approval_summary"].head(20))

## V5 Session and Time-of-Day Chart

This section plots trade results by time of day.

Each point is one closed trade. Vertical dotted lines mark the configured Asia, London, and New York session boundaries.

In [ ]:
# ---------------------------------------------------------------------
# V5 time-of-day / session chart
# ---------------------------------------------------------------------

def v5_time_to_decimal_hour(time_text: str) -> float:
    """
    Convert HH:MM text into decimal hours.
    """
    hour, minute = str(time_text).split(":")
    return int(hour) + int(minute) / 60.0


def v5_add_session_boundaries_to_axis(ax, config: dict) -> None:
    """
    Add vertical dotted session boundary lines to a matplotlib axis.
    """
    session_windows = config.get("session_windows", {})

    for session_name, window in session_windows.items():
        try:
            start_text, _ = window
            boundary_hour = v5_time_to_decimal_hour(start_text)
        except Exception:
            continue

        ax.axvline(boundary_hour, linestyle="--", linewidth=1)
        ax.text(
            boundary_hour,
            ax.get_ylim()[1],
            session_name,
            rotation=90,
            va="top",
            ha="right",
        )


def plot_v5_session_time_chart(
    trades: pd.DataFrame,
    config: dict,
    report_charts_dir: Path | None = None,
) -> Path | None:
    """
    Plot closed-trade R by entry time of day.
    """
    if not config.get("plot_session_time_chart", True):
        print("V5 session/time-of-day chart disabled in CONFIG.")
        return None

    trades = v5_safe_trades_df(trades)

    if trades.empty:
        print("No trades available for V5 session/time-of-day chart.")
        return None

    chart_df = trades.copy()

    if "entry_hour" in chart_df.columns and "entry_minute" in chart_df.columns:
        chart_df["entry_decimal_hour"] = (
            pd.to_numeric(chart_df["entry_hour"], errors="coerce")
            + pd.to_numeric(chart_df["entry_minute"], errors="coerce") / 60.0
        )
    else:
        entry_times = pd.to_datetime(chart_df["entry_time"], errors="coerce")
        chart_df["entry_decimal_hour"] = (
            entry_times.dt.hour
            + entry_times.dt.minute / 60.0
        )

    chart_df = chart_df.dropna(subset=["entry_decimal_hour", "r_multiple"]).copy()

    if chart_df.empty:
        print("No valid entry times available for V5 session/time-of-day chart.")
        return None

    chart_df = chart_df.sort_values("entry_decimal_hour")

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.scatter(
        chart_df["entry_decimal_hour"],
        chart_df["r_multiple"],
        alpha=0.75,
    )

    ax.axhline(0.0, linewidth=1)
    v5_add_session_boundaries_to_axis(ax, config)

    ax.set_title("V5 trade R by entry time of day")
    ax.set_xlabel(f"Entry time of day ({config.get('session_timezone', 'session time')})")
    ax.set_ylabel("Trade R")
    ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 2))
    ax.grid(True, linestyle=":", linewidth=0.8)

    plt.tight_layout()

    chart_path = None

    if report_charts_dir is not None:
        report_charts_dir.mkdir(parents=True, exist_ok=True)

        safe_strategy = make_safe_filename_part(config["strategy_version"])
        safe_dataset = make_safe_filename_part(config["dataset_name"])

        chart_path = (
            report_charts_dir
            / f"{safe_strategy}__{safe_dataset}__v5_session_time_r_scatter.png"
        )

        fig.savefig(
            chart_path,
            dpi=180,
            bbox_inches="tight",
        )

        print(f"Saved V5 session/time-of-day chart: {chart_path.relative_to(PROJECT_ROOT)}")

    plt.show()

    return chart_path


v5_session_time_chart_path = plot_v5_session_time_chart(
    trades_df,
    CONFIG,
    report_charts_dir=ARTIFACTS_DIR / "report_charts",
)

In [ ]:
def run_v3_backtest_for_dataset(base_config: dict, dataset_config: dict) -> dict:
    """
    Run the active automated VWAP V3 strategy on one comparison dataset.
    """
    run_config = base_config.copy()
    run_config.update(dataset_config)

    raw_df_local, data_file_local = load_market_data(run_config)

    engine_df_local = build_existing_engine_output(raw_df_local, ENGINE_CONFIG)
    automation_df_local = prepare_automation_dataframe(engine_df_local)
    features_df_local = add_automation_features(automation_df_local, run_config)
    signals_df_local = add_v1_green_signals(features_df_local, run_config)

    skipped_signal_candidates_df_local = signals_df_local[
        (signals_df_local["raw_signal_name"] != "NO_SIGNAL")
        & (signals_df_local["v1_signal_name"] == "NO_SIGNAL")
    ].copy()

    trades_df_local, daily_summary_df_local, skipped_execution_signals_df_local = run_v1_trade_simulation_with_daily_controls(
        signals_df_local,
        run_config,
    )

    trades_df_local = add_v5_original_tp_hit_diagnostics(
    trades_df_local,
    signals_df_local,
    run_config,
    )

    summary = build_performance_summary(
        raw_ohlc_df=raw_df_local,
        signals_df=signals_df_local,
        trades_df=trades_df_local,
        skipped_signal_candidates_df=skipped_signal_candidates_df_local,
        skipped_execution_signals_df=skipped_execution_signals_df_local,
        config=run_config,
        data_file=data_file_local,
    )

    return summary

In [ ]:
# Compare active V5 dynamic regime strategy across configured datasets

comparison_summaries = []

for dataset_config in CONFIG["comparison_datasets"]:
    print("=" * 90)
    print(f"Running active V4 dynamic regime strategy on: {dataset_config['dataset_name']}")
    print("=" * 90)

    summary = run_v3_backtest_for_dataset(CONFIG, dataset_config)
    comparison_summaries.append(summary)


comparison_table = pd.DataFrame(
    [
        {
            "Dataset": summary["dataset_name"],
            "Rows": summary["number_of_rows"],
            "Raw Signals": summary["number_of_raw_signals"],
            "Final Signals": summary["number_of_final_signals"],
            "Trades": summary["number_of_trades"],
            "Full TP": summary["target_tp_count"],
            "Original TP Hits": summary.get("original_tp_hits", 0),
            "SL_FULL": summary["sl_full_count"],
            "BE_FLAT": summary["be_flat_count"],
            "BE_PLUS": summary["be_plus_count"],
            "TRAILED_SL_PROFIT": summary["trailed_profit_count"],
            "TIME_EXIT": summary["time_exit_count"],
            "Full TP Rate": f"{summary['target_tp_rate']:.2%}",
            "Original TP Hit Rate": f"{summary.get('original_tp_hit_rate', 0.0):.2%}",
            "Full SL Rate": f"{summary['full_sl_rate']:.2%}",
            "Positive Exit Rate": f"{summary['positive_exit_rate']:.2%}",
            "WR ex Flat BE": f"{summary['win_rate_ex_flat_be']:.2%}",
            "Non-Loss Rate": f"{summary['non_loss_rate']:.2%}",
            "Total R": f"{summary['total_r']:.2f}R",
            "Avg R": f"{summary['average_r']:.3f}R",
            "Account %": f"{summary['total_account_pct']:.2f}%",
            "Points": f"{summary['total_points']:.2f}",
            "Avg Bars": f"{summary['average_bars_held']:.2f}",
            "Skipped": summary["total_skipped_signals"],
        }
        for summary in comparison_summaries
    ]
)

comparison_table

In [ ]:
# ---------------------------------------------------------------------
# Automatic V5 experiment log append
# ---------------------------------------------------------------------

V5_EXPERIMENT_CONFIG_KEYS = [
    "engine_mode",
    "tp_points",
    "runner_mode",
    "runner_target_r",
    "runner_trail_rules_r",
    "runner_tp_points",
    "trail_5r_tp_points",

    "enable_legacy_early_runner_lock",
    "legacy_early_runner_lock_only_when_tp_points_above",
    "legacy_runner_reference_sl_points",
    "legacy_runner_lock_trigger_r",
    "legacy_runner_lock_r",
    "legacy_runner_lock_trigger_points",
    "legacy_runner_lock_points",

    "enable_v1_s_tier",
    "enable_v2_trend_health_filter",
    "enable_v3_a_tier_second_close",

    "enable_v5_mr_orange_failure_entry",
    "mr_orange_confirm_closes",
    "mr_orange_touch_lookback_bars",
    "mr_use_locked_orange_touch",
    "mr_block_if_strong_continuation",
    "mr_require_weak_or_compressing_trend",
    "mr_target_mode",
    "mr_sl_mode",
    "mr_max_sl_points",
    "mr_min_target_rr",
    "mr_entry_priority",

    "s_tier_use_entry_red_shift_floor",
    "s_tier_min_directional_red_shift_points",
    "a_tier_use_entry_red_shift_floor",
    "a_tier_min_directional_red_shift_points",

    "enable_dynamic_s_tier_touch_diagnostics",
    "dynamic_s_tier_touch_mode",
    "enable_s_tier_clean_state_filter",
    "s_tier_green_failure_lookback_bars",
    "s_tier_required_clean_closes_after_failure",

    "enable_a_tier_retrace_entry",
    "a_tier_retrace_entry_mode",

    "enable_a_tier_delayed_pullback_entry",
    "a_tier_delayed_pullback_entry_bar",
    "a_tier_delayed_pullback_min_hold_bars",
    "a_tier_delayed_pullback_min_points",
    "a_tier_delayed_pullback_max_points",
    "a_tier_delayed_pullback_must_hold_green",
    "a_tier_delayed_pullback_allow_intrabar_green_pierce_points",
    "a_tier_delayed_pullback_require_trend_health",
    "enable_a_tier_delayed_pullback_strength_filter",
    "a_tier_delayed_pullback_allowed_red_shift_buckets",
    "a_tier_delayed_pullback_block_track_only",
    "a_tier_delayed_pullback_require_v2_pass",

    "v2_trend_health_activation_mode",
    "v2_force_after_time_enabled",

    "enable_session_filter",
    "enabled_sessions",
    "no_new_trades_after",

    "max_daily_loss_r",
    "max_no_tp_run_r",
    "max_consecutive_sl",
]

V5_EXPERIMENT_RESULT_COLUMNS = [
    "Dataset",
    "Raw Signals",
    "Final Signals",
    "Trades",
    "Full TP",
    "Original TP Hits",
    "SL_FULL",
    "BE_FLAT",
    "BE_PLUS",
    "TRAILED_SL_PROFIT",
    "TIME_EXIT",
    "Full TP Rate",
    "Original TP Hit Rate",
    "Full SL Rate",
    "Positive Exit Rate",
    "WR ex Flat BE",
    "Non-Loss Rate",
    "Total R",
    "Avg R",
    "Account %",
    "Points",
    "Avg Bars",
    "Skipped",
]


def v5_log_json_default(value):
    """
    JSON serializer for config snapshots used by the experiment log.
    """
    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, Path):
        return str(value)

    if hasattr(value, "item"):
        return value.item()

    if hasattr(value, "isoformat"):
        return value.isoformat()

    return str(value)


def v5_config_value_for_log(value):
    """
    Store nested config values in a CSV-friendly way.
    """
    if isinstance(value, (list, tuple, dict)):
        return json.dumps(value, default=v5_log_json_default)

    return value


def append_v5_experiment_log(
    comparison_df: pd.DataFrame,
    config: dict,
    selected_preset: str,
    experiment_name: str,
    experiment_tag: str,
    experiment_notes: str,
    log_path: Path,
) -> pd.DataFrame:
    """
    Append one experiment-log row per comparison dataset.
    """
    if comparison_df is None or len(comparison_df) == 0:
        print("V5 experiment log skipped: comparison table is empty.")
        return pd.DataFrame()

    log_path.parent.mkdir(parents=True, exist_ok=True)

    run_timestamp = pd.Timestamp.now(tz="Europe/London").isoformat()
    run_id = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
    runner_target_r = get_runner_target_r(config)
    config_snapshot_json = json.dumps(config, default=v5_log_json_default, sort_keys=True)

    rows = []

    for _, result_row in comparison_df.iterrows():
        log_row = {
            "run_timestamp": run_timestamp,
            "run_id": run_id,
            "experiment_name": experiment_name,
            "experiment_tag": experiment_tag,
            "experiment_notes": experiment_notes,
            "selected_preset": selected_preset,
            "runner_target_r": runner_target_r,
            "config_snapshot_json": config_snapshot_json,
        }

        for key in V5_EXPERIMENT_CONFIG_KEYS:
            log_row[key] = v5_config_value_for_log(config.get(key))

        for column in V5_EXPERIMENT_RESULT_COLUMNS:
            if column in comparison_df.columns:
                log_row[column] = result_row.get(column)

        rows.append(log_row)

    log_df = pd.DataFrame(rows)

    if log_path.exists():
        existing_log_df = pd.read_csv(log_path)

        existing_log_df = existing_log_df.loc[
            :,
            ~existing_log_df.columns.astype(str).str.startswith("Unnamed:"),
        ]

        all_columns = list(dict.fromkeys(list(existing_log_df.columns) + list(log_df.columns)))

        combined_log_df = pd.concat(
            [
                existing_log_df.reindex(columns=all_columns),
                log_df.reindex(columns=all_columns),
            ],
            ignore_index=True,
        )

        combined_log_df.to_csv(log_path, index=False)
    else:
        log_df.to_csv(log_path, index=False)

    print("Saved V5 experiment log:")
    print(log_path.relative_to(PROJECT_ROOT))
    print(f"Rows appended: {len(log_df)}")
    print(f"Experiment: {experiment_name}")

    return log_df


if SAVE_EXPERIMENT_LOG:
    v5_experiment_log_rows = append_v5_experiment_log(
        comparison_df=comparison_table,
        config=CONFIG,
        selected_preset=SELECTED_V5_PRESET,
        experiment_name=EXPERIMENT_NAME,
        experiment_tag=EXPERIMENT_TAG,
        experiment_notes=EXPERIMENT_NOTES,
        log_path=EXPERIMENT_LOG_PATH,
    )
else:
    print("V5 experiment logging disabled: SAVE_EXPERIMENT_LOG is False.")

## Synthetic FTMO-Style $200k Account Curve

This section creates a GitHub-ready synthetic account curve using the 1Y comparison dataset.

The chart starts from a hypothetical $200,000 account and converts each closed trade's account percentage return into synthetic dollar PnL.

This is a closed-trade proxy only. It is not an official FTMO rule checker because it does not include floating PnL, intratrade equity movement, commissions, swaps, spreads, or real-time equity rule checks.

In [ ]:
# ------------------------------------------------------------------
# Synthetic FTMO-style $200k account report chart
# ------------------------------------------------------------------

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter


# ------------------------------------------------------------------
# Report chart dataset selector
# ------------------------------------------------------------------
# Choose which configured comparison dataset gets the flashy FTMO-style chart.
#
# Option 1: select by exact dataset name
# Option 2: select by row/index from CONFIG["comparison_datasets"]
#
# For your current table:
# 0 = US100_cash_M1_NY_session_30d
# 1 = US100_cash_M1_NY_session_1y

REPORT_CHART_DATASET_NAME = "US100_cash_M1_NY_session_1y"
REPORT_CHART_DATASET_INDEX = None


FTMO_SYNTHETIC_CONFIG = {
    "initial_balance": 200_000.0,
    "profit_target_pct": 10.0,
    "max_loss_pct": 10.0,
    "chart_dpi": 240,
}


def select_report_chart_dataset_config(
    config: dict,
    dataset_name: str | None = None,
    dataset_index: int | None = None,
) -> dict:
    """
    Select one dataset from CONFIG['comparison_datasets'] for the report chart.
    """
    comparison_datasets = config["comparison_datasets"]

    if dataset_index is not None:
        return comparison_datasets[int(dataset_index)]

    if dataset_name is not None:
        matches = [
            dataset_config
            for dataset_config in comparison_datasets
            if dataset_config.get("dataset_name") == dataset_name
        ]

        if len(matches) == 0:
            available_names = [
                dataset_config.get("dataset_name")
                for dataset_config in comparison_datasets
            ]

            raise ValueError(
                f"No dataset matched REPORT_CHART_DATASET_NAME={dataset_name!r}.\n"
                f"Available datasets: {available_names}"
            )

        return matches[0]

    return comparison_datasets[-1]


def safe_report_filename_part(value: str) -> str:
    """
    Safe filename helper for report chart exports.
    """
    return (
        str(value)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(":", "_")
        .replace("__", "_")
    )


def run_active_strategy_details_for_dataset(base_config: dict, dataset_config: dict) -> dict:
    """
    Run the active notebook strategy on the selected dataset and return the full trade log.

    This version does not depend on run_v1/v2/v3_backtest_details_for_dataset being loaded.
    It directly uses the same core pipeline already used by the notebook comparison table.
    """
    run_config = base_config.copy()
    run_config.update(dataset_config)

    raw_df_local, data_file_local = load_market_data(run_config)

    engine_df_local = build_existing_engine_output(raw_df_local, ENGINE_CONFIG)
    automation_df_local = prepare_automation_dataframe(engine_df_local)
    features_df_local = add_automation_features(automation_df_local, run_config)
    signals_df_local = add_v1_green_signals(features_df_local, run_config)

    trades_df_local, daily_summary_df_local, skipped_execution_signals_df_local = (
        run_v1_trade_simulation_with_daily_controls(
            signals_df_local,
            run_config,
        )
    )

    trades_df_local = add_v5_original_tp_hit_diagnostics(
    trades_df_local,
    signals_df_local,
    run_config,
    )

    return {
        "config": run_config,
        "dataset_name": run_config["dataset_name"],
        "trades_df": trades_df_local,
        "daily_summary_df": daily_summary_df_local,
        "data_file": data_file_local,
    }


def build_synthetic_account_curve(
    trades: pd.DataFrame,
    config: dict,
    synthetic_config: dict,
) -> pd.DataFrame:
    """
    Build a closed-trade synthetic $200k account curve.

    This is a synthetic reporting curve, not an official prop-firm rule engine.
    """
    if trades is None or len(trades) == 0:
        return pd.DataFrame()

    curve = trades.sort_values("entry_time").reset_index(drop=True).copy()

    initial_balance = float(synthetic_config["initial_balance"])

    curve["trade_number"] = np.arange(1, len(curve) + 1)

    if "account_pct" in curve.columns:
        curve["trade_account_pct"] = curve["account_pct"].astype(float)
    else:
        curve["trade_account_pct"] = (
            curve["r_multiple"].astype(float) * float(config["risk_per_trade_pct"])
        )

    curve["trade_pnl_usd"] = initial_balance * curve["trade_account_pct"] / 100.0

    curve["balance_before_trade"] = (
        initial_balance + curve["trade_pnl_usd"].cumsum().shift(fill_value=0.0)
    )
    curve["balance_after_trade"] = initial_balance + curve["trade_pnl_usd"].cumsum()

    curve["return_pct"] = 100.0 * (
        curve["balance_after_trade"] / initial_balance - 1.0
    )

    curve["running_peak_balance"] = curve["balance_after_trade"].cummax()
    curve["drawdown_usd"] = (
        curve["balance_after_trade"] - curve["running_peak_balance"]
    )
    curve["drawdown_pct_initial"] = (
        100.0 * curve["drawdown_usd"] / initial_balance
    )

    if "session_date" in curve.columns:
        curve["session_date"] = pd.to_datetime(curve["session_date"]).dt.date
    else:
        curve["session_date"] = pd.to_datetime(curve["exit_time"]).dt.date

    curve["daily_closed_pnl_usd"] = curve.groupby("session_date")[
        "trade_pnl_usd"
    ].cumsum()
    curve["daily_closed_pnl_pct_initial"] = (
        100.0 * curve["daily_closed_pnl_usd"] / initial_balance
    )

    curve["is_new_equity_high"] = (
        curve["balance_after_trade"] >= curve["balance_after_trade"].cummax()
    )

    curve["strategy_version"] = config["strategy_version"]
    curve["dataset_name"] = config["dataset_name"]

    return curve


def usd_formatter(value, _):
    return f"${value:,.0f}"


def pct_formatter(value, _):
    return f"{value:.1f}%"


def plot_synthetic_ftmo_account_curve(
    account_curve: pd.DataFrame,
    config: dict,
    synthetic_config: dict,
    report_charts_dir: Path,
) -> Path | None:
    """
    Save a polished synthetic FTMO-style account chart for GitHub.
    """
    if account_curve is None or len(account_curve) == 0:
        print("No trades available, so no synthetic account chart was created.")
        return None

    report_charts_dir.mkdir(parents=True, exist_ok=True)

    initial_balance = float(synthetic_config["initial_balance"])
    profit_target_pct = float(synthetic_config["profit_target_pct"])
    max_loss_pct = float(synthetic_config["max_loss_pct"])

    profit_target_balance = initial_balance * (1.0 + profit_target_pct / 100.0)
    max_loss_floor = initial_balance * (1.0 - max_loss_pct / 100.0)

    final_balance = float(account_curve["balance_after_trade"].iloc[-1])
    final_return_pct = 100.0 * (final_balance / initial_balance - 1.0)

    max_balance = float(account_curve["balance_after_trade"].max())
    min_balance = float(account_curve["balance_after_trade"].min())

    max_drawdown_pct = float(account_curve["drawdown_pct_initial"].min())
    worst_daily_closed_pnl_pct = float(
        account_curve["daily_closed_pnl_pct_initial"].min()
    )

    total_r = float(account_curve["r_multiple"].sum())
    average_r = float(account_curve["r_multiple"].mean())

    full_tp_count = int((account_curve["result"].astype(str).str.upper() == "TP").sum())

    original_tp_hit_count = int(
        account_curve.get(
            "hit_original_tp_before_exit",
            pd.Series(False, index=account_curve.index),
        )
        .fillna(False)
        .astype(bool)
        .sum()
    )

    full_sl_count = int(
        (
            account_curve.get(
                "outcome_group",
                account_curve.get("result", pd.Series("", index=account_curve.index)),
            )
            .fillna("")
            .astype(str)
            .str.upper()
            .eq("SL_FULL")
        ).sum()
    )

    if full_sl_count == 0:
        full_sl_count = int(
            (account_curve["result"].astype(str).str.upper() == "SL").sum()
        )

    be_trail_count = int(
        (
            account_curve.get("outcome_group", pd.Series("", index=account_curve.index))
            .fillna("")
            .astype(str)
            .str.upper()
            .isin(["BE_FLAT", "BE_PLUS", "TRAILED_SL_PROFIT"])
        ).sum()
    )

    if be_trail_count == 0:
        be_trail_count = int(
            (account_curve["result"].astype(str).str.upper() == "BE").sum()
        )

    hit_profit_target = bool(
        (account_curve["balance_after_trade"] >= profit_target_balance).any()
    )
    hit_max_loss = bool(
        (account_curve["balance_after_trade"] <= max_loss_floor).any()
    )

    if hit_max_loss:
        status = "MAX LOSS HIT"
        badge_colour = "#ef4444"
    elif hit_profit_target:
        status = "TARGET HIT"
        badge_colour = "#22c55e"
    else:
        status = "TARGET NOT HIT"
        badge_colour = "#f59e0b"

    safe_strategy = safe_report_filename_part(config["strategy_version"])
    safe_dataset = safe_report_filename_part(config["dataset_name"])

    chart_path = (
        report_charts_dir
        / f"{safe_strategy}__{safe_dataset}__ftmo_200k_synthetic_equity.png"
    )

    x = account_curve["trade_number"].to_numpy()
    balance = account_curve["balance_after_trade"].astype(float).to_numpy()
    drawdown = account_curve["drawdown_pct_initial"].astype(float).to_numpy()
    pnl = account_curve["trade_pnl_usd"].astype(float).to_numpy()

    bar_colours = np.where(pnl >= 0, "#22c55e", "#ef4444")

    fig = plt.figure(figsize=(17, 10.5), facecolor="#020617")
    grid = fig.add_gridspec(
        3,
        1,
        height_ratios=[3.2, 1.15, 1.25],
        hspace=0.16,
    )

    ax_equity = fig.add_subplot(grid[0])
    ax_dd = fig.add_subplot(grid[1], sharex=ax_equity)
    ax_pnl = fig.add_subplot(grid[2], sharex=ax_equity)

    for ax in [ax_equity, ax_dd, ax_pnl]:
        ax.set_facecolor("#0f172a")
        ax.grid(True, alpha=0.16, linewidth=0.8)
        ax.tick_params(colors="#e5e7eb", labelsize=10)

        for spine in ax.spines.values():
            spine.set_color("#334155")
            spine.set_linewidth(0.9)

    # ------------------------------------------------------------------
    # Equity panel
    # ------------------------------------------------------------------
    y_top = max(max_balance, profit_target_balance) * 1.004
    y_bottom = min(min_balance, max_loss_floor) * 0.996

    ax_equity.axhspan(
        profit_target_balance,
        y_top,
        color="#22c55e",
        alpha=0.08,
    )

    ax_equity.axhspan(
        y_bottom,
        max_loss_floor,
        color="#ef4444",
        alpha=0.10,
    )

    ax_equity.plot(
        x,
        balance,
        linewidth=3.1,
        color="#38bdf8",
        label="Closed-trade balance",
        zorder=4,
    )

    ax_equity.scatter(
        x,
        balance,
        s=18,
        color="#7dd3fc",
        alpha=0.70,
        zorder=5,
    )

    new_highs = account_curve[account_curve["is_new_equity_high"]].copy()

    if len(new_highs) > 0:
        ax_equity.scatter(
            new_highs["trade_number"],
            new_highs["balance_after_trade"],
            s=28,
            color="#facc15",
            alpha=0.85,
            label="New equity high",
            zorder=6,
        )

    ax_equity.axhline(
        initial_balance,
        color="#cbd5e1",
        linestyle="--",
        linewidth=1.2,
        alpha=0.75,
        label="Starting balance",
    )

    ax_equity.axhline(
        profit_target_balance,
        color="#22c55e",
        linestyle="--",
        linewidth=1.7,
        alpha=0.95,
        label=f"+{profit_target_pct:.0f}% target",
    )

    ax_equity.axhline(
        max_loss_floor,
        color="#ef4444",
        linestyle="--",
        linewidth=1.7,
        alpha=0.95,
        label=f"-{max_loss_pct:.0f}% max loss floor",
    )

    target_hits = account_curve[
        account_curve["balance_after_trade"] >= profit_target_balance
    ]

    if len(target_hits) > 0:
        first_target_trade = int(target_hits["trade_number"].iloc[0])

        ax_equity.axvline(
            first_target_trade,
            color="#22c55e",
            linestyle=":",
            linewidth=2.0,
            alpha=0.95,
            zorder=3,
        )

        ax_equity.text(
            first_target_trade,
            profit_target_balance,
            "  Target hit",
            color="#bbf7d0",
            fontsize=10,
            fontweight="bold",
            va="bottom",
            ha="left",
        )

    loss_hits = account_curve[
        account_curve["balance_after_trade"] <= max_loss_floor
    ]

    if len(loss_hits) > 0:
        first_loss_trade = int(loss_hits["trade_number"].iloc[0])

        ax_equity.axvline(
            first_loss_trade,
            color="#ef4444",
            linestyle=":",
            linewidth=2.0,
            alpha=0.95,
            zorder=3,
        )

        ax_equity.text(
            first_loss_trade,
            max_loss_floor,
            "  Max loss hit",
            color="#fecaca",
            fontsize=10,
            fontweight="bold",
            va="top",
            ha="left",
        )

    ax_equity.set_ylim(y_bottom, y_top)
    ax_equity.yaxis.set_major_formatter(FuncFormatter(usd_formatter))
    ax_equity.set_ylabel("Synthetic balance", color="#e5e7eb", fontsize=11)

    title = (
        f"{config['strategy_version'].upper()} | "
        f"Synthetic $200k FTMO-Style Account Curve"
    )

    subtitle = (
        f"{config['dataset_name']} | "
        f"Closed-trade proxy | "
        f"Risk per trade: {config['risk_per_trade_pct']:.2f}%"
    )

    ax_equity.set_title(
        title,
        color="white",
        fontsize=18,
        fontweight="bold",
        pad=24,
    )

    ax_equity.text(
        0.0,
        1.015,
        subtitle,
        transform=ax_equity.transAxes,
        ha="left",
        va="bottom",
        fontsize=10.5,
        color="#94a3b8",
    )

    ax_equity.text(
        0.985,
        1.015,
        status,
        transform=ax_equity.transAxes,
        ha="right",
        va="bottom",
        fontsize=11,
        color="#020617",
        fontweight="bold",
        bbox=dict(
            boxstyle="round,pad=0.38",
            facecolor=badge_colour,
            edgecolor=badge_colour,
            alpha=0.98,
        ),
    )

    stats_text = (
        f"Final balance: ${final_balance:,.0f}\n"
        f"Return: {final_return_pct:.2f}%\n"
        f"Total R: {total_r:.2f}R\n"
        f"Avg R/trade: {average_r:.3f}R\n"
        f"Max DD: {max_drawdown_pct:.2f}%\n"
        f"Worst daily closed PnL: {worst_daily_closed_pnl_pct:.2f}%\n"
        f"Trades: {len(account_curve)} | FullTP: {full_tp_count} | OrigTP: {original_tp_hit_count} | SL: {full_sl_count} | BE/Trail: {be_trail_count}"
    )

    ax_equity.text(
        0.985,
        0.045,
        stats_text,
        transform=ax_equity.transAxes,
        ha="right",
        va="bottom",
        fontsize=10.5,
        color="#e5e7eb",
        linespacing=1.35,
        bbox=dict(
            boxstyle="round,pad=0.58",
            facecolor="#020617",
            edgecolor="#475569",
            alpha=0.94,
        ),
    )

    ax_equity.legend(
        loc="upper left",
        frameon=True,
        facecolor="#020617",
        edgecolor="#334155",
        labelcolor="#e5e7eb",
        fontsize=9,
    )

    # ------------------------------------------------------------------
    # Drawdown panel
    # ------------------------------------------------------------------
    ax_dd.fill_between(
        x,
        drawdown,
        0,
        color="#f97316",
        alpha=0.35,
        zorder=2,
    )

    ax_dd.plot(
        x,
        drawdown,
        color="#fb923c",
        linewidth=1.9,
        zorder=3,
    )

    ax_dd.axhline(
        -max_loss_pct,
        color="#ef4444",
        linestyle="--",
        linewidth=1.2,
        alpha=0.9,
    )

    ax_dd.yaxis.set_major_formatter(FuncFormatter(pct_formatter))
    ax_dd.set_ylabel("Drawdown", color="#e5e7eb", fontsize=11)

    ax_dd.text(
        0.01,
        0.12,
        "Closed-trade drawdown from running peak",
        transform=ax_dd.transAxes,
        ha="left",
        va="bottom",
        fontsize=9,
        color="#94a3b8",
    )

    # ------------------------------------------------------------------
    # Trade PnL panel
    # ------------------------------------------------------------------
    ax_pnl.bar(
        x,
        pnl,
        color=bar_colours,
        alpha=0.88,
        width=0.82,
        zorder=3,
    )

    ax_pnl.axhline(
        0,
        color="#e5e7eb",
        linewidth=1.0,
        alpha=0.8,
    )

    ax_pnl.yaxis.set_major_formatter(FuncFormatter(usd_formatter))
    ax_pnl.set_ylabel("Trade PnL", color="#e5e7eb", fontsize=11)
    ax_pnl.set_xlabel("Trade number", color="#e5e7eb", fontsize=11)

    ax_pnl.text(
        0.01,
        0.88,
        "Per-trade synthetic PnL",
        transform=ax_pnl.transAxes,
        ha="left",
        va="top",
        fontsize=9,
        color="#94a3b8",
    )

    fig.text(
        0.5,
        0.012,
        (
            "Synthetic closed-trade proxy only. "
            "Does not include floating PnL, commissions, swaps, spreads, "
            "or intratrade equity movement."
        ),
        ha="center",
        color="#64748b",
        fontsize=9,
    )

    fig.savefig(
        chart_path,
        dpi=int(synthetic_config["chart_dpi"]),
        bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )

    plt.show()

    print(f"Saved synthetic FTMO-style chart: {chart_path.relative_to(PROJECT_ROOT)}")

    return chart_path


REPORT_DATASET_CONFIG = select_report_chart_dataset_config(
    config=CONFIG,
    dataset_name=REPORT_CHART_DATASET_NAME,
    dataset_index=REPORT_CHART_DATASET_INDEX,
)

print("Running synthetic FTMO-style chart for:")
print(REPORT_DATASET_CONFIG)

report_backtest = run_active_strategy_details_for_dataset(
    base_config=CONFIG,
    dataset_config=REPORT_DATASET_CONFIG,
)

synthetic_account_curve_df = build_synthetic_account_curve(
    trades=report_backtest["trades_df"],
    config=report_backtest["config"],
    synthetic_config=FTMO_SYNTHETIC_CONFIG,
)

synthetic_ftmo_chart_path = plot_synthetic_ftmo_account_curve(
    account_curve=synthetic_account_curve_df,
    config=report_backtest["config"],
    synthetic_config=FTMO_SYNTHETIC_CONFIG,
    report_charts_dir=ARTIFACTS_DIR / "report_charts",
)

## Compact Trade Audit

This section prints compact trade diagnostics for the selected report dataset.

The goal is to identify bad periods without flooding the notebook output with hundreds of trade rows. The full trade log is saved to `artifacts/trade_audits/`, while the notebook only displays monthly performance, worst rolling windows, loss streaks, and a truncated trade preview.

In [ ]:
# ------------------------------------------------------------------
# Compact trade audit diagnostics
# ------------------------------------------------------------------

TRADE_AUDIT_CONFIG = {
    # Uses the same selected dataset as the FTMO-style report chart by default.
    "dataset_name": REPORT_CHART_DATASET_NAME,
    "dataset_index": REPORT_CHART_DATASET_INDEX,

    # Notebook display controls
    "trade_log_preview_rows": 80,
    "bad_months_to_show": 12,
    "loss_streaks_to_show": 12,
    "worst_windows_to_show": 15,

    # Rolling windows measured in number of closed trades
    "rolling_trade_windows": [5, 10, 20],

    # Save full detailed trade log to artifacts
    "save_full_trade_log_csv": True,
}


def select_trade_audit_dataset_config(
    config: dict,
    dataset_name: str | None = None,
    dataset_index: int | None = None,
) -> dict:
    """
    Select one dataset from CONFIG['comparison_datasets'] for compact trade audit diagnostics.
    """
    comparison_datasets = config["comparison_datasets"]

    if dataset_index is not None:
        return comparison_datasets[int(dataset_index)]

    if dataset_name is not None:
        matches = [
            dataset_config
            for dataset_config in comparison_datasets
            if dataset_config.get("dataset_name") == dataset_name
        ]

        if len(matches) == 0:
            available_names = [
                dataset_config.get("dataset_name")
                for dataset_config in comparison_datasets
            ]

            raise ValueError(
                f"No dataset matched TRADE_AUDIT_CONFIG['dataset_name']={dataset_name!r}.\n"
                f"Available datasets: {available_names}"
            )

        return matches[0]

    return comparison_datasets[-1]


def max_consecutive_losses(results: pd.Series, r_multiples: pd.Series) -> int:
    """
    Calculate max consecutive losing trades for a grouped period.
    """
    is_loss = (results.astype(str).str.upper() == "SL") | (r_multiples.astype(float) < 0)

    max_streak = 0
    current_streak = 0

    for loss in is_loss:
        if loss:
            current_streak += 1
            max_streak = max(max_streak, current_streak)
        else:
            current_streak = 0

    return int(max_streak)


def build_trade_audit_dataframe(trades_df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Build a clean trade audit dataframe with useful date/time columns.
    """
    if trades_df is None or len(trades_df) == 0:
        return pd.DataFrame()

    audit_df = trades_df.copy()

    audit_df["entry_time"] = pd.to_datetime(audit_df["entry_time"], utc=True).dt.tz_convert(None)
    audit_df["exit_time"] = pd.to_datetime(audit_df["exit_time"], utc=True).dt.tz_convert(None)

    if "session_date" in audit_df.columns:
        audit_df["session_date"] = pd.to_datetime(audit_df["session_date"]).dt.date
    else:
        audit_df["session_date"] = audit_df["exit_time"].dt.date

    audit_df = audit_df.sort_values("entry_time").reset_index(drop=True)

    audit_df["trade_number"] = np.arange(1, len(audit_df) + 1)
    audit_df["exit_month"] = audit_df["exit_time"].dt.to_period("M").astype(str)
    audit_df["exit_week"] = audit_df["exit_time"].dt.to_period("W").astype(str)

    if "account_pct" not in audit_df.columns:
        audit_df["account_pct"] = (
            audit_df["r_multiple"].astype(float) * float(config["risk_per_trade_pct"])
        )

    audit_df["is_win"] = audit_df["r_multiple"].astype(float) > 0
    audit_df["is_loss"] = (
        (audit_df["result"].astype(str).str.upper() == "SL")
        | (audit_df["r_multiple"].astype(float) < 0)
    )

    audit_df["cumulative_r"] = audit_df["r_multiple"].astype(float).cumsum()
    audit_df["cumulative_account_pct"] = audit_df["account_pct"].astype(float).cumsum()

    audit_df["running_peak_r"] = audit_df["cumulative_r"].cummax()
    audit_df["drawdown_r"] = audit_df["cumulative_r"] - audit_df["running_peak_r"]

    return audit_df


def build_monthly_trade_audit(audit_df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarise performance by calendar month.
    """
    if audit_df is None or len(audit_df) == 0:
        return pd.DataFrame()

    monthly_rows = []

    for month, month_df in audit_df.groupby("exit_month"):
        trades = len(month_df)
        wins = int((month_df["r_multiple"] > 0).sum())
        losses = int((month_df["r_multiple"] < 0).sum())

        monthly_rows.append(
            {
                "month": month,
                "first_trade": month_df["entry_time"].min(),
                "last_trade": month_df["exit_time"].max(),
                "trades": trades,
                "wins": wins,
                "losses": losses,
                "win_rate_pct": 100.0 * wins / trades if trades > 0 else 0.0,
                "total_r": float(month_df["r_multiple"].sum()),
                "account_pct": float(month_df["account_pct"].sum()),
                "avg_r": float(month_df["r_multiple"].mean()),
                "worst_trade_r": float(month_df["r_multiple"].min()),
                "best_trade_r": float(month_df["r_multiple"].max()),
                "max_consecutive_losses": max_consecutive_losses(
                    month_df["result"],
                    month_df["r_multiple"],
                ),
            }
        )

    monthly_audit_df = pd.DataFrame(monthly_rows)

    return monthly_audit_df.sort_values("month").reset_index(drop=True)


def build_loss_streak_audit(audit_df: pd.DataFrame) -> pd.DataFrame:
    """
    Find consecutive losing streaks and summarise when they happened.
    """
    if audit_df is None or len(audit_df) == 0:
        return pd.DataFrame()

    streak_df = audit_df.copy()
    streak_df["loss_block_id"] = (
        streak_df["is_loss"] != streak_df["is_loss"].shift(fill_value=False)
    ).cumsum()

    loss_streak_rows = []

    for _, block_df in streak_df[streak_df["is_loss"]].groupby("loss_block_id"):
        loss_streak_rows.append(
            {
                "start_trade": int(block_df["trade_number"].min()),
                "end_trade": int(block_df["trade_number"].max()),
                "start_time": block_df["entry_time"].min(),
                "end_time": block_df["exit_time"].max(),
                "start_date": block_df["entry_time"].min().date(),
                "end_date": block_df["exit_time"].max().date(),
                "loss_count": int(len(block_df)),
                "total_r": float(block_df["r_multiple"].sum()),
                "account_pct": float(block_df["account_pct"].sum()),
                "results": " → ".join(block_df["result"].astype(str).tolist()),
            }
        )

    loss_streaks_df = pd.DataFrame(loss_streak_rows)

    if len(loss_streaks_df) == 0:
        return loss_streaks_df

    return loss_streaks_df.sort_values(
        by=["loss_count", "total_r"],
        ascending=[False, True],
    ).reset_index(drop=True)


def build_worst_rolling_windows(
    audit_df: pd.DataFrame,
    windows: list[int],
    top_n: int,
) -> pd.DataFrame:
    """
    Find the worst rolling windows measured by number of closed trades.
    """
    if audit_df is None or len(audit_df) == 0:
        return pd.DataFrame()

    rolling_rows = []

    for window in windows:
        if len(audit_df) < window:
            continue

        rolling_r = audit_df["r_multiple"].astype(float).rolling(window).sum()
        rolling_account_pct = audit_df["account_pct"].astype(float).rolling(window).sum()

        temp_df = audit_df.copy()
        temp_df["rolling_total_r"] = rolling_r
        temp_df["rolling_account_pct"] = rolling_account_pct

        temp_df = temp_df.dropna(subset=["rolling_total_r"]).copy()

        for idx, row in temp_df.nsmallest(top_n, "rolling_total_r").iterrows():
            end_position = int(idx)
            start_position = end_position - window + 1

            start_row = audit_df.iloc[start_position]
            end_row = audit_df.iloc[end_position]

            window_slice = audit_df.iloc[start_position : end_position + 1]

            rolling_rows.append(
                {
                    "window_trades": int(window),
                    "start_trade": int(start_row["trade_number"]),
                    "end_trade": int(end_row["trade_number"]),
                    "start_time": start_row["entry_time"],
                    "end_time": end_row["exit_time"],
                    "start_date": start_row["entry_time"].date(),
                    "end_date": end_row["exit_time"].date(),
                    "total_r": float(row["rolling_total_r"]),
                    "account_pct": float(row["rolling_account_pct"]),
                    "wins": int((window_slice["r_multiple"] > 0).sum()),
                    "losses": int((window_slice["r_multiple"] < 0).sum()),
                    "results": " → ".join(window_slice["result"].astype(str).tolist()),
                }
            )

    worst_windows_df = pd.DataFrame(rolling_rows)

    if len(worst_windows_df) == 0:
        return worst_windows_df

    return worst_windows_df.sort_values("total_r").reset_index(drop=True)


def save_full_trade_audit_csv(
    audit_df: pd.DataFrame,
    config: dict,
    output_dir: Path,
) -> Path | None:
    """
    Save the full trade audit log to artifacts/trade_audits.
    """
    if audit_df is None or len(audit_df) == 0:
        return None

    trade_audit_dir = output_dir / "trade_audits"
    trade_audit_dir.mkdir(parents=True, exist_ok=True)

    safe_strategy = safe_report_filename_part(config["strategy_version"])
    safe_dataset = safe_report_filename_part(config["dataset_name"])

    output_path = trade_audit_dir / f"{safe_strategy}__{safe_dataset}__trade_audit.csv"

    audit_df.to_csv(output_path, index=False)

    print(f"Saved full trade audit log: {output_path.relative_to(PROJECT_ROOT)}")

    return output_path


TRADE_AUDIT_DATASET_CONFIG = select_trade_audit_dataset_config(
    config=CONFIG,
    dataset_name=TRADE_AUDIT_CONFIG["dataset_name"],
    dataset_index=TRADE_AUDIT_CONFIG["dataset_index"],
)

print("Running compact trade audit for:")
print(TRADE_AUDIT_DATASET_CONFIG)

trade_audit_backtest = run_active_strategy_details_for_dataset(
    base_config=CONFIG,
    dataset_config=TRADE_AUDIT_DATASET_CONFIG,
)

trade_audit_df = build_trade_audit_dataframe(
    trades_df=trade_audit_backtest["trades_df"],
    config=trade_audit_backtest["config"],
)

monthly_trade_audit_df = build_monthly_trade_audit(trade_audit_df)

loss_streak_audit_df = build_loss_streak_audit(trade_audit_df)

worst_rolling_windows_df = build_worst_rolling_windows(
    audit_df=trade_audit_df,
    windows=TRADE_AUDIT_CONFIG["rolling_trade_windows"],
    top_n=TRADE_AUDIT_CONFIG["worst_windows_to_show"],
)

if TRADE_AUDIT_CONFIG["save_full_trade_log_csv"]:
    full_trade_audit_csv_path = save_full_trade_audit_csv(
        audit_df=trade_audit_df,
        config=trade_audit_backtest["config"],
        output_dir=ARTIFACTS_DIR,
    )

print("\nWorst monthly performance audit:")
display(
    monthly_trade_audit_df.sort_values("total_r")
    .head(TRADE_AUDIT_CONFIG["bad_months_to_show"])
    .reset_index(drop=True)
)

print("\nMonthly performance audit, chronological:")
monthly_chronological_df = monthly_trade_audit_df.sort_values("month").reset_index(drop=True)

monthly_total_row = pd.DataFrame(
    [
        {
            "month": "TOTAL",
            "first_trade": monthly_chronological_df["first_trade"].min(),
            "last_trade": monthly_chronological_df["last_trade"].max(),
            "trades": int(monthly_chronological_df["trades"].sum()),
            "wins": int(monthly_chronological_df["wins"].sum()),
            "losses": int(monthly_chronological_df["losses"].sum()),
            "win_rate_pct": 100.0
            * monthly_chronological_df["wins"].sum()
            / monthly_chronological_df["trades"].sum(),
            "total_r": float(monthly_chronological_df["total_r"].sum()),
            "account_pct": float(monthly_chronological_df["account_pct"].sum()),
            "avg_r": float(trade_audit_df["r_multiple"].mean()),
            "worst_trade_r": float(trade_audit_df["r_multiple"].min()),
            "best_trade_r": float(trade_audit_df["r_multiple"].max()),
            "max_consecutive_losses": int(
                monthly_chronological_df["max_consecutive_losses"].max()
            ),
        }
    ]
)

display(
    pd.concat(
        [monthly_chronological_df, monthly_total_row],
        ignore_index=True,
    )
)

print("\nTrade audit total check:")
print(f"Trade rows: {len(trade_audit_df)}")
print(f"Trade Total R: {trade_audit_df['r_multiple'].sum():.2f}R")
print(f"Monthly Total R: {monthly_chronological_df['total_r'].sum():.2f}R")
print(f"Trade Account %: {trade_audit_df['account_pct'].sum():.2f}%")
print(f"Monthly Account %: {monthly_chronological_df['account_pct'].sum():.2f}%")

print("\nWorst consecutive loss streaks:")
display(
    loss_streak_audit_df.head(
        TRADE_AUDIT_CONFIG["loss_streaks_to_show"]
    )
)

print("\nWorst rolling trade windows:")
display(
    worst_rolling_windows_df.head(
        TRADE_AUDIT_CONFIG["worst_windows_to_show"]
    )
)

trade_preview_columns = [
    "trade_number",
    "entry_time",
    "exit_time",
    "session_date",
    "result",
    "r_multiple",
    "account_pct",
    "cumulative_r",
    "drawdown_r",
]

trade_preview_columns = [
    col for col in trade_preview_columns if col in trade_audit_df.columns
]

print("\nTruncated trade log preview:")
with pd.option_context(
    "display.max_rows",
    TRADE_AUDIT_CONFIG["trade_log_preview_rows"],
    "display.max_columns",
    30,
    "display.width",
    180,
):
    display(
        trade_audit_df[trade_preview_columns].tail(
            TRADE_AUDIT_CONFIG["trade_log_preview_rows"]
        )
    )

## Risk Diagnostics

This section checks losing streaks, no-TP runs, and max drawdown in R for each comparison dataset.

The diagnostics use the active V5 modular strategy configuration.

### V5 Account Survival Diagnostics

This table focuses on starting-balance survival risk.

`Min Equity From Start R` measures how far cumulative R went below the dataset starting point.

`Worst Start-to-Trough R` asks how bad the loss could be if trading began at the worst possible point in the sequence.

### Risk diagnostics note

In runner mode, a trade can exit by stop after the stop has already moved into profit.  
Therefore, technical stop exits are not the same as full losing stop losses.

`Max Consecutive Full SL` counts only true losing stop losses.  
`Max Consecutive Stop Exits` counts technical stop-based exits, including profitable trail-lock exits.  
`Worst No-TP Run R` measures cumulative R between full target hits, so it will not necessarily match the stop-exit streak count.

In [ ]:
# Active V5 risk diagnostic: max no-TP run and max drawdown for each comparison dataset

def run_v3_backtest_details_for_dataset(base_config: dict, dataset_config: dict) -> dict:
    run_config = base_config.copy()
    run_config.update(dataset_config)

    raw_df, data_file = load_market_data(run_config)

    engine_df_local = build_existing_engine_output(raw_df, ENGINE_CONFIG)
    automation_df_local = prepare_automation_dataframe(engine_df_local)
    features_df_local = add_automation_features(automation_df_local, run_config)
    signals_df_local = add_v1_green_signals(features_df_local, run_config)

    skipped_generation_df_local = signals_df_local[
        (signals_df_local["raw_signal_name"] != "NO_SIGNAL")
        & (signals_df_local["v1_signal_name"] == "NO_SIGNAL")
    ].copy()

    trades_df_local, daily_summary_df_local, skipped_execution_df_local = run_v1_trade_simulation_with_daily_controls(
        signals_df_local,
        run_config,
    )

    trades_df_local = add_v5_original_tp_hit_diagnostics(
        trades_df_local,
        signals_df_local,
        run_config,
    )

    return {
        "dataset_name": run_config["dataset_name"],
        "trades_df": trades_df_local,
        "daily_summary_df": daily_summary_df_local,
    }


def v5_is_full_sl_trade(trade: pd.Series) -> bool:
    """
    True losing full SL only.

    Runner-mode trail-lock exits can have result == SL but positive R,
    so they must not count as full losing stop losses.
    """
    result = str(trade.get("result", "")).upper()
    exit_reason = str(trade.get("exit_reason", trade.get("result_reason", ""))).upper()
    result_reason = str(trade.get("result_reason", "")).upper()
    outcome_group = str(trade.get("outcome_group", "")).upper()
    r = float(trade.get("r_multiple", 0.0))

    if outcome_group:
        return outcome_group == "SL_FULL"

    return (
        r < 0
        and (
            result == "SL"
            or exit_reason in {"SL", "SL_FULL"}
            or result_reason in {"SL", "SL_FULL"}
        )
    )


def v5_is_stop_exit_trade(trade: pd.Series) -> bool:
    """
    Technical stop exit diagnostic.

    This can include full SL, BE stop, BE-plus stop, and profitable trail-lock stop exits.
    """
    result = str(trade.get("result", "")).upper()
    exit_reason = str(trade.get("exit_reason", trade.get("result_reason", ""))).upper()
    result_reason = str(trade.get("result_reason", "")).upper()

    return (
        result in {"SL", "BE"}
        or exit_reason in {"SL", "SL_FULL", "BE_TRAIL", "BE_PLUS_1R_TRAIL"}
        or result_reason in {"SL", "SL_FULL", "BE_TRAIL", "BE_PLUS_1R_TRAIL"}
        or exit_reason.startswith("TRAIL_LOCK_")
        or result_reason.startswith("TRAIL_LOCK_")
    )


def calculate_risk_diagnostics(trades: pd.DataFrame) -> dict:
    if trades is None or len(trades) == 0:
        return {
            "Trades": 0,
            "Max Consecutive Full SL": 0,
            "Max Consecutive Stop Exits": 0,
            "Worst No-TP Run R": 0.0,
            "Max Equity Drawdown R": 0.0,
        }

    trades = trades.sort_values("entry_time").reset_index(drop=True).copy()

    max_consecutive_full_sl = 0
    current_consecutive_full_sl = 0

    max_consecutive_stop_exits = 0
    current_consecutive_stop_exits = 0

    worst_no_tp_run_r = 0.0
    current_no_tp_run_r = 0.0

    equity = 0.0
    peak_equity = 0.0
    max_drawdown = 0.0

    for _, trade in trades.iterrows():
        result = str(trade.get("result", "")).upper()
        outcome_group = str(trade.get("outcome_group", "")).upper()
        r = float(trade.get("r_multiple", 0.0))

        is_full_sl = v5_is_full_sl_trade(trade)
        is_stop_exit = v5_is_stop_exit_trade(trade)

        if is_full_sl:
            current_consecutive_full_sl += 1
            max_consecutive_full_sl = max(
                max_consecutive_full_sl,
                current_consecutive_full_sl,
            )
        else:
            current_consecutive_full_sl = 0

        if is_stop_exit:
            current_consecutive_stop_exits += 1
            max_consecutive_stop_exits = max(
                max_consecutive_stop_exits,
                current_consecutive_stop_exits,
            )
        else:
            current_consecutive_stop_exits = 0

        is_full_tp = outcome_group == "TP" or result == "TP"

        if is_full_tp:
            current_no_tp_run_r = 0.0
        else:
            current_no_tp_run_r += r
            worst_no_tp_run_r = min(worst_no_tp_run_r, current_no_tp_run_r)

        equity += r
        peak_equity = max(peak_equity, equity)
        drawdown = equity - peak_equity
        max_drawdown = min(max_drawdown, drawdown)

    return {
        "Trades": len(trades),
        "Max Consecutive Full SL": max_consecutive_full_sl,
        "Max Consecutive Stop Exits": max_consecutive_stop_exits,
        "Worst No-TP Run R": worst_no_tp_run_r,
        "Max Equity Drawdown R": max_drawdown,
    }


def calculate_account_survival_diagnostics(trades: pd.DataFrame) -> dict:
    if trades is None or len(trades) == 0:
        return {
            "Trades": 0,
            "Total R": 0.0,
            "Max Consecutive Full SL": 0,
            "SL Streaks >= 5": 0,
            "Worst Consecutive Loss R": 0.0,
            "Min Equity From Start R": 0.0,
            "Worst Start-to-Trough R": 0.0,
            "Breach -5R?": False,
            "Breach -8R?": False,
            "Breach -10R?": False,
        }

    trades = trades.sort_values("entry_time").reset_index(drop=True).copy()
    r_values = pd.to_numeric(trades["r_multiple"], errors="coerce").fillna(0.0)

    total_r = float(r_values.sum())

    max_consecutive_full_sl = 0
    current_consecutive_full_sl = 0
    sl_streaks_gte_5 = 0
    active_sl_streak_already_counted = False

    worst_consecutive_loss_r = 0.0
    current_consecutive_loss_r = 0.0

    for idx, trade in trades.iterrows():
        r = float(r_values.iloc[idx])
        is_full_sl = v5_is_full_sl_trade(trade)

        if is_full_sl:
            current_consecutive_full_sl += 1
            max_consecutive_full_sl = max(
                max_consecutive_full_sl,
                current_consecutive_full_sl,
            )

            if current_consecutive_full_sl >= 5 and not active_sl_streak_already_counted:
                sl_streaks_gte_5 += 1
                active_sl_streak_already_counted = True
        else:
            current_consecutive_full_sl = 0
            active_sl_streak_already_counted = False

        if r < 0:
            current_consecutive_loss_r += r
            worst_consecutive_loss_r = min(
                worst_consecutive_loss_r,
                current_consecutive_loss_r,
            )
        else:
            current_consecutive_loss_r = 0.0

    equity_curve_r = r_values.cumsum()

    min_equity_from_start_r = (
        float(equity_curve_r.min())
        if len(equity_curve_r) > 0
        else 0.0
    )

    running_peak_r = equity_curve_r.cummax()
    drawdown_r = equity_curve_r - running_peak_r

    worst_start_to_trough_r = (
        float(drawdown_r.min())
        if len(drawdown_r) > 0
        else 0.0
    )

    worst_survival_r = min(
        min_equity_from_start_r,
        worst_start_to_trough_r,
    )

    return {
        "Trades": len(trades),
        "Total R": total_r,
        "Max Consecutive Full SL": max_consecutive_full_sl,
        "SL Streaks >= 5": sl_streaks_gte_5,
        "Worst Consecutive Loss R": worst_consecutive_loss_r,
        "Min Equity From Start R": min_equity_from_start_r,
        "Worst Start-to-Trough R": worst_start_to_trough_r,
        "Breach -5R?": bool(worst_survival_r <= -5),
        "Breach -8R?": bool(worst_survival_r <= -8),
        "Breach -10R?": bool(worst_survival_r <= -10),
    }


risk_rows = []
survival_rows = []

for dataset_config in CONFIG["comparison_datasets"]:
    details = run_v3_backtest_details_for_dataset(CONFIG, dataset_config)

    diagnostics = calculate_risk_diagnostics(details["trades_df"])
    survival_diagnostics = calculate_account_survival_diagnostics(details["trades_df"])

    risk_rows.append(
        {
            "Dataset": details["dataset_name"],
            **diagnostics,
            "Worst No-TP Run %": f"{diagnostics['Worst No-TP Run R']:.2f}%",
            "Max Drawdown %": f"{diagnostics['Max Equity Drawdown R']:.2f}%",
        }
    )

    survival_rows.append(
        {
            "Dataset": details["dataset_name"],
            **survival_diagnostics,
        }
    )

risk_diagnostics_table = pd.DataFrame(risk_rows)

print("V5 Risk Diagnostics")
display(risk_diagnostics_table)

print(
    "\nV5 Account Survival Diagnostics\n\n"
    "This table focuses on starting-balance survival risk.\n\n"
    "`Max Equity Drawdown R` is peak-to-trough drawdown and measures strategy smoothness.\n\n"
    "`Min Equity From Start R` measures how far cumulative R went below the dataset starting point.\n\n"
    "`Worst Start-to-Trough R` asks how bad the loss could be if trading began at the worst possible point in the sequence.\n\n"
    "This helps separate long-run profitability from the risk of starting during a bad trade cluster."
)

v5_account_survival_diagnostics_table = pd.DataFrame(survival_rows)

display(v5_account_survival_diagnostics_table)

V5_ACCOUNT_SURVIVAL_DIAGNOSTICS_PATH = (
    ARTIFACTS_DIR / "tables" / "v5_account_survival_diagnostics.csv"
)
V5_ACCOUNT_SURVIVAL_DIAGNOSTICS_PATH.parent.mkdir(parents=True, exist_ok=True)

v5_account_survival_diagnostics_table.to_csv(
    V5_ACCOUNT_SURVIVAL_DIAGNOSTICS_PATH,
    index=False,
)

print(f"Saved V5 account survival diagnostics to: {V5_ACCOUNT_SURVIVAL_DIAGNOSTICS_PATH}")

## V5 Trade Attribution Report

This section explains where performance is coming from.

It groups executed trades by setup type, entry timing, S-tier touch type, A-tier timing type, exit reason, and outcome group.

The goal is to answer questions like:

- Which setup made the most R?
- Which setup caused the most full SLs?
- Are S-tier band-shift touches good or bad?
- Is delayed pullback helping or adding drawdown?
- Which exit types are driving performance?
- Which signals were blocked most often?

In [ ]:
# ---------------------------------------------------------------------
# V5 Trade Attribution Report
# ---------------------------------------------------------------------
import gc
import re

RUN_V5_TRADE_ATTRIBUTION_REPORT = True
SAVE_V5_TRADE_ATTRIBUTION_TABLES = True

V5_TRADE_ATTRIBUTION_DATASETS = "all"
# Or use a list like:
# V5_TRADE_ATTRIBUTION_DATASETS = [
#     "US100_cash_M1_NY_session_1y",
#     "US100_cash_M1_NY_session_recent_2025_2026",
# ]

V5_TRADE_ATTRIBUTION_DIR = ARTIFACTS_DIR / "tables" / "v5_trade_attribution"
V5_TRADE_ATTRIBUTION_DIR.mkdir(parents=True, exist_ok=True)


def v5_slugify_dataset_name(name: str) -> str:
    """
    Make a safe filename stem from a dataset name.
    """
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(name)).strip("_")


def v5_find_r_column(df: pd.DataFrame) -> str | None:
    """
    Find the trade R column safely.
    """
    for candidate in ["r_multiple", "R", "trade_r", "result_r", "R_result"]:
        if candidate in df.columns:
            return candidate

    return None


def v5_find_bars_column(df: pd.DataFrame) -> str | None:
    """
    Find bars-held column safely.
    """
    for candidate in ["bars_held", "holding_bars", "trade_bars", "duration_bars"]:
        if candidate in df.columns:
            return candidate

    return None


def v5_add_managed_exit_category_for_report(trades_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add a managed exit category for reporting only.

    This is reporting-only and does not affect trade simulation.
    """
    if trades_df is None or len(trades_df) == 0:
        return pd.DataFrame()

    df = trades_df.copy()

    r_col = v5_find_r_column(df)
    if r_col is None:
        df["_report_r"] = 0.0
    else:
        df["_report_r"] = pd.to_numeric(df[r_col], errors="coerce").fillna(0.0)

    result_text = (
        df.get("result", pd.Series("", index=df.index))
        .fillna("")
        .astype(str)
        .str.upper()
    )

    exit_reason_text = (
        df.get("exit_reason", pd.Series("", index=df.index))
        .fillna("")
        .astype(str)
        .str.upper()
    )

    outcome_group_text = (
        df.get("outcome_group", pd.Series("", index=df.index))
        .fillna("")
        .astype(str)
        .str.upper()
    )

    if "exit_category" in df.columns:
        managed = (
            df["exit_category"]
            .fillna("")
            .astype(str)
            .str.upper()
        )
    else:
        managed = pd.Series("", index=df.index, dtype="object")

    managed = managed.replace(
        {
            "": "UNKNOWN_EXIT",
            "BE": "BE_FLAT",
            "BE_TRAIL": "BE_PLUS",
            "TRAIL": "TRAILED_SL_PROFIT",
            "TRAIL_LOCK": "TRAILED_SL_PROFIT",
            "CLOSE_PROFIT": "TIME_EXIT",
            "CLOSE_LOSS": "TIME_EXIT",
            "CLOSE_FLAT": "TIME_EXIT",
        }
    )

    managed.loc[result_text.eq("TP")] = "TP"
    managed.loc[result_text.eq("SL")] = "SL_FULL"
    managed.loc[result_text.eq("BE") & df["_report_r"].eq(0)] = "BE_FLAT"
    managed.loc[result_text.eq("BE") & df["_report_r"].gt(0)] = "BE_PLUS"

    managed.loc[exit_reason_text.eq("BE_PLUS_1R_TRAIL")] = "BE_PLUS"
    managed.loc[exit_reason_text.str.startswith("TRAIL_LOCK_", na=False)] = (
        "TRAILED_SL_PROFIT"
    )
    managed.loc[exit_reason_text.eq("TIME_EXIT")] = "TIME_EXIT"

    managed.loc[outcome_group_text.eq("TP")] = "TP"
    managed.loc[outcome_group_text.eq("SL_FULL")] = "SL_FULL"
    managed.loc[outcome_group_text.eq("BE_FLAT")] = "BE_FLAT"
    managed.loc[outcome_group_text.eq("BE_PLUS")] = "BE_PLUS"
    managed.loc[outcome_group_text.eq("TRAILED_SL_PROFIT")] = "TRAILED_SL_PROFIT"

    df["_managed_exit_category"] = managed

    return df


def summarize_trade_group(
    trades_df: pd.DataFrame,
    group_cols: list[str],
) -> pd.DataFrame:
    """
    Summarise trade performance by one or more grouping columns.

    Memory-safe:
    - no row-wise apply
    - simple groupby/pivot only
    - missing columns skipped safely
    """
    if trades_df is None or len(trades_df) == 0:
        return pd.DataFrame()

    df = v5_add_managed_exit_category_for_report(trades_df)

    if "hit_original_tp_before_exit" not in df.columns:
        df["hit_original_tp_before_exit"] = False

    df["hit_original_tp_before_exit"] = (
        df["hit_original_tp_before_exit"]
        .fillna(False)
        .astype(bool)
    )

    existing_group_cols = [col for col in group_cols if col in df.columns]

    if not existing_group_cols:
        return pd.DataFrame()

    r_col = "_report_r"
    bars_col = v5_find_bars_column(df)

    grouped = df.groupby(existing_group_cols, dropna=False)

    summary = (
        grouped
        .agg(
            trades=(r_col, "count"),
            original_tp_hits=("hit_original_tp_before_exit", "sum"),
            total_R=(r_col, "sum"),
            avg_R=(r_col, "mean"),
            positive_exits=(r_col, lambda x: int((x > 0).sum())),
            non_loss_exits=(r_col, lambda x: int((x >= 0).sum())),
        )
        .reset_index()
    )

    if bars_col is not None:
        avg_bars = (
            grouped[bars_col]
            .mean()
            .reset_index(name="avg_bars")
        )
        summary = summary.merge(avg_bars, on=existing_group_cols, how="left")
    else:
        summary["avg_bars"] = np.nan

    exit_counts = (
        df.pivot_table(
            index=existing_group_cols,
            columns="_managed_exit_category",
            values=r_col,
            aggfunc="count",
            fill_value=0,
        )
        .reset_index()
    )

    summary = summary.merge(exit_counts, on=existing_group_cols, how="left")

    for col in [
        "TP",
        "SL_FULL",
        "BE_FLAT",
        "BE_PLUS",
        "TRAILED_SL_PROFIT",
        "TIME_EXIT",
    ]:
        if col not in summary.columns:
            summary[col] = 0

    summary["target_tp_rate"] = np.where(
        summary["trades"] > 0,
        summary["TP"] / summary["trades"],
        0.0,
    )

    summary["original_tp_hit_rate"] = np.where(
        summary["trades"] > 0,
        summary["original_tp_hits"] / summary["trades"],
        0.0,
    )

    summary["full_sl_rate"] = np.where(
        summary["trades"] > 0,
        summary["SL_FULL"] / summary["trades"],
        0.0,
    )

    summary["positive_exit_rate"] = np.where(
        summary["trades"] > 0,
        summary["positive_exits"] / summary["trades"],
        0.0,
    )

    summary["non_loss_rate"] = np.where(
        summary["trades"] > 0,
        summary["non_loss_exits"] / summary["trades"],
        0.0,
    )

    dd_rows = []

    sort_col = "entry_time" if "entry_time" in df.columns else None

    for group_key, group_df in df.groupby(existing_group_cols, dropna=False):
        group_df = group_df.copy()

        if sort_col is not None:
            group_df = group_df.sort_values(sort_col)

        r_values = pd.to_numeric(group_df[r_col], errors="coerce").fillna(0.0)
        equity = r_values.cumsum()
        drawdown = equity - equity.cummax()

        no_tp_run = 0.0
        worst_no_tp_run = 0.0

        managed_exit = group_df["_managed_exit_category"].astype(str)

        for exit_type, r_value in zip(managed_exit, r_values):
            if exit_type == "TP":
                no_tp_run = 0.0
            else:
                no_tp_run += float(r_value)
                worst_no_tp_run = min(worst_no_tp_run, no_tp_run)

        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        dd_rows.append(
            {
                **dict(zip(existing_group_cols, group_key)),
                "max_dd_R": float(drawdown.min()) if len(drawdown) else 0.0,
                "worst_no_tp_run_R": float(worst_no_tp_run),
            }
        )

    dd_table = pd.DataFrame(dd_rows)

    if not dd_table.empty:
        summary = summary.merge(dd_table, on=existing_group_cols, how="left")
    else:
        summary["max_dd_R"] = 0.0
        summary["worst_no_tp_run_R"] = 0.0

    ordered_cols = (
        existing_group_cols
        + [
            "trades",
            "original_tp_hits",
            "original_tp_hit_rate",
            "total_R",
            "avg_R",
            "TP",
            "SL_FULL",
            "BE_FLAT",
            "BE_PLUS",
            "TRAILED_SL_PROFIT",
            "TIME_EXIT",
            "target_tp_rate",
            "full_sl_rate",
            "positive_exit_rate",
            "non_loss_rate",
            "max_dd_R",
            "worst_no_tp_run_R",
            "avg_bars",
        ]
    )

    ordered_cols = [col for col in ordered_cols if col in summary.columns]

    return (
        summary[ordered_cols]
        .sort_values("total_R", ascending=False)
        .reset_index(drop=True)
    )


def summarize_setup_by_exit(
    trades_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Summarise setup source by exit reason/category.
    """
    if trades_df is None or len(trades_df) == 0:
        return pd.DataFrame()

    df = v5_add_managed_exit_category_for_report(trades_df)

    if "hit_original_tp_before_exit" not in df.columns:
        df["hit_original_tp_before_exit"] = False

    df["hit_original_tp_before_exit"] = (
        df["hit_original_tp_before_exit"]
        .fillna(False)
        .astype(bool)
    )

    if "setup_source" not in df.columns:
        return pd.DataFrame()

    exit_group_col = None

    for candidate in ["exit_reason", "outcome_group", "_managed_exit_category", "result"]:
        if candidate in df.columns:
            exit_group_col = candidate
            break

    if exit_group_col is None:
        return pd.DataFrame()

    grouped = (
        df.groupby(["setup_source", exit_group_col], dropna=False)
        .agg(
            trades=("_report_r", "count"),
            original_tp_hits=("hit_original_tp_before_exit", "sum"),
            total_R=("_report_r", "sum"),
            avg_R=("_report_r", "mean"),
        )
        .reset_index()
        .sort_values(["setup_source", "total_R"], ascending=[True, False])
    )

    grouped["original_tp_hit_rate"] = np.where(
        grouped["trades"] > 0,
        grouped["original_tp_hits"] / grouped["trades"],
        0.0,
    )
    
    return grouped


def summarize_blocked_signals(signals_df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarise blocked signal reasons safely.
    """
    if signals_df is None or len(signals_df) == 0:
        return pd.DataFrame()

    df = signals_df.copy()

    reason_col = None
    for candidate in ["v1_signal_block_reason", "block_reason", "signal_block_reason"]:
        if candidate in df.columns:
            reason_col = candidate
            break

    raw_col = None
    for candidate in ["raw_signal_name", "raw_signal", "signal_name"]:
        if candidate in df.columns:
            raw_col = candidate
            break

    if reason_col is None:
        return pd.DataFrame()

    if raw_col is None:
        df["_raw_signal_for_report"] = "UNKNOWN_RAW_SIGNAL"
        raw_col = "_raw_signal_for_report"

    if "setup_source" not in df.columns:
        df["_setup_source_for_report"] = "UNKNOWN_SETUP_SOURCE"
        setup_col = "_setup_source_for_report"
    else:
        setup_col = "setup_source"

    blocked_df = df[
        df[reason_col]
        .fillna("NO_BLOCK")
        .astype(str)
        .ne("NO_BLOCK")
    ].copy()

    if blocked_df.empty:
        return pd.DataFrame()

    blocked_summary = (
        blocked_df
        .groupby([reason_col, raw_col, setup_col], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    return blocked_summary


def v5_save_attribution_table(
    table: pd.DataFrame,
    dataset_name: str,
    table_name: str,
) -> None:
    """
    Save attribution table if enabled.
    """
    if not SAVE_V5_TRADE_ATTRIBUTION_TABLES:
        return

    if table is None or table.empty:
        return

    dataset_slug = v5_slugify_dataset_name(dataset_name)
    output_path = V5_TRADE_ATTRIBUTION_DIR / f"{dataset_slug}__{table_name}.csv"
    table.to_csv(output_path, index=False)


def v5_build_current_dataset_attribution_details() -> dict:
    """
    Use already-computed current notebook outputs.

    This avoids rerunning the engine for the current active dataset.
    """
    return {
        "dataset_name": CONFIG.get("dataset_name", "current_dataset"),
        "trades_df": trades_df,
        "signals_df": signals_df,
        "skipped_execution_signals_df": (
            skipped_execution_signals_df
            if "skipped_execution_signals_df" in globals()
            else pd.DataFrame()
        ),
    }


def v5_build_attribution_details_for_dataset(
    base_config: dict,
    dataset_config: dict,
) -> dict:
    """
    Re-run one comparison dataset at a time for attribution reporting.

    This is reporting-only and mirrors the existing backtest-detail helper,
    but keeps the signal dataframe available for blocked-signal reporting.
    """
    run_config = base_config.copy()
    run_config.update(dataset_config)

    raw_df, _ = load_market_data(run_config)

    engine_df_local = build_existing_engine_output(raw_df, ENGINE_CONFIG)
    automation_df_local = prepare_automation_dataframe(engine_df_local)
    features_df_local = add_automation_features(automation_df_local, run_config)
    signals_df_local = add_v1_green_signals(features_df_local, run_config)

    trades_df_local, daily_summary_df_local, skipped_execution_df_local = (
        run_v1_trade_simulation_with_daily_controls(
            signals_df_local,
            run_config,
        )
    )

    trades_df_local = add_v5_original_tp_hit_diagnostics(
    trades_df_local,
    signals_df_local,
    run_config,
    )

    return {
        "dataset_name": run_config["dataset_name"],
        "trades_df": trades_df_local,
        "signals_df": signals_df_local,
        "daily_summary_df": daily_summary_df_local,
        "skipped_execution_signals_df": skipped_execution_df_local,
    }


def v5_display_and_save_attribution_report(details: dict) -> None:
    """
    Build, display, and optionally save attribution tables for one dataset.
    """
    dataset_name = details.get("dataset_name", "unknown_dataset")
    trades = details.get("trades_df", pd.DataFrame())
    signals = details.get("signals_df", pd.DataFrame())

    print(f"\nV5 Trade Attribution Report: {dataset_name}")

    setup_source_table = summarize_trade_group(trades, ["setup_source"])
    s_tier_touch_table = summarize_trade_group(trades, ["s_tier_touch_type"])
    a_tier_timing_table = summarize_trade_group(trades, ["a_tier_entry_timing_type"])
    entry_timing_table = summarize_trade_group(trades, ["entry_timing_type"])
    setup_by_exit_table = summarize_setup_by_exit(trades)
    blocked_signals_table = summarize_blocked_signals(signals)

    print("Setup source performance:")
    display(setup_source_table.round(4) if not setup_source_table.empty else setup_source_table)

    print("S-tier Dynamic Touch performance:")
    display(s_tier_touch_table.round(4) if not s_tier_touch_table.empty else s_tier_touch_table)

    print("A-tier timing performance:")
    display(a_tier_timing_table.round(4) if not a_tier_timing_table.empty else a_tier_timing_table)

    if not entry_timing_table.empty:
        print("Generic entry timing performance:")
        display(entry_timing_table.round(4))

    print("Setup source by exit reason/category:")
    display(setup_by_exit_table.round(4) if not setup_by_exit_table.empty else setup_by_exit_table)

    print("Blocked signal report:")
    display(blocked_signals_table if not blocked_signals_table.empty else blocked_signals_table)

    v5_save_attribution_table(setup_source_table, dataset_name, "setup_source")
    v5_save_attribution_table(s_tier_touch_table, dataset_name, "s_tier_touch_type")
    v5_save_attribution_table(a_tier_timing_table, dataset_name, "a_tier_entry_timing")
    v5_save_attribution_table(entry_timing_table, dataset_name, "entry_timing")
    v5_save_attribution_table(setup_by_exit_table, dataset_name, "setup_by_exit_reason")
    v5_save_attribution_table(blocked_signals_table, dataset_name, "blocked_signals")


if RUN_V5_TRADE_ATTRIBUTION_REPORT:
    selected_dataset_names = V5_TRADE_ATTRIBUTION_DATASETS

    current_details = v5_build_current_dataset_attribution_details()
    v5_display_and_save_attribution_report(current_details)

    if selected_dataset_names == "all":
        attribution_dataset_configs = CONFIG.get("comparison_datasets", [])
    else:
        selected_dataset_names = set(selected_dataset_names)
        attribution_dataset_configs = [
            dataset_config
            for dataset_config in CONFIG.get("comparison_datasets", [])
            if dataset_config.get("dataset_name") in selected_dataset_names
        ]

    for dataset_config in attribution_dataset_configs:
        if dataset_config.get("dataset_name") == CONFIG.get("dataset_name"):
            continue

        details = v5_build_attribution_details_for_dataset(CONFIG, dataset_config)
        v5_display_and_save_attribution_report(details)

        del details
        gc.collect()

    print(f"V5 attribution tables saved to: {V5_TRADE_ATTRIBUTION_DIR}")
else:
    print("V5 Trade Attribution Report skipped.")

## V5 Blocked Signal Counterfactual Audit

This section analyses blocked signals as shadow trades.

Blocked signals are not real executed trades, so they do not have true outcomes. This audit asks a counterfactual question:

> If this blocked signal had been allowed and entered on the next candle open, what would have happened under the same TP/SL/runner exit rules?

These results are diagnostic only. They should not be mixed into the official backtest totals.

In [ ]:
# ---------------------------------------------------------------------
# V5 Blocked Signal Counterfactual Audit
# ---------------------------------------------------------------------
import gc
import re

RUN_V5_BLOCKED_SIGNAL_COUNTERFACTUAL_AUDIT = True
SAVE_V5_BLOCKED_SIGNAL_COUNTERFACTUAL_AUDIT = True

# V5_BLOCKED_SIGNAL_AUDIT_DATASETS = "all"
# For safer first test, use:
V5_BLOCKED_SIGNAL_AUDIT_DATASETS = []

V5_BLOCKED_SIGNAL_AUDIT_REASONS = [
    "BLOCKED_S_TIER_NOT_CLEAN_AFTER_GREEN_FAILURE",
    "BLOCKED_S_TIER_RECENT_GREEN_BREAK",
    "STRATEGY_FILTER_BLOCKED",
    "BLOCKED_BY_S_TIER_REDSHIFT_FLOOR",
    "BLOCKED_BY_A_TIER_REDSHIFT_FLOOR",
]

V5_BLOCKED_SIGNAL_AUDIT_DIR = ARTIFACTS_DIR / "tables" / "v5_blocked_signal_audit"
V5_BLOCKED_SIGNAL_AUDIT_DIR.mkdir(parents=True, exist_ok=True)


def v5_blocked_audit_slugify(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(name)).strip("_")


def v5_find_signal_block_reason_col(signals: pd.DataFrame) -> str | None:
    for candidate in ["v1_signal_block_reason", "block_reason", "signal_block_reason"]:
        if candidate in signals.columns:
            return candidate
    return None


def v5_find_raw_signal_col(signals: pd.DataFrame) -> str | None:
    for candidate in ["raw_signal_name", "raw_signal", "signal_name"]:
        if candidate in signals.columns:
            return candidate
    return None


def v5_shadow_outcome_group(result: str, result_reason: str, r_value: float) -> str:
    result = str(result).upper()
    result_reason = str(result_reason).upper()
    r_value = float(r_value)

    if result_reason.startswith("TRAIL_LOCK_"):
        return "TRAILED_SL_PROFIT"

    if result_reason == "BE_PLUS_1R_TRAIL":
        return "BE_PLUS"

    if result == "TP":
        return "TP"

    if result == "BE":
        return "BE_PLUS" if r_value > 0 else "BE_FLAT"

    if result == "SL":
        return "TRAILED_SL_PROFIT" if r_value > 0 else "SL_FULL"

    if result_reason in {"FORCE_CLOSED_AT_DATASET_END", "TIME_EXIT"}:
        return "TIME_EXIT"

    return "UNKNOWN_OUTCOME"

def v5_shadow_trade_setup_source(row: pd.Series) -> str:
    """
    Infer setup source for blocked/shadow signal rows.
    Reporting only.
    """
    if "setup_source" in row.index and pd.notna(row.get("setup_source")):
        return str(row.get("setup_source"))

    if "v5_setup_source_from_signal_row" in globals():
        try:
            return v5_setup_source_from_signal_row(row)
        except Exception:
            pass

    raw_name = str(
        row.get("raw_signal_name", row.get("v1_signal_name", ""))
    ).upper()

    if "DELAYED_PULLBACK" in raw_name:
        return "V5_A_TIER_DELAYED_PULLBACK"

    if "FAST_RECLAIM" in raw_name:
        return "V5_A_TIER_FAST_RECLAIM"

    if "V3" in raw_name:
        return "V3_A_TIER_SECOND_CLOSE"

    if "GREEN_RECLAIM" in raw_name or "GREEN_REJECTION" in raw_name:
        return "V1_S_TIER"

    return "UNKNOWN_SETUP_SOURCE"


def v5_direction_from_blocked_row(row: pd.Series) -> str:
    """
    Infer long/short side for a blocked signal row.
    """
    side = str(
        row.get("raw_signal_side", row.get("v1_signal_side", ""))
    ).lower()

    if side in {"long", "short"}:
        return side

    raw_name = str(
        row.get("raw_signal_name", row.get("v1_signal_name", ""))
    ).upper()

    if raw_name.startswith("LONG"):
        return "long"

    if raw_name.startswith("SHORT"):
        return "short"

    return "unknown"


def v5_real_trade_open_flags(
    blocked_rows: pd.DataFrame,
    official_trades: pd.DataFrame,
) -> pd.Series:
    """
    Optional diagnostic: was a real official trade open at the blocked signal timestamp?
    """
    if (
        blocked_rows is None
        or blocked_rows.empty
        or official_trades is None
        or official_trades.empty
        or "datetime" not in blocked_rows.columns
        or "entry_time" not in official_trades.columns
        or "exit_time" not in official_trades.columns
    ):
        return pd.Series(False, index=blocked_rows.index if blocked_rows is not None else [])

    trade_windows = official_trades[["entry_time", "exit_time"]].copy()
    trade_windows["entry_time"] = pd.to_datetime(
        trade_windows["entry_time"],
        errors="coerce",
    )
    trade_windows["exit_time"] = pd.to_datetime(
        trade_windows["exit_time"],
        errors="coerce",
    )
    trade_windows = trade_windows.dropna()

    open_flags = []

    for _, row in blocked_rows.iterrows():
        timestamp = pd.to_datetime(row["datetime"], errors="coerce")

        if pd.isna(timestamp):
            open_flags.append(False)
            continue

        is_open = bool(
            (
                (trade_windows["entry_time"] <= timestamp)
                & (trade_windows["exit_time"] >= timestamp)
            ).any()
        )

        open_flags.append(is_open)

    return pd.Series(open_flags, index=blocked_rows.index)


def v5_select_blocked_signal_rows_for_audit(
    signals: pd.DataFrame,
    audit_reasons: list[str],
) -> pd.DataFrame:
    """
    Select blocked raw signals matching the requested audit block reasons.
    """
    if signals is None or signals.empty:
        return pd.DataFrame()

    reason_col = v5_find_signal_block_reason_col(signals)
    raw_col = v5_find_raw_signal_col(signals)

    if reason_col is None or raw_col is None:
        return pd.DataFrame()

    report = signals.copy()

    report[reason_col] = (
        report[reason_col]
        .fillna("NO_BLOCK")
        .astype(str)
    )

    report[raw_col] = (
        report[raw_col]
        .fillna("NO_SIGNAL")
        .astype(str)
    )

    audit_reason_set = set(str(reason) for reason in audit_reasons)

    blocked_mask = (
        report[raw_col].ne("NO_SIGNAL")
        & report[reason_col].isin(audit_reason_set)
    )

    return report.loc[blocked_mask].copy()


def v5_prepare_shadow_signal_dataframe(
    signals: pd.DataFrame,
    blocked_index: pd.Index,
) -> pd.DataFrame:
    """
    Prepare a copy of signals where selected blocked rows are converted
    into shadow-valid signals for isolated simulation.

    This does not alter official signals_df.
    """
    shadow_df = signals.reset_index(drop=True).copy()

    for col in ["raw_signal_name", "raw_signal_side", "v1_signal_name", "v1_signal_side"]:
        if col not in shadow_df.columns:
            shadow_df[col] = "NO_SIGNAL" if "name" in col else "none"

    valid_blocked_index = [idx for idx in blocked_index if idx in shadow_df.index]

    shadow_df.loc[valid_blocked_index, "v1_signal_name"] = (
        shadow_df.loc[valid_blocked_index, "raw_signal_name"]
        .fillna("NO_SIGNAL")
        .astype(str)
    )

    shadow_df.loc[valid_blocked_index, "v1_signal_side"] = (
        shadow_df.loc[valid_blocked_index, "raw_signal_side"]
        .fillna("none")
        .astype(str)
    )

    if "v1_signal_valid" in shadow_df.columns:
        shadow_df.loc[valid_blocked_index, "v1_signal_valid"] = True

    return shadow_df


def v5_build_shadow_config(base_config: dict) -> dict:
    """
    Build isolated counterfactual config.

    Keeps TP/SL/runner exit rules.
    Avoids portfolio/daily lockout because this is one-signal-at-a-time audit.
    """
    shadow_config = base_config.copy()

    # Reporting-only counterfactual: test the blocked signal itself.
    shadow_config["enable_a_tier_retrace_entry"] = False

    # Avoid session/cutoff skips masking the pure signal outcome.
    shadow_config["enable_session_filter"] = False
    shadow_config["allow_out_of_session_trades"] = True
    shadow_config["no_new_trades_after"] = "23:59"

    return shadow_config

def v5_summarize_blocked_shadow_trades(
    shadow_trades: pd.DataFrame,
    group_cols: list[str],
) -> pd.DataFrame:
    """
    Summarise isolated blocked-signal shadow trades.
    """
    if shadow_trades is None or shadow_trades.empty:
        return pd.DataFrame()

    existing_group_cols = [
        col for col in group_cols
        if col in shadow_trades.columns
    ]

    if not existing_group_cols or "R" not in shadow_trades.columns:
        return pd.DataFrame()

    df = shadow_trades.copy()
    df["R"] = pd.to_numeric(df["R"], errors="coerce")

    # Exclude pure simulator skips from R metrics, but keep them countable separately.
    trade_df = df[df["R"].notna()].copy()

    if trade_df.empty:
        return pd.DataFrame()

    grouped = trade_df.groupby(existing_group_cols, dropna=False)

    summary = (
        grouped
        .agg(
            shadow_trades=("R", "count"),
            total_shadow_R=("R", "sum"),
            avg_shadow_R=("R", "mean"),
            positive_exits=("R", lambda x: int((x > 0).sum())),
        )
        .reset_index()
    )

    outcome_counts = (
        trade_df.pivot_table(
            index=existing_group_cols,
            columns="outcome_group",
            values="R",
            aggfunc="count",
            fill_value=0,
        )
        .reset_index()
    )

    summary = summary.merge(
        outcome_counts,
        on=existing_group_cols,
        how="left",
    )

    for col in [
        "TP",
        "SL_FULL",
        "BE_FLAT",
        "BE_PLUS",
        "TRAILED_SL_PROFIT",
        "TIME_EXIT",
    ]:
        if col not in summary.columns:
            summary[col] = 0

    summary["positive_exit_rate"] = np.where(
        summary["shadow_trades"] > 0,
        summary["positive_exits"] / summary["shadow_trades"],
        0.0,
    )

    summary["full_sl_rate"] = np.where(
        summary["shadow_trades"] > 0,
        summary["SL_FULL"] / summary["shadow_trades"],
        0.0,
    )

    ordered_cols = (
        existing_group_cols
        + [
            "shadow_trades",
            "total_shadow_R",
            "avg_shadow_R",
            "TP",
            "SL_FULL",
            "BE_FLAT",
            "BE_PLUS",
            "TRAILED_SL_PROFIT",
            "TIME_EXIT",
            "positive_exit_rate",
            "full_sl_rate",
        ]
    )

    ordered_cols = [col for col in ordered_cols if col in summary.columns]

    return (
        summary[ordered_cols]
        .sort_values("total_shadow_R", ascending=False)
        .reset_index(drop=True)
    )


def v5_save_blocked_signal_audit_table(
    table: pd.DataFrame,
    dataset_name: str,
    table_name: str,
) -> None:
    if not SAVE_V5_BLOCKED_SIGNAL_COUNTERFACTUAL_AUDIT:
        return

    if table is None or table.empty:
        return

    dataset_slug = v5_blocked_audit_slugify(dataset_name)
    output_path = V5_BLOCKED_SIGNAL_AUDIT_DIR / f"{dataset_slug}__{table_name}.csv"
    table.to_csv(output_path, index=False)


def v5_simulate_blocked_shadow_trades_for_details(
    details: dict,
    base_config: dict,
    audit_reasons: list[str],
) -> pd.DataFrame:
    """
    Simulate blocked signals as isolated shadow trades.

    Reporting only. Does not alter official trades or results.
    """
    dataset_name = details.get("dataset_name", "unknown_dataset")
    signals = details.get("signals_df", pd.DataFrame())
    official_trades = details.get("trades_df", pd.DataFrame())

    if signals is None or signals.empty:
        return pd.DataFrame()

    signals_reset = signals.reset_index(drop=True).copy()

    blocked_rows = v5_select_blocked_signal_rows_for_audit(
        signals_reset,
        audit_reasons,
    )

    if blocked_rows.empty:
        return pd.DataFrame()

    shadow_df = v5_prepare_shadow_signal_dataframe(
        signals_reset,
        blocked_rows.index,
    )

    shadow_config = v5_build_shadow_config(base_config)

    real_trade_open_flags = v5_real_trade_open_flags(
        blocked_rows,
        official_trades,
    )

    shadow_records = []

    reason_col = v5_find_signal_block_reason_col(signals_reset)
    raw_col = v5_find_raw_signal_col(signals_reset)

    for signal_index, blocked_row in blocked_rows.iterrows():
        direction = v5_direction_from_blocked_row(blocked_row)

        if direction not in {"long", "short"}:
            continue

        trade, skipped = simulate_single_v1_trade(
            shadow_df,
            int(signal_index),
            shadow_config,
        )

        base_record = {
            "dataset_name": dataset_name,
            "timestamp": blocked_row.get("datetime", pd.NaT),
            "direction": direction,
            "raw_signal": blocked_row.get(raw_col, "UNKNOWN_RAW_SIGNAL"),
            "setup_source": v5_shadow_trade_setup_source(blocked_row),
            "block_reason": blocked_row.get(reason_col, "UNKNOWN_BLOCK_REASON"),
            "real_trade_open_at_signal_time": bool(
                real_trade_open_flags.loc[signal_index]
                if signal_index in real_trade_open_flags.index
                else False
            ),
            "s_tier_touch_type": blocked_row.get("s_tier_touch_type", "UNKNOWN"),
            "a_tier_entry_timing_type": blocked_row.get(
                "a_tier_entry_timing_type",
                "UNKNOWN",
            ),
            "s_tier_clean_state_recent_failure_seen": blocked_row.get(
                "s_tier_clean_state_recent_failure_seen",
                False,
            ),
            "s_tier_clean_state_clean_closes_after_failure": blocked_row.get(
                "s_tier_clean_state_clean_closes_after_failure",
                np.nan,
            ),
            "s_tier_clean_state_pass": blocked_row.get(
                "s_tier_clean_state_pass",
                False,
            ),
            "s_tier_clean_state_blocked": blocked_row.get(
                "s_tier_clean_state_blocked",
                False,
            ),
            "s_tier_band_shift_touch": blocked_row.get(
                "s_tier_band_shift_touch",
                False,
            ),
        }

        if trade is None:
            skip_reason = (
                skipped.get("skip_reason", "SHADOW_SKIPPED")
                if skipped is not None
                else "SHADOW_SKIPPED"
            )

            shadow_records.append(
                {
                    **base_record,
                    "entry_price": np.nan,
                    "sl_price": np.nan,
                    "tp_price": np.nan,
                    "exit_price": np.nan,
                    "exit_time": pd.NaT,
                    "exit_reason": skip_reason,
                    "outcome_group": "SHADOW_SKIPPED",
                    "R": np.nan,
                    "bars_held": np.nan,
                }
            )

            continue

        r_value = float(trade.get("r_multiple", 0.0))
        result = str(trade.get("result", "UNKNOWN"))
        result_reason = str(trade.get("result_reason", "UNKNOWN"))
        outcome_group = v5_shadow_outcome_group(result, result_reason, r_value)

        shadow_records.append(
            {
                **base_record,
                "entry_price": trade.get("entry_price", np.nan),
                "sl_price": trade.get("stop_price_initial", np.nan),
                "tp_price": trade.get("target_price", np.nan),
                "exit_price": trade.get("exit_price", np.nan),
                "exit_time": trade.get("exit_time", pd.NaT),
                "exit_reason": result_reason,
                "outcome_group": outcome_group,
                "R": r_value,
                "bars_held": trade.get("bars_held", np.nan),
            }
        )

    return pd.DataFrame(shadow_records)

def v5_display_and_save_blocked_signal_audit(
    details: dict,
    base_config: dict,
) -> None:
    dataset_name = details.get("dataset_name", "unknown_dataset")

    print(f"\nV5 Blocked Signal Counterfactual Audit: {dataset_name}")

    shadow_trades = v5_simulate_blocked_shadow_trades_for_details(
        details=details,
        base_config=base_config,
        audit_reasons=V5_BLOCKED_SIGNAL_AUDIT_REASONS,
    )

    if shadow_trades.empty:
        print("No blocked signals matched the requested audit reasons.")
        return

    summary_by_reason = v5_summarize_blocked_shadow_trades(
        shadow_trades,
        ["block_reason"],
    )

    summary_by_reason_setup = v5_summarize_blocked_shadow_trades(
        shadow_trades,
        ["block_reason", "setup_source"],
    )

    summary_by_reason_touch = v5_summarize_blocked_shadow_trades(
        shadow_trades,
        ["block_reason", "s_tier_touch_type"],
    )

    print("Blocked shadow trade summary by block reason:")
    display(summary_by_reason.round(4) if not summary_by_reason.empty else summary_by_reason)

    print("Blocked shadow trade summary by block reason × setup source:")
    display(
        summary_by_reason_setup.round(4)
        if not summary_by_reason_setup.empty
        else summary_by_reason_setup
    )

    print("Blocked shadow trade summary by block reason × S-tier touch type:")
    display(
        summary_by_reason_touch.round(4)
        if not summary_by_reason_touch.empty
        else summary_by_reason_touch
    )

    print("Blocked shadow trades:")
    display(shadow_trades)

    v5_save_blocked_signal_audit_table(
        shadow_trades,
        dataset_name,
        "blocked_shadow_trades",
    )

    v5_save_blocked_signal_audit_table(
        summary_by_reason,
        dataset_name,
        "blocked_summary_by_reason",
    )

    v5_save_blocked_signal_audit_table(
        summary_by_reason_setup,
        dataset_name,
        "blocked_summary_by_reason_setup",
    )

    v5_save_blocked_signal_audit_table(
        summary_by_reason_touch,
        dataset_name,
        "blocked_summary_by_reason_s_tier_touch",
    )


def v5_build_blocked_audit_current_details() -> dict:
    return {
        "dataset_name": CONFIG.get("dataset_name", "current_dataset"),
        "signals_df": signals_df,
        "trades_df": trades_df,
    }


def v5_build_blocked_audit_details_for_dataset(
    base_config: dict,
    dataset_config: dict,
) -> dict:
    """
    Build signal/trade details one dataset at a time for reporting only.
    """
    if "v5_build_attribution_details_for_dataset" in globals():
        return v5_build_attribution_details_for_dataset(
            base_config,
            dataset_config,
        )

    run_config = base_config.copy()
    run_config.update(dataset_config)

    raw_df, _ = load_market_data(run_config)

    engine_df_local = build_existing_engine_output(raw_df, ENGINE_CONFIG)
    automation_df_local = prepare_automation_dataframe(engine_df_local)
    features_df_local = add_automation_features(automation_df_local, run_config)
    signals_df_local = add_v1_green_signals(features_df_local, run_config)

    trades_df_local, daily_summary_df_local, skipped_execution_df_local = (
        run_v1_trade_simulation_with_daily_controls(
            signals_df_local,
            run_config,
        )
    )

    trades_df_local = add_v5_original_tp_hit_diagnostics(
    trades_df_local,
    signals_df_local,
    run_config,
    )

    return {
        "dataset_name": run_config["dataset_name"],
        "signals_df": signals_df_local,
        "trades_df": trades_df_local,
        "daily_summary_df": daily_summary_df_local,
        "skipped_execution_signals_df": skipped_execution_df_local,
    }


if RUN_V5_BLOCKED_SIGNAL_COUNTERFACTUAL_AUDIT:
    current_details = v5_build_blocked_audit_current_details()
    v5_display_and_save_blocked_signal_audit(current_details, CONFIG)

    selected_dataset_names = V5_BLOCKED_SIGNAL_AUDIT_DATASETS

    if selected_dataset_names == "all":
        audit_dataset_configs = CONFIG.get("comparison_datasets", [])
    else:
        selected_dataset_names = set(selected_dataset_names)
        audit_dataset_configs = [
            dataset_config
            for dataset_config in CONFIG.get("comparison_datasets", [])
            if dataset_config.get("dataset_name") in selected_dataset_names
        ]

    for dataset_config in audit_dataset_configs:
        if dataset_config.get("dataset_name") == CONFIG.get("dataset_name"):
            continue

        details = v5_build_blocked_audit_details_for_dataset(CONFIG, dataset_config)
        v5_display_and_save_blocked_signal_audit(details, CONFIG)

        del details
        gc.collect()

    print(f"V5 blocked signal audit tables saved to: {V5_BLOCKED_SIGNAL_AUDIT_DIR}")
else:
    print("V5 Blocked Signal Counterfactual Audit skipped.")

## Recent V5 Experiment Log Entries

This optional final display shows the latest saved experiment-log rows from `artifacts/tables/v5_experiment_log.csv`.

In [ ]:
if SAVE_EXPERIMENT_LOG and EXPERIMENT_LOG_PATH.exists():
    experiment_log_df = pd.read_csv(EXPERIMENT_LOG_PATH)
    display(experiment_log_df.tail(10))